Part 1 — Study Setup, Dataset Quality, Cohort Definition and Leakage Prevention (Steps 1–10)
Purpose
Establish a reproducible environment, document the Sygen dataset, define the clinical prediction task, assess data quality, prevent future-information leakage, and prepare patient-level modelling datasets.
Workflow
- Step 1 — Project and environment setup: initialize project architecture, random seeds, package versions, and hardware documentation.
- Step 2 — Raw dataset inspection: locate, load, and characterize the dataset; verify patient identifiers, the primary target, longitudinal motor scores, age, AIS, and sex coding.
- Step 3 — Study documentation: document the data source, longitudinal assessment points, planned dynamic models, and core features.
- Step 4 — Data quality: examine missingness, duplicates, score ranges, age and categorical-variable quality, numerical distributions, outliers, plots, and complete-case availability by landmark.
- Step 5 — Clinical prediction specification: define the primary prediction question, analytic cohort, outcome and predictor missing-data policies, baseline and dynamic features, landmark feature sets, and planned analyses.
- Step 6 — Leakage prevention: inspect suspicious variables and precomputed change scores; define feature-availability timelines, allowed inputs, forbidden future variables, and automated leakage checks.
- Step 7 — Patient-level splitting: create train, validation, and internal test partitions; check patient overlap, cohort preservation, and split characteristics; save split metadata.
- Step 8 — Feature engineering: construct and verify dynamic features, define landmark-specific feature sets, prepare model datasets, and save feature definitions.
- Step 9 — Preprocessing: define numerical/categorical preprocessing, fit preprocessing on training data only, inspect processed matrices, and save preprocessing artifacts.
- Step 10 — Baseline models: calculate mean-baseline predictions and save baseline performance for context.
Key reporting requirements
Describe cohort eligibility and exclusions, outcome availability, missing-data handling, feature availability at each prediction landmark, and the patient-level split strategy. Do not imply that a variable is leakage-free solely because its name appears appropriate; verify its provenance and timing.

In [ ]:
# ============================================================
# STEP 1.1: IMPORT LIBRARIES
# SCI Dynamic Recovery Prediction
# ============================================================


# -----------------------------
# Basic libraries
# -----------------------------
import os
import sys
import random
import warnings
import platform
import time
from pathlib import Path


# -----------------------------
# Data manipulation
# -----------------------------
import numpy as np
import pandas as pd


# -----------------------------
# Visualization
# -----------------------------
import matplotlib.pyplot as plt
import seaborn as sns


# -----------------------------
# Machine Learning
# -----------------------------
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_val_score
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline


# Regression metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# -----------------------------
# Baseline Models
# -----------------------------
from sklearn.linear_model import (
    ElasticNet
)


# -----------------------------
# Machine Learning Models
# -----------------------------
from sklearn.ensemble import (
    RandomForestRegressor
)


from sklearn.neural_network import (
    MLPRegressor
)


# -----------------------------
# Gradient Boosting Models
# -----------------------------
from xgboost import XGBRegressor

from lightgbm import LGBMRegressor


# -----------------------------
# Optimization
# -----------------------------
import optuna


# -----------------------------
# Explainability
# -----------------------------
import shap


# -----------------------------
# Statistics
# -----------------------------
from scipy import stats


# -----------------------------
# Calibration
# -----------------------------
from sklearn.calibration import calibration_curve


# -----------------------------
# Display settings
# -----------------------------
pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    100
)


warnings.filterwarnings(
    "ignore"
)


print("All libraries imported successfully")

In [ ]:
# ============================================================
# STEP 1.2 : COMPLETE PROJECT ARCHITECTURE
# ============================================================

from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/SCI_Recovery_ML_Project")


# ============================================================
# 01_data
# ============================================================

DATA_RAW = PROJECT_DIR / "01_data/primary/raw"
DATA_PROCESSED = PROJECT_DIR / "01_data/primary/processed"
DATA_SPLITS = PROJECT_DIR / "01_data/primary/splits"

DATA_EXTERNAL_RAW = PROJECT_DIR / "01_data/external_validation/raw"
DATA_EXTERNAL_PROCESSED = PROJECT_DIR / "01_data/external_validation/processed"

DATA_METADATA = PROJECT_DIR / "01_data/metadata"


# ============================================================
# 02_notebooks
# ============================================================

NOTEBOOK_DIR = PROJECT_DIR / "02_notebooks"


# ============================================================
# 03_configs
# ============================================================

CONFIG_DIR = PROJECT_DIR / "03_configs"


# ============================================================
# 04_code
# ============================================================

CODE_PREPROCESSING = PROJECT_DIR / "04_code/preprocessing"
CODE_FEATURE_ENGINEERING = PROJECT_DIR / "04_code/feature_engineering"

CODE_MODELS = PROJECT_DIR / "04_code/models"

CODE_TRAINING = PROJECT_DIR / "04_code/training"
CODE_EVALUATION = PROJECT_DIR / "04_code/evaluation"
CODE_EXPLAINABILITY = PROJECT_DIR / "04_code/explainability"


# ============================================================
# 05_models
# ============================================================

MODEL_BASELINE = PROJECT_DIR / "05_models/baseline"

MODEL_BENCHMARK = PROJECT_DIR / "05_models/benchmark"

MODEL_FINAL = (
    PROJECT_DIR /
    "05_models/proposed/optimized_lightgbm_B16"
)

MODEL_PREPROCESSING = PROJECT_DIR / "05_models/preprocessing"


# ============================================================
# 06_results / tables
# ============================================================

TABLE_DATASET = PROJECT_DIR / "06_results/tables/dataset"
TABLE_BENCHMARK = PROJECT_DIR / "06_results/tables/benchmarks"
TABLE_ABLATION = PROJECT_DIR / "06_results/tables/ablation"
TABLE_VALIDATION = PROJECT_DIR / "06_results/tables/validation"
TABLE_SUBGROUP = PROJECT_DIR / "06_results/tables/subgroups"
TABLE_XAI = PROJECT_DIR / "06_results/tables/explainability"


# ============================================================
# 06_results / metrics
# ============================================================

METRIC_BASELINE = PROJECT_DIR / "06_results/metrics/baseline"
METRIC_BENCHMARK = PROJECT_DIR / "06_results/metrics/benchmark"
METRIC_NESTED_CV = PROJECT_DIR / "06_results/metrics/nested_cv"
METRIC_BOOTSTRAP = PROJECT_DIR / "06_results/metrics/bootstrap"
METRIC_EXTERNAL = PROJECT_DIR / "06_results/metrics/external"


# ============================================================
# 06_results / figures
# ============================================================

FIG_DATA = PROJECT_DIR / "06_results/figures/data_analysis"
FIG_TRAINING = PROJECT_DIR / "06_results/figures/training"
FIG_EVALUATION = PROJECT_DIR / "06_results/figures/evaluation"
FIG_CALIBRATION = PROJECT_DIR / "06_results/figures/calibration"
FIG_LEARNING = PROJECT_DIR / "06_results/figures/learning_curve"
FIG_ERROR = PROJECT_DIR / "06_results/figures/error_analysis"
FIG_LANDMARK = PROJECT_DIR / "06_results/figures/dynamic_landmark"
FIG_SHAP = PROJECT_DIR / "06_results/figures/shap"


# ============================================================
# CREATE ALL DIRECTORIES
# ============================================================

DIRECTORIES = [

    DATA_RAW,
    DATA_PROCESSED,
    DATA_SPLITS,
    DATA_EXTERNAL_RAW,
    DATA_EXTERNAL_PROCESSED,
    DATA_METADATA,

    NOTEBOOK_DIR,

    CONFIG_DIR,

    CODE_PREPROCESSING,
    CODE_FEATURE_ENGINEERING,
    CODE_MODELS,
    CODE_TRAINING,
    CODE_EVALUATION,
    CODE_EXPLAINABILITY,

    MODEL_BASELINE,
    MODEL_BENCHMARK,
    MODEL_FINAL,
    MODEL_PREPROCESSING,

    TABLE_DATASET,
    TABLE_BENCHMARK,
    TABLE_ABLATION,
    TABLE_VALIDATION,
    TABLE_SUBGROUP,
    TABLE_XAI,

    METRIC_BASELINE,
    METRIC_BENCHMARK,
    METRIC_NESTED_CV,
    METRIC_BOOTSTRAP,
    METRIC_EXTERNAL,

    FIG_DATA,
    FIG_TRAINING,
    FIG_EVALUATION,
    FIG_CALIBRATION,
    FIG_LEARNING,
    FIG_ERROR,
    FIG_LANDMARK,
    FIG_SHAP
]

for directory in DIRECTORIES:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("Complete project architecture created successfully.")
print("\nMain folders:")

for folder in sorted(PROJECT_DIR.iterdir()):
    if folder.is_dir():
        print(" -", folder.name)

In [ ]:
# ============================================================
# STEP 1.3: GLOBAL RANDOM SEED & REPRODUCIBILITY
# ============================================================

import os
import random
import numpy as np

SEED = 42

# Python hash seed
os.environ["PYTHONHASHSEED"] = str(SEED)

# Python random
random.seed(SEED)

# NumPy random
np.random.seed(SEED)

# Optional environment variables for deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print("Global random seed set successfully.")
print("SEED =", SEED)

In [ ]:
# ============================================================
# STEP 1.4: RECORD PYTHON & PACKAGE VERSIONS
# ============================================================

import sys
import platform
import importlib.metadata as importlib_metadata
import pandas as pd

packages_to_check = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "xgboost",
    "lightgbm",
    "optuna",
    "shap",
    "scipy",
    "statsmodels"
]

version_records = []

for pkg in packages_to_check:
    try:
        version = importlib_metadata.version(pkg)
    except importlib_metadata.PackageNotFoundError:
        version = "Not installed"

    version_records.append({
        "Package": pkg,
        "Version": version
    })

environment_info = pd.DataFrame(version_records)

print("Python Version :", sys.version.split()[0])
print("Platform       :", platform.platform())
print("\nPackage Versions:")
display(environment_info)

# ------------------------------------------------------------
# Save environment information
# ------------------------------------------------------------

env_file = CONFIG_DIR / "environment_versions.csv"
environment_info.to_csv(env_file, index=False)

text_file = CONFIG_DIR / "environment_summary.txt"

with open(text_file, "w") as f:
    f.write(f"Python Version: {sys.version}\n")
    f.write(f"Platform: {platform.platform()}\n\n")
    f.write("Package Versions:\n")
    
    for _, row in environment_info.iterrows():
        f.write(f"{row['Package']}: {row['Version']}\n")

print("\nSaved:")
print(" -", env_file)
print(" -", text_file)

In [ ]:
# ============================================================
# STEP 1.5: HARDWARE INFORMATION
# ============================================================

import os
import platform
import psutil
import subprocess
import pandas as pd


hardware_info = {}


# ------------------------------------------------------------
# Operating system
# ------------------------------------------------------------

hardware_info["Operating System"] = platform.platform()


# ------------------------------------------------------------
# CPU information
# ------------------------------------------------------------

hardware_info["Processor"] = platform.processor()
hardware_info["Physical CPU Cores"] = psutil.cpu_count(logical=False)
hardware_info["Logical CPU Cores"] = psutil.cpu_count(logical=True)


# ------------------------------------------------------------
# RAM information
# ------------------------------------------------------------

total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)

hardware_info["Total RAM (GB)"] = round(total_ram_gb, 2)


# ------------------------------------------------------------
# GPU information
# ------------------------------------------------------------

gpu_name = "Not available"

try:
    gpu_output = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name",
            "--format=csv,noheader"
        ],
        stderr=subprocess.DEVNULL
    ).decode().strip()

    if gpu_output:
        gpu_name = gpu_output

except Exception:
    pass

hardware_info["GPU"] = gpu_name


# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

hardware_df = pd.DataFrame(
    list(hardware_info.items()),
    columns=["Component", "Information"]
)

display(hardware_df)


# ------------------------------------------------------------
# Save hardware information
# ------------------------------------------------------------

hardware_file = CONFIG_DIR / "hardware_information.csv"

hardware_df.to_csv(
    hardware_file,
    index=False
)


hardware_text_file = CONFIG_DIR / "hardware_information.txt"

with open(hardware_text_file, "w") as f:
    for key, value in hardware_info.items():
        f.write(f"{key}: {value}\n")


print("\nHardware information saved successfully.")

print("Saved:")
print(" -", hardware_file)
print(" -", hardware_text_file)

In [ ]:
# ============================================================
# STEP 2.1: LOCATE RAW DATASET
# ============================================================

from pathlib import Path

# Possible known location
known_path = Path(
    "/kaggle/input/datasets/shizansarkar/sci-recovery-data/"
    "SCI_Neurological_Recovery-master/test/spss_v1_2.csv"
)

dataset_candidates = []

# Check known path first
if known_path.exists():
    dataset_candidates.append(known_path)

# Search all Kaggle input directories if needed
if not dataset_candidates:
    dataset_candidates = list(
        Path("/kaggle/input").rglob("spss_v1_2.csv")
    )

print(f"Number of matching dataset files found: {len(dataset_candidates)}")

for i, path in enumerate(dataset_candidates, start=1):
    print(f"{i}. {path}")

if len(dataset_candidates) == 0:
    raise FileNotFoundError(
        "spss_v1_2.csv was not found under /kaggle/input."
    )

RAW_SOURCE_PATH = dataset_candidates[0]

print("\nSelected raw dataset:")
print(RAW_SOURCE_PATH)

In [ ]:
# ============================================================
# STEP 2.2: COPY RAW DATASET INTO PROJECT DIRECTORY
# ============================================================

import shutil

RAW_PROJECT_PATH = DATA_RAW / "spss_v1_2.csv"

# Copy only if it does not already exist
if not RAW_PROJECT_PATH.exists():
    shutil.copy2(
        RAW_SOURCE_PATH,
        RAW_PROJECT_PATH
    )
    print("Raw dataset copied successfully.")
else:
    print("Raw dataset already exists in project directory.")

print("Project raw dataset path:")
print(RAW_PROJECT_PATH)

In [ ]:
# ============================================================
# STEP 2.3: LOAD RAW DATASET
# ============================================================

import pandas as pd

raw_df = pd.read_csv(
    RAW_PROJECT_PATH,
    low_memory=False
)

print("Dataset loaded successfully.")

print("\nDataset shape:")
print(raw_df.shape)

print("\nNumber of rows:")
print(raw_df.shape[0])

print("\nNumber of columns:")
print(raw_df.shape[1])

In [ ]:
# ============================================================
# STEP 2.4: BASIC DATASET INSPECTION
# ============================================================

print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)

print("\nShape:")
print(raw_df.shape)


In [ ]:
# ============================================================
# STEP 2.5: VERIFY CRITICAL STUDY VARIABLES
# ============================================================

required_variables = [
    "PTID",
    "AGE",
    "SEXCD",
    "AIS.1",
    "TOTMOT01",
    "TOTMOT04",
    "TOTMOT08",
    "TOTMOT16",
    "TOTMOT26",
    "TOTMOT52"
]

variable_check = pd.DataFrame({
    "Variable": required_variables,
    "Present": [
        var in raw_df.columns
        for var in required_variables
    ]
})

display(variable_check)

missing_required = variable_check.loc[
    ~variable_check["Present"],
    "Variable"
].tolist()

if len(missing_required) == 0:
    print("\nAll critical variables are available.")
else:
    print("\nVariables not found:")
    print(missing_required)

In [ ]:
# ============================================================
# STEP 2.6: VERIFY PATIENT IDENTIFIER
# ============================================================

ID_COLUMN = "PTID"

if ID_COLUMN not in raw_df.columns:
    raise KeyError(
        f"Patient ID column '{ID_COLUMN}' was not found."
    )

total_rows = len(raw_df)

unique_patients = raw_df[ID_COLUMN].nunique(
    dropna=True
)

missing_patient_ids = raw_df[ID_COLUMN].isna().sum()

duplicated_patient_ids = raw_df[ID_COLUMN].duplicated(
    keep=False
).sum()

print("Patient ID assessment")
print("-" * 50)

print(f"Total dataset rows        : {total_rows}")
print(f"Unique patient IDs        : {unique_patients}")
print(f"Missing patient IDs       : {missing_patient_ids}")
print(f"Rows with repeated PTID    : {duplicated_patient_ids}")

In [ ]:
# ============================================================
# STEP 2.7: PRIMARY TARGET VARIABLE INSPECTION
# ============================================================

TARGET = "TOTMOT26"

if TARGET not in raw_df.columns:
    raise KeyError(
        f"Target variable '{TARGET}' was not found."
    )

target_summary = raw_df[TARGET].describe()

print("=" * 60)
print("PRIMARY TARGET: TOTMOT26")
print("=" * 60)

print(target_summary)

print("\nTarget missing values:")
print(raw_df[TARGET].isna().sum())

print("\nTarget available values:")
print(raw_df[TARGET].notna().sum())

print("\nTarget missing percentage:")
print(
    round(
        raw_df[TARGET].isna().mean() * 100,
        2
    ),
    "%"
)

In [ ]:
# ============================================================
# STEP 2.8: CHECK LONGITUDINAL MOTOR VARIABLES
# ============================================================

motor_timepoints = [
    "TOTMOT01",
    "TOTMOT04",
    "TOTMOT08",
    "TOTMOT16",
    "TOTMOT26",
    "TOTMOT52"
]

motor_summary_records = []

for col in motor_timepoints:

    if col in raw_df.columns:

        motor_summary_records.append({
            "Variable": col,
            "Available_N": raw_df[col].notna().sum(),
            "Missing_N": raw_df[col].isna().sum(),
            "Missing_%": round(
                raw_df[col].isna().mean() * 100,
                2
            ),
            "Mean": raw_df[col].mean(),
            "SD": raw_df[col].std(),
            "Min": raw_df[col].min(),
            "Max": raw_df[col].max()
        })

motor_summary = pd.DataFrame(
    motor_summary_records
)

display(motor_summary)

In [ ]:
# ============================================================
# STEP 2.9: INSPECT BASELINE CLINICAL VARIABLES
# ============================================================

baseline_variables = [
    "PTID",
    "AGE",
    "SEXCD",
    "AIS.1",
    "TOTMOT01"
]

existing_baseline_variables = [
    col
    for col in baseline_variables
    if col in raw_df.columns
]

baseline_preview = raw_df[
    existing_baseline_variables
].copy()

display(
    baseline_preview.head(10)
)

print("\nData types:")
print(
    baseline_preview.dtypes
)

print("\nMissing values:")
print(
    baseline_preview.isna().sum()
)

In [ ]:
# ============================================================
# STEP 2.10: INSPECT AIS CODING
# ============================================================

AIS_COLUMN = "AIS.1"

if AIS_COLUMN in raw_df.columns:

    print("AIS value counts INCLUDING missing values:")
    print(
        raw_df[AIS_COLUMN]
        .value_counts(
            dropna=False
        )
    )

    print("\nUnique AIS values:")
    print(
        raw_df[AIS_COLUMN]
        .unique()
    )

In [ ]:
# ============================================================
# STEP 2.11: INSPECT SEX VARIABLE CODING
# ============================================================

SEX_COLUMN = "SEXCD"

if SEX_COLUMN in raw_df.columns:

    print("SEXCD value counts:")
    print(
        raw_df[SEX_COLUMN]
        .value_counts(
            dropna=False
        )
        .sort_index()
    )

    print("\nUnique SEXCD values:")
    print(
        raw_df[SEX_COLUMN]
        .unique()
    )

In [ ]:
# ============================================================
# STEP 2.12: DATASET MEMORY INFORMATION
# ============================================================

memory_bytes = raw_df.memory_usage(
    deep=True
).sum()

memory_mb = memory_bytes / (1024 ** 2)

print(f"Dataset memory usage: {memory_mb:.2f} MB")

In [ ]:
# ============================================================
# STEP 2.13: SAVE RAW DATASET SUMMARY
# ============================================================

dataset_summary = pd.DataFrame({
    "Characteristic": [
        "Raw rows",
        "Raw columns",
        "Unique patient IDs",
        "Missing patient IDs",
        "Primary target",
        "Target available N",
        "Target missing N",
        "Target missing %",
        "Dataset memory MB"
    ],

    "Value": [
        raw_df.shape[0],
        raw_df.shape[1],
        raw_df["PTID"].nunique(dropna=True),
        raw_df["PTID"].isna().sum(),
        TARGET,
        raw_df[TARGET].notna().sum(),
        raw_df[TARGET].isna().sum(),
        round(
            raw_df[TARGET].isna().mean() * 100,
            2
        ),
        round(memory_mb, 2)
    ]
})

display(dataset_summary)


dataset_summary_file = (
    TABLE_DATASET /
    "raw_dataset_summary.csv"
)

dataset_summary.to_csv(
    dataset_summary_file,
    index=False
)


motor_summary_file = (
    TABLE_DATASET /
    "longitudinal_motor_summary.csv"
)

motor_summary.to_csv(
    motor_summary_file,
    index=False
)


variable_check_file = (
    TABLE_DATASET /
    "critical_variable_availability.csv"
)

variable_check.to_csv(
    variable_check_file,
    index=False
)


print("\nSaved:")
print(" -", dataset_summary_file)
print(" -", motor_summary_file)
print(" -", variable_check_file)

In [ ]:
# ============================================================
# STEP 2.14: SAVE COMPLETE COLUMN INVENTORY
# ============================================================

column_inventory = pd.DataFrame({
    "Column_Number": range(
        1,
        len(raw_df.columns) + 1
    ),

    "Variable": raw_df.columns,

    "Data_Type": [
        str(raw_df[col].dtype)
        for col in raw_df.columns
    ],

    "Non_Missing_N": [
        raw_df[col].notna().sum()
        for col in raw_df.columns
    ],

    "Missing_N": [
        raw_df[col].isna().sum()
        for col in raw_df.columns
    ],

    "Missing_Percent": [
        round(
            raw_df[col].isna().mean() * 100,
            2
        )
        for col in raw_df.columns
    ]
})


column_inventory_file = (
    TABLE_DATASET /
    "raw_column_inventory.csv"
)

column_inventory.to_csv(
    column_inventory_file,
    index=False
)

print(
    "Complete column inventory saved:"
)

print(column_inventory_file)

print(
    "\nTotal variables documented:",
    len(column_inventory)
)

In [ ]:
# ============================================================
# STEP 2.15: SAVE DATASET LOADING METADATA
# ============================================================

import json
from datetime import datetime

dataset_metadata = {

    "dataset_name":
        "Sygen Traumatic Spinal Cord Injury Cohort",

    "source_file":
        str(RAW_SOURCE_PATH),

    "project_raw_file":
        str(RAW_PROJECT_PATH),

    "rows":
        int(raw_df.shape[0]),

    "columns":
        int(raw_df.shape[1]),

    "patient_id":
        "PTID",

    "primary_target":
        "TOTMOT26",

    "problem_type":
        "Regression",

    "baseline_landmark":
        "Week 1",

    "dynamic_landmarks": [
        "Week 4",
        "Week 8",
        "Week 16"
    ],

    "primary_outcome_landmark":
        "Week 26",

    "secondary_outcome_landmark":
        "Week 52",

    "random_seed":
        SEED,

    "loaded_at":
        datetime.now().isoformat()
}


metadata_file = (
    DATA_METADATA /
    "dataset_metadata.json"
)


with open(
    metadata_file,
    "w"
) as f:

    json.dump(
        dataset_metadata,
        f,
        indent=4
    )


print("Dataset metadata saved:")
print(metadata_file)

In [ ]:
# ============================================================
# STEP 2.16: STEP 2 COMPLETION CHECK
# ============================================================

checks = {
    "Dataset loaded":
        isinstance(raw_df, pd.DataFrame),

    "Dataset has rows":
        len(raw_df) > 0,

    "PTID available":
        "PTID" in raw_df.columns,

    "TOTMOT01 available":
        "TOTMOT01" in raw_df.columns,

    "TOTMOT04 available":
        "TOTMOT04" in raw_df.columns,

    "TOTMOT08 available":
        "TOTMOT08" in raw_df.columns,

    "TOTMOT16 available":
        "TOTMOT16" in raw_df.columns,

    "TOTMOT26 available":
        "TOTMOT26" in raw_df.columns,

    "Raw dataset preserved":
        RAW_PROJECT_PATH.exists(),

    "Metadata saved":
        metadata_file.exists()
}


step2_check = pd.DataFrame({
    "Check": checks.keys(),
    "Passed": checks.values()
})

display(step2_check)


if step2_check["Passed"].all():

    print(
        "\nSTEP 2 COMPLETED SUCCESSFULLY."
    )

else:

    print(
        "\nSTEP 2 contains unresolved issues."
    )

In [ ]:
# ============================================================
# STEP 3.1: CORE DATASET DOCUMENTATION
# ============================================================

dataset_documentation = pd.DataFrame({
    "Item": [
        "Dataset name",
        "Raw cohort size",
        "Unique patients",
        "Patient identifier",
        "Primary prediction target",
        "Problem type",
        "Baseline landmark",
        "Dynamic landmarks",
        "Primary outcome landmark",
        "Secondary outcome landmark",
        "Primary target available N",
        "Primary target missing N",
        "Primary target missing %",
        "AIS blank-coded N",
        "SEXCD observed categories"
    ],
    "Value": [
        "Sygen Traumatic Spinal Cord Injury Longitudinal Cohort",
        raw_df.shape[0],
        raw_df["PTID"].nunique(dropna=True),
        "PTID",
        "TOTMOT26",
        "Regression",
        "Week 1",
        "Week 4, Week 8, Week 16",
        "Week 26",
        "Week 52",
        raw_df["TOTMOT26"].notna().sum(),
        raw_df["TOTMOT26"].isna().sum(),
        round(raw_df["TOTMOT26"].isna().mean() * 100, 2),
        (raw_df["AIS.1"].astype(str).str.strip() == "").sum(),
        sorted(raw_df["SEXCD"].dropna().unique().tolist())
    ]
})

display(dataset_documentation)

In [ ]:
# ============================================================
# STEP 3.2: LONGITUDINAL TIMEPOINT DOCUMENTATION
# ============================================================

timepoint_documentation = pd.DataFrame({
    "Clinical_Timepoint": [
        "Week 1",
        "Week 4",
        "Week 8",
        "Week 16",
        "Week 26",
        "Week 52"
    ],
    "Variable": [
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08",
        "TOTMOT16",
        "TOTMOT26",
        "TOTMOT52"
    ],
    "Role": [
        "Baseline neurological status",
        "Early dynamic landmark",
        "Early dynamic landmark",
        "Early dynamic landmark",
        "Primary prediction outcome",
        "Secondary / sensitivity outcome"
    ],
    "Available_N": [
        raw_df["TOTMOT01"].notna().sum(),
        raw_df["TOTMOT04"].notna().sum(),
        raw_df["TOTMOT08"].notna().sum(),
        raw_df["TOTMOT16"].notna().sum(),
        raw_df["TOTMOT26"].notna().sum(),
        raw_df["TOTMOT52"].notna().sum()
    ],
    "Missing_N": [
        raw_df["TOTMOT01"].isna().sum(),
        raw_df["TOTMOT04"].isna().sum(),
        raw_df["TOTMOT08"].isna().sum(),
        raw_df["TOTMOT16"].isna().sum(),
        raw_df["TOTMOT26"].isna().sum(),
        raw_df["TOTMOT52"].isna().sum()
    ]
})

timepoint_documentation["Missing_%"] = (
    timepoint_documentation["Missing_N"] /
    len(raw_df) * 100
).round(2)

display(timepoint_documentation)

In [ ]:
# ============================================================
# STEP 3.3: DOCUMENT PLANNED DYNAMIC MODELS
# ============================================================

model_documentation = pd.DataFrame({
    "Model": [
        "Model A",
        "Model B4",
        "Model B8",
        "Model B16"
    ],
    "Prediction_Landmark": [
        "Week 1",
        "Week 4",
        "Week 8",
        "Week 16"
    ],
    "Planned_Information": [
        "Baseline clinical + neurological features",
        "Baseline features + Week 1-to-4 motor change",
        "Baseline features + Week 1-to-4 and Week 1-to-8 motor change",
        "Baseline features + Week 1-to-4, Week 1-to-8, and Week 1-to-16 motor change"
    ],
    "Target": [
        "TOTMOT26",
        "TOTMOT26",
        "TOTMOT26",
        "TOTMOT26"
    ]
})

display(model_documentation)

In [ ]:
# ============================================================
# STEP 3.4: CORE FEATURE DOCUMENTATION
# ============================================================

feature_documentation = pd.DataFrame({
    "Variable": [
        "PTID",
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08",
        "TOTMOT16",
        "TOTMOT26",
        "TOTMOT52"
    ],
    "Role": [
        "Patient identifier",
        "Baseline demographic predictor",
        "Baseline demographic predictor",
        "Baseline neurological severity predictor",
        "Baseline motor score",
        "Week 4 motor assessment",
        "Week 8 motor assessment",
        "Week 16 motor assessment",
        "Primary outcome",
        "Secondary outcome"
    ],
    "Data_Type": [
        str(raw_df[col].dtype)
        for col in [
            "PTID",
            "AGE",
            "SEXCD",
            "AIS.1",
            "TOTMOT01",
            "TOTMOT04",
            "TOTMOT08",
            "TOTMOT16",
            "TOTMOT26",
            "TOTMOT52"
        ]
    ],
    "Missing_N": [
        raw_df[col].isna().sum()
        for col in [
            "PTID",
            "AGE",
            "SEXCD",
            "AIS.1",
            "TOTMOT01",
            "TOTMOT04",
            "TOTMOT08",
            "TOTMOT16",
            "TOTMOT26",
            "TOTMOT52"
        ]
    ]
})

display(feature_documentation)

In [ ]:
# ============================================================
# STEP 3.5: SAVE DATASET DOCUMENTATION
# ============================================================

dataset_documentation.to_csv(
    TABLE_DATASET / "dataset_documentation.csv",
    index=False
)

timepoint_documentation.to_csv(
    TABLE_DATASET / "longitudinal_timepoint_documentation.csv",
    index=False
)

model_documentation.to_csv(
    TABLE_DATASET / "dynamic_model_documentation.csv",
    index=False
)

feature_documentation.to_csv(
    TABLE_DATASET / "core_feature_documentation.csv",
    index=False
)

print("Dataset documentation saved successfully.")

print("\nSaved files:")
print(" -", TABLE_DATASET / "dataset_documentation.csv")
print(" -", TABLE_DATASET / "longitudinal_timepoint_documentation.csv")
print(" -", TABLE_DATASET / "dynamic_model_documentation.csv")
print(" -", TABLE_DATASET / "core_feature_documentation.csv")

In [ ]:
# ============================================================
# STEP 3.6: STEP 3 COMPLETION CHECK
# ============================================================

step3_checks = {
    "Raw cohort documented": raw_df.shape[0] == 791,
    "Patient ID documented": "PTID" in raw_df.columns,
    "Primary target documented": "TOTMOT26" in raw_df.columns,
    "Baseline landmark documented": "TOTMOT01" in raw_df.columns,
    "Week 4 landmark documented": "TOTMOT04" in raw_df.columns,
    "Week 8 landmark documented": "TOTMOT08" in raw_df.columns,
    "Week 16 landmark documented": "TOTMOT16" in raw_df.columns,
    "Week 52 secondary outcome documented": "TOTMOT52" in raw_df.columns
}

step3_check_df = pd.DataFrame({
    "Check": step3_checks.keys(),
    "Passed": step3_checks.values()
})

display(step3_check_df)

if step3_check_df["Passed"].all():
    print("\nSTEP 3 COMPLETED SUCCESSFULLY.")
else:
    print("\nSTEP 3 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 4.1: CREATE WORKING COPY FOR QUALITY ASSESSMENT
# ============================================================

quality_df = raw_df.copy()

print("Working copy created successfully.")
print("Shape:", quality_df.shape)

In [ ]:
# ============================================================
# STEP 4.2: NORMALIZE BLANK STRINGS AS MISSING
# ============================================================

import numpy as np

# Count blank strings before normalization
blank_counts_before = {}

for col in quality_df.select_dtypes(include="object").columns:
    blank_count = (
        quality_df[col]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    
    if blank_count > 0:
        blank_counts_before[col] = blank_count

print("Blank-string counts before normalization:")
print(blank_counts_before)

# Convert empty / whitespace-only strings to NaN
object_cols = quality_df.select_dtypes(include="object").columns

quality_df[object_cols] = quality_df[object_cols].replace(
    r"^\s*$",
    np.nan,
    regex=True
)

print("\nBlank strings converted to NaN successfully.")

print("\nAIS.1 missing after normalization:")
print(quality_df["AIS.1"].isna().sum())

In [ ]:
# ============================================================
# STEP 4.3: GLOBAL MISSING VALUE ANALYSIS
# ============================================================

missing_summary = pd.DataFrame({
    "Variable": quality_df.columns,
    "Missing_N": quality_df.isna().sum().values
})

missing_summary["Missing_%"] = (
    missing_summary["Missing_N"] / len(quality_df) * 100
).round(2)

missing_summary = (
    missing_summary
    .sort_values(
        by="Missing_%",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top 30 variables by missingness:")
display(missing_summary.head(30))

In [ ]:
# ============================================================
# STEP 4.4: CORE VARIABLE MISSINGNESS
# ============================================================

core_variables = [
    "PTID",
    "AGE",
    "SEXCD",
    "AIS.1",
    "TOTMOT01",
    "TOTMOT04",
    "TOTMOT08",
    "TOTMOT16",
    "TOTMOT26",
    "TOTMOT52"
]

core_missingness = pd.DataFrame({
    "Variable": core_variables,
    "Available_N": [
        quality_df[col].notna().sum()
        for col in core_variables
    ],
    "Missing_N": [
        quality_df[col].isna().sum()
        for col in core_variables
    ]
})

core_missingness["Missing_%"] = (
    core_missingness["Missing_N"] / len(quality_df) * 100
).round(2)

display(core_missingness)

In [ ]:
# ============================================================
# STEP 4.5: MISSINGNESS CATEGORY DISTRIBUTION
# ============================================================

missing_pct = quality_df.isna().mean() * 100

missingness_categories = pd.Series({
    "0% missing": (missing_pct == 0).sum(),
    ">0 to 20% missing": ((missing_pct > 0) & (missing_pct <= 20)).sum(),
    ">20 to 50% missing": ((missing_pct > 20) & (missing_pct <= 50)).sum(),
    ">50 to 80% missing": ((missing_pct > 50) & (missing_pct <= 80)).sum(),
    ">80 to <100% missing": ((missing_pct > 80) & (missing_pct < 100)).sum(),
    "100% missing": (missing_pct == 100).sum()
})

missingness_category_df = (
    missingness_categories
    .rename("Number_of_Variables")
    .reset_index()
    .rename(columns={"index": "Missingness_Category"})
)

display(missingness_category_df)

In [ ]:
# ============================================================
# STEP 4.6: IDENTIFY COMPLETELY MISSING COLUMNS
# ============================================================

completely_missing_cols = quality_df.columns[
    quality_df.isna().all()
].tolist()

print("Number of completely missing columns:")
print(len(completely_missing_cols))

print("\nFirst 50 completely missing columns:")
print(completely_missing_cols[:50])

In [ ]:
# ============================================================
# STEP 4.7: PATIENT-LEVEL DUPLICATE ASSESSMENT
# ============================================================

duplicate_ptid_mask = quality_df["PTID"].duplicated(
    keep=False
)

duplicate_ptid_rows = quality_df.loc[
    duplicate_ptid_mask,
    ["PTID"]
]

print("Total rows:", len(quality_df))
print("Unique PTIDs:", quality_df["PTID"].nunique(dropna=True))
print("Repeated PTID rows:", duplicate_ptid_mask.sum())

if duplicate_ptid_mask.sum() == 0:
    print("\nNo duplicate patient IDs detected.")
else:
    display(duplicate_ptid_rows)

In [ ]:
# ============================================================
# STEP 4.8: EXACT FULL-ROW DUPLICATE CHECK
# ============================================================

exact_duplicate_rows = quality_df.duplicated(
    keep=False
).sum()

print("Exact duplicated rows:")
print(exact_duplicate_rows)

In [ ]:
# ============================================================
# STEP 4.9: MOTOR SCORE RANGE CHECK
# ============================================================

motor_variables = [
    "TOTMOT01",
    "TOTMOT04",
    "TOTMOT08",
    "TOTMOT16",
    "TOTMOT26",
    "TOTMOT52"
]

range_check_records = []

for col in motor_variables:
    
    invalid_low = (quality_df[col] < 0).sum()
    invalid_high = (quality_df[col] > 100).sum()
    
    range_check_records.append({
        "Variable": col,
        "Min": quality_df[col].min(),
        "Max": quality_df[col].max(),
        "Values_<0": invalid_low,
        "Values_>100": invalid_high,
        "Invalid_Total": invalid_low + invalid_high
    })

motor_range_check = pd.DataFrame(
    range_check_records
)

display(motor_range_check)

In [ ]:
# ============================================================
# STEP 4.10: AGE QUALITY CHECK
# ============================================================

print("AGE summary:")
display(
    quality_df["AGE"].describe()
)

print("\nUnique minimum age values:")
print(
    sorted(
        quality_df["AGE"]
        .dropna()
        .unique()
    )[:10]
)

print("\nUnique maximum age values:")
print(
    sorted(
        quality_df["AGE"]
        .dropna()
        .unique()
    )[-10:]
)

In [ ]:
# ============================================================
# STEP 4.11: CATEGORICAL VARIABLE QUALITY CHECK
# ============================================================

print("AIS.1 distribution after blank normalization:")
print(
    quality_df["AIS.1"]
    .value_counts(dropna=False)
)

print("\nSEXCD distribution:")
print(
    quality_df["SEXCD"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
# ============================================================
# STEP 4.12: NUMERICAL DISTRIBUTION SUMMARY
# ============================================================

numerical_core = [
    "AGE",
    "TOTMOT01",
    "TOTMOT04",
    "TOTMOT08",
    "TOTMOT16",
    "TOTMOT26",
    "TOTMOT52"
]

numerical_summary = (
    quality_df[numerical_core]
    .describe()
    .T
)

numerical_summary["Missing_N"] = (
    quality_df[numerical_core]
    .isna()
    .sum()
)

numerical_summary["Missing_%"] = (
    quality_df[numerical_core]
    .isna()
    .mean() * 100
).round(2)

display(numerical_summary)

In [ ]:
# ============================================================
# STEP 4.13: IQR-BASED OUTLIER SCREENING
# ============================================================

outlier_records = []

for col in numerical_core:
    
    series = quality_df[col].dropna()
    
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outlier_n = (
        (series < lower_bound) |
        (series > upper_bound)
    ).sum()
    
    outlier_records.append({
        "Variable": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_N": outlier_n,
        "Outlier_%": round(
            outlier_n / len(series) * 100,
            2
        )
    })

outlier_summary = pd.DataFrame(
    outlier_records
)

display(outlier_summary)

In [ ]:
# ============================================================
# STEP 4.14: CORE NUMERICAL DISTRIBUTION PLOTS
# ============================================================

for col in numerical_core:
    
    plt.figure(figsize=(7, 5))
    
    sns.histplot(
        data=quality_df,
        x=col,
        kde=True
    )
    
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    
    pdf_path = FIG_DATA / f"{col}_distribution.pdf"
    png_path = FIG_DATA / f"{col}_distribution.png"
    
    plt.savefig(
        pdf_path,
        bbox_inches="tight"
    )
    
    plt.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight"
    )
    
    plt.show()
    plt.close()

print("Distribution plots saved successfully.")

In [ ]:
# ============================================================
# STEP 4.15: LONGITUDINAL MISSINGNESS PLOT
# ============================================================

longitudinal_missing = pd.DataFrame({
    "Timepoint": [
        "Week 1",
        "Week 4",
        "Week 8",
        "Week 16",
        "Week 26",
        "Week 52"
    ],
    "Missing_Percent": [
        quality_df["TOTMOT01"].isna().mean() * 100,
        quality_df["TOTMOT04"].isna().mean() * 100,
        quality_df["TOTMOT08"].isna().mean() * 100,
        quality_df["TOTMOT16"].isna().mean() * 100,
        quality_df["TOTMOT26"].isna().mean() * 100,
        quality_df["TOTMOT52"].isna().mean() * 100
    ]
})

plt.figure(figsize=(8, 5))

plt.plot(
    longitudinal_missing["Timepoint"],
    longitudinal_missing["Missing_Percent"],
    marker="o"
)

plt.xlabel("Follow-up Timepoint")
plt.ylabel("Missing Data (%)")
plt.title("Longitudinal Missingness Across Motor Assessments")
plt.tight_layout()

plt.savefig(
    FIG_DATA / "longitudinal_motor_missingness.pdf",
    bbox_inches="tight"
)

plt.savefig(
    FIG_DATA / "longitudinal_motor_missingness.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

display(longitudinal_missing)

In [ ]:
# ============================================================
# STEP 4.16: COMPLETE-CASE AVAILABILITY BY LANDMARK
# ============================================================

landmark_requirements = {
    
    "Model_A": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT26"
    ],
    
    "Model_B4": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT26"
    ],
    
    "Model_B8": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08",
        "TOTMOT26"
    ],
    
    "Model_B16": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08",
        "TOTMOT16",
        "TOTMOT26"
    ]
}

landmark_availability_records = []

for model_name, cols in landmark_requirements.items():
    
    complete_mask = quality_df[cols].notna().all(axis=1)
    
    n_complete = complete_mask.sum()
    
    landmark_availability_records.append({
        "Model": model_name,
        "Required_Variables": ", ".join(cols),
        "Complete_N": int(n_complete),
        "Excluded_Due_to_Missing": int(len(quality_df) - n_complete),
        "Complete_%": round(
            n_complete / len(quality_df) * 100,
            2
        )
    })

landmark_availability = pd.DataFrame(
    landmark_availability_records
)

display(landmark_availability)

In [ ]:
# ============================================================
# STEP 4.17: SAVE QUALITY ASSESSMENT OUTPUTS
# ============================================================

missing_summary.to_csv(
    TABLE_DATASET / "global_missingness_summary.csv",
    index=False
)

core_missingness.to_csv(
    TABLE_DATASET / "core_variable_missingness.csv",
    index=False
)

missingness_category_df.to_csv(
    TABLE_DATASET / "missingness_categories.csv",
    index=False
)

motor_range_check.to_csv(
    TABLE_DATASET / "motor_score_range_check.csv",
    index=False
)

numerical_summary.to_csv(
    TABLE_DATASET / "numerical_distribution_summary.csv"
)

outlier_summary.to_csv(
    TABLE_DATASET / "iqr_outlier_screening.csv",
    index=False
)

longitudinal_missing.to_csv(
    TABLE_DATASET / "longitudinal_missingness.csv",
    index=False
)

landmark_availability.to_csv(
    TABLE_DATASET / "landmark_complete_case_availability.csv",
    index=False
)

print("STEP 4 quality-assessment outputs saved successfully.")

In [ ]:
# ============================================================
# STEP 4.18: SAVE QUALITY-NORMALIZED DATASET
# ============================================================

quality_dataset_path = (
    DATA_PROCESSED /
    "quality_normalized_dataset.csv"
)

quality_df.to_csv(
    quality_dataset_path,
    index=False
)

print("Quality-normalized dataset saved:")
print(quality_dataset_path)

print("\nShape:", quality_df.shape)

In [ ]:
# ============================================================
# STEP 4.19: STEP 4 COMPLETION CHECK
# ============================================================

step4_checks = {
    "Raw cohort preserved":
        len(quality_df) == len(raw_df),

    "Unique PTIDs preserved":
        quality_df["PTID"].nunique(dropna=True) == 791,

    "No repeated PTIDs":
        quality_df["PTID"].duplicated().sum() == 0,

    "AIS blanks normalized":
        (
            quality_df["AIS.1"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
            ==
            quality_df["AIS.1"].isna().sum()
        ),

    "Primary target present":
        "TOTMOT26" in quality_df.columns,

    "No invalid TOTMOT26 below 0":
        (quality_df["TOTMOT26"].dropna() < 0).sum() == 0,

    "No invalid TOTMOT26 above 100":
        (quality_df["TOTMOT26"].dropna() > 100).sum() == 0,

    "Quality dataset saved":
        quality_dataset_path.exists()
}

step4_check_df = pd.DataFrame({
    "Check": step4_checks.keys(),
    "Passed": step4_checks.values()
})

display(step4_check_df)

if step4_check_df["Passed"].all():
    print("\nSTEP 4 COMPLETED SUCCESSFULLY.")
else:
    print("\nSTEP 4 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 5.1: DEFINE PRIMARY CLINICAL PREDICTION PROBLEM
# ============================================================

PREDICTION_CONFIG = {
    "study_name":
        "SCI Dynamic Recovery Prediction",

    "problem_type":
        "Regression",

    "patient_id":
        "PTID",

    "baseline_landmark":
        "Week 1",

    "primary_target":
        "TOTMOT26",

    "primary_outcome_landmark":
        "Week 26",

    "secondary_target":
        "TOTMOT52",

    "secondary_outcome_landmark":
        "Week 52",

    "dynamic_landmarks": [
        "Week 4",
        "Week 8",
        "Week 16"
    ],

    "primary_research_question":
        (
            "Does incorporating early longitudinal neurological "
            "change improve prediction of Week-26 motor recovery "
            "beyond baseline neurological and clinical characteristics?"
        ),

    "primary_hypothesis":
        (
            "Early neurological motor change provides incremental "
            "prognostic information beyond baseline neurological status."
        )
}

for key, value in PREDICTION_CONFIG.items():
    print(f"{key}: {value}")

In [ ]:
# ============================================================
# STEP 5.2: DEFINE PRIMARY ANALYTIC COHORT
# ============================================================

PRIMARY_TARGET = "TOTMOT26"

primary_cohort = quality_df.loc[
    quality_df[PRIMARY_TARGET].notna()
].copy()

print("Primary analytic cohort created.")

print("\nRaw cohort N:")
print(len(quality_df))

print("\nPrimary outcome available N:")
print(len(primary_cohort))

print("\nExcluded because TOTMOT26 missing:")
print(len(quality_df) - len(primary_cohort))

print("\nPrimary cohort percentage:")
print(
    round(
        len(primary_cohort) / len(quality_df) * 100,
        2
    ),
    "%"
)

In [ ]:
# ============================================================
# STEP 5.3: TARGET MISSING-DATA POLICY
# ============================================================

target_policy = {
    "Primary target": "TOTMOT26",
    "Target imputation allowed": False,
    "Patients with missing TOTMOT26": "Excluded from primary modelling",
    "Predictor imputation": "Allowed inside training pipeline only"
}

target_policy_df = pd.DataFrame(
    target_policy.items(),
    columns=["Rule", "Decision"]
)

display(target_policy_df)

In [ ]:
# ============================================================
# STEP 5.4: DEFINE BASELINE FEATURE SET
# ============================================================

BASELINE_FEATURES = [
    "AGE",
    "SEXCD",
    "AIS.1",
    "TOTMOT01"
]

print("Baseline features:")

for feature in BASELINE_FEATURES:
    print(" -", feature)

In [ ]:
# ============================================================
# STEP 5.5: DEFINE DYNAMIC MOTOR CHANGE VARIABLES
# ============================================================

DYNAMIC_CHANGE_DEFINITIONS = {
    "TOTMOT_change_4":
        "TOTMOT04 - TOTMOT01",

    "TOTMOT_change_8":
        "TOTMOT08 - TOTMOT01",

    "TOTMOT_change_16":
        "TOTMOT16 - TOTMOT01"
}

for name, definition in DYNAMIC_CHANGE_DEFINITIONS.items():
    print(f"{name}: {definition}")

In [ ]:
# ============================================================
# STEP 5.6: DEFINE LANDMARK MODEL FEATURE SETS
# ============================================================

MODEL_FEATURES = {
    "Model_A": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01"
    ],

    "Model_B4": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT_change_4"
    ],

    "Model_B8": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT_change_4",
        "TOTMOT_change_8"
    ],

    "Model_B16": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT_change_4",
        "TOTMOT_change_8",
        "TOTMOT_change_16"
    ]
}

for model_name, features in MODEL_FEATURES.items():
    print(f"\n{model_name}")
    for feature in features:
        print(" -", feature)

In [ ]:
# ============================================================
# STEP 5.7: LANDMARK FEATURE AVAILABILITY MAP
# ============================================================

feature_timeline = pd.DataFrame({
    "Feature": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT_change_4",
        "TOTMOT_change_8",
        "TOTMOT_change_16",
        "TOTMOT26"
    ],

    "Available_By": [
        "Week 1",
        "Week 1",
        "Week 1",
        "Week 1",
        "Week 4",
        "Week 8",
        "Week 16",
        "Week 26"
    ],

    "Role": [
        "Baseline predictor",
        "Baseline predictor",
        "Baseline predictor",
        "Baseline predictor",
        "Dynamic predictor",
        "Dynamic predictor",
        "Dynamic predictor",
        "Outcome"
    ]
})

display(feature_timeline)

In [ ]:
# ============================================================
# STEP 5.8: DEFINE FORMAL LEAKAGE RULES
# ============================================================

leakage_rules = pd.DataFrame({
    "Model": [
        "Model_A",
        "Model_B4",
        "Model_B8",
        "Model_B16"
    ],

    "Prediction_Landmark": [
        "Week 1",
        "Week 4",
        "Week 8",
        "Week 16"
    ],

    "Latest_Allowed_Information": [
        "Week 1",
        "Week 4",
        "Week 8",
        "Week 16"
    ],

    "Future_Information_Forbidden": [
        "Week 4 and later",
        "After Week 4",
        "After Week 8",
        "After Week 16"
    ]
})

display(leakage_rules)

In [ ]:
# ============================================================
# STEP 5.9: DEFINE PREDICTOR MISSING-DATA POLICY
# ============================================================

missing_data_policy = pd.DataFrame({
    "Variable_Type": [
        "Continuous predictors",
        "Categorical predictors",
        "Primary outcome"
    ],

    "Handling": [
        "Median imputation fitted on training data only",
        "Most-frequent imputation fitted on training data only",
        "No imputation; exclude missing target"
    ]
})

display(missing_data_policy)

In [ ]:
# ============================================================
# STEP 5.10: DEFINE AGE POLICY
# ============================================================

age_policy = pd.DataFrame({
    "Issue": [
        "Observed age range",
        "Primary analysis",
        "Sensitivity analysis"
    ],

    "Decision": [
        f"{quality_df['AGE'].min()} to {quality_df['AGE'].max()} years",
        "Retain all eligible patients unless dataset protocol specifies otherwise",
        "Consider adult-only analysis (AGE >= 18) as sensitivity analysis"
    ]
})

display(age_policy)

In [ ]:
# ============================================================
# STEP 5.11: DEFINE PRIMARY AND SECONDARY ANALYSES
# ============================================================

analysis_plan = pd.DataFrame({
    "Analysis": [
        "Primary",
        "Dynamic ablation",
        "Secondary outcome",
        "Age sensitivity",
        "Clinical subgroup"
    ],

    "Definition": [
        "Predict TOTMOT26 using dynamic landmark models",
        "Compare Model A vs B4 vs B8 vs B16",
        "Repeat framework for TOTMOT52 if feasible",
        "Repeat primary analysis in AGE >= 18 subgroup",
        "Evaluate AIS, age, sex, baseline severity subgroups"
    ]
})

display(analysis_plan)

In [ ]:
# ============================================================
# STEP 5.12: PRIMARY COHORT PREDICTOR MISSINGNESS
# ============================================================

primary_predictor_variables = [
    "AGE",
    "SEXCD",
    "AIS.1",
    "TOTMOT01",
    "TOTMOT04",
    "TOTMOT08",
    "TOTMOT16"
]

primary_predictor_missingness = pd.DataFrame({
    "Variable": primary_predictor_variables,

    "Available_N": [
        primary_cohort[col].notna().sum()
        for col in primary_predictor_variables
    ],

    "Missing_N": [
        primary_cohort[col].isna().sum()
        for col in primary_predictor_variables
    ]
})

primary_predictor_missingness["Missing_%"] = (
    primary_predictor_missingness["Missing_N"] /
    len(primary_cohort) * 100
).round(2)

display(primary_predictor_missingness)

In [ ]:
# ============================================================
# STEP 5.13: VERIFY PRIMARY COHORT PATIENT IDs
# ============================================================

print("Primary cohort rows:")
print(len(primary_cohort))

print("\nUnique primary cohort PTIDs:")
print(primary_cohort["PTID"].nunique())

print("\nMissing PTIDs:")
print(primary_cohort["PTID"].isna().sum())

print("\nRepeated PTIDs:")
print(primary_cohort["PTID"].duplicated().sum())

In [ ]:
# ============================================================
# STEP 5.14: SAVE CLINICAL PREDICTION CONFIGURATION
# ============================================================

import json

prediction_config_file = (
    CONFIG_DIR /
    "prediction_config.json"
)

config_to_save = {
    "study_name":
        PREDICTION_CONFIG["study_name"],

    "problem_type":
        "regression",

    "patient_id":
        "PTID",

    "primary_target":
        "TOTMOT26",

    "secondary_target":
        "TOTMOT52",

    "baseline_landmark":
        "Week 1",

    "dynamic_landmarks":
        ["Week 4", "Week 8", "Week 16"],

    "baseline_features":
        BASELINE_FEATURES,

    "model_features":
        MODEL_FEATURES,

    "dynamic_change_definitions":
        DYNAMIC_CHANGE_DEFINITIONS,

    "target_imputation":
        False,

    "predictor_imputation":
        "Training-fold only",

    "random_seed":
        SEED
}

with open(
    prediction_config_file,
    "w"
) as f:

    json.dump(
        config_to_save,
        f,
        indent=4
    )

print("Prediction configuration saved:")
print(prediction_config_file)

In [ ]:
# ============================================================
# STEP 5.15: SAVE PRIMARY ANALYTIC COHORT
# ============================================================

primary_cohort_path = (
    DATA_PROCESSED /
    "primary_week26_analytic_cohort.csv"
)

primary_cohort.to_csv(
    primary_cohort_path,
    index=False
)

print("Primary analytic cohort saved:")
print(primary_cohort_path)

print("\nShape:")
print(primary_cohort.shape)

In [ ]:
# ============================================================
# STEP 5.16: SAVE STUDY-DEFINITION TABLES
# ============================================================

feature_timeline.to_csv(
    TABLE_DATASET / "feature_timeline.csv",
    index=False
)

leakage_rules.to_csv(
    TABLE_DATASET / "landmark_leakage_rules.csv",
    index=False
)

missing_data_policy.to_csv(
    TABLE_DATASET / "missing_data_policy.csv",
    index=False
)

analysis_plan.to_csv(
    TABLE_DATASET / "analysis_plan.csv",
    index=False
)

primary_predictor_missingness.to_csv(
    TABLE_DATASET / "primary_cohort_predictor_missingness.csv",
    index=False
)

print("Study-definition tables saved successfully.")

In [ ]:
# ============================================================
# STEP 5.17: STEP 5 COMPLETION CHECK
# ============================================================

step5_checks = {
    "Primary target defined":
        PRIMARY_TARGET == "TOTMOT26",

    "Primary cohort has observed target only":
        primary_cohort["TOTMOT26"].isna().sum() == 0,

    "Primary cohort size is valid":
        len(primary_cohort) > 0,

    "Patient IDs unique":
        primary_cohort["PTID"].duplicated().sum() == 0,

    "Baseline features defined":
        len(BASELINE_FEATURES) == 4,

    "Four landmark models defined":
        len(MODEL_FEATURES) == 4,

    "Prediction config saved":
        prediction_config_file.exists(),

    "Primary cohort saved":
        primary_cohort_path.exists()
}

step5_check_df = pd.DataFrame({
    "Check": step5_checks.keys(),
    "Passed": step5_checks.values()
})

display(step5_check_df)

if step5_check_df["Passed"].all():
    print("\nSTEP 5 COMPLETED SUCCESSFULLY.")
else:
    print("\nSTEP 5 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 6.1: IDENTIFY POTENTIAL LEAKAGE VARIABLES BY NAME
# ============================================================

import re
import pandas as pd

all_columns = pd.Series(primary_cohort.columns, name="Variable")

leakage_patterns = [
    r"52",
    r"26",
    r"change52",
    r"follow",
    r"outcome",
    r"recovery",
    r"future"
]

pattern_regex = "|".join(leakage_patterns)

potential_leakage_by_name = all_columns[
    all_columns.str.contains(
        pattern_regex,
        case=False,
        regex=True,
        na=False
    )
].reset_index(drop=True)

print("Potential leakage-related variables found:")
print(len(potential_leakage_by_name))

display(
    potential_leakage_by_name.to_frame()
)

In [ ]:
# ============================================================
# STEP 6.2: INSPECT SUSPICIOUS PRECOMPUTED CHANGE VARIABLES
# ============================================================

suspicious_terms = [
    "change52",
    "lemschange52",
    "uemschange52"
]

suspicious_columns = [
    col
    for col in primary_cohort.columns
    if any(term.lower() in col.lower() for term in suspicious_terms)
]

print("Suspicious precomputed change variables:")
print(len(suspicious_columns))

for col in suspicious_columns:
    print(" -", col)

In [ ]:
# ============================================================
# STEP 6.3: DEFINE VARIABLE AVAILABILITY TIMELINE
# ============================================================

variable_timeline = pd.DataFrame({
    "Variable": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08",
        "TOTMOT16",
        "TOTMOT26",
        "TOTMOT52"
    ],

    "Time_Available": [
        "Week 1",
        "Week 1",
        "Week 1",
        "Week 1",
        "Week 4",
        "Week 8",
        "Week 16",
        "Week 26",
        "Week 52"
    ],

    "Allowed_As_Predictor": [
        True,
        True,
        True,
        True,
        True,
        True,
        True,
        False,
        False
    ],

    "Role": [
        "Baseline predictor",
        "Baseline predictor",
        "Baseline predictor",
        "Baseline predictor",
        "Dynamic source",
        "Dynamic source",
        "Dynamic source",
        "Primary outcome",
        "Secondary outcome"
    ]
})

display(variable_timeline)

In [ ]:
# ============================================================
# STEP 6.4: DEFINE ALLOWED RAW INPUTS FOR EACH LANDMARK
# ============================================================

ALLOWED_RAW_FEATURES = {

    "Model_A": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01"
    ],

    "Model_B4": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04"
    ],

    "Model_B8": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08"
    ],

    "Model_B16": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08",
        "TOTMOT16"
    ]
}

for model_name, features in ALLOWED_RAW_FEATURES.items():
    print(f"\n{model_name}")
    for feature in features:
        print(" -", feature)

In [ ]:
# ============================================================
# STEP 6.5: DEFINE FORBIDDEN FUTURE VARIABLES
# ============================================================

FORBIDDEN_BY_MODEL = {

    "Model_A": [
        "TOTMOT04",
        "TOTMOT08",
        "TOTMOT16",
        "TOTMOT26",
        "TOTMOT52"
    ],

    "Model_B4": [
        "TOTMOT08",
        "TOTMOT16",
        "TOTMOT26",
        "TOTMOT52"
    ],

    "Model_B8": [
        "TOTMOT16",
        "TOTMOT26",
        "TOTMOT52"
    ],

    "Model_B16": [
        "TOTMOT26",
        "TOTMOT52"
    ]
}

for model_name, forbidden in FORBIDDEN_BY_MODEL.items():
    print(f"\n{model_name} forbidden variables:")
    for feature in forbidden:
        print(" -", feature)

In [ ]:
# ============================================================
# STEP 6.6: AUTOMATED LEAKAGE CHECK
# ============================================================

planned_raw_features = {
    "Model_A": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01"
    ],

    "Model_B4": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04"
    ],

    "Model_B8": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08"
    ],

    "Model_B16": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08",
        "TOTMOT16"
    ]
}

leakage_check_records = []

for model_name, features in planned_raw_features.items():

    forbidden = FORBIDDEN_BY_MODEL[model_name]

    detected = [
        feature
        for feature in features
        if feature in forbidden
    ]

    leakage_check_records.append({
        "Model": model_name,
        "Number_of_Features": len(features),
        "Leakage_Detected": len(detected) > 0,
        "Forbidden_Features_Found":
            ", ".join(detected) if detected else "None"
    })

leakage_check_df = pd.DataFrame(
    leakage_check_records
)

display(leakage_check_df)

In [ ]:
# ============================================================
# STEP 6.7: EXPLICIT TARGET LEAKAGE CHECK
# ============================================================

TARGET_VARIABLES = [
    "TOTMOT26",
    "TOTMOT52"
]

target_leakage_records = []

for model_name, features in planned_raw_features.items():

    leaked_targets = [
        target
        for target in TARGET_VARIABLES
        if target in features
    ]

    target_leakage_records.append({
        "Model": model_name,
        "Target_Leakage": len(leaked_targets) > 0,
        "Leaked_Targets":
            ", ".join(leaked_targets) if leaked_targets else "None"
    })

target_leakage_df = pd.DataFrame(
    target_leakage_records
)

display(target_leakage_df)

In [ ]:
# ============================================================
# STEP 6.8: FLAG PRECOMPUTED CHANGE VARIABLES
# ============================================================

change_like_columns = [
    col
    for col in primary_cohort.columns
    if "change" in col.lower()
]

change_variable_audit = []

for col in change_like_columns:

    suspicious_future_reference = (
        "52" in col.lower()
        or "26" in col.lower()
    )

    change_variable_audit.append({
        "Variable": col,
        "Contains_26_or_52":
            suspicious_future_reference,

        "Default_Decision":
            "EXCLUDE until verified"
            if suspicious_future_reference
            else "Manual review required"
    })

change_variable_audit_df = pd.DataFrame(
    change_variable_audit
)

display(
    change_variable_audit_df.head(100)
)

print("\nTotal change-like variables:")
print(len(change_variable_audit_df))

In [ ]:
# ============================================================
# STEP 6.9: DEFINE SAFE DYNAMIC FEATURE POLICY
# ============================================================

safe_dynamic_feature_policy = pd.DataFrame({
    "Derived_Feature": [
        "TOTMOT_change_4",
        "TOTMOT_change_8",
        "TOTMOT_change_16"
    ],

    "Formula": [
        "TOTMOT04 - TOTMOT01",
        "TOTMOT08 - TOTMOT01",
        "TOTMOT16 - TOTMOT01"
    ],

    "Latest_Information_Used": [
        "Week 4",
        "Week 8",
        "Week 16"
    ],

    "Leakage_Status": [
        "Safe for B4/B8/B16",
        "Safe for B8/B16",
        "Safe for B16"
    ]
})

display(safe_dynamic_feature_policy)

In [ ]:
# ============================================================
# STEP 6.10: BASELINE FEATURE LEAKAGE CHECK
# ============================================================

future_tokens = [
    "04",
    "08",
    "16",
    "26",
    "52",
    "change"
]

baseline_leakage_suspects = []

for feature in BASELINE_FEATURES:

    if any(token in feature.lower() for token in future_tokens):
        baseline_leakage_suspects.append(feature)

print("Baseline leakage suspects:")
print(baseline_leakage_suspects)

if len(baseline_leakage_suspects) == 0:
    print("\nBaseline feature set passes temporal leakage screening.")

In [ ]:
# ============================================================
# STEP 6.11: FORMAL LEAKAGE PREVENTION TABLE
# ============================================================

leakage_prevention_table = pd.DataFrame({
    "Model": [
        "Model_A",
        "Model_B4",
        "Model_B8",
        "Model_B16"
    ],

    "Prediction_Time": [
        "Week 1",
        "Week 4",
        "Week 8",
        "Week 16"
    ],

    "Latest_Allowed_Raw_Motor_Assessment": [
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08",
        "TOTMOT16"
    ],

    "Derived_Change_Features": [
        "None",
        "TOTMOT04 - TOTMOT01",
        "TOTMOT04 - TOTMOT01; TOTMOT08 - TOTMOT01",
        (
            "TOTMOT04 - TOTMOT01; "
            "TOTMOT08 - TOTMOT01; "
            "TOTMOT16 - TOTMOT01"
        )
    ],

    "Outcome_Excluded": [
        "TOTMOT26, TOTMOT52",
        "TOTMOT26, TOTMOT52",
        "TOTMOT26, TOTMOT52",
        "TOTMOT26, TOTMOT52"
    ]
})

display(leakage_prevention_table)

In [ ]:
# ============================================================
# STEP 6.12: SAVE LEAKAGE AUDIT OUTPUTS
# ============================================================

variable_timeline.to_csv(
    TABLE_DATASET / "variable_timeline.csv",
    index=False
)

leakage_check_df.to_csv(
    TABLE_DATASET / "planned_feature_leakage_check.csv",
    index=False
)

target_leakage_df.to_csv(
    TABLE_DATASET / "target_leakage_check.csv",
    index=False
)

change_variable_audit_df.to_csv(
    TABLE_DATASET / "precomputed_change_variable_audit.csv",
    index=False
)

safe_dynamic_feature_policy.to_csv(
    TABLE_DATASET / "safe_dynamic_feature_policy.csv",
    index=False
)

leakage_prevention_table.to_csv(
    TABLE_DATASET / "leakage_prevention_table.csv",
    index=False
)

print("Leakage audit outputs saved successfully.")

In [ ]:
# ============================================================
# STEP 6.13: FINAL LEAKAGE DECISION
# ============================================================

all_planned_models_safe = (
    (~leakage_check_df["Leakage_Detected"]).all()
    and
    (~target_leakage_df["Target_Leakage"]).all()
)

print("=" * 70)
print("DATA LEAKAGE VERIFICATION")
print("=" * 70)

print("\nPlanned landmark feature sets safe:")
print(all_planned_models_safe)

print("\nPolicy:")
print(
    "Only raw measurements available at or before each "
    "prediction landmark will be used."
)

print(
    "Precomputed change variables referring to Week 26 or "
    "Week 52 are excluded unless independently verified."
)

print(
    "Dynamic changes will be derived directly from "
    "TOTMOT01, TOTMOT04, TOTMOT08, and TOTMOT16."
)

if all_planned_models_safe:
    print("\nSTEP 6 LEAKAGE VERIFICATION PASSED.")
else:
    print("\nSTEP 6 contains unresolved leakage issues.")

In [ ]:
# ============================================================
# STEP 7.1: DEFINE PATIENT-LEVEL SPLIT PROPORTIONS
# ============================================================

TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

print("Split proportions:")
print(f"Train      : {TRAIN_SIZE:.0%}")
print(f"Validation : {VALIDATION_SIZE:.0%}")
print(f"Test       : {TEST_SIZE:.0%}")

In [ ]:
# ============================================================
# STEP 7.2: PREPARE PRIMARY COHORT FOR SPLITTING
# ============================================================

# Make sure quality_df exists
print("quality_df available:", "quality_df" in globals())

PRIMARY_TARGET = "TOTMOT26"

primary_cohort = quality_df.loc[
    quality_df[PRIMARY_TARGET].notna()
].copy()

print("Primary analytic cohort recreated.")

print("\nShape:")
print(primary_cohort.shape)

print("\nUnique PTIDs:")
print(primary_cohort["PTID"].nunique())

print("\nMissing TOTMOT26:")
print(primary_cohort["TOTMOT26"].isna().sum())

In [ ]:
# ============================================================
# STEP 7.3: CREATE PRIMARY COHORT + TRAIN/TEMPORARY SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# -----------------------------
# Reproducibility
# -----------------------------

if "SEED" not in globals():
    SEED = 42


# -----------------------------
# Recreate primary cohort if missing
# -----------------------------

if "primary_cohort" not in globals():

    primary_cohort = quality_df.loc[
        quality_df["TOTMOT26"].notna()
    ].copy()


# -----------------------------
# Prepare split dataset
# -----------------------------

split_df = primary_cohort.copy()


print("="*60)
print("PRIMARY COHORT CHECK")
print("="*60)

print("Primary cohort shape:")
print(split_df.shape)

print("\nUnique PTIDs:")
print(split_df["PTID"].nunique())

print("\nMissing TOTMOT26:")
print(split_df["TOTMOT26"].isna().sum())


# -----------------------------
# Train / Temporary Holdout Split
# -----------------------------

train_df, temp_df = train_test_split(
    split_df,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)


print("\n" + "="*60)
print("TRAIN / TEMPORARY SPLIT")
print("="*60)

print("Train shape:")
print(train_df.shape)

print("\nTemporary holdout shape:")
print(temp_df.shape)

In [ ]:
# ============================================================
# STEP 7.4: VALIDATION / INTERNAL TEST SPLIT
# ============================================================

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Train N      :", len(train_df))
print("Validation N :", len(validation_df))
print("Test N       :", len(test_df))

print("\nTotal:")
print(
    len(train_df) +
    len(validation_df) +
    len(test_df)
)

In [ ]:
# ============================================================
# STEP 7.5: RESET DATAFRAME INDICES
# ============================================================

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Indices reset successfully.")

In [ ]:
# ============================================================
# STEP 7.6: VERIFY ZERO PATIENT OVERLAP
# ============================================================

train_ids = set(train_df["PTID"])
validation_ids = set(validation_df["PTID"])
test_ids = set(test_df["PTID"])

train_validation_overlap = train_ids.intersection(validation_ids)
train_test_overlap = train_ids.intersection(test_ids)
validation_test_overlap = validation_ids.intersection(test_ids)

print("Train vs Validation overlap:",
      len(train_validation_overlap))

print("Train vs Test overlap:",
      len(train_test_overlap))

print("Validation vs Test overlap:",
      len(validation_test_overlap))

if (
    len(train_validation_overlap) == 0
    and len(train_test_overlap) == 0
    and len(validation_test_overlap) == 0
):
    print("\nNo patient overlap detected.")
else:
    print("\nWARNING: Patient overlap detected.")

In [ ]:
# ============================================================
# STEP 7.7: VERIFY FULL COHORT PRESERVATION
# ============================================================

combined_ids = (
    train_ids |
    validation_ids |
    test_ids
)

print("Original primary cohort patients:",
      primary_cohort["PTID"].nunique())

print("Patients after splitting:",
      len(combined_ids))

print(
    "All patients preserved:",
    len(combined_ids) ==
    primary_cohort["PTID"].nunique()
)

In [ ]:
# ============================================================
# STEP 7.8: SPLIT SUMMARY
# ============================================================

split_summary = pd.DataFrame({
    "Split": [
        "Train",
        "Validation",
        "Internal Test",
        "Total"
    ],
    "N": [
        len(train_df),
        len(validation_df),
        len(test_df),
        (
            len(train_df) +
            len(validation_df) +
            len(test_df)
        )
    ]
})

split_summary["Percent"] = [
    round(len(train_df) / len(split_df) * 100, 2),
    round(len(validation_df) / len(split_df) * 100, 2),
    round(len(test_df) / len(split_df) * 100, 2),
    100.00
]

display(split_summary)

In [ ]:
# ============================================================
# STEP 7.9: TARGET DISTRIBUTION ACROSS SPLITS
# ============================================================

def summarize_target(df, split_name):
    return {
        "Split": split_name,
        "N": len(df),
        "Mean": df["TOTMOT26"].mean(),
        "SD": df["TOTMOT26"].std(),
        "Median": df["TOTMOT26"].median(),
        "Min": df["TOTMOT26"].min(),
        "Max": df["TOTMOT26"].max()
    }


target_split_summary = pd.DataFrame([
    summarize_target(train_df, "Train"),
    summarize_target(validation_df, "Validation"),
    summarize_target(test_df, "Internal Test")
])

display(target_split_summary)

In [ ]:
# ============================================================
# STEP 7.10: BASELINE CHARACTERISTICS ACROSS SPLITS
# ============================================================

split_characteristics = []

for split_name, df in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Internal Test", test_df)
]:

    split_characteristics.append({
        "Split": split_name,

        "N": len(df),

        "Age_Mean":
            df["AGE"].mean(),

        "Age_SD":
            df["AGE"].std(),

        "TOTMOT01_Mean":
            df["TOTMOT01"].mean(),

        "TOTMOT01_SD":
            df["TOTMOT01"].std(),

        "AIS_Missing_N":
            df["AIS.1"].isna().sum(),

        "SEXCD_1_N":
            (df["SEXCD"] == 1).sum(),

        "SEXCD_2_N":
            (df["SEXCD"] == 2).sum()
    })

split_characteristics_df = pd.DataFrame(
    split_characteristics
)

display(split_characteristics_df)

In [ ]:
# ============================================================
# STEP 7.11: AIS DISTRIBUTION ACROSS SPLITS
# ============================================================

ais_split_distribution = pd.concat(
    [
        train_df["AIS.1"]
        .value_counts(dropna=False)
        .rename("Train"),

        validation_df["AIS.1"]
        .value_counts(dropna=False)
        .rename("Validation"),

        test_df["AIS.1"]
        .value_counts(dropna=False)
        .rename("Internal_Test")
    ],
    axis=1
).fillna(0).astype(int)

display(ais_split_distribution)

In [ ]:
# ============================================================
# STEP 7.12: SAVE PATIENT-LEVEL SPLITS
# ============================================================

TRAIN_PATH = DATA_SPLITS / "train.csv"
VALIDATION_PATH = DATA_SPLITS / "validation.csv"
TEST_PATH = DATA_SPLITS / "internal_test.csv"

train_df.to_csv(
    TRAIN_PATH,
    index=False
)

validation_df.to_csv(
    VALIDATION_PATH,
    index=False
)

test_df.to_csv(
    TEST_PATH,
    index=False
)

print("Patient-level splits saved successfully.")

print("\nSaved files:")
print(" -", TRAIN_PATH)
print(" -", VALIDATION_PATH)
print(" -", TEST_PATH)

In [ ]:
# ============================================================
# STEP 7.13: SAVE PATIENT ID SPLIT LISTS
# ============================================================

pd.DataFrame({
    "PTID": train_df["PTID"]
}).to_csv(
    DATA_SPLITS / "train_ptids.csv",
    index=False
)

pd.DataFrame({
    "PTID": validation_df["PTID"]
}).to_csv(
    DATA_SPLITS / "validation_ptids.csv",
    index=False
)

pd.DataFrame({
    "PTID": test_df["PTID"]
}).to_csv(
    DATA_SPLITS / "internal_test_ptids.csv",
    index=False
)

print("PTID split files saved successfully.")

In [ ]:
# ============================================================
# STEP 7.14: SAVE SPLIT METADATA
# ============================================================

import json

split_metadata = {
    "primary_cohort_n":
        int(len(primary_cohort)),

    "train_n":
        int(len(train_df)),

    "validation_n":
        int(len(validation_df)),

    "internal_test_n":
        int(len(test_df)),

    "train_fraction":
        TRAIN_SIZE,

    "validation_fraction":
        VALIDATION_SIZE,

    "test_fraction":
        TEST_SIZE,

    "random_seed":
        SEED,

    "patient_identifier":
        "PTID",

    "target":
        "TOTMOT26",

    "split_level":
        "patient",

    "patient_overlap":
        False
}

split_metadata_path = (
    CONFIG_DIR /
    "patient_split_metadata.json"
)

with open(
    split_metadata_path,
    "w"
) as f:
    json.dump(
        split_metadata,
        f,
        indent=4
    )

print("Split metadata saved:")
print(split_metadata_path)

In [ ]:
# ============================================================
# STEP 7.15: SAVE SPLIT SUMMARY TABLES
# ============================================================

split_summary.to_csv(
    TABLE_DATASET /
    "patient_split_summary.csv",
    index=False
)

target_split_summary.to_csv(
    TABLE_DATASET /
    "target_distribution_by_split.csv",
    index=False
)

split_characteristics_df.to_csv(
    TABLE_DATASET /
    "baseline_characteristics_by_split.csv",
    index=False
)

ais_split_distribution.to_csv(
    TABLE_DATASET /
    "ais_distribution_by_split.csv"
)

print("Split summary tables saved successfully.")

In [ ]:
# ============================================================
# STEP 7.16: FINAL STEP 7 INTEGRITY CHECK
# ============================================================

step7_checks = {
    "Primary cohort N = 659":
        len(primary_cohort) == 659,

    "Train/validation/test total = 659":
        (
            len(train_df) +
            len(validation_df) +
            len(test_df)
        ) == 659,

    "Train PTIDs unique":
        train_df["PTID"].duplicated().sum() == 0,

    "Validation PTIDs unique":
        validation_df["PTID"].duplicated().sum() == 0,

    "Test PTIDs unique":
        test_df["PTID"].duplicated().sum() == 0,

    "No train-validation overlap":
        len(train_validation_overlap) == 0,

    "No train-test overlap":
        len(train_test_overlap) == 0,

    "No validation-test overlap":
        len(validation_test_overlap) == 0,

    "Train target complete":
        train_df["TOTMOT26"].isna().sum() == 0,

    "Validation target complete":
        validation_df["TOTMOT26"].isna().sum() == 0,

    "Test target complete":
        test_df["TOTMOT26"].isna().sum() == 0,

    "Split files saved":
        (
            TRAIN_PATH.exists()
            and VALIDATION_PATH.exists()
            and TEST_PATH.exists()
        )
}

step7_check_df = pd.DataFrame({
    "Check": step7_checks.keys(),
    "Passed": step7_checks.values()
})

display(step7_check_df)

if step7_check_df["Passed"].all():
    print("\nSTEP 7 COMPLETED SUCCESSFULLY.")
else:
    print("\nSTEP 7 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 8.1: FEATURE ENGINEERING FUNCTION
# ============================================================

import numpy as np
import pandas as pd


def create_dynamic_features(df):
    
    data = df.copy()
    
    # Dynamic motor recovery features
    data["TOTMOT_change_4"] = (
        data["TOTMOT04"] -
        data["TOTMOT01"]
    )
    
    data["TOTMOT_change_8"] = (
        data["TOTMOT08"] -
        data["TOTMOT01"]
    )
    
    data["TOTMOT_change_16"] = (
        data["TOTMOT16"] -
        data["TOTMOT01"]
    )
    
    return data


print("Feature engineering function created.")

In [ ]:
# ============================================================
# STEP 8.2: CREATE ENGINEERED DATASETS
# ============================================================


train_fe = create_dynamic_features(train_df)

validation_fe = create_dynamic_features(validation_df)

test_fe = create_dynamic_features(test_df)


print("Train shape:")
print(train_fe.shape)

print("\nValidation shape:")
print(validation_fe.shape)

print("\nTest shape:")
print(test_fe.shape)

In [ ]:
# ============================================================
# STEP 8.3: CHECK ENGINEERED FEATURES
# ============================================================

dynamic_features = [
    "TOTMOT_change_4",
    "TOTMOT_change_8",
    "TOTMOT_change_16"
]


feature_check = pd.DataFrame({
    
    "Feature": dynamic_features,
    
    "Train_Missing":
    [
        train_fe[col].isna().sum()
        for col in dynamic_features
    ],
    
    "Validation_Missing":
    [
        validation_fe[col].isna().sum()
        for col in dynamic_features
    ],
    
    "Test_Missing":
    [
        test_fe[col].isna().sum()
        for col in dynamic_features
    ]
})


display(feature_check)

In [ ]:
# ============================================================
# STEP 8.4: MANUAL FEATURE VERIFICATION
# ============================================================


sample_check = train_fe[
    [
        "PTID",
        "TOTMOT01",
        "TOTMOT04",
        "TOTMOT08",
        "TOTMOT16",
        "TOTMOT_change_4",
        "TOTMOT_change_8",
        "TOTMOT_change_16"
    ]
].head(10)


display(sample_check)

In [ ]:
# ============================================================
# STEP 8.5: DEFINE LANDMARK FEATURE SETS
# ============================================================


MODEL_FEATURES = {


    "Model_A": [

        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01"

    ],



    "Model_B4": [

        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT_change_4"

    ],



    "Model_B8": [

        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT_change_4",
        "TOTMOT_change_8"

    ],



    "Model_B16": [

        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT_change_4",
        "TOTMOT_change_8",
        "TOTMOT_change_16"

    ]

}


for model, features in MODEL_FEATURES.items():
    
    print("\n", model)
    
    for f in features:
        print(" -", f)

In [ ]:
# ============================================================
# STEP 8.6: CREATE MODEL DATASETS
# ============================================================


TARGET = "TOTMOT26"


MODEL_DATASETS = {}


for model_name, features in MODEL_FEATURES.items():

    MODEL_DATASETS[model_name] = {

        "X_train":
            train_fe[features].copy(),

        "y_train":
            train_fe[TARGET].copy(),


        "X_validation":
            validation_fe[features].copy(),

        "y_validation":
            validation_fe[TARGET].copy(),


        "X_test":
            test_fe[features].copy(),

        "y_test":
            test_fe[TARGET].copy()

    }


print("Models created:")

for model in MODEL_DATASETS.keys():
    print("-", model)

In [ ]:
# ============================================================
# STEP 8.7: MODEL DATASET DIMENSIONS
# ============================================================


dimension_summary = []


for model_name, data in MODEL_DATASETS.items():

    dimension_summary.append({

        "Model":
            model_name,

        "Train_features":
            data["X_train"].shape[1],

        "Train_N":
            data["X_train"].shape[0],

        "Validation_N":
            data["X_validation"].shape[0],

        "Test_N":
            data["X_test"].shape[0]

    })


dimension_summary = pd.DataFrame(
    dimension_summary
)


display(dimension_summary)

In [ ]:
# ============================================================
# STEP 8.8: MODEL INPUT MISSINGNESS
# ============================================================


for model_name, data in MODEL_DATASETS.items():

    print("\n===================")
    print(model_name)
    print("===================")

    print(
        data["X_train"]
        .isna()
        .sum()
    )

In [ ]:
# ============================================================
# STEP 8.9: SAVE FEATURE ENGINEERED DATASETS
# ============================================================


FEATURE_DIR = DATA_PROCESSED / "feature_engineered"

FEATURE_DIR.mkdir(
    exist_ok=True
)


train_fe.to_csv(
    FEATURE_DIR / "train_feature_engineered.csv",
    index=False
)


validation_fe.to_csv(
    FEATURE_DIR / "validation_feature_engineered.csv",
    index=False
)


test_fe.to_csv(
    FEATURE_DIR / "test_feature_engineered.csv",
    index=False
)


print("Feature engineered datasets saved.")

In [ ]:
# ============================================================
# STEP 8.10: SAVE FEATURE DEFINITION TABLE
# ============================================================


feature_definition = pd.DataFrame({

    "Feature":[
        "TOTMOT_change_4",
        "TOTMOT_change_8",
        "TOTMOT_change_16"
    ],


    "Formula":[
        "TOTMOT04 - TOTMOT01",
        "TOTMOT08 - TOTMOT01",
        "TOTMOT16 - TOTMOT01"
    ],


    "Clinical_Meaning":[
        "Early neurological improvement by Week 4",
        "Intermediate neurological improvement by Week 8",
        "Late neurological improvement by Week 16"
    ]
})


display(feature_definition)


feature_definition.to_csv(
    TABLE_DATASET /
    "dynamic_feature_definition.csv",
    index=False
)

In [ ]:
# ============================================================
# STEP 8.11: STEP 8 COMPLETION CHECK
# ============================================================


step8_checks = {

    "Dynamic feature 4 created":
        "TOTMOT_change_4" in train_fe.columns,


    "Dynamic feature 8 created":
        "TOTMOT_change_8" in train_fe.columns,


    "Dynamic feature 16 created":
        "TOTMOT_change_16" in train_fe.columns,


    "Four models created":
        len(MODEL_DATASETS) == 4,


    "Model A features correct":
        len(MODEL_FEATURES["Model_A"]) == 4,


    "Model B4 features correct":
        len(MODEL_FEATURES["Model_B4"]) == 5,


    "Model B8 features correct":
        len(MODEL_FEATURES["Model_B8"]) == 6,


    "Model B16 features correct":
        len(MODEL_FEATURES["Model_B16"]) == 7,


    "No target leakage":
        TARGET not in MODEL_FEATURES["Model_B16"]

}


step8_check_df = pd.DataFrame({

    "Check":
        step8_checks.keys(),

    "Passed":
        step8_checks.values()

})


display(step8_check_df)


if step8_check_df["Passed"].all():

    print("\nSTEP 8 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 8 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 9.1: IMPORT PREPROCESSING LIBRARIES
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

import numpy as np
import pandas as pd

print("Preprocessing libraries imported.")

In [ ]:
# ============================================================
# STEP 9.2: DEFINE NUMERICAL AND CATEGORICAL FEATURES
# ============================================================


NUMERICAL_FEATURES = {

    "Model_A": [
        "AGE",
        "SEXCD",
        "TOTMOT01"
    ],

    "Model_B4": [
        "AGE",
        "SEXCD",
        "TOTMOT01",
        "TOTMOT_change_4"
    ],

    "Model_B8": [
        "AGE",
        "SEXCD",
        "TOTMOT01",
        "TOTMOT_change_4",
        "TOTMOT_change_8"
    ],

    "Model_B16": [
        "AGE",
        "SEXCD",
        "TOTMOT01",
        "TOTMOT_change_4",
        "TOTMOT_change_8",
        "TOTMOT_change_16"
    ]
}


CATEGORICAL_FEATURES = {

    "Model_A": [
        "AIS.1"
    ],

    "Model_B4": [
        "AIS.1"
    ],

    "Model_B8": [
        "AIS.1"
    ],

    "Model_B16": [
        "AIS.1"
    ]
}


print("Feature types defined.")

In [ ]:
# ============================================================
# STEP 9.3: VERIFY FEATURE SEPARATION
# ============================================================


for model in MODEL_FEATURES.keys():

    print("\n====================")
    print(model)
    print("====================")

    print(
        "Numerical:",
        NUMERICAL_FEATURES[model]
    )

    print(
        "Categorical:",
        CATEGORICAL_FEATURES[model]
    )

In [ ]:
# ============================================================
# STEP 9.4: CREATE PREPROCESSOR FUNCTION
# ============================================================


def create_preprocessor(model_name):

    numerical_pipeline = Pipeline(
        steps=[

            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    categorical_pipeline = Pipeline(
        steps=[

            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),

            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )


    preprocessor = ColumnTransformer(

        transformers=[

            (
                "num",
                numerical_pipeline,
                NUMERICAL_FEATURES[model_name]
            ),


            (
                "cat",
                categorical_pipeline,
                CATEGORICAL_FEATURES[model_name]
            )
        ],

        remainder="drop"

    )


    return preprocessor


print("Preprocessor function created.")

In [ ]:
# ============================================================
# STEP 9.5: CREATE MODEL-SPECIFIC PREPROCESSORS
# ============================================================


PREPROCESSORS = {}


for model_name in MODEL_FEATURES.keys():

    PREPROCESSORS[model_name] = (
        create_preprocessor(model_name)
    )


print("Created preprocessors:")

for key in PREPROCESSORS.keys():
    print("-", key)

In [ ]:
# ============================================================
# STEP 9.6: PREPARE MODEL INPUT DICTIONARY
# ============================================================


MODEL_INPUTS = {}


for model_name, features in MODEL_FEATURES.items():

    MODEL_INPUTS[model_name] = {


        "X_train":
            train_fe[features].copy(),


        "X_validation":
            validation_fe[features].copy(),


        "X_test":
            test_fe[features].copy(),


        "y_train":
            train_fe[TARGET].copy(),


        "y_validation":
            validation_fe[TARGET].copy(),


        "y_test":
            test_fe[TARGET].copy()
    }


print("Model inputs prepared.")

In [ ]:
# ============================================================
# STEP 9.7: FIT PREPROCESSOR ON TRAIN ONLY
# ============================================================


b16_preprocessor = PREPROCESSORS["Model_B16"]


X_train_B16_processed = (
    b16_preprocessor
    .fit_transform(
        MODEL_INPUTS["Model_B16"]["X_train"]
    )
)


X_validation_B16_processed = (
    b16_preprocessor
    .transform(
        MODEL_INPUTS["Model_B16"]["X_validation"]
    )
)


X_test_B16_processed = (
    b16_preprocessor
    .transform(
        MODEL_INPUTS["Model_B16"]["X_test"]
    )
)


print(
    "Train processed shape:",
    X_train_B16_processed.shape
)

print(
    "Validation processed shape:",
    X_validation_B16_processed.shape
)

print(
    "Test processed shape:",
    X_test_B16_processed.shape
)

In [ ]:
# ============================================================
# STEP 9.8: CHECK PROCESSED MATRIX
# ============================================================


print(
    "Contains NaN:"
)

print(
    np.isnan(
        X_train_B16_processed
    ).sum()
)


print(
    "\nContains infinity:"
)

print(
    np.isinf(
        X_train_B16_processed
    ).sum()
)

In [ ]:
# ============================================================
# STEP 9.9: EXTRACT FEATURE NAMES
# ============================================================


feature_names_B16 = (
    b16_preprocessor
    .get_feature_names_out()
)


feature_names_B16 = list(
    feature_names_B16
)


print(
    "Number of processed features:",
    len(feature_names_B16)
)


print(
    feature_names_B16
)

In [ ]:
# ============================================================
# STEP 9.10: SAVE PROCESSED MATRICES
# ============================================================


PROCESSED_DATA = {


    "Model_B16": {

        "X_train":
            X_train_B16_processed,

        "X_validation":
            X_validation_B16_processed,

        "X_test":
            X_test_B16_processed,

        "y_train":
            MODEL_INPUTS["Model_B16"]["y_train"],

        "y_validation":
            MODEL_INPUTS["Model_B16"]["y_validation"],

        "y_test":
            MODEL_INPUTS["Model_B16"]["y_test"],

        "feature_names":
            feature_names_B16
    }

}


print("Processed dataset dictionary created.")

In [ ]:
# ============================================================
# STEP 9.11: SAVE PREPROCESSING INFORMATION
# ============================================================


preprocessing_summary = pd.DataFrame({

    "Component":[
        "Numerical preprocessing",
        "Categorical preprocessing",
        "Scaling",
        "Unknown category handling"
    ],

    "Method":[
        "Median imputation",
        "Most frequent imputation + OneHotEncoder",
        "StandardScaler",
        "ignore"
    ]

})


display(preprocessing_summary)


preprocessing_summary.to_csv(
    TABLE_DATASET /
    "preprocessing_summary.csv",
    index=False
)

In [ ]:
# ============================================================
# STEP 9.12: STEP 9 COMPLETION CHECK
# ============================================================


step9_checks = {


    "Preprocessors created":
        len(PREPROCESSORS)==4,


    "Model inputs created":
        len(MODEL_INPUTS)==4,


    "No NaN after preprocessing":
        np.isnan(
            X_train_B16_processed
        ).sum()==0,


    "No infinity after preprocessing":
        np.isinf(
            X_train_B16_processed
        ).sum()==0,


    "Feature names extracted":
        len(feature_names_B16)>0

}


step9_check_df = pd.DataFrame({

    "Check":
        step9_checks.keys(),

    "Passed":
        step9_checks.values()

})


display(step9_check_df)


if step9_check_df["Passed"].all():

    print("\nSTEP 9 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 9 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 10.1: IMPORT REGRESSION METRICS
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd

print("Regression metrics imported.")

In [ ]:
# ============================================================
# STEP 10.2: CALCULATE MEAN BASELINE VALUE
# ============================================================


baseline_prediction_value = (
    train_df["TOTMOT26"]
    .mean()
)


print(
    "Training TOTMOT26 mean:"
)

print(
    baseline_prediction_value
)

In [ ]:
# ============================================================
# STEP 10.3: BASELINE PREDICTION FUNCTION
# ============================================================


def evaluate_mean_baseline(
    y_true,
    baseline_value
):

    y_pred = np.full(
        shape=len(y_true),
        fill_value=baseline_value
    )


    mae = mean_absolute_error(
        y_true,
        y_pred
    )


    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


    r2 = r2_score(
        y_true,
        y_pred
    )


    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }



print("Baseline evaluation function created.")

In [ ]:
# ============================================================
# STEP 10.4: BASELINE PERFORMANCE
# ============================================================


baseline_results = []


for split_name, y_data in [
    
    (
        "Validation",
        validation_df["TOTMOT26"]
    ),
    
    (
        "Internal Test",
        test_df["TOTMOT26"]
    )
]:

    result = evaluate_mean_baseline(
        y_data,
        baseline_prediction_value
    )


    result["Split"] = split_name
    result["Model"] = "Mean Baseline"


    baseline_results.append(
        result
    )



baseline_results_df = pd.DataFrame(
    baseline_results
)


baseline_results_df = baseline_results_df[
    [
        "Split",
        "Model",
        "MAE",
        "RMSE",
        "R2"
    ]
]


display(
    baseline_results_df
)

In [ ]:
# ============================================================
# STEP 10.5: LANDMARK BASELINE TABLE
# ============================================================


landmark_baseline_results = []


for landmark in [
    "Model_A",
    "Model_B4",
    "Model_B8",
    "Model_B16"
]:

    result = evaluate_mean_baseline(
        test_df["TOTMOT26"],
        baseline_prediction_value
    )


    landmark_baseline_results.append({

        "Landmark":
            landmark,

        "Model":
            "Mean Baseline",

        "MAE":
            result["MAE"],

        "RMSE":
            result["RMSE"],

        "R2":
            result["R2"]
    })


landmark_baseline_df = pd.DataFrame(
    landmark_baseline_results
)


display(
    landmark_baseline_df
)

In [ ]:
# ============================================================
# STEP 10.6: BASELINE CONTEXT
# ============================================================


baseline_context = pd.DataFrame({

    "Metric":[
        "Training TOTMOT26 mean",
        "Training TOTMOT26 SD",
        "Validation TOTMOT26 mean",
        "Test TOTMOT26 mean"
    ],


    "Value":[

        train_df["TOTMOT26"].mean(),

        train_df["TOTMOT26"].std(),

        validation_df["TOTMOT26"].mean(),

        test_df["TOTMOT26"].mean()

    ]

})


display(
    baseline_context
)

In [ ]:
# ============================================================
# STEP 10.7: SAVE BASELINE RESULTS
# ============================================================


landmark_baseline_df.to_excel(

    TABLE_DATASET /
    "Table_Baseline_Mean_Model.xlsx",

    index=False

)


baseline_results_df.to_excel(

    TABLE_DATASET /
    "Table_Baseline_Split_Performance.xlsx",

    index=False

)


print(
    "Baseline tables saved successfully."
)

In [ ]:
# ============================================================
# STEP 10.8: STEP 10 COMPLETION CHECK
# ============================================================


step10_checks = {


    "Baseline value calculated":
        baseline_prediction_value is not None,


    "Validation evaluated":
        len(baseline_results_df) >= 1,


    "Test evaluated":
        len(baseline_results_df) >= 2,


    "All metrics present":
        all(
            col in landmark_baseline_df.columns
            for col in [
                "MAE",
                "RMSE",
                "R2"
            ]
        ),


    "Baseline table saved":
        (
            TABLE_DATASET /
            "Table_Baseline_Mean_Model.xlsx"
        ).exists()

}



step10_check_df = pd.DataFrame({

    "Check":
        step10_checks.keys(),

    "Passed":
        step10_checks.values()

})


display(step10_check_df)


if step10_check_df["Passed"].all():

    print("\nSTEP 10 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 10 contains unresolved issues.")

Part 2 — Benchmarking, Model Optimization, Nested Cross-Validation and Statistical Evaluation (Steps 11–20)
Purpose
Compare candidate regression algorithms, evaluate the incremental value of longitudinal neurological information, optimize the primary model, and estimate performance using clearly distinguished validation procedures.
Workflow
- Step 11 — Benchmark models: define leakage-safe landmark pipelines, train benchmark algorithms, evaluate their performance, and save benchmark results. The patient-overlap audit checks current partition separation but cannot establish historical test-set independence.
- Step 12 — Landmark ablation and matched comparisons: compare landmark feature sets, evaluate changes in performance, and use matched-patient and paired-bootstrap analyses where implemented.
- Step 13 — Hyperparameter optimization: tune the Week-16 LightGBM model using the documented Optuna procedure and save the study results and optimization history.
- Step 14 — Final optimized model: train the selected LightGBM configuration, generate validation and internal-test predictions, calculate performance metrics, and export model results.
- Step 15 — Nested and repeated cross-validation: run patient-disjoint nested cross-validation, summarize outer-fold metrics, save out-of-fold predictions, audit split integrity and feature timing, and assess missingness and Week-16 sensitivity.
- Step 16 — Bootstrap uncertainty: use bootstrap resampling to quantify uncertainty in the specified test-performance estimates.
- Step 17 — Statistical model comparisons: compare prediction errors using the implemented paired statistical tests and bootstrap error differences.
- Step 18 — Model explainability: generate SHAP summaries, grouped importance, dependence plots, and individual-patient explanations; save the corresponding outputs.
- Step 19 — Subgroup robustness: assess performance across the specified AIS, age, sex, and baseline motor-severity subgroups, subject to sample-size limitations.
- Step 20 — Calibration: assess calibration slope/intercept, calibration plots, and prediction-quartile performance, and export the results.
Key reporting requirements
Report the evaluation partition, number of patients, model configuration, metric definitions, and uncertainty procedure for every result. Keep validation results, outer-fold means and standard deviations, pooled out-of-fold metrics, and internal-test results separate. Calibration analyses do not by themselves establish formal conformal coverage or clinical utility.

In [ ]:
# ============================================================
# STEP 11.1: IMPORT MACHINE LEARNING LIBRARIES
# ============================================================


from sklearn.linear_model import ElasticNet

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from lightgbm import LGBMRegressor

from sklearn.neural_network import MLPRegressor


from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


import numpy as np
import pandas as pd


print("ML libraries imported successfully.")

In [ ]:
# ============================================================
# STEP 11.2: MODEL EVALUATION FUNCTION
# ============================================================


def evaluate_regression(
    y_true,
    y_pred
):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )


    return {

        "MAE": mae,

        "RMSE": rmse,

        "R2": r2
    }


print("Evaluation function created.")

In [ ]:
# ============================================================
# STEP 11.3: DEFINE BENCHMARK MODELS
# ============================================================


ML_MODELS = {


    "Elastic Net":

    ElasticNet(

        alpha=0.01,

        l1_ratio=0.5,

        random_state=SEED,

        max_iter=5000
    ),



    "Random Forest":

    RandomForestRegressor(

        n_estimators=500,

        random_state=SEED,

        n_jobs=-1
    ),



    "XGBoost":

    XGBRegressor(

        n_estimators=500,

        learning_rate=0.03,

        max_depth=4,

        subsample=0.8,

        colsample_bytree=0.8,

        random_state=SEED,

        objective="reg:squarederror",

        n_jobs=-1
    ),



    "LightGBM":

    LGBMRegressor(

        n_estimators=500,

        learning_rate=0.03,

        max_depth=4,

        random_state=SEED,

        verbosity=-1
    ),



    "MLP":

    MLPRegressor(

        hidden_layer_sizes=(64,32),

        activation="relu",

        solver="adam",

        learning_rate_init=0.001,

        max_iter=1000,

        random_state=SEED

    )

}



print("Models defined:")

for model in ML_MODELS.keys():

    print("-", model)

In [ ]:

# ============================================================
# STEP 11.4: PREPARE LEAKAGE-SAFE LANDMARK PIPELINES
# ============================================================

import numpy as np
from sklearn.base import clone
from sklearn.pipeline import Pipeline

print("=" * 70)
print("STEP 11.4: LANDMARK PREPROCESSING")
print("=" * 70)

PROCESSED_MODELS = {}
LANDMARK_PIPELINES = {}

for landmark in MODEL_FEATURES.keys():

    data = MODEL_INPUTS[landmark]

    # Fit preprocessing on training data only.
    # These matrices are retained for existing downstream code.
    preprocessor = clone(create_preprocessor(landmark))

    X_train = preprocessor.fit_transform(data["X_train"])
    X_validation = preprocessor.transform(data["X_validation"])
    X_test = preprocessor.transform(data["X_test"])

    PROCESSED_MODELS[landmark] = {
        "X_train": X_train,
        "X_validation": X_validation,
        "X_test": X_test,
        "y_train": data["y_train"],
        "y_validation": data["y_validation"],
        "y_test": data["y_test"],
        "feature_names": list(
            preprocessor.get_feature_names_out()
        )
    }

    # Unfitted preprocessing template for CV pipelines
    LANDMARK_PIPELINES[landmark] = (
        create_preprocessor(landmark)
    )

    print(
        f"{landmark}: "
        f"Train={X_train.shape}, "
        f"Validation={X_validation.shape}, "
        f"Test={X_test.shape}"
    )

print("\nSTEP 11.4 COMPLETED")


In [ ]:

# ============================================================
# STEP 11.5: TRAIN BENCHMARK MODELS
# Validation-Based Model Selection
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline

print("=" * 70)
print("STEP 11.5: BENCHMARK MODEL EVALUATION")
print("=" * 70)

benchmark_results = []
trained_models = {}

for landmark, data in MODEL_INPUTS.items():

    print("\nLandmark:", landmark)

    trained_models[landmark] = {}

    for model_name, model in ML_MODELS.items():

        print("Training:", model_name)

        estimator = clone(model)

        # Reduce LightGBM console output
        if model_name == "LightGBM":
            estimator.set_params(
                verbosity=-1,
                n_jobs=1
            )

        pipeline = Pipeline([
            (
                "preprocessor",
                clone(LANDMARK_PIPELINES[landmark])
            ),
            ("model", estimator)
        ])

        pipeline.fit(
            data["X_train"],
            np.asarray(data["y_train"]).ravel()
        )

        prediction = pipeline.predict(
            data["X_validation"]
        )

        metrics = evaluate_regression(
            data["y_validation"],
            prediction
        )

        benchmark_results.append({
            "Landmark": landmark,
            "Model": model_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"]
        })

        trained_models[landmark][model_name] = {
            "model": pipeline,
            "prediction": prediction,
            "evaluation_split": "Validation"
        }

benchmark_results_df = pd.DataFrame(
    benchmark_results
)

print("\nBenchmark evaluation completed.")

display(
    benchmark_results_df.round(4)
)

print("\nSTEP 11.5 COMPLETED")


In [ ]:

# ============================================================
# STEP 11.5.1 — TRAIN / VALIDATION / TEST PATIENT-OVERLAP AUDIT
# Placement: Immediately after the three data partitions are created
# Purpose:
#   Verify patient-level separation between partitions.
# Output:
#   JSON audit file.
# Limitation:
#   Cannot establish historical test-set independence.
# ============================================================

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

splits = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df
}

# 1. Validate the patient identifier column.
for name, df in splits.items():

    if "PTID" not in df.columns:
        raise KeyError(f"{name} is missing PTID.")

    if df["PTID"].isna().any():
        raise ValueError(f"{name} has missing patient IDs.")

    if not df["PTID"].is_unique:
        raise ValueError(
            f"{name} contains duplicate PTIDs. "
            "Check whether rows represent visits rather than unique patients."
        )

# 2. Normalize IDs and calculate pairwise overlap.
ids = {
    name: set(df["PTID"].astype(str).str.strip())
    for name, df in splits.items()
}

overlap = {
    "train_validation": len(ids["train"] & ids["validation"]),
    "train_test": len(ids["train"] & ids["test"]),
    "validation_test": len(ids["validation"] & ids["test"])
}

partition_integrity_passed = all(
    count == 0 for count in overlap.values()
)

# 3. Create transparent audit metadata.
audit = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "n_train": len(train_df),
    "n_validation": len(validation_df),
    "n_test": len(test_df),
    "unique_patients": {
        name: len(patient_ids)
        for name, patient_ids in ids.items()
    },
    "overlap_counts": overlap,
    "partition_integrity_passed": partition_integrity_passed,
    "historical_test_set_independence": "UNKNOWN",
    "external_validation_status": (
        "NOT_ESTABLISHED_BY_THIS_AUDIT"
    ),
    "interpretation": (
        "This audit checks current patient partition separation. "
        "It cannot establish whether the test cohort was inspected "
        "during historical model development or selection."
    )
}

print("STEP 11.5.1 — SPLIT INTEGRITY AUDIT")
print(json.dumps(audit, indent=2))

if not partition_integrity_passed:
    raise RuntimeError(
        "Patient overlap detected. Stop downstream evaluation "
        "until the partitioning issue is resolved."
    )

output_dir = Path("final_manuscript_results/audit")
output_dir.mkdir(parents=True, exist_ok=True)

audit_path = output_dir / "split_integrity_audit.json"
audit_path.write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8"
)

print(f"Split audit saved to: {audit_path}")
print("Historical test-set independence remains unverified.")


In [ ]:
# ============================================================
# STEP 11.6: BENCHMARK PERFORMANCE TABLE
# ============================================================


benchmark_results_df = pd.DataFrame(
    benchmark_results
)


benchmark_results_df = benchmark_results_df.sort_values(
    [
        "Landmark",
        "R2"
    ],
    ascending=[
        True,
        False
    ]
)


display(
    benchmark_results_df
)

In [ ]:
# ============================================================
# STEP 11.7: BEST MODEL SELECTION
# ============================================================


best_models_df = (
    benchmark_results_df
    .sort_values(
        "R2",
        ascending=False
    )
    .groupby(
        "Landmark"
    )
    .first()
    .reset_index()
)


display(best_models_df)

In [ ]:
# ============================================================
# STEP 11.8: SAVE BENCHMARK RESULTS
# ============================================================


benchmark_results_df.to_excel(

    TABLE_DATASET /
    "Table11_ML_Benchmark_Performance.xlsx",

    index=False

)


best_models_df.to_excel(

    TABLE_DATASET /
    "Table11_Best_Model_Per_Landmark.xlsx",

    index=False

)


print("Benchmark tables saved.")

In [ ]:
# ============================================================
# STEP 11.9: STEP 11 COMPLETION CHECK
# ============================================================


step11_checks = {


    "Five ML models defined":
        len(ML_MODELS)==5,


    "Four landmark datasets processed":
        len(PROCESSED_MODELS)==4,


    "Models trained":
        len(trained_models)==4,


    "Performance table created":
        len(benchmark_results_df)>0,


    "Required metrics present":

        all(
            col in benchmark_results_df.columns

            for col in [
                "MAE",
                "RMSE",
                "R2"
            ]
        )

}



step11_check_df = pd.DataFrame({

    "Check":
        step11_checks.keys(),

    "Passed":
        step11_checks.values()

})


display(step11_check_df)


if step11_check_df["Passed"].all():

    print("\nSTEP 11 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 11 contains unresolved issues.")

In [ ]:

# ============================================================
# STEP 12.1: LEAKAGE-SAFE LANDMARK ABLATION
# Same LightGBM Algorithm Across All Landmarks
# Validation-Based Comparison | No Test-Set Usage
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from lightgbm import LGBMRegressor

print("=" * 70)
print("STEP 12.1: LANDMARK ABLATION ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define fixed algorithm
# ------------------------------------------------------------

ablation_estimator = LGBMRegressor(
    random_state=42,
    verbosity=-1,
    n_jobs=1
)

# ------------------------------------------------------------
# 2. Verify inputs
# ------------------------------------------------------------

required_objects = [
    "MODEL_INPUTS",
    "LANDMARK_PIPELINES"
]

for obj in required_objects:
    if obj not in globals():
        raise RuntimeError(
            f"{obj} is missing. Run STEP 11.4 first."
        )

# ------------------------------------------------------------
# 3. Train and evaluate each landmark
# ------------------------------------------------------------

ablation_records = []
ablation_models = {}

for landmark, data in MODEL_INPUTS.items():

    print(f"\nProcessing {landmark}...")

    pipeline = Pipeline([
        (
            "preprocessor",
            clone(LANDMARK_PIPELINES[landmark])
        ),
        (
            "model",
            clone(ablation_estimator)
        )
    ])

    y_train = np.asarray(
        data["y_train"]
    ).ravel()

    y_validation = np.asarray(
        data["y_validation"]
    ).ravel()

    pipeline.fit(
        data["X_train"],
        y_train
    )

    y_pred = pipeline.predict(
        data["X_validation"]
    )

    mae = mean_absolute_error(
        y_validation,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            y_pred
        )
    )

    r2 = r2_score(
        y_validation,
        y_pred
    )

    ablation_records.append({
        "Landmark": landmark,
        "Algorithm": "LightGBM",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Train_N": len(y_train),
        "Validation_N": len(y_validation)
    })

    ablation_models[landmark] = pipeline

    print(
        f"MAE={mae:.4f} | "
        f"RMSE={rmse:.4f} | "
        f"R2={r2:.4f}"
    )

# ------------------------------------------------------------
# 4. Create ablation table
# ------------------------------------------------------------

landmark_ablation_df = pd.DataFrame(
    ablation_records
)

landmark_order = [
    "Model_A",
    "Model_B4",
    "Model_B8",
    "Model_B16"
]

landmark_ablation_df["Landmark"] = (
    pd.Categorical(
        landmark_ablation_df["Landmark"],
        categories=landmark_order,
        ordered=True
    )
)

landmark_ablation_df = (
    landmark_ablation_df
    .sort_values("Landmark")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\nLANDMARK ABLATION RESULTS")

display(
    landmark_ablation_df.round(4)
)

print("\nSTEP 12.1 COMPLETED")


In [ ]:

# ============================================================
# STEP 12.1.1 — CORRECTED MATCHED REPRESENTATION COMPARISON
# Same target, patients, preprocessing strategy, and LightGBM
# configuration across baseline, absolute-score, and change-score
# representations.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from lightgbm import LGBMRegressor

TARGET = "TOTMOT26"
BASE = ["AGE", "SEXCD", "AIS.1", "TOTMOT01"]

FEATURE_SETS = {
    "Baseline": BASE,
    "Week4_Absolute": BASE + ["TOTMOT04"],
    "Week4_Change": BASE + ["TOTMOT_change_4"],
    "Week8_Absolute": BASE + ["TOTMOT04", "TOTMOT08"],
    "Week8_Change": BASE + [
        "TOTMOT_change_4", "TOTMOT_change_8"
    ],
    "Week16_Absolute": BASE + [
        "TOTMOT04", "TOTMOT08", "TOTMOT16"
    ],
    "Week16_Change": BASE + [
        "TOTMOT_change_4",
        "TOTMOT_change_8",
        "TOTMOT_change_16"
    ]
}

# Confirm that every required column exists.
required = sorted(
    set([TARGET] + [
        c for features in FEATURE_SETS.values()
        for c in features
    ])
)

for name, df in [
    ("train_fe", train_fe),
    ("validation_fe", validation_fe)
]:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

train = train_fe.loc[train_fe[TARGET].notna()].copy()
valid = validation_fe.loc[validation_fe[TARGET].notna()].copy()

# Check patient separation and preserve row order.
if "PTID" in train.columns and "PTID" in valid.columns:
    train_ids = set(train["PTID"].astype(str).str.strip())
    valid_ids = set(valid["PTID"].astype(str).str.strip())
    assert train_ids.isdisjoint(valid_ids), "Patient overlap detected."

y_train = train[TARGET].astype(float).to_numpy()
y_valid = valid[TARGET].astype(float).to_numpy()

if not np.isfinite(y_train).all() or not np.isfinite(y_valid).all():
    raise ValueError("Target contains non-finite values.")

# Match the main analysis configuration.
# Verify SEED against the existing STEP 12.1 definition.
if "SEED" not in globals():
    raise RuntimeError("Run the setup cell defining SEED first.")

base_estimator = ML_MODELS["LightGBM"].get_params(deep=False).copy()
base_estimator.update({
    "verbosity": -1,
    "n_jobs": 1
})

results = []
representation_predictions = {}
representation_models = {}

for name, features in FEATURE_SETS.items():

    numeric = [c for c in features if c != "AIS.1"]

    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric),
        ("ais", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), ["AIS.1"])
    ])

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LGBMRegressor(**base_estimator))
    ])

    pipeline.fit(train[features], y_train)
    pred = pipeline.predict(valid[features])

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite predictions for {name}.")

    representation_models[name] = pipeline
    representation_predictions[name] = np.asarray(pred)

    results.append({
        "Representation": name,
        "Predictor_Count": len(features),
        "Training_N": len(train),
        "Validation_N": len(valid),
        "MAE": mean_absolute_error(y_valid, pred),
        "RMSE": np.sqrt(mean_squared_error(y_valid, pred)),
        "R2": r2_score(y_valid, pred)
    })

representation_results_df = pd.DataFrame(results)

print("STEP 12.1.1 — CORRECTED REPRESENTATION COMPARISON")
display(representation_results_df.round(4))

out = Path("final_manuscript_results/representation_comparison")
out.mkdir(parents=True, exist_ok=True)

representation_results_df.to_csv(
    out / "absolute_vs_change_validation_corrected.csv",
    index=False
)

prediction_table = pd.DataFrame({
    "Observed_TOTMOT26": y_valid
})

if "PTID" in valid.columns:
    prediction_table.insert(
        0, "PTID", valid["PTID"].astype(str).to_numpy()
    )

for name, pred in representation_predictions.items():
    prediction_table[f"Pred_{name}"] = pred

prediction_table.to_csv(
    out / "absolute_vs_change_predictions_corrected.csv",
    index=False
)

print("Corrected representation comparison saved.")
print("Rerun STEP 12.1.2 to recompute bootstrap intervals.")


In [ ]:

# ============================================================
# STEP 12.1.2 — CORRECTED PAIRED BOOTSTRAP COMPARISONS
# Uses predictions from corrected STEP 12.1.1
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

if "representation_predictions" not in globals():
    raise RuntimeError(
        "Run corrected STEP 12.1.1 before this cell."
    )

if "y_valid" not in globals():
    raise RuntimeError("Validation targets y_valid not found.")

predictions = representation_predictions
y = np.asarray(y_valid, dtype=float).ravel()

required_models = [
    "Baseline",
    "Week4_Absolute",
    "Week4_Change",
    "Week8_Absolute",
    "Week8_Change",
    "Week16_Absolute",
    "Week16_Change"
]

missing = [m for m in required_models if m not in predictions]
if missing:
    raise ValueError(f"Missing predictions: {missing}")

pred = {}
for name in required_models:
    pred[name] = np.asarray(predictions[name], dtype=float).ravel()
    if len(pred[name]) != len(y):
        raise ValueError(f"Patient count mismatch for {name}.")
    if not np.isfinite(pred[name]).all():
        raise ValueError(f"Non-finite predictions for {name}.")

comparisons = [
    ("Baseline", "Week4_Absolute"),
    ("Baseline", "Week4_Change"),
    ("Baseline", "Week8_Absolute"),
    ("Baseline", "Week8_Change"),
    ("Baseline", "Week16_Absolute"),
    ("Baseline", "Week16_Change"),
    ("Week4_Absolute", "Week4_Change"),
    ("Week8_Absolute", "Week8_Change"),
    ("Week16_Absolute", "Week16_Change")
]

rng = np.random.default_rng(2026)
B = 10000
n = len(y)
rows = []

for reference, comparison in comparisons:
    err_ref = np.abs(y - pred[reference])
    err_cmp = np.abs(y - pred[comparison])

    # Difference < 0 favours the comparison model.
    difference = err_cmp - err_ref
    boot_means = np.empty(B)

    for b in range(B):
        idx = rng.integers(0, n, size=n)
        boot_means[b] = difference[idx].mean()

    mae_ref = err_ref.mean()
    mae_cmp = err_cmp.mean()
    sd_diff = difference.std(ddof=1)

    rows.append({
        "Reference_Model": reference,
        "Comparison_Model": comparison,
        "N_Validation": n,
        "MAE_Reference": mae_ref,
        "MAE_Comparison": mae_cmp,
        "MAE_Difference_Comparison_Minus_Reference":
            difference.mean(),
        "Bootstrap_95CI_Lower":
            np.quantile(boot_means, 0.025),
        "Bootstrap_95CI_Upper":
            np.quantile(boot_means, 0.975),
        "MAE_Reduction_Percent":
            100 * (mae_ref - mae_cmp) / mae_ref,
        "Paired_Cohens_dz":
            (err_ref - err_cmp).mean() / sd_diff
            if sd_diff > 0 else np.nan,
        "Bootstrap_Replicates": B
    })

paired_bootstrap_df = pd.DataFrame(rows)

print("STEP 12.1.2 — CORRECTED PAIRED BOOTSTRAP RESULTS")
display(paired_bootstrap_df.round(4))

out = Path("final_manuscript_results/representation_comparison")
out.mkdir(parents=True, exist_ok=True)

paired_bootstrap_df.to_csv(
    out / "paired_bootstrap_corrected.csv",
    index=False
)

print("Saved corrected paired bootstrap results.")
print(
    "Intervals are conditional on fitted models; "
    "they do not capture model-fitting uncertainty."
)
print(
    "No multiple-comparison adjustment has been applied."
)


In [ ]:

# ============================================================
# STEP 12.2: LANDMARK PERFORMANCE IMPROVEMENT ANALYSIS
# Validation-Based | Same Algorithm: LightGBM
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 12.2: LANDMARK PERFORMANCE COMPARISON")
print("=" * 70)

df = landmark_ablation_df.copy()

# Ensure correct chronological order
landmark_order = [
    "Model_A",
    "Model_B4",
    "Model_B8",
    "Model_B16"
]

df["Landmark"] = df["Landmark"].astype(str)

df = (
    df.set_index("Landmark")
      .loc[landmark_order]
      .reset_index()
)

# Compare against baseline landmark
baseline_mae = df.loc[0, "MAE"]
baseline_rmse = df.loc[0, "RMSE"]
baseline_r2 = df.loc[0, "R2"]

df["MAE_Reduction_vs_Baseline_%"] = (
    (baseline_mae - df["MAE"])
    / baseline_mae
) * 100

df["RMSE_Reduction_vs_Baseline_%"] = (
    (baseline_rmse - df["RMSE"])
    / baseline_rmse
) * 100

df["R2_Gain_vs_Baseline"] = (
    df["R2"] - baseline_r2
)

# Improvements from preceding landmark
df["MAE_Reduction_vs_Previous_%"] = (
    (df["MAE"].shift(1) - df["MAE"])
    / df["MAE"].shift(1)
) * 100

df["RMSE_Reduction_vs_Previous_%"] = (
    (df["RMSE"].shift(1) - df["RMSE"])
    / df["RMSE"].shift(1)
) * 100

df["R2_Gain_vs_Previous"] = (
    df["R2"].diff()
)

df = df.fillna(0)

landmark_improvement_df = df.copy()

print("\nLANDMARK IMPROVEMENT TABLE")

display(
    landmark_improvement_df.round(4)
)

# Summary
best = df.iloc[-1]

print("\nFINAL LANDMARK SUMMARY")
print("-" * 50)

print("Best landmark:", best["Landmark"])

print(
    f"MAE reduction vs baseline: "
    f"{best['MAE_Reduction_vs_Baseline_%']:.2f}%"
)

print(
    f"RMSE reduction vs baseline: "
    f"{best['RMSE_Reduction_vs_Baseline_%']:.2f}%"
)

print(
    f"R2 gain vs baseline: "
    f"{best['R2_Gain_vs_Baseline']:.4f}"
)

print("\nSTEP 12.2 COMPLETED")


In [ ]:

# ============================================================
# STEP 12.3: LANDMARK ABLATION VISUALIZATION
# Validation-Based LightGBM Comparison
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 70)
print("STEP 12.3: LANDMARK ABLATION VISUALIZATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare results
# ------------------------------------------------------------

df = landmark_improvement_df.copy()

landmark_order = [
    "Model_A",
    "Model_B4",
    "Model_B8",
    "Model_B16"
]

df["Landmark"] = df["Landmark"].astype(str)

df = (
    df.set_index("Landmark")
      .loc[landmark_order]
      .reset_index()
)

labels = ["Baseline", "Week 4", "Week 8", "Week 16"]
x = np.arange(len(labels))

# ------------------------------------------------------------
# 2. Create output directory
# ------------------------------------------------------------

FIGURE_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project/04_figures/ablation"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 3. Plot MAE and RMSE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    x,
    df["MAE"],
    marker="o",
    linewidth=2,
    label="MAE"
)

ax.plot(
    x,
    df["RMSE"],
    marker="s",
    linewidth=2,
    label="RMSE"
)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Prediction Error (Motor Score Points)")
ax.set_xlabel("Prediction Landmark")
ax.set_title("Longitudinal Landmark Ablation: Prediction Error")
ax.legend()
ax.grid(alpha=0.25)

fig.tight_layout()

error_path = FIGURE_DIR / "landmark_ablation_errors.png"

fig.savefig(
    error_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 4. Plot R-squared
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    x,
    df["R2"],
    marker="o",
    linewidth=2
)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Validation R²")
ax.set_xlabel("Prediction Landmark")
ax.set_title("Longitudinal Landmark Ablation: R²")
ax.grid(alpha=0.25)

fig.tight_layout()

r2_path = FIGURE_DIR / "landmark_ablation_r2.png"

fig.savefig(
    r2_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 5. Export table
# ------------------------------------------------------------

TABLE_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project/03_tables/ablation"
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

table_path = TABLE_DIR / "landmark_ablation_improvement.csv"

df.to_csv(
    table_path,
    index=False
)

print("\nSaved files:")
print(error_path)
print(r2_path)
print(table_path)

print("\nSTEP 12.3 COMPLETED")


In [ ]:

# ============================================================
# STEP 12.4: MATCHED-PATIENT BOOTSTRAP
# Longitudinal Landmark Ablation
# Validation Only | No Model Retraining
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error

print("=" * 70)
print("STEP 12.4: PAIRED BOOTSTRAP LANDMARK ANALYSIS")
print("=" * 70)

LANDMARK_ORDER = [
    "Model_A",
    "Model_B4",
    "Model_B8",
    "Model_B16"
]

N_BOOTSTRAP = 2000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------
# 1. Check required objects
# ------------------------------------------------------------

if "MODEL_INPUTS" not in globals():
    raise RuntimeError("MODEL_INPUTS is missing.")

if "ablation_models" not in globals():
    raise RuntimeError(
        "ablation_models is missing. Run Step 12.1."
    )

if "validation_df" not in globals():
    raise RuntimeError("validation_df is missing.")

if "PTID" not in validation_df.columns:
    raise KeyError(
        "Validation dataset must contain PTID."
    )

# ------------------------------------------------------------
# 2. Verify source patient IDs
# ------------------------------------------------------------

source_ids = (
    validation_df["PTID"]
    .astype("string")
    .str.strip()
)

assert source_ids.notna().all()
assert source_ids.is_unique
assert validation_df.index.is_unique

pred_by_model = {}
y_by_model = {}

# ------------------------------------------------------------
# 3. Collect validation predictions
# ------------------------------------------------------------

for model_name in LANDMARK_ORDER:

    if model_name not in MODEL_INPUTS:
        raise KeyError(
            f"Missing model inputs: {model_name}"
        )

    if model_name not in ablation_models:
        raise KeyError(
            f"Missing fitted ablation model: {model_name}"
        )

    data = MODEL_INPUTS[model_name]

    X = data["X_validation"]

    if not X.index.equals(validation_df.index):
        raise AssertionError(
            f"{model_name}: validation row index mismatch"
        )

    if not X.index.is_unique:
        raise AssertionError(
            f"{model_name}: duplicated row indices"
        )

    mapped_ids = source_ids.reindex(X.index)

    if mapped_ids.isna().any():
        raise AssertionError(
            f"{model_name}: missing mapped PTIDs"
        )

    y = np.asarray(
        data["y_validation"],
        dtype=float
    ).ravel()

    source_y = pd.to_numeric(
        validation_df.loc[X.index, "TOTMOT26"],
        errors="coerce"
    ).to_numpy(dtype=float)

    if not np.allclose(
        y,
        source_y,
        equal_nan=True,
        rtol=0,
        atol=1e-8
    ):
        raise AssertionError(
            f"{model_name}: target/source mismatch"
        )

    predictions = np.asarray(
        ablation_models[model_name].predict(X),
        dtype=float
    ).ravel()

    if len(predictions) != len(y):
        raise AssertionError(
            f"{model_name}: prediction length mismatch"
        )

    if not np.isfinite(predictions).all():
        raise AssertionError(
            f"{model_name}: invalid predictions"
        )

    if not np.isfinite(y).all():
        raise AssertionError(
            f"{model_name}: invalid target values"
        )

    pred_by_model[model_name] = predictions
    y_by_model[model_name] = y

# ------------------------------------------------------------
# 4. Verify targets match across all landmarks
# ------------------------------------------------------------

reference_y = y_by_model["Model_A"]

for model_name in LANDMARK_ORDER:

    if not np.array_equal(
        reference_y,
        y_by_model[model_name]
    ):
        raise AssertionError(
            f"{model_name}: unmatched validation targets"
        )

print("\nPATIENT ALIGNMENT CHECKS PASSED")
print("Validation patients:", len(reference_y))
print("Landmarks:", len(LANDMARK_ORDER))

# ------------------------------------------------------------
# 5. Paired bootstrap comparisons
# ------------------------------------------------------------

comparisons = [
    ("Model_A", "Model_B4"),
    ("Model_B4", "Model_B8"),
    ("Model_B8", "Model_B16"),
    ("Model_A", "Model_B16")
]

bootstrap_rows = []

for earlier, later in comparisons:

    earlier_prediction = pred_by_model[earlier]
    later_prediction = pred_by_model[later]

    # Positive improvement means later model
    # produces a lower absolute prediction error.

    per_patient_improvement = (
        np.abs(reference_y - earlier_prediction)
        -
        np.abs(reference_y - later_prediction)
    )

    sample_indices = rng.integers(
        low=0,
        high=len(per_patient_improvement),
        size=(
            N_BOOTSTRAP,
            len(per_patient_improvement)
        )
    )

    bootstrap_improvements = (
        per_patient_improvement[sample_indices]
        .mean(axis=1)
    )

    ci_lower, ci_upper = np.quantile(
        bootstrap_improvements,
        [0.025, 0.975]
    )

    mae_earlier = mean_absolute_error(
        reference_y,
        earlier_prediction
    )

    mae_later = mean_absolute_error(
        reference_y,
        later_prediction
    )

    bootstrap_rows.append({
        "Earlier_Model": earlier,
        "Later_Model": later,
        "Patients": len(reference_y),
        "MAE_Earlier": mae_earlier,
        "MAE_Later": mae_later,
        "MAE_Improvement": (
            mae_earlier - mae_later
        ),
        "CI_95_Lower": ci_lower,
        "CI_95_Upper": ci_upper,
        "CI_Excludes_Zero": bool(
            ci_lower > 0 or ci_upper < 0
        )
    })

# ------------------------------------------------------------
# 6. Create results table
# ------------------------------------------------------------

landmark_bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

print("\nPAIRED BOOTSTRAP RESULTS")

display(
    landmark_bootstrap_df.round(4)
)

# ------------------------------------------------------------
# 7. Save manuscript results
# ------------------------------------------------------------

output_path = Path(
    "/kaggle/working/SCI_Recovery_ML_Project/"
    "06_results/tables/landmark_paired_bootstrap.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

landmark_bootstrap_df.to_csv(
    output_path,
    index=False
)

print("\nResults saved:", output_path)

print("\nINTERPRETATION")
print("Positive MAE improvement favors the later landmark.")
print("Confidence intervals are paired and patient-level.")
print("Intervals are exploratory and not multiplicity-adjusted.")
print("Patient mapping assumes preserved source row indices.")

print("\nSTEP 12.4 COMPLETED")


In [ ]:

# ============================================================
# STEP 12.5: EXPORT LANDMARK ABLATION RESULTS
# Validation Performance + Improvement + Bootstrap CI
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("STEP 12.5: LANDMARK ABLATION RESULTS EXPORT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify required outputs
# ------------------------------------------------------------

required_dataframes = [
    "landmark_ablation_df",
    "landmark_improvement_df",
    "landmark_bootstrap_df"
]

for name in required_dataframes:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Run STEP 12.1-12.4 first."
        )

# ------------------------------------------------------------
# 2. Define output directory
# ------------------------------------------------------------

ABLATION_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project/"
    "03_tables/ablation"
)

ABLATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 3. Prepare tables
# ------------------------------------------------------------

performance_df = landmark_ablation_df.copy()
improvement_df = landmark_improvement_df.copy()
bootstrap_df = landmark_bootstrap_df.copy()

for df in [
    performance_df,
    improvement_df
]:
    df["Landmark"] = df["Landmark"].astype(str)

# ------------------------------------------------------------
# 4. Save CSV files
# ------------------------------------------------------------

performance_path = (
    ABLATION_DIR /
    "landmark_validation_performance.csv"
)

improvement_path = (
    ABLATION_DIR /
    "landmark_improvement_analysis.csv"
)

bootstrap_path = (
    ABLATION_DIR /
    "landmark_bootstrap_confidence_intervals.csv"
)

performance_df.to_csv(
    performance_path,
    index=False
)

improvement_df.to_csv(
    improvement_path,
    index=False
)

bootstrap_df.to_csv(
    bootstrap_path,
    index=False
)

# ------------------------------------------------------------
# 5. Save combined Excel workbook
# ------------------------------------------------------------

excel_path = (
    ABLATION_DIR /
    "landmark_ablation_complete_results.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    performance_df.to_excel(
        writer,
        sheet_name="Validation_Performance",
        index=False
    )

    improvement_df.to_excel(
        writer,
        sheet_name="Improvement_Analysis",
        index=False
    )

    bootstrap_df.to_excel(
        writer,
        sheet_name="Bootstrap_95CI",
        index=False
    )

# ------------------------------------------------------------
# 6. Validate exported files
# ------------------------------------------------------------

exported_files = [
    performance_path,
    improvement_path,
    bootstrap_path,
    excel_path
]

print("\nEXPORTED FILES")
print("-" * 60)

for file_path in exported_files:
    print(
        f"{file_path.name}: "
        f"{'OK' if file_path.exists() else 'FAILED'}"
    )

print("\nDATA SUMMARY")
print("-" * 60)

print(
    "Landmark performance rows:",
    len(performance_df)
)

print(
    "Improvement analysis rows:",
    len(improvement_df)
)

print(
    "Bootstrap comparison rows:",
    len(bootstrap_df)
)

print("\nSTEP 12.5 COMPLETED")


In [ ]:

# ============================================================
# STEP 12.6: FINAL LANDMARK ABLATION SUMMARY
# FIXED: BOOTSTRAP COLUMN COMPATIBILITY
# Validation-Based | No Model Retraining
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 12.6: FINAL LANDMARK ABLATION SUMMARY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify required previous outputs
# ------------------------------------------------------------

required_objects = [
    "landmark_ablation_df",
    "landmark_improvement_df",
    "landmark_bootstrap_df"
]

for name in required_objects:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. "
            "Run Steps 12.1-12.5 first."
        )

LANDMARK_ORDER = [
    "Model_A",
    "Model_B4",
    "Model_B8",
    "Model_B16"
]

# ------------------------------------------------------------
# 2. Prepare landmark performance results
# ------------------------------------------------------------

results = landmark_improvement_df.copy()

required_performance_cols = [
    "Landmark",
    "MAE",
    "RMSE",
    "R2",
    "Train_N",
    "Validation_N",
    "MAE_Reduction_vs_Baseline_%",
    "RMSE_Reduction_vs_Baseline_%",
    "R2_Gain_vs_Baseline"
]

missing_cols = [
    col for col in required_performance_cols
    if col not in results.columns
]

if missing_cols:
    raise KeyError(
        f"Missing performance columns: {missing_cols}"
    )

results["Landmark"] = (
    results["Landmark"].astype(str)
)

if results["Landmark"].duplicated().any():
    raise AssertionError(
        "Duplicate landmark rows detected."
    )

if set(results["Landmark"]) != set(LANDMARK_ORDER):
    raise AssertionError(
        "Expected exactly four landmark models."
    )

results = (
    results.set_index("Landmark")
    .loc[LANDMARK_ORDER]
    .reset_index()
)

numeric_cols = [
    "MAE",
    "RMSE",
    "R2",
    "Train_N",
    "Validation_N",
    "MAE_Reduction_vs_Baseline_%",
    "RMSE_Reduction_vs_Baseline_%",
    "R2_Gain_vs_Baseline"
]

for col in numeric_cols:
    results[col] = pd.to_numeric(
        results[col],
        errors="raise"
    )

if not np.isfinite(
    results[numeric_cols].to_numpy(dtype=float)
).all():
    raise AssertionError(
        "Non-finite landmark performance detected."
    )

baseline = results.iloc[0]
final_landmark = results.iloc[-1]

# ------------------------------------------------------------
# 3. Determine best landmark
# ------------------------------------------------------------

best_mae_row = results.loc[
    results["MAE"].idxmin()
]

best_rmse_row = results.loc[
    results["RMSE"].idxmin()
]

best_r2_row = results.loc[
    results["R2"].idxmax()
]

best_landmark = str(
    best_mae_row["Landmark"]
)

# Do not automatically claim Week 16 is best
# for every metric unless supported by results.

week16_best_all_metrics = (
    best_mae_row["Landmark"] == "Model_B16"
    and best_rmse_row["Landmark"] == "Model_B16"
    and best_r2_row["Landmark"] == "Model_B16"
)


# ------------------------------------------------------------
# 4. Normalize bootstrap dataframe columns
# FIXED FOR ACTUAL STEP 12.4 OUTPUT
# ------------------------------------------------------------

bootstrap = landmark_bootstrap_df.copy()

print("\nORIGINAL BOOTSTRAP COLUMNS")
print(bootstrap.columns.tolist())

# Standardize confidence interval columns.

if "CI_Lower" not in bootstrap.columns:

    if "CI_95_Lower" in bootstrap.columns:
        bootstrap["CI_Lower"] = bootstrap["CI_95_Lower"]

    else:
        raise KeyError(
            "Missing CI_Lower or CI_95_Lower."
        )

if "CI_Upper" not in bootstrap.columns:

    if "CI_95_Upper" in bootstrap.columns:
        bootstrap["CI_Upper"] = bootstrap["CI_95_Upper"]

    else:
        raise KeyError(
            "Missing CI_Upper or CI_95_Upper."
        )

# Support all three possible comparison formats:
#
# 1. Comparison
# 2. Earlier_Model + Later_Model
# 3. Earlier + Later

if "Earlier_Model" in bootstrap.columns and \
   "Later_Model" in bootstrap.columns:

    bootstrap["Comparison"] = (
        bootstrap["Earlier_Model"].astype(str)
        + " vs "
        + bootstrap["Later_Model"].astype(str)
    )

elif "Earlier" in bootstrap.columns and \
     "Later" in bootstrap.columns:

    bootstrap["Comparison"] = (
        bootstrap["Earlier"].astype(str)
        + " vs "
        + bootstrap["Later"].astype(str)
    )

elif "Comparison" not in bootstrap.columns:

    raise KeyError(
        "Missing comparison labels. Expected "
        "Earlier_Model/Later_Model, "
        "Earlier/Later, or Comparison."
    )

# Check required improvement field.

if "MAE_Improvement" not in bootstrap.columns:

    raise KeyError(
        "Missing MAE_Improvement column."
    )

print("\nNORMALIZED BOOTSTRAP COLUMNS")
print(bootstrap.columns.tolist())

print("\nBOOTSTRAP COMPARISON LABELS")
print(bootstrap["Comparison"].tolist())

# ------------------------------------------------------------
# 5. Verify bootstrap values
# ------------------------------------------------------------

bootstrap_numeric = [
    "MAE_Improvement",
    "CI_Lower",
    "CI_Upper"
]

for col in bootstrap_numeric:

    bootstrap[col] = pd.to_numeric(
        bootstrap[col],
        errors="raise"
    )

if not np.isfinite(
    bootstrap[bootstrap_numeric].to_numpy(
        dtype=float
    )
).all():

    raise AssertionError(
        "Invalid bootstrap values detected."
    )

if (
    bootstrap["CI_Lower"] >
    bootstrap["CI_Upper"]
).any():

    raise AssertionError(
        "Bootstrap CI lower bound exceeds upper bound."
    )

if len(bootstrap) != 4:
    raise AssertionError(
        "Expected four bootstrap comparisons."
    )

expected_comparisons = [
    "Model_A vs Model_B4",
    "Model_B4 vs Model_B8",
    "Model_B8 vs Model_B16",
    "Model_A vs Model_B16"
]

if set(bootstrap["Comparison"]) != set(
    expected_comparisons
):
    raise AssertionError(
        "Unexpected bootstrap comparison labels."
    )

bootstrap = (
    bootstrap.set_index("Comparison")
    .loc[expected_comparisons]
    .reset_index()
)

# ------------------------------------------------------------
# 6. Calculate exploratory CI interpretation
# ------------------------------------------------------------

bootstrap["CI_Excludes_Zero"] = (
    (bootstrap["CI_Lower"] > 0)
    |
    (bootstrap["CI_Upper"] < 0)
)

bootstrap["Direction"] = np.select(
    [
        bootstrap["CI_Lower"] > 0,
        bootstrap["CI_Upper"] < 0
    ],
    [
        "Later landmark favored",
        "Earlier landmark favored"
    ],
    default="Inconclusive"
)

# The intervals are descriptive and exploratory.
# They are not adjusted for multiple comparisons
# or landmark selection.

bootstrap["Statistical_Status"] = (
    "Exploratory; multiplicity unadjusted"
)

# ------------------------------------------------------------
# 7. Verify improvement arithmetic
# ------------------------------------------------------------

baseline_mae = float(baseline["MAE"])
baseline_rmse = float(baseline["RMSE"])
baseline_r2 = float(baseline["R2"])

week16_mae = float(final_landmark["MAE"])
week16_rmse = float(final_landmark["RMSE"])
week16_r2 = float(final_landmark["R2"])

assert baseline_mae > 0
assert baseline_rmse > 0

calculated_mae_reduction = (
    (baseline_mae - week16_mae)
    / baseline_mae
) * 100

calculated_rmse_reduction = (
    (baseline_rmse - week16_rmse)
    / baseline_rmse
) * 100

calculated_r2_gain = (
    week16_r2 - baseline_r2
)

np.testing.assert_allclose(
    calculated_mae_reduction,
    float(
        final_landmark[
            "MAE_Reduction_vs_Baseline_%"
        ]
    ),
    atol=1e-6,
    rtol=0
)

np.testing.assert_allclose(
    calculated_rmse_reduction,
    float(
        final_landmark[
            "RMSE_Reduction_vs_Baseline_%"
        ]
    ),
    atol=1e-6,
    rtol=0
)

np.testing.assert_allclose(
    calculated_r2_gain,
    float(
        final_landmark["R2_Gain_vs_Baseline"]
    ),
    atol=1e-6,
    rtol=0
)

# ------------------------------------------------------------
# 8. Create accurate research interpretation
# ------------------------------------------------------------

if week16_best_all_metrics:

    performance_statement = (
        "The nominal Week-16 landmark achieved "
        "the lowest MAE and RMSE and the highest "
        "R2 among the four evaluated landmarks."
    )

else:

    performance_statement = (
        f"The lowest MAE occurred at {best_landmark}, "
        f"the lowest RMSE at "
        f"{best_rmse_row['Landmark']}, "
        f"and the highest R2 at "
        f"{best_r2_row['Landmark']}."
    )

stepwise_mae_improvement = bool(
    np.all(
        np.diff(
            results["MAE"].to_numpy(dtype=float)
        ) < 0
    )
)

stepwise_statement = (
    "MAE decreased at every successive nominal landmark."
    if stepwise_mae_improvement
    else
    "MAE did not decrease at every successive landmark."
)

# ------------------------------------------------------------
# 9. Generate final textual summary
# ------------------------------------------------------------

summary = f"""
LANDMARK ABLATION ANALYSIS — VALIDATION RESULTS
================================================

Algorithm:
LightGBM with the same estimator configuration
across the four prediction landmarks.

Study population:
Training patients = {int(baseline['Train_N'])}
Validation patients = {int(baseline['Validation_N'])}

BASELINE MODEL (Model_A)
MAE  = {baseline_mae:.4f}
RMSE = {baseline_rmse:.4f}
R2   = {baseline_r2:.4f}

NOMINAL WEEK-16 MODEL (Model_B16)
MAE  = {week16_mae:.4f}
RMSE = {week16_rmse:.4f}
R2   = {week16_r2:.4f}

BASELINE TO WEEK-16 CHANGE
MAE reduction  = {calculated_mae_reduction:.2f}%
RMSE reduction = {calculated_rmse_reduction:.2f}%
Absolute R2 gain = {calculated_r2_gain:.4f}

BEST VALIDATION LANDMARK BY MAE
{best_landmark}

INTERPRETATION
{performance_statement}

{stepwise_statement}

The analysis suggests that information from
successive nominal neurological assessments
may improve prediction of motor recovery
at the nominal Week-26 study visit.

The comparisons are observational predictive
analyses, not evidence of causality.

BOOTSTRAP UNCERTAINTY
Four matched-patient MAE comparisons were
evaluated using paired bootstrap resampling.

Positive MAE improvement favors the later
assessment landmark.

The confidence intervals are exploratory,
not adjusted for multiple comparisons,
and do not account for the full uncertainty
arising from model or landmark selection.

LIMITATIONS
1. All results are based on internal validation.
2. Patient alignment must be preserved across
   prediction landmarks.
3. Bootstrap intervals are exploratory.
4. There is no external validation in this analysis.
5. Week-16 prediction must not be described
   as baseline-only prediction.
6. The visit labels describe nominal study
   assessments, not verified individual dates.
"""

# ------------------------------------------------------------
# 10. Display final tables
# ------------------------------------------------------------

print("\nLANDMARK PERFORMANCE")
display(
    results[
        [
            "Landmark",
            "MAE",
            "RMSE",
            "R2",
            "Train_N",
            "Validation_N"
        ]
    ].round(4)
)

print("\nBOOTSTRAP COMPARISON SUMMARY")

display(
    bootstrap[
        [
            "Comparison",
            "MAE_Improvement",
            "CI_Lower",
            "CI_Upper",
            "CI_Excludes_Zero",
            "Direction"
        ]
    ].round(4)
)

print("\nFINAL RESEARCH SUMMARY")
print(summary)

# ------------------------------------------------------------
# 11. Export research-ready results
# ------------------------------------------------------------

SUMMARY_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project/"
    "03_tables/ablation"
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

summary_path = (
    SUMMARY_DIR /
    "landmark_ablation_final_summary.txt"
)

bootstrap_path = (
    SUMMARY_DIR /
    "landmark_bootstrap_interpretation.csv"
)

results_path = (
    SUMMARY_DIR /
    "landmark_ablation_final_performance.csv"
)

summary_path.write_text(
    summary,
    encoding="utf-8"
)

bootstrap.to_csv(
    bootstrap_path,
    index=False
)

results.to_csv(
    results_path,
    index=False
)

# ------------------------------------------------------------
# 12. Completion verification
# ------------------------------------------------------------

assert summary_path.is_file()
assert bootstrap_path.is_file()
assert results_path.is_file()

print("\nVERIFICATION SUMMARY")
print("Bootstrap schema compatibility: PASSED")
print("Bootstrap CI bounds: PASSED")
print("Landmark ordering: PASSED")
print("Improvement arithmetic: PASSED")

print("\nSAVED FILES")
print(summary_path)
print(bootstrap_path)
print(results_path)

print("\nModel retrained: No")
print("Original predictions modified: No")

print("\nSTEP 12.6 COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# STEP 12.7: STEP 12 COMPLETION CHECK
# Corrected for Updated Landmark Ablation Workflow
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("STEP 12.7: LANDMARK ABLATION COMPLETION CHECK")
print("=" * 70)

expected_landmarks = [
    "Model_A",
    "Model_B4",
    "Model_B8",
    "Model_B16"
]

step12_checks = {}

# ------------------------------------------------------------
# 1. Verify results dataframe
# ------------------------------------------------------------

results_exist = (
    "landmark_ablation_df" in globals()
    and isinstance(landmark_ablation_df, pd.DataFrame)
)

step12_checks["Ablation results available"] = results_exist

if results_exist:

    df = landmark_ablation_df.copy()

    required_columns = [
        "Landmark",
        "Algorithm",
        "MAE",
        "RMSE",
        "R2",
        "Train_N",
        "Validation_N"
    ]

    step12_checks["Required columns available"] = (
        all(col in df.columns for col in required_columns)
    )

    if "Landmark" in df.columns:

        actual_landmarks = df["Landmark"].astype(str).tolist()

        step12_checks["Four landmark models evaluated"] = (
            len(df) == 4
            and len(set(actual_landmarks)) == 4
        )

        for landmark in expected_landmarks:
            step12_checks[f"{landmark} included"] = (
                landmark in actual_landmarks
            )

    else:
        step12_checks["Four landmark models evaluated"] = False

        for landmark in expected_landmarks:
            step12_checks[f"{landmark} included"] = False

    if all(col in df.columns for col in ["MAE", "RMSE", "R2"]):

        metrics = df[["MAE", "RMSE", "R2"]].apply(
            pd.to_numeric,
            errors="coerce"
        )

        step12_checks["Metrics finite"] = (
            np.isfinite(metrics.to_numpy()).all()
        )

    else:
        step12_checks["Metrics finite"] = False

    if "Algorithm" in df.columns:

        step12_checks["Same LightGBM algorithm"] = (
            df["Algorithm"].eq("LightGBM").all()
        )

    else:
        step12_checks["Same LightGBM algorithm"] = False

else:

    step12_checks["Required columns available"] = False
    step12_checks["Four landmark models evaluated"] = False
    step12_checks["Metrics finite"] = False
    step12_checks["Same LightGBM algorithm"] = False

    for landmark in expected_landmarks:
        step12_checks[f"{landmark} included"] = False

# ------------------------------------------------------------
# 2. Verify improvement and bootstrap outputs
# ------------------------------------------------------------

step12_checks["Improvement analysis available"] = (
    "landmark_improvement_df" in globals()
    and isinstance(landmark_improvement_df, pd.DataFrame)
    and len(landmark_improvement_df) == 4
)

step12_checks["Bootstrap analysis available"] = (
    "landmark_bootstrap_df" in globals()
    and isinstance(landmark_bootstrap_df, pd.DataFrame)
    and len(landmark_bootstrap_df) == 4
)

# ------------------------------------------------------------
# 3. Verify exported files
# ------------------------------------------------------------

project_dir = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

ablation_dir = project_dir / "03_tables" / "ablation"
figure_dir = project_dir / "04_figures" / "ablation"

expected_files = {
    "Performance CSV saved":
        ablation_dir / "landmark_validation_performance.csv",

    "Improvement CSV saved":
        ablation_dir / "landmark_improvement_analysis.csv",

    "Bootstrap CSV saved":
        ablation_dir / "landmark_bootstrap_confidence_intervals.csv",

    "Excel workbook saved":
        ablation_dir / "landmark_ablation_complete_results.xlsx",

    "Final summary saved":
        ablation_dir / "landmark_ablation_final_summary.txt",

    "Error figure saved":
        figure_dir / "landmark_ablation_errors.png",

    "R2 figure saved":
        figure_dir / "landmark_ablation_r2.png"
}

for check_name, file_path in expected_files.items():
    step12_checks[check_name] = file_path.is_file()

# ------------------------------------------------------------
# 4. Display completion checklist
# ------------------------------------------------------------

step12_check_df = pd.DataFrame({
    "Check": list(step12_checks.keys()),
    "Passed": list(step12_checks.values())
})

display(step12_check_df)

passed = int(step12_check_df["Passed"].sum())
total = len(step12_check_df)

print(f"\nChecks passed: {passed}/{total}")

if step12_check_df["Passed"].all():

    print("\nSTEP 12 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 12 contains unresolved issues.")

    print("\nFailed checks:")

    display(
        step12_check_df.loc[
            ~step12_check_df["Passed"]
        ]
    )

print(
    "\nNOTE: Patient-ID alignment, temporal leakage, "
    "and independent validation require separate checks."
)


In [ ]:
# ============================================================
# STEP 13.1: IMPORT OPTUNA + LIGHTGBM
# ============================================================

import optuna
import lightgbm as lgb

from sklearn.model_selection import KFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd


print("Optuna and LightGBM imported.")

In [ ]:
# ============================================================
# STEP 13.2: PREPARE B16 DATA FOR OPTIMIZATION
# ============================================================


X_train_B16_opt = PROCESSED_MODELS["Model_B16"]["X_train"]

y_train_B16_opt = PROCESSED_MODELS["Model_B16"]["y_train"]


X_validation_B16_opt = PROCESSED_MODELS["Model_B16"]["X_validation"]

y_validation_B16_opt = PROCESSED_MODELS["Model_B16"]["y_validation"]


X_test_B16_opt = PROCESSED_MODELS["Model_B16"]["X_test"]

y_test_B16_opt = PROCESSED_MODELS["Model_B16"]["y_test"]



print("Training shape:")
print(X_train_B16_opt.shape)

print("Validation shape:")
print(X_validation_B16_opt.shape)

print("Test shape:")
print(X_test_B16_opt.shape)

In [ ]:

# ============================================================
# STEP 13.3: LEAKAGE-SAFE OPTUNA HYPERPARAMETER TUNING
# LightGBM | Model_B16 | Training Data Only
# ============================================================

import numpy as np
import pandas as pd
import optuna

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from lightgbm import LGBMRegressor

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

print("=" * 70)
print("STEP 13.3: LEAKAGE-SAFE OPTUNA TUNING")
print("=" * 70)

# ------------------------------------------------------------
# 1. Retrieve raw training data
# ------------------------------------------------------------

LANDMARK = "Model_B16"

X_train_raw = MODEL_INPUTS[LANDMARK]["X_train"]
y_train_raw = np.asarray(
    MODEL_INPUTS[LANDMARK]["y_train"]
).ravel()

print("Training samples:", len(y_train_raw))

# ------------------------------------------------------------
# 2. Define internal CV
# ------------------------------------------------------------

inner_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# IMPORTANT:
# If multiple observations belong to the same patient,
# replace KFold with group-aware CV.

# ------------------------------------------------------------
# 3. Define Optuna objective
# ------------------------------------------------------------

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 600
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 3, 10
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves", 7, 63
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 5, 40
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.7, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-5, 10.0, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-5, 10.0, log=True
        ),
        "random_state": 42,
        "verbosity": -1,
        "n_jobs": 1
    }

    model_pipeline = Pipeline([
        (
            "preprocessor",
            clone(LANDMARK_PIPELINES[LANDMARK])
        ),
        (
            "model",
            LGBMRegressor(**params)
        )
    ])

    scores = cross_val_score(
        model_pipeline,
        X_train_raw,
        y_train_raw,
        cv=inner_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=1,
        error_score="raise"
    )

    return -float(np.mean(scores))

# ------------------------------------------------------------
# 4. Run optimization
# ------------------------------------------------------------

study_B16 = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=42
    )
)

study_B16.optimize(
    objective,
    n_trials=30,
    show_progress_bar=False
)

# ------------------------------------------------------------
# 5. Save best parameters
# ------------------------------------------------------------

best_params_B16 = study_B16.best_params.copy()

best_params_B16.update({
    "random_state": 42,
    "verbosity": -1,
    "n_jobs": 1
})

print("\nBEST CV MAE:")
print(round(study_B16.best_value, 4))

print("\nBEST HYPERPARAMETERS:")
for key, value in best_params_B16.items():
    print(f"{key}: {value}")

# ------------------------------------------------------------
# 6. Fit optimized pipeline on training data
# ------------------------------------------------------------

optimized_pipeline_B16 = Pipeline([
    (
        "preprocessor",
        clone(LANDMARK_PIPELINES[LANDMARK])
    ),
    (
        "model",
        LGBMRegressor(**best_params_B16)
    )
])

optimized_pipeline_B16.fit(
    X_train_raw,
    y_train_raw
)

print("\nSTEP 13.3 COMPLETED")


In [ ]:

# ============================================================
# STEP 13.4: FINALIZE AND EXPORT OPTUNA STUDY
# Reuse completed leakage-safe optimization
# ============================================================

import optuna
import pandas as pd
import json
from pathlib import Path

print("=" * 70)
print("STEP 13.4: OPTUNA STUDY FINALIZATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Reuse completed study
# ------------------------------------------------------------

study = study_B16

assert study.direction == optuna.study.StudyDirection.MINIMIZE, (
    "Study direction must be MINIMIZE for MAE."
)

completed_trials = [
    t for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
]

print("\nStudy name:", study.study_name)
print("Completed trials:", len(completed_trials))

# ------------------------------------------------------------
# 2. Retrieve optimized results
# ------------------------------------------------------------

best_trial = study.best_trial

best_cv_mae = float(study.best_value)

best_params = study.best_params.copy()

print("\nBEST OPTUNA RESULTS")
print("-" * 60)

print(f"Best trial number: {best_trial.number}")
print(f"Best 5-fold CV MAE: {best_cv_mae:.4f}")

print("\nOptimized parameters:")

for parameter, value in best_params.items():
    print(f"{parameter}: {value}")

# ------------------------------------------------------------
# 3. Create trials dataframe
# ------------------------------------------------------------

optuna_trials_df = study.trials_dataframe()

print("\nOPTUNA TRIAL SUMMARY")

display(
    optuna_trials_df[
        ["number", "value", "state"]
    ].sort_values("value").head(10)
)

# ------------------------------------------------------------
# 4. Export optimization results
# ------------------------------------------------------------

OPTUNA_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project/"
    "03_tables/optimization"
)

OPTUNA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

trials_path = (
    OPTUNA_DIR / "optuna_B16_trials.csv"
)

params_path = (
    OPTUNA_DIR / "optuna_B16_best_params.json"
)

summary_path = (
    OPTUNA_DIR / "optuna_B16_summary.csv"
)

optuna_trials_df.to_csv(
    trials_path,
    index=False
)

with open(params_path, "w") as f:
    json.dump(
        best_params,
        f,
        indent=4
    )

optuna_summary_df = pd.DataFrame([{
    "Landmark": "Model_B16",
    "Algorithm": "LightGBM",
    "Optimization": "Optuna TPE",
    "Objective": "Minimize MAE",
    "CV_Folds": 5,
    "Completed_Trials": len(completed_trials),
    "Best_Trial": best_trial.number,
    "Best_CV_MAE": best_cv_mae,
    "Preprocessing_Inside_CV": True
}])

optuna_summary_df.to_csv(
    summary_path,
    index=False
)

# ------------------------------------------------------------
# 5. Preserve downstream compatibility
# ------------------------------------------------------------

optuna_best_params = best_params.copy()

optimized_lgbm_pipeline = optimized_pipeline_B16

print("\nEXPORTED FILES")
print("-" * 60)

for path in [
    trials_path,
    params_path,
    summary_path
]:
    print(
        f"{path.name}: "
        f"{'OK' if path.exists() else 'FAILED'}"
    )

print("\nSTEP 13.4 COMPLETED")


In [ ]:

# ============================================================
# STEP 13.5: BEST OPTUNA RESULTS
# Correct Objective Reporting | Model_B16
# ============================================================

import pandas as pd
import optuna

print("=" * 70)
print("STEP 13.5: BEST OPTUNA RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify optimization direction
# ------------------------------------------------------------

assert study.direction == optuna.study.StudyDirection.MINIMIZE, (
    "Expected MAE minimization study."
)

# ------------------------------------------------------------
# 2. Retrieve best trial
# ------------------------------------------------------------

best_trial = study.best_trial

best_cv_mae = float(study.best_value)

best_parameters = study.best_params.copy()

# ------------------------------------------------------------
# 3. Display optimization results
# ------------------------------------------------------------

print("\nOPTIMIZATION SUMMARY")
print("-" * 60)

print("Landmark: Model_B16")
print("Algorithm: LightGBM")
print("Optimization method: Optuna TPE")
print("Objective: Minimize 5-fold CV MAE")

print(f"\nBest Trial: {best_trial.number}")
print(f"Best CV MAE: {best_cv_mae:.4f}")

print("\nBEST HYPERPARAMETERS")
print("-" * 60)

for key, value in best_parameters.items():
    print(f"{key}: {value}")

# ------------------------------------------------------------
# 4. Create formatted parameter table
# ------------------------------------------------------------

best_optuna_results_df = pd.DataFrame({
    "Parameter": list(best_parameters.keys()),
    "Optimized_Value": list(best_parameters.values())
})

print("\nOPTIMIZED PARAMETER TABLE")

display(best_optuna_results_df)

# ------------------------------------------------------------
# 5. Preserve downstream variables
# ------------------------------------------------------------

best_optuna_trial = best_trial.number
best_optuna_cv_mae = best_cv_mae
best_optuna_params = best_parameters.copy()

print("\nMODEL STATUS")
print("-" * 60)

print("Hyperparameter tuning: Completed")
print("Optimized pipeline: Available")
print("Held-out test evaluation: Not performed in this step")
print("Nested cross-validation: Not performed in this step")

print("\nSTEP 13.5 COMPLETED")


In [ ]:

# ============================================================
# STEP 13.6: SAVE OPTUNA RESULTS
# Model_B16 | LightGBM | Leakage-Safe CV Optimization
# ============================================================

import pandas as pd
import json
from pathlib import Path

print("=" * 70)
print("STEP 13.6: SAVE OPTUNA RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare output directory
# ------------------------------------------------------------

if "TABLE_DATASET" in globals():
    SAVE_DIR = Path(TABLE_DATASET)
else:
    SAVE_DIR = Path(
        "/kaggle/working/SCI_Recovery_ML_Project/"
        "03_tables/optimization"
    )

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Export complete Optuna trial history
# ------------------------------------------------------------

optuna_results = study.trials_dataframe()

trials_file = (
    SAVE_DIR / "Table13_Optuna_Trials.xlsx"
)

optuna_results.to_excel(
    trials_file,
    index=False
)

# ------------------------------------------------------------
# 3. Export best hyperparameters
# ------------------------------------------------------------

best_parameters = pd.DataFrame({
    "Parameter": list(study.best_params.keys()),
    "Value": list(study.best_params.values())
})

parameters_file = (
    SAVE_DIR /
    "Table13_Best_LightGBM_Parameters.xlsx"
)

best_parameters.to_excel(
    parameters_file,
    index=False
)

# ------------------------------------------------------------
# 4. Export optimization summary
# ------------------------------------------------------------

completed_trials = sum(
    trial.state.name == "COMPLETE"
    for trial in study.trials
)

optuna_final_summary = pd.DataFrame([{
    "Landmark": "Model_B16",
    "Algorithm": "LightGBM",
    "Optimization_Method": "Optuna TPE",
    "Objective": "Minimize MAE",
    "CV_Folds": 5,
    "Completed_Trials": completed_trials,
    "Best_Trial": study.best_trial.number,
    "Best_CV_MAE": study.best_value,
    "Training_Samples": len(
        MODEL_INPUTS["Model_B16"]["y_train"]
    ),
    "Preprocessing_Inside_CV": True
}])

summary_file = (
    SAVE_DIR /
    "Table13_Optuna_Optimization_Summary.xlsx"
)

optuna_final_summary.to_excel(
    summary_file,
    index=False
)

# ------------------------------------------------------------
# 5. Preserve important downstream variables
# ------------------------------------------------------------

best_optuna_params = study.best_params.copy()

best_optuna_cv_mae = float(study.best_value)

# ------------------------------------------------------------
# 6. Verify exported files
# ------------------------------------------------------------

print("\nEXPORTED FILES")
print("-" * 60)

for file_path in [
    trials_file,
    parameters_file,
    summary_file
]:
    status = (
        "OK" if file_path.exists() else "FAILED"
    )

    print(f"{file_path.name}: {status}")

print("\nOPTIMIZATION SUMMARY")
print("-" * 60)

print("Best trial:", study.best_trial.number)
print("Completed trials:", completed_trials)
print(f"Best CV MAE: {study.best_value:.4f}")

print("\nSTEP 13.6 COMPLETED")


In [ ]:

# ============================================================
# STEP 13.7: OPTUNA OPTIMIZATION HISTORY PLOT
# LightGBM | Model_B16 | 5-Fold CV MAE
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from pathlib import Path

print("=" * 70)
print("STEP 13.7: OPTUNA OPTIMIZATION HISTORY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare output directory
# ------------------------------------------------------------

if "FIG_TRAINING" in globals():
    FIG_DIR = Path(FIG_TRAINING)
else:
    FIG_DIR = Path(
        "/kaggle/working/SCI_Recovery_ML_Project/"
        "04_figures/training"
    )

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Retrieve completed trials
# ------------------------------------------------------------

trials_df = study.trials_dataframe()

completed_df = (
    trials_df[
        trials_df["state"] == "COMPLETE"
    ][["number", "value"]]
    .dropna()
    .sort_values("number")
    .copy()
)

if completed_df.empty:
    raise ValueError(
        "No completed Optuna trials found."
    )

# Running best for a minimization objective
completed_df["Running_Best_MAE"] = (
    completed_df["value"].cummin()
)

best_trial_number = study.best_trial.number
best_cv_mae = float(study.best_value)

# ------------------------------------------------------------
# 3. Create optimization history figure
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    completed_df["number"],
    completed_df["value"],
    marker="o",
    markersize=4,
    linewidth=1.2,
    alpha=0.7,
    label="Trial CV MAE"
)

ax.plot(
    completed_df["number"],
    completed_df["Running_Best_MAE"],
    linewidth=2.5,
    label="Best MAE So Far"
)

ax.scatter(
    best_trial_number,
    best_cv_mae,
    s=110,
    marker="*",
    color="red",
    zorder=5,
    label=f"Best Trial {best_trial_number}"
)

ax.annotate(
    f"Best MAE = {best_cv_mae:.4f}",
    xy=(best_trial_number, best_cv_mae),
    xytext=(0, -28),
    textcoords="offset points",
    ha="center",
    fontsize=9
)

ax.set_xlabel(
    "Optuna Trial Number",
    fontsize=11
)

ax.set_ylabel(
    "5-Fold Cross-Validation MAE",
    fontsize=11
)

ax.set_title(
    "LightGBM Hyperparameter Optimization History\n"
    "Week-16 Landmark Model",
    fontsize=12
)

ax.grid(
    True,
    linestyle="--",
    alpha=0.35
)

ax.legend(
    loc="best",
    fontsize=9
)

plt.tight_layout()

# ------------------------------------------------------------
# 4. Save publication-ready figures
# ------------------------------------------------------------

pdf_path = (
    FIG_DIR / "optuna_lightgbm_history.pdf"
)

png_path = (
    FIG_DIR / "optuna_lightgbm_history.png"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. Save plotting data
# ------------------------------------------------------------

history_csv = (
    FIG_DIR / "optuna_lightgbm_history_data.csv"
)

completed_df.to_csv(
    history_csv,
    index=False
)

# ------------------------------------------------------------
# 6. Print summary
# ------------------------------------------------------------

print("\nOPTIMIZATION HISTORY SUMMARY")
print("-" * 60)

print("Completed trials:", len(completed_df))
print("Best trial:", best_trial_number)
print(f"Best CV MAE: {best_cv_mae:.4f}")

print("\nSAVED FILES")
print("-" * 60)

for file_path in [
    pdf_path,
    png_path,
    history_csv
]:
    print(
        f"{file_path.name}: "
        f"{'OK' if file_path.exists() else 'FAILED'}"
    )

print("\nSTEP 13.7 COMPLETED")


In [ ]:

# ============================================================
# STEP 13.8: STEP 13 COMPLETION AND INTEGRITY CHECK
# LightGBM | Model_B16 | Optuna
# ============================================================

import numpy as np
import pandas as pd
import optuna

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted

print("=" * 70)
print("STEP 13.8: OPTUNA COMPLETION AND INTEGRITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define expected output paths
# ------------------------------------------------------------

if "TABLE_DATASET" in globals():
    TABLE_DIR = Path(TABLE_DATASET)
else:
    TABLE_DIR = Path(
        "/kaggle/working/SCI_Recovery_ML_Project/"
        "03_tables/optimization"
    )

if "FIG_TRAINING" in globals():
    FIG_DIR = Path(FIG_TRAINING)
else:
    FIG_DIR = Path(
        "/kaggle/working/SCI_Recovery_ML_Project/"
        "04_figures/training"
    )

# ------------------------------------------------------------
# 2. Count completed trials
# ------------------------------------------------------------

completed_trials = [
    trial for trial in study.trials
    if trial.state == optuna.trial.TrialState.COMPLETE
]

# ------------------------------------------------------------
# 3. Check fitted optimized pipeline
# ------------------------------------------------------------

pipeline_fitted = False

if isinstance(
    optimized_pipeline_B16,
    Pipeline
):
    try:
        check_is_fitted(
            optimized_pipeline_B16
        )
        pipeline_fitted = True
    except Exception:
        pipeline_fitted = False

# ------------------------------------------------------------
# 4. Build completion checks
# ------------------------------------------------------------

step13_checks = {

    "Optuna study created":
        study is not None,

    "MAE minimization objective":
        study.direction ==
        optuna.study.StudyDirection.MINIMIZE,

    "30 trials completed":
        len(completed_trials) == 30,

    "Best parameters available":
        len(study.best_params) > 0,

    "Best CV MAE is finite":
        np.isfinite(study.best_value),

    "Best trial completed":
        study.best_trial.state ==
        optuna.trial.TrialState.COMPLETE,

    "Optimized pipeline fitted":
        pipeline_fitted,

    "Trial history Excel saved":
        (
            TABLE_DIR /
            "Table13_Optuna_Trials.xlsx"
        ).exists(),

    "Best parameters Excel saved":
        (
            TABLE_DIR /
            "Table13_Best_LightGBM_Parameters.xlsx"
        ).exists(),

    "Optimization summary saved":
        (
            TABLE_DIR /
            "Table13_Optuna_Optimization_Summary.xlsx"
        ).exists(),

    "Optimization PDF saved":
        (
            FIG_DIR /
            "optuna_lightgbm_history.pdf"
        ).exists(),

    "Optimization PNG saved":
        (
            FIG_DIR /
            "optuna_lightgbm_history.png"
        ).exists()
}

# ------------------------------------------------------------
# 5. Create check dataframe
# ------------------------------------------------------------

step13_check_df = pd.DataFrame({
    "Check": list(step13_checks.keys()),
    "Passed": list(step13_checks.values())
})

display(step13_check_df)

# ------------------------------------------------------------
# 6. Final summary
# ------------------------------------------------------------

passed_count = int(
    step13_check_df["Passed"].sum()
)

total_checks = len(step13_check_df)

print("\nCOMPLETION SUMMARY")
print("-" * 60)

print(f"Checks passed: {passed_count}/{total_checks}")
print(f"Best trial: {study.best_trial.number}")
print(f"Best CV MAE: {study.best_value:.4f}")

if step13_check_df["Passed"].all():
    print("\nSTEP 13 COMPLETED SUCCESSFULLY.")
else:
    print("\nSTEP 13 CONTAINS UNRESOLVED ISSUES.")

    failed_checks = step13_check_df.loc[
        ~step13_check_df["Passed"],
        "Check"
    ]

    print("\nFailed checks:")

    for check in failed_checks:
        print("-", check)

# ------------------------------------------------------------
# 7. Research validation reminder
# ------------------------------------------------------------

print("\nMETHODOLOGICAL STATUS")
print("-" * 60)

print("Optuna optimization: Completed")
print("CV preprocessing: Inside pipeline")
print("Best CV MAE: Optimization-selected estimate")
print("Nested CV: Pending independent verification")
print("Held-out test evaluation: Not performed here")
print("Patient-group leakage audit: Still required")

print("\nSTEP 13.8 CHECK FINISHED")


In [ ]:
# ============================================================
# STEP 14.1: IMPORT LIGHTGBM
# ============================================================

import lightgbm as lgb

import numpy as np
import pandas as pd


print("LightGBM imported.")

In [ ]:

# ============================================================
# STEP 14.2: DEFINE OPTIMIZED LIGHTGBM PARAMETERS
# ============================================================

from lightgbm import LGBMRegressor
import optuna

print("=" * 70)
print("STEP 14.2: OPTIMIZED LIGHTGBM PARAMETERS")
print("=" * 70)

assert study.direction == optuna.study.StudyDirection.MINIMIZE

# Preserve every Optuna-selected hyperparameter
optimized_params = study.best_params.copy()

# Add fixed model configuration
optimized_params.update({
    "objective": "regression",
    "random_state": 42,
    "verbosity": -1,
    "n_jobs": 1
})

# Ensure selected estimator count is preserved
assert optimized_params["n_estimators"] == study.best_params["n_estimators"]

print("\nOPTIMIZED PARAMETERS")
print("-" * 60)

for key, value in optimized_params.items():
    print(f"{key}: {value}")

print("\nBest Optuna trial:", study.best_trial.number)
print(f"Best CV MAE: {study.best_value:.4f}")

print("\nSTEP 14.2 COMPLETED")


In [ ]:

# ============================================================
# STEP 14.3: TRAIN FINAL OPTIMIZED LIGHTGBM
# Model_B16 | Raw Data | Leakage-Safe Pipeline
# ============================================================

import numpy as np
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted
from lightgbm import LGBMRegressor

print("=" * 70)
print("STEP 14.3: TRAIN FINAL OPTIMIZED LIGHTGBM")
print("=" * 70)

# ------------------------------------------------------------
# 1. Select landmark and raw training data
# ------------------------------------------------------------

LANDMARK = "Model_B16"

X_train_B16_raw = MODEL_INPUTS[LANDMARK]["X_train"]

y_train_B16_raw = np.asarray(
    MODEL_INPUTS[LANDMARK]["y_train"]
).ravel()

assert len(X_train_B16_raw) == len(y_train_B16_raw), (
    "Training features and targets have different lengths."
)

assert np.isfinite(y_train_B16_raw).all(), (
    "Training targets contain non-finite values."
)

# ------------------------------------------------------------
# 2. Build optimized preprocessing + model pipeline
# ------------------------------------------------------------

final_lgbm_model = Pipeline([
    (
        "preprocessor",
        clone(LANDMARK_PIPELINES[LANDMARK])
    ),
    (
        "model",
        LGBMRegressor(**optimized_params)
    )
])

# ------------------------------------------------------------
# 3. Fit on training data only
# ------------------------------------------------------------

final_lgbm_model.fit(
    X_train_B16_raw,
    y_train_B16_raw
)

# ------------------------------------------------------------
# 4. Verify fitted components
# ------------------------------------------------------------

check_is_fitted(final_lgbm_model)

check_is_fitted(
    final_lgbm_model.named_steps["model"]
)

assert (
    final_lgbm_model.named_steps["model"]
    .get_params()["n_estimators"]
    == study.best_params["n_estimators"]
), "Optimized n_estimators was not preserved."

# ------------------------------------------------------------
# 5. Preserve compatible model references
# ------------------------------------------------------------

final_lgbm_pipeline = final_lgbm_model

optimized_lgbm_pipeline = final_lgbm_model

# ------------------------------------------------------------
# 6. Training summary
# ------------------------------------------------------------

print("\nFINAL MODEL SUMMARY")
print("-" * 60)

print("Landmark:", LANDMARK)
print("Algorithm: LightGBM")
print("Model type: sklearn Pipeline")
print("Training samples:", len(y_train_B16_raw))
print("Input features:", X_train_B16_raw.shape[1])

print(
    "Preprocessing:",
    type(
        final_lgbm_model.named_steps["preprocessor"]
    ).__name__
)

print(
    "Estimators:",
    final_lgbm_model.named_steps["model"]
    .get_params()["n_estimators"]
)

print(
    "Best Optuna CV MAE:",
    f"{study.best_value:.4f}"
)

print("\nTraining data: Training partition only")
print("Validation data used for fitting: No")
print("Test data used for fitting: No")

print("\nSTEP 14.3 COMPLETED")


In [ ]:

# ============================================================
# STEP 14.4: GENERATE OPTIMIZED LIGHTGBM PREDICTIONS
# Model_B16 | Raw Features | Fitted Pipeline
# ============================================================

import numpy as np
import pandas as pd
from sklearn.utils.validation import check_is_fitted

print("=" * 70)
print("STEP 14.4: GENERATE OPTIMIZED LIGHTGBM PREDICTIONS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Retrieve raw validation and test data
# ------------------------------------------------------------

LANDMARK = "Model_B16"

X_validation_B16_raw = MODEL_INPUTS[LANDMARK]["X_validation"]
X_test_B16_raw = MODEL_INPUTS[LANDMARK]["X_test"]

y_validation_B16_raw = np.asarray(
    MODEL_INPUTS[LANDMARK]["y_validation"]
).ravel()

y_test_B16_raw = np.asarray(
    MODEL_INPUTS[LANDMARK]["y_test"]
).ravel()

# ------------------------------------------------------------
# 2. Verify feature and target alignment
# ------------------------------------------------------------

assert len(X_validation_B16_raw) == len(y_validation_B16_raw), (
    "Validation feature/target length mismatch."
)

assert len(X_test_B16_raw) == len(y_test_B16_raw), (
    "Test feature/target length mismatch."
)

assert list(X_validation_B16_raw.columns) == list(
    X_train_B16_raw.columns
), "Validation feature columns differ from training."

assert list(X_test_B16_raw.columns) == list(
    X_train_B16_raw.columns
), "Test feature columns differ from training."

# ------------------------------------------------------------
# 3. Verify fitted model
# ------------------------------------------------------------

check_is_fitted(final_lgbm_model)

# ------------------------------------------------------------
# 4. Generate predictions using raw features
# ------------------------------------------------------------

lgbm_validation_prediction = np.asarray(
    final_lgbm_model.predict(X_validation_B16_raw)
).ravel()

lgbm_test_prediction = np.asarray(
    final_lgbm_model.predict(X_test_B16_raw)
).ravel()

# ------------------------------------------------------------
# 5. Validate predictions
# ------------------------------------------------------------

assert len(lgbm_validation_prediction) == len(
    y_validation_B16_raw
)

assert len(lgbm_test_prediction) == len(
    y_test_B16_raw
)

assert np.isfinite(
    lgbm_validation_prediction
).all(), "Non-finite validation predictions."

assert np.isfinite(
    lgbm_test_prediction
).all(), "Non-finite test predictions."

# ------------------------------------------------------------
# 6. Preserve downstream target variables
# ------------------------------------------------------------

y_validation_B16_opt = y_validation_B16_raw.copy()
y_test_B16_opt = y_test_B16_raw.copy()

# ------------------------------------------------------------
# 7. Display prediction summary
# ------------------------------------------------------------

print("\nPREDICTION SUMMARY")
print("-" * 60)

print("Landmark:", LANDMARK)

print(
    "Validation samples:",
    len(lgbm_validation_prediction)
)

print(
    "Test samples:",
    len(lgbm_test_prediction)
)

print("\nValidation prediction range:")
print(
    f"{lgbm_validation_prediction.min():.2f} to "
    f"{lgbm_validation_prediction.max():.2f}"
)

print("\nTest prediction range:")
print(
    f"{lgbm_test_prediction.min():.2f} to "
    f"{lgbm_test_prediction.max():.2f}"
)

print("\nPrediction validation: PASSED")
print("Preprocessing applied by fitted pipeline: YES")

print("\nSTEP 14.4 COMPLETED")


In [ ]:

# ============================================================
# STEP 14.5: FINAL MODEL EVALUATION
# Model_B16 | MAE | RMSE | R2
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 14.5: FINAL MODEL EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define reusable evaluation function
# ------------------------------------------------------------

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    if len(y_true) != len(y_pred):
        raise ValueError(
            "Target and prediction lengths do not match."
        )

    if len(y_true) < 2:
        raise ValueError(
            "At least two observations are required for R2."
        )

    if not (
        np.isfinite(y_true).all()
        and np.isfinite(y_pred).all()
    ):
        raise ValueError(
            "Targets or predictions contain non-finite values."
        )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return float(mae), float(rmse), float(r2)

# ------------------------------------------------------------
# 2. Evaluate validation predictions
# ------------------------------------------------------------

validation_mae, validation_rmse, validation_r2 = (
    calculate_metrics(
        y_validation_B16_raw,
        lgbm_validation_prediction
    )
)

# ------------------------------------------------------------
# 3. Evaluate held-out test predictions
# ------------------------------------------------------------

test_mae, test_rmse, test_r2 = (
    calculate_metrics(
        y_test_B16_raw,
        lgbm_test_prediction
    )
)

# ------------------------------------------------------------
# 4. Create evaluation table
# ------------------------------------------------------------

final_model_metrics_df = pd.DataFrame([
    {
        "Dataset": "Validation",
        "N": len(y_validation_B16_raw),
        "MAE": validation_mae,
        "RMSE": validation_rmse,
        "R2": validation_r2
    },
    {
        "Dataset": "Held-out Test",
        "N": len(y_test_B16_raw),
        "MAE": test_mae,
        "RMSE": test_rmse,
        "R2": test_r2
    }
])

print("\nFINAL OPTIMIZED LIGHTGBM PERFORMANCE")
print("-" * 60)

display(
    final_model_metrics_df.round(4)
)

print("\nSTEP 14.5 COMPLETED")


In [ ]:

# ============================================================
# STEP 14.6: VALIDATION PERFORMANCE
# Model_B16 | Optimized LightGBM
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 14.6: VALIDATION PERFORMANCE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Reuse metrics from STEP 14.5
# ------------------------------------------------------------

required_vars = [
    "validation_mae",
    "validation_rmse",
    "validation_r2",
    "y_validation_B16_raw",
    "lgbm_validation_prediction"
]

missing_vars = [
    name for name in required_vars
    if name not in globals()
]

if missing_vars:
    raise RuntimeError(
        "Run STEP 14.5 first. Missing: "
        + ", ".join(missing_vars)
    )

val_mae = float(validation_mae)
val_rmse = float(validation_rmse)
val_r2 = float(validation_r2)

# ------------------------------------------------------------
# 2. Verify validation sample alignment
# ------------------------------------------------------------

validation_n = len(y_validation_B16_raw)

assert validation_n == len(
    lgbm_validation_prediction
), "Validation sample count mismatch."

assert np.isfinite(
    [val_mae, val_rmse, val_r2]
).all(), "Validation metrics contain non-finite values."

# ------------------------------------------------------------
# 3. Create downstream-compatible results table
# ------------------------------------------------------------

validation_result = pd.DataFrame({
    "Dataset": ["Validation"],
    "Model": ["Optimized LightGBM"],
    "MAE": [val_mae],
    "RMSE": [val_rmse],
    "R2": [val_r2]
})

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

print("\nOPTIMIZED LIGHTGBM VALIDATION RESULTS")
print("-" * 60)

display(validation_result.round(4))

print("\nValidation samples:", validation_n)
print("Landmark: Model_B16")
print("Prediction target: Week-26 total motor score")

print("\nSTEP 14.6 COMPLETED")


In [ ]:

# ============================================================
# STEP 14.7: INTERNAL TEST PERFORMANCE
# Model_B16 | Optimized LightGBM
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 14.7: INTERNAL TEST PERFORMANCE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify STEP 14.5 outputs
# ------------------------------------------------------------

required_vars = [
    "test_mae",
    "test_rmse",
    "test_r2",
    "y_test_B16_raw",
    "lgbm_test_prediction"
]

missing_vars = [
    name for name in required_vars
    if name not in globals()
]

if missing_vars:
    raise RuntimeError(
        "Run STEP 14.5 first. Missing: "
        + ", ".join(missing_vars)
    )

# ------------------------------------------------------------
# 2. Reuse previously calculated test metrics
# ------------------------------------------------------------

test_mae = float(test_mae)
test_rmse = float(test_rmse)
test_r2 = float(test_r2)

# ------------------------------------------------------------
# 3. Verify prediction integrity
# ------------------------------------------------------------

test_n = len(y_test_B16_raw)

assert test_n == len(lgbm_test_prediction), (
    "Test feature/target prediction count mismatch."
)

assert np.isfinite(
    [test_mae, test_rmse, test_r2]
).all(), "Test metrics contain non-finite values."

# ------------------------------------------------------------
# 4. Preserve original downstream table structure
# ------------------------------------------------------------

test_result = pd.DataFrame({
    "Dataset": ["Internal Test"],
    "Model": ["Optimized LightGBM"],
    "MAE": [test_mae],
    "RMSE": [test_rmse],
    "R2": [test_r2]
})

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\nOPTIMIZED LIGHTGBM INTERNAL TEST RESULTS")
print("-" * 60)

display(test_result.round(4))

print("\nTest samples:", test_n)
print("Landmark: Model_B16")
print("Prediction target: Week-26 total motor score")

print("\nEVALUATION STATUS")
print("-" * 60)

print("Model retraining: No")
print("Hyperparameter retuning: No")
print("Test data used for model fitting: No")
print("Evaluation type: Internal held-out test")
print("External validation: Not performed")

print("\nSTEP 14.7 COMPLETED")


In [ ]:

# ============================================================
# STEP 14.8: FINAL MODEL COMPARISON
# Model_B16 | Internal Held-Out Test Only
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 14.8: FINAL MODEL COMPARISON")
print("=" * 70)

comparison_rows = []

# ------------------------------------------------------------
# 1. Mean baseline: internal test only
# ------------------------------------------------------------

if "baseline_results_df" in globals():

    baseline_df = baseline_results_df.copy()

    if "Split" in baseline_df.columns:

        baseline_test = baseline_df.loc[
            baseline_df["Split"] == "Internal Test"
        ]

        if len(baseline_test) == 1:

            row = baseline_test.iloc[0]

            comparison_rows.append({
                "Model": "Mean Baseline",
                "MAE": float(row["MAE"]),
                "RMSE": float(row["RMSE"]),
                "R2": float(row["R2"]),
                "Evaluation_Split": "Internal Test"
            })

# ------------------------------------------------------------
# 2. Optimized LightGBM: internal test
# ------------------------------------------------------------

comparison_rows.append({
    "Model": "Optimized_LightGBM_B16",
    "MAE": float(test_mae),
    "RMSE": float(test_rmse),
    "R2": float(test_r2),
    "Evaluation_Split": "Internal Test"
})

# ------------------------------------------------------------
# 3. Optional MLP comparison
# Only include independently verified test metrics
# ------------------------------------------------------------

# Do not use ablation_results_df as an MLP result.
# Earlier benchmark results were validation-based.
# MLP can be added only after evaluating its fitted
# training-only pipeline on the identical test cohort.

# ------------------------------------------------------------
# 4. Create final comparison table
# ------------------------------------------------------------

final_comparison = pd.DataFrame(comparison_rows)

if final_comparison.empty:
    raise RuntimeError(
        "No valid internal-test results available."
    )

assert final_comparison[
    ["MAE", "RMSE", "R2"]
].apply(
    lambda col: np.isfinite(col).all()
).all(), "Non-finite comparison metrics detected."

# ------------------------------------------------------------
# 5. Sort and display
# ------------------------------------------------------------

final_comparison = (
    final_comparison
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("\nINTERNAL TEST MODEL COMPARISON")
print("-" * 60)

display(final_comparison.round(4))

print("\nCOMPARISON INTEGRITY")
print("-" * 60)

print("Evaluation split: Internal Test")
print("Validation-only metrics included: No")
print("Ablation metrics mislabeled as MLP: No")
print("External validation: Not performed")

if len(final_comparison) < 3:
    print(
        "\nNote: MLP is excluded until its "
        "internal-test metrics are verified."
    )

print("\nSTEP 14.8 COMPLETED")


In [ ]:

# ============================================================
# STEP 14.9: SAVE FINAL MODEL RESULTS
# Model_B16 | Optimized LightGBM
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 14.9: SAVE FINAL MODEL RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare output directory
# ------------------------------------------------------------

if "TABLE_DATASET" in globals():
    SAVE_DIR = Path(TABLE_DATASET)
else:
    SAVE_DIR = Path(
        "/kaggle/working/SCI_Recovery_ML_Project/"
        "03_tables"
    )

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Verify required result tables
# ------------------------------------------------------------

result_tables = {
    "Table14_Final_Model_Performance": final_comparison,
    "Table14_Validation_Performance": validation_result,
    "Table14_Internal_Test_Performance": test_result
}

required_columns = [
    "MAE",
    "RMSE",
    "R2"
]

for table_name, df in result_tables.items():

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f"{table_name} is not a DataFrame."
        )

    if df.empty:
        raise ValueError(
            f"{table_name} is empty."
        )

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{table_name} missing columns: {missing}"
        )

    metric_values = df[
        required_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(metric_values).all():
        raise ValueError(
            f"{table_name} contains non-finite metrics."
        )

# ------------------------------------------------------------
# 3. Export Excel and CSV files
# ------------------------------------------------------------

exported_files = []

for table_name, df in result_tables.items():

    excel_path = SAVE_DIR / f"{table_name}.xlsx"
    csv_path = SAVE_DIR / f"{table_name}.csv"

    df.to_excel(
        excel_path,
        index=False
    )

    df.to_csv(
        csv_path,
        index=False
    )

    exported_files.extend([
        excel_path,
        csv_path
    ])

# ------------------------------------------------------------
# 4. Verify exports
# ------------------------------------------------------------

print("\nEXPORTED FILES")
print("-" * 60)

all_saved = True

for file_path in exported_files:

    saved = (
        file_path.exists()
        and file_path.stat().st_size > 0
    )

    all_saved = all_saved and saved

    print(
        f"{file_path.name}: "
        f"{'OK' if saved else 'FAILED'}"
    )

# ------------------------------------------------------------
# 5. Display final summary
# ------------------------------------------------------------

print("\nFINAL MODEL SUMMARY")
print("-" * 60)

print("Landmark: Model_B16")
print("Algorithm: Optimized LightGBM")

print(f"Validation MAE: {val_mae:.4f}")
print(f"Validation RMSE: {val_rmse:.4f}")
print(f"Validation R2: {val_r2:.4f}")

print(f"\nInternal Test MAE: {test_mae:.4f}")
print(f"Internal Test RMSE: {test_rmse:.4f}")
print(f"Internal Test R2: {test_r2:.4f}")

print("\nExternal validation: Not performed")

if not all_saved:
    raise RuntimeError(
        "One or more result files were not saved."
    )

print("\nSTEP 14.9 COMPLETED")


In [ ]:

# ============================================================
# STEP 14.10: SAVE FINAL TEST PREDICTIONS
# Model_B16 | Week-26 Total Motor Score
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 14.10: SAVE FINAL TEST PREDICTIONS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare actual and predicted values
# ------------------------------------------------------------

y_true = np.asarray(
    y_test_B16_raw
).ravel()

y_pred = np.asarray(
    lgbm_test_prediction
).ravel()

if len(y_true) != len(y_pred):
    raise ValueError(
        "Actual and predicted sample counts differ."
    )

if not (
    np.isfinite(y_true).all()
    and np.isfinite(y_pred).all()
):
    raise ValueError(
        "Actual or predicted values contain NaN/Inf."
    )

# ------------------------------------------------------------
# 2. Build patient-level prediction table
# ------------------------------------------------------------

final_predictions_df = pd.DataFrame({
    "Actual_TOTMOT26": y_true,
    "Predicted_TOTMOT26": y_pred
})

# Signed prediction error
final_predictions_df["Error"] = (
    final_predictions_df["Predicted_TOTMOT26"]
    - final_predictions_df["Actual_TOTMOT26"]
)

# Absolute prediction error
final_predictions_df["Absolute_Error"] = (
    final_predictions_df["Error"].abs()
)

# ------------------------------------------------------------
# 3. Verify prediction table
# ------------------------------------------------------------

assert len(final_predictions_df) == len(y_true)

calculated_mae = float(
    final_predictions_df["Absolute_Error"].mean()
)

assert np.isclose(
    calculated_mae,
    test_mae,
    atol=1e-8
), "Prediction-table MAE differs from STEP 14.7."

# ------------------------------------------------------------
# 4. Prepare export directory
# ------------------------------------------------------------

if "TABLE_DATASET" in globals():
    SAVE_DIR = Path(TABLE_DATASET)
else:
    SAVE_DIR = Path(
        "/kaggle/working/SCI_Recovery_ML_Project/"
        "03_tables"
    )

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 5. Export Excel and CSV
# ------------------------------------------------------------

excel_path = (
    SAVE_DIR / "Table14_Final_Test_Predictions.xlsx"
)

csv_path = (
    SAVE_DIR / "Table14_Final_Test_Predictions.csv"
)

final_predictions_df.to_excel(
    excel_path,
    index=False
)

final_predictions_df.to_csv(
    csv_path,
    index=False
)

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("\nTEST PREDICTION SUMMARY")
print("-" * 60)

print("Landmark: Model_B16")
print("Target: TOTMOT26")
print("Test samples:", len(final_predictions_df))

print(f"MAE from prediction table: {calculated_mae:.4f}")
print(f"Expected test MAE: {test_mae:.4f}")

print("\nFIRST FIVE PREDICTIONS")
display(final_predictions_df.head())

print("\nEXPORTED FILES")
print("-" * 60)

for file_path in [excel_path, csv_path]:
    print(
        f"{file_path.name}: "
        f"{'OK' if file_path.exists() else 'FAILED'}"
    )

print("\nSTEP 14.10 COMPLETED")


In [ ]:

# ============================================================
# STEP 14.11: STEP 14 COMPLETION AND INTEGRITY CHECK
# Model_B16 | Optimized LightGBM
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import mean_absolute_error

print("=" * 70)
print("STEP 14.11: COMPLETION AND INTEGRITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify fitted final pipeline
# ------------------------------------------------------------

model_fitted = False

if isinstance(final_lgbm_model, Pipeline):
    try:
        check_is_fitted(final_lgbm_model)
        check_is_fitted(
            final_lgbm_model.named_steps["model"]
        )
        model_fitted = True
    except Exception:
        model_fitted = False

# ------------------------------------------------------------
# 2. Verify predictions and metrics
# ------------------------------------------------------------

validation_aligned = (
    len(lgbm_validation_prediction)
    == len(y_validation_B16_raw)
)

test_aligned = (
    len(lgbm_test_prediction)
    == len(y_test_B16_raw)
)

predictions_finite = (
    np.isfinite(lgbm_validation_prediction).all()
    and np.isfinite(lgbm_test_prediction).all()
)

validation_mae_verified = np.isclose(
    mean_absolute_error(
        y_validation_B16_raw,
        lgbm_validation_prediction
    ),
    val_mae,
    atol=1e-8
)

test_mae_verified = np.isclose(
    mean_absolute_error(
        y_test_B16_raw,
        lgbm_test_prediction
    ),
    test_mae,
    atol=1e-8
)

# ------------------------------------------------------------
# 3. Verify optimized parameters
# ------------------------------------------------------------

trained_params = (
    final_lgbm_model.named_steps["model"]
    .get_params()
)

parameters_preserved = all(
    trained_params.get(key) == value
    for key, value in study.best_params.items()
)

# ------------------------------------------------------------
# 4. Verify exported files
# ------------------------------------------------------------

if "TABLE_DATASET" in globals():
    SAVE_DIR = Path(TABLE_DATASET)
else:
    SAVE_DIR = Path(
        "/kaggle/working/SCI_Recovery_ML_Project/"
        "03_tables"
    )

required_files = [
    "Table14_Final_Model_Performance.xlsx",
    "Table14_Validation_Performance.xlsx",
    "Table14_Internal_Test_Performance.xlsx",
    "Table14_Final_Test_Predictions.xlsx",
    "Table14_Final_Test_Predictions.csv"
]

files_saved = all(
    (SAVE_DIR / filename).is_file()
    and (SAVE_DIR / filename).stat().st_size > 0
    for filename in required_files
)

# ------------------------------------------------------------
# 5. Verify exported prediction table
# ------------------------------------------------------------

prediction_table_valid = (
    len(final_predictions_df) == len(y_test_B16_raw)
    and np.allclose(
        final_predictions_df["Actual_TOTMOT26"].to_numpy(),
        np.asarray(y_test_B16_raw).ravel()
    )
    and np.allclose(
        final_predictions_df["Predicted_TOTMOT26"].to_numpy(),
        np.asarray(lgbm_test_prediction).ravel()
    )
)

# ------------------------------------------------------------
# 6. Build completion checklist
# ------------------------------------------------------------

step14_checks = {
    "Final pipeline fitted": model_fitted,
    "Optimized parameters preserved": parameters_preserved,
    "Validation prediction count correct": validation_aligned,
    "Test prediction count correct": test_aligned,
    "Predictions finite": predictions_finite,
    "Validation MAE consistent": validation_mae_verified,
    "Test MAE consistent": test_mae_verified,
    "Validation results available": (
        isinstance(validation_result, pd.DataFrame)
        and not validation_result.empty
    ),
    "Test results available": (
        isinstance(test_result, pd.DataFrame)
        and not test_result.empty
    ),
    "Final comparison available": (
        isinstance(final_comparison, pd.DataFrame)
        and not final_comparison.empty
    ),
    "Prediction table consistent": prediction_table_valid,
    "Required result files saved": files_saved
}

step14_check_df = pd.DataFrame({
    "Check": list(step14_checks.keys()),
    "Passed": list(step14_checks.values())
})

display(step14_check_df)

# ------------------------------------------------------------
# 7. Completion summary
# ------------------------------------------------------------

passed_count = int(step14_check_df["Passed"].sum())
total_checks = len(step14_check_df)

print("\nCOMPLETION SUMMARY")
print("-" * 60)

print(f"Checks passed: {passed_count}/{total_checks}")

print(f"\nValidation MAE: {val_mae:.4f}")
print(f"Validation RMSE: {val_rmse:.4f}")
print(f"Validation R2: {val_r2:.4f}")

print(f"\nInternal Test MAE: {test_mae:.4f}")
print(f"Internal Test RMSE: {test_rmse:.4f}")
print(f"Internal Test R2: {test_r2:.4f}")

if step14_check_df["Passed"].all():
    print("\nSTEP 14 COMPLETED SUCCESSFULLY.")
else:
    print("\nSTEP 14 CONTAINS UNRESOLVED ISSUES.")

    failed_checks = step14_check_df.loc[
        ~step14_check_df["Passed"],
        "Check"
    ]

    for item in failed_checks:
        print("-", item)

print("\nMETHODOLOGICAL STATUS")
print("-" * 60)

print("Optimized LightGBM pipeline: Trained")
print("Internal test evaluation: Completed")
print("External validation: Not performed")
print("Patient-level leakage audit: Pending")
print("True nested cross-validation: Pending correction")

print("\nSTEP 14.11 CHECK FINISHED")


In [ ]:
# ============================================================
# STEP 15.1: IMPORT NESTED CV LIBRARIES
# ============================================================


from sklearn.model_selection import KFold

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import lightgbm as lgb

import numpy as np
import pandas as pd


print("Nested CV libraries imported.")

In [ ]:

# ============================================================
# STEP 15.2: PATIENT-VERIFIED NESTED CV DATA
# Model_B16 | Training Partition Only
# ============================================================

import numpy as np
import pandas as pd
from sklearn.base import clone

print("=" * 70)
print("STEP 15.2: PATIENT-VERIFIED NESTED CV DATA")
print("=" * 70)

NESTED_LANDMARK = "Model_B16"

# ------------------------------------------------------------
# 1. Check required variables
# ------------------------------------------------------------

required = [
    "MODEL_INPUTS",
    "LANDMARK_PIPELINES",
    "train_df"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing earlier objects: " + str(missing)
    )

# ------------------------------------------------------------
# 2. Load raw training inputs
# ------------------------------------------------------------

data = MODEL_INPUTS[NESTED_LANDMARK]

X_nested = data["X_train"].copy()

y_nested = np.asarray(
    data["y_train"],
    dtype=float
).ravel().copy()

assert isinstance(X_nested, pd.DataFrame)
assert len(X_nested) == len(y_nested)
assert X_nested.index.is_unique
assert X_nested.columns.is_unique
assert np.isfinite(y_nested).all()

# ------------------------------------------------------------
# 3. Verify original patient identities
# ------------------------------------------------------------

assert "PTID" in train_df.columns
assert "TOTMOT26" in train_df.columns

assert train_df.index.is_unique

# Index equality must be accompanied by source
# feature and outcome checks.

assert X_nested.index.equals(
    train_df.index
), "Training row-index mismatch."

source_y = pd.to_numeric(
    train_df.loc[X_nested.index, "TOTMOT26"],
    errors="coerce"
).to_numpy(dtype=float)

np.testing.assert_allclose(
    y_nested,
    source_y,
    rtol=0,
    atol=1e-8
)

# Check unchanged predictors against source.
# Engineered changes are handled separately.

common_features = [
    col for col in X_nested.columns
    if col in train_df.columns
]

for col in common_features:

    left = X_nested[col].reset_index(drop=True)
    right = train_df.loc[
        X_nested.index, col
    ].reset_index(drop=True)

    if (
        pd.api.types.is_numeric_dtype(left)
        and pd.api.types.is_numeric_dtype(right)
    ):
        equal = np.allclose(
            pd.to_numeric(left, errors="coerce"),
            pd.to_numeric(right, errors="coerce"),
            equal_nan=True
        )
    else:
        equal = (
            left.astype("string").equals(
                right.astype("string")
            )
        )

    assert equal, (
        f"Source feature mismatch: {col}"
    )

# ------------------------------------------------------------
# 4. Restore aligned patient IDs
# ------------------------------------------------------------

nested_groups = (
    train_df.loc[X_nested.index, "PTID"]
    .astype("string")
    .str.strip()
    .to_numpy()
)

assert pd.notna(nested_groups).all()

unique_patient_count = len(
    np.unique(nested_groups)
)

duplicate_patient_rows = (
    len(nested_groups) - unique_patient_count
)

# PTID must never become a predictor.

assert "PTID" not in X_nested.columns

# ------------------------------------------------------------
# 5. Prepare fresh preprocessing template
# ------------------------------------------------------------

nested_preprocessor = clone(
    LANDMARK_PIPELINES[NESTED_LANDMARK]
)

nested_feature_names = (
    X_nested.columns.tolist()
)

nested_sample_count = len(X_nested)

# ------------------------------------------------------------
# 6. Final checks
# ------------------------------------------------------------

assert len(X_nested) == len(nested_groups)
assert len(X_nested) == len(y_nested)

assert unique_patient_count >= 5

print("\nNESTED CV DATA SUMMARY")
print("Landmark:", NESTED_LANDMARK)
print("Training rows:", len(X_nested))
print("Unique patients:", unique_patient_count)
print("Duplicate patient rows:", duplicate_patient_rows)
print("Features:", len(nested_feature_names))
print("PTID included as predictor: No")
print("Target alignment: PASSED")
print("Source feature checks: PASSED")
print("Training partition only: Yes")
print("Test data used: No")

print("\nFEATURE LIST")

for name in nested_feature_names:
    print("-", name)

print("\nPATIENT GROUPING")
print("nested_groups created: Yes")
print("Grouping source: train_df['PTID']")
print("CV preprocessing template: Unfitted clone")
print("Engineered-feature lineage: Separate audit required")

print("\nSTEP 15.2 COMPLETED")


In [ ]:

# ============================================================
# STEP 15.3: DEFINE NESTED CV LIGHTGBM MODEL FACTORY
# Independent Hyperparameters | Unfitted Pipeline
# ============================================================

import optuna
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.pipeline import Pipeline

print("=" * 70)
print("STEP 15.3: NESTED CV LIGHTGBM MODEL FACTORY")
print("=" * 70)

NESTED_SEED = 42

# ------------------------------------------------------------
# 1. Define independent Optuna search space
# ------------------------------------------------------------

def suggest_nested_lgbm_params(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 600
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 3, 10
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves", 7, 63
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 5, 40
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.7, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-5, 10.0, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-5, 10.0, log=True
        )
    }

    return params

# ------------------------------------------------------------
# 2. Define model factory
# ------------------------------------------------------------

def create_lgbm_model(params=None):

    if params is None:
        params = {
            "n_estimators": 300,
            "max_depth": 6,
            "num_leaves": 31,
            "learning_rate": 0.05,
            "min_child_samples": 20,
            "subsample": 1.0,
            "colsample_bytree": 1.0,
            "reg_alpha": 0.0,
            "reg_lambda": 0.0
        }

    model_params = params.copy()

    model_params.update({
        "objective": "regression",
        "random_state": NESTED_SEED,
        "verbosity": -1,
        "n_jobs": 1
    })

    pipeline = Pipeline([
        (
            "preprocessor",
            clone(nested_preprocessor)
        ),
        (
            "model",
            LGBMRegressor(**model_params)
        )
    ])

    return pipeline

# ------------------------------------------------------------
# 3. Verify factory output
# ------------------------------------------------------------

test_pipeline = create_lgbm_model()

assert isinstance(test_pipeline, Pipeline)

assert "preprocessor" in test_pipeline.named_steps

assert "model" in test_pipeline.named_steps

assert not hasattr(
    test_pipeline.named_steps["model"],
    "booster_"
), "Model factory unexpectedly returned a fitted model."

print("\nMODEL FACTORY SUMMARY")
print("-" * 60)

print("Algorithm: LightGBM")
print("Pipeline: Preprocessing + Regression")
print("Search method: Optuna TPE")
print("Optimization objective: Minimize MAE")
print("Hyperparameter selection: Inner CV only")
print("Outer validation used for tuning: No")
print("Global Optuna best parameters reused: No")
print("Model factory returns: Unfitted pipeline")

print("\nSTEP 15.3 COMPLETED")


In [ ]:

# ============================================================
# STEP 15.4 — PATIENT-DISJOINT NESTED CROSS-VALIDATION
#               FOR ALL LANDMARK MODELS
# Placement: After STEP 15.3 (Nested-CV Model Factory)
# Requires:
#   MODEL_INPUTS, LANDMARK_PIPELINES, train_df,
#   suggest_nested_lgbm_params
# Output:
#   Fold-level metrics, summary metrics, and OOF predictions.
# ============================================================

import numpy as np
import pandas as pd
import optuna
from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor

LANDMARKS = ["Model_A", "Model_B4", "Model_B8", "Model_B16"]
OUTER_FOLDS = 5
INNER_FOLDS = 5
N_TRIALS = 20
RANDOM_SEED = 42

required_objects = [
    "MODEL_INPUTS",
    "LANDMARK_PIPELINES",
    "train_df",
    "suggest_nested_lgbm_params"
]

for name in required_objects:
    if name not in globals():
        raise RuntimeError(
            f"Required object '{name}' is missing. "
            "Run the preceding setup cells first."
        )

if "PTID" not in train_df.columns:
    raise KeyError("train_df must contain PTID.")

if train_df["PTID"].isna().any():
    raise ValueError("Missing PTID values in training data.")

fold_results = []
oof_results = []

for landmark in LANDMARKS:

    print(f"\nRunning nested CV for {landmark}")

    X = MODEL_INPUTS[landmark]["X_train"].copy()
    y = np.asarray(
        MODEL_INPUTS[landmark]["y_train"],
        dtype=float
    ).ravel()

    if len(X) != len(y) or X.index.has_duplicates:
        raise ValueError(f"{landmark}: invalid feature/target alignment.")

    # This assertion is intentional: do not silently reorder patients.
    if not X.index.equals(train_df.index):
        raise ValueError(
            f"{landmark}: X_train index does not match train_df index. "
            "Align the feature rows and patient IDs before running CV."
        )

    groups = (
        train_df.loc[X.index, "PTID"]
        .astype(str)
        .str.strip()
        .to_numpy()
    )

    if pd.isna(groups).any():
        raise ValueError(f"{landmark}: missing patient-group labels.")

    if len(np.unique(groups)) < max(OUTER_FOLDS, INNER_FOLDS):
        raise ValueError(f"{landmark}: insufficient unique patient groups.")

    outer_cv = GroupKFold(n_splits=OUTER_FOLDS)
    oof_pred = np.full(len(y), np.nan)
    oof_count = np.zeros(len(y), dtype=int)

    for outer_fold, (train_idx, valid_idx) in enumerate(
        outer_cv.split(X, y, groups),
        start=1
    ):

        X_outer_train = X.iloc[train_idx]
        X_outer_valid = X.iloc[valid_idx]
        y_outer_train = y[train_idx]
        y_outer_valid = y[valid_idx]
        groups_outer_train = groups[train_idx]
        groups_outer_valid = groups[valid_idx]

        if not set(groups_outer_train).isdisjoint(
            set(groups_outer_valid)
        ):
            raise RuntimeError(
                f"{landmark}, outer fold {outer_fold}: patient overlap."
            )

        inner_cv = GroupKFold(n_splits=INNER_FOLDS)
        inner_splits = list(
            inner_cv.split(
                X_outer_train,
                y_outer_train,
                groups_outer_train
            )
        )

        def objective(trial):
            params = suggest_nested_lgbm_params(trial).copy()

            params.update({
                "objective": "regression",
                "random_state": RANDOM_SEED + outer_fold,
                "verbosity": -1,
                "n_jobs": 1
            })

            inner_scores = []

            for inner_train_idx, inner_valid_idx in inner_splits:

                X_inner_train = X_outer_train.iloc[inner_train_idx]
                X_inner_valid = X_outer_train.iloc[inner_valid_idx]
                y_inner_train = y_outer_train[inner_train_idx]
                y_inner_valid = y_outer_train[inner_valid_idx]

                groups_inner_train = groups_outer_train[inner_train_idx]
                groups_inner_valid = groups_outer_train[inner_valid_idx]

                if not set(groups_inner_train).isdisjoint(
                    set(groups_inner_valid)
                ):
                    raise RuntimeError(
                        "Patient overlap detected in inner CV."
                    )

                inner_model = Pipeline([
                    (
                        "preprocessor",
                        clone(LANDMARK_PIPELINES[landmark])
                    ),
                    ("regressor", LGBMRegressor(**params))
                ])

                inner_model.fit(X_inner_train, y_inner_train)
                inner_pred = inner_model.predict(X_inner_valid)

                inner_scores.append(
                    mean_absolute_error(y_inner_valid, inner_pred)
                )

            return float(np.mean(inner_scores))

        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(
                seed=RANDOM_SEED + outer_fold
            )
        )

        study.optimize(
            objective,
            n_trials=N_TRIALS,
            show_progress_bar=False
        )

        best_params = study.best_params.copy()
        best_params.update({
            "objective": "regression",
            "random_state": RANDOM_SEED + outer_fold,
            "verbosity": -1,
            "n_jobs": 1
        })

        outer_model = Pipeline([
            (
                "preprocessor",
                clone(LANDMARK_PIPELINES[landmark])
            ),
            ("regressor", LGBMRegressor(**best_params))
        ])

        outer_model.fit(X_outer_train, y_outer_train)
        outer_pred = outer_model.predict(X_outer_valid)

        oof_pred[valid_idx] = outer_pred
        oof_count[valid_idx] += 1

        fold_results.append({
            "Landmark": landmark,
            "Outer_Fold": outer_fold,
            "N_Training_Rows": len(train_idx),
            "N_Validation_Rows": len(valid_idx),
            "MAE": mean_absolute_error(y_outer_valid, outer_pred),
            "RMSE": np.sqrt(
                mean_squared_error(y_outer_valid, outer_pred)
            ),
            "R2": r2_score(y_outer_valid, outer_pred),
            "Best_Inner_MAE": study.best_value,
            "Best_Params": str(best_params)
        })

    if not np.isfinite(oof_pred).all():
        raise RuntimeError(f"{landmark}: incomplete OOF predictions.")

    if not np.all(oof_count == 1):
        raise RuntimeError(
            f"{landmark}: each patient row must receive one OOF prediction."
        )

    for i in range(len(y)):
        oof_results.append({
            "Landmark": landmark,
            "PTID": groups[i],
            "Observed_TOTMOT26": y[i],
            "OOF_Prediction": oof_pred[i]
        })

nested_cv_folds_df = pd.DataFrame(fold_results)
nested_cv_oof_df = pd.DataFrame(oof_results)

nested_cv_summary_df = (
    nested_cv_folds_df
    .groupby("Landmark", sort=False)
    .agg(
        MAE_Mean=("MAE", "mean"),
        MAE_SD=("MAE", "std"),
        RMSE_Mean=("RMSE", "mean"),
        RMSE_SD=("RMSE", "std"),
        R2_Mean=("R2", "mean"),
        R2_SD=("R2", "std")
    )
    .reset_index()
)

print("STEP 15.4 — NESTED-CV SUMMARY")
display(nested_cv_summary_df.round(4))

print("Fold-level results")
display(nested_cv_folds_df.round(4))

output_dir = Path("final_manuscript_results/nested_cv_all_landmarks")
output_dir.mkdir(parents=True, exist_ok=True)

nested_cv_summary_df.to_csv(
    output_dir / "nested_cv_summary.csv",
    index=False
)
nested_cv_folds_df.to_csv(
    output_dir / "nested_cv_fold_results.csv",
    index=False
)
nested_cv_oof_df.to_csv(
    output_dir / "nested_cv_oof_predictions.csv",
    index=False
)

print("Nested-CV results saved.")
print("These are internal training-partition estimates, not external validation.")


In [ ]:

# ============================================================
# STEP 15.5 — NESTED CV PERFORMANCE SUMMARY (FIXED)
# Model_B16 | Existing outer-fold results only
# No model retraining or retuning
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 15.5: NESTED CV PERFORMANCE SUMMARY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Locate existing fold-level results
# ------------------------------------------------------------

fold_df = None
source_used = None

candidate_names = [
    "nested_cv_results_df",
    "nested_cv_folds_df",
]

for name in candidate_names:
    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame) and not obj.empty:
        fold_df = obj.copy()
        source_used = f"In-memory variable: {name}"
        break

# If no in-memory result exists, try saved fold-level CSVs.
if fold_df is None:

    candidate_paths = [
        Path(
            "/kaggle/working/final_manuscript_results/"
            "nested_cv_all_landmarks/nested_cv_fold_results.csv"
        ),
        Path(
            "/kaggle/working/SCI_Recovery_ML_Project/"
            "06_results/tables/primary/"
            "Table15_Nested_CV_Fold_Performance.csv"
        ),
        Path(
            "/kaggle/working/SCI_Recovery_ML_Project/"
            "06_results/tables/"
            "Table15_Nested_CV_Fold_Performance.csv"
        ),
    ]

    for path in candidate_paths:
        if path.is_file():
            fold_df = pd.read_csv(path)
            source_used = str(path)
            break

if fold_df is None:
    raise FileNotFoundError(
        "No saved nested-CV fold results found. "
        "Run the original STEP 15.4/15.6 first, or inspect "
        "TABLE_DATASET for the actual fold-results CSV."
    )

print("Source:", source_used)
print("Available columns:", fold_df.columns.tolist())

# ------------------------------------------------------------
# 2. Select Model_B16 when the file contains multiple models
# ------------------------------------------------------------

if "Landmark" in fold_df.columns:

    available_landmarks = (
        fold_df["Landmark"].astype(str).unique().tolist()
    )

    if "Model_B16" not in available_landmarks:
        raise ValueError(
            "Model_B16 was not found. Available landmarks: "
            f"{available_landmarks}"
        )

    fold_df = fold_df.loc[
        fold_df["Landmark"].astype(str) == "Model_B16"
    ].copy()

# ------------------------------------------------------------
# 3. Validate the five outer-fold metric rows
# ------------------------------------------------------------

required_metrics = ["MAE", "RMSE", "R2"]

missing_columns = [
    col for col in required_metrics
    if col not in fold_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing metric columns: {missing_columns}. "
        "Check the original fold-results schema."
    )

if len(fold_df) != 5:
    raise ValueError(
        f"Expected 5 outer-fold rows for Model_B16; "
        f"found {len(fold_df)}. "
        "Do not calculate a five-fold summary from an "
        "unexpected number of rows."
    )

metric_data = fold_df[required_metrics].apply(
    pd.to_numeric, errors="coerce"
)

if not np.isfinite(metric_data.to_numpy()).all():
    raise ValueError(
        "Fold metrics contain missing or non-finite values."
    )

if (metric_data["MAE"] < 0).any() or (
    metric_data["RMSE"] < 0
).any():
    raise ValueError("MAE/RMSE cannot be negative.")

# ------------------------------------------------------------
# 4. Calculate mean and sample SD across outer folds
# ------------------------------------------------------------

nested_cv_summary = pd.DataFrame({
    "Metric": required_metrics,
    "Mean": [
        metric_data[col].mean()
        for col in required_metrics
    ],
    "SD": [
        metric_data[col].std(ddof=1)
        for col in required_metrics
    ],
})

nested_cv_summary["N_Outer_Folds"] = 5
nested_cv_summary["Validation_Type"] = (
    "Nested 5x5 CV - Training Partition"
)
nested_cv_summary["Landmark"] = "Model_B16"

print("\nNESTED CV SUMMARY: MEAN ± SD")
display(nested_cv_summary.round(4))

for _, row in nested_cv_summary.iterrows():
    print(
        f"{row['Metric']}: "
        f"{row['Mean']:.4f} ± {row['SD']:.4f}"
    )

# ------------------------------------------------------------
# 5. Save the corrected summary
# ------------------------------------------------------------

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
OUT_DIR = ROOT / "06_results" / "tables" / "primary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUT_DIR / "Table15_Nested_CV_Mean_SD.csv"
fold_path = OUT_DIR / "Table15_Nested_CV_Fold_Performance.csv"

nested_cv_summary.to_csv(summary_path, index=False)
fold_df.to_csv(fold_path, index=False)

print("\nSaved summary:", summary_path)
print("Saved fold results:", fold_path)

print("\nREPORTING NOTES")
print("Outer folds: 5")
print("Inner folds: 5 (as configured in the original nested CV)")
print("Summary SD: Sample SD (ddof=1)")
print("Held-out internal test: Not used by this summary cell")
print("External validation: Not performed")
print("Historical test-set independence: Not established")
print("\nSTEP 15.5 COMPLETED")


In [ ]:
# ============================================================
# STEP 15.6 — FIXED NESTED-CV EXPORT
# Uses STEP 15.4 results; no retraining or retuning
# ============================================================

import ast
import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
SAVE_DIR = ROOT / "06_results" / "tables" / "primary"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Use existing STEP 15.4 fold results
if "nested_cv_folds_df" not in globals():
    raise RuntimeError(
        "nested_cv_folds_df is missing. Run STEP 15.4 successfully first."
    )

all_folds = nested_cv_folds_df.copy()

required = {
    "Landmark", "Outer_Fold", "MAE", "RMSE", "R2", "Best_Params"
}
missing = required - set(all_folds.columns)

if missing:
    raise ValueError(f"STEP 15.4 results missing columns: {sorted(missing)}")

# 2. Select Model_B16 for the existing single-landmark plot
nested_cv_results_df = (
    all_folds.loc[all_folds["Landmark"].eq("Model_B16")]
    .copy()
    .rename(columns={"Outer_Fold": "Fold"})
    .sort_values("Fold")
    .reset_index(drop=True)
)

if len(nested_cv_results_df) != 5:
    raise ValueError(
        f"Expected 5 Model_B16 folds; found {len(nested_cv_results_df)}."
    )

metrics = ["MAE", "RMSE", "R2"]
if not np.isfinite(
    nested_cv_results_df[metrics].to_numpy(dtype=float)
).all():
    raise ValueError("Non-finite nested-CV metrics detected.")

# 3. Build the matching summary from the same five folds
nested_cv_summary = pd.DataFrame({
    "Metric": metrics,
    "Mean": [
        nested_cv_results_df[m].mean() for m in metrics
    ],
    "SD": [
        nested_cv_results_df[m].std(ddof=1) for m in metrics
    ],
})
nested_cv_summary["N_Outer_Folds"] = 5
nested_cv_summary["Validation_Type"] = (
    "Nested 5-fold CV, training partition"
)
nested_cv_summary["Landmark"] = "Model_B16"

# 4. Export all-landmark results and Model_B16 results
all_folds.to_csv(
    SAVE_DIR / "Table15_Nested_CV_All_Landmarks_Folds.csv",
    index=False
)
all_folds.to_excel(
    SAVE_DIR / "Table15_Nested_CV_All_Landmarks_Folds.xlsx",
    index=False
)

nested_cv_results_df.to_csv(
    SAVE_DIR / "Table15_Nested_CV_Fold_Performance.csv",
    index=False
)
nested_cv_results_df.to_excel(
    SAVE_DIR / "Table15_Nested_CV_Fold_Performance.xlsx",
    index=False
)

nested_cv_summary.to_csv(
    SAVE_DIR / "Table15_Nested_CV_Mean_SD.csv",
    index=False
)
nested_cv_summary.to_excel(
    SAVE_DIR / "Table15_Nested_CV_Mean_SD.xlsx",
    index=False
)

# 5. Export recorded best parameters without claiming study objects exist
parameter_rows = []
for _, row in nested_cv_results_df.iterrows():
    try:
        params = ast.literal_eval(row["Best_Params"])
        if not isinstance(params, dict):
            params = {"Recorded_Best_Params": str(row["Best_Params"])}
    except (ValueError, SyntaxError, TypeError):
        params = {"Recorded_Best_Params": str(row["Best_Params"])}

    parameter_rows.append({
        "Fold": int(row["Fold"]),
        **params
    })

pd.DataFrame(parameter_rows).to_csv(
    SAVE_DIR / "Table15_Nested_CV_Best_Parameters.csv",
    index=False
)

print("STEP 15.6 PASSED")
print("Model_B16 outer folds:", len(nested_cv_results_df))
print("All-landmark folds:", len(all_folds))
display(nested_cv_summary.round(4))
print("Saved to:", SAVE_DIR)

In [ ]:

# ============================================================
# STEP 15.7: PUBLICATION-QUALITY NESTED CV R2 PLOT
# Model_B16 | Modern Scientific Style | 600 DPI
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from pathlib import Path

print("=" * 70)
print("STEP 15.7: PUBLICATION-QUALITY NESTED CV R2 PLOT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Output directory
# ------------------------------------------------------------

if "FIG_EVALUATION" not in globals():
    raise NameError("FIG_EVALUATION is not defined.")

FIG_EVALUATION = Path(FIG_EVALUATION)
FIG_EVALUATION.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Prepare data
# ------------------------------------------------------------

plot_df = (
    nested_cv_results_df[["Fold", "R2"]]
    .copy()
    .sort_values("Fold")
    .reset_index(drop=True)
)

assert len(plot_df) == 5
assert plot_df["Fold"].is_unique
assert np.isfinite(plot_df["R2"]).all()

folds = plot_df["Fold"].to_numpy()
r2_values = plot_df["R2"].to_numpy(dtype=float)

mean_r2 = float(np.mean(r2_values))
sd_r2 = float(np.std(r2_values, ddof=1))

# ------------------------------------------------------------
# 3. Scientific publication style
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.linewidth": 0.9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.facecolor": "white"
})

fig, ax = plt.subplots(figsize=(8.5, 5.5))
fig.patch.set_facecolor("white")
ax.set_facecolor("#FAFCFF")

# ------------------------------------------------------------
# 4. Gradient-inspired blue palette
# ------------------------------------------------------------

bar_colors = [
    "#A7C7E7",
    "#7DAED8",
    "#568FC7",
    "#3573AE",
    "#1F548B"
]

bars = ax.bar(
    folds,
    r2_values,
    width=0.58,
    color=bar_colors,
    edgecolor="#24476A",
    linewidth=0.8,
    zorder=3
)

# ------------------------------------------------------------
# 5. Mean R2 reference line
# ------------------------------------------------------------

ax.axhline(
    mean_r2,
    color="#C44747",
    linestyle=(0, (6, 4)),
    linewidth=1.8,
    label=f"Mean R² = {mean_r2:.4f}",
    zorder=4
)

# ------------------------------------------------------------
# 6. Individual value labels
# ------------------------------------------------------------

for bar, value in zip(bars, r2_values):

    ax.annotate(
        f"{value:.4f}",
        xy=(
            bar.get_x() + bar.get_width() / 2,
            value
        ),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color="#243746"
    )

# ------------------------------------------------------------
# 7. Axes and grid formatting
# ------------------------------------------------------------

ax.set_xlabel(
    "Outer Cross-Validation Fold",
    fontweight="medium",
    labelpad=11
)

ax.set_ylabel(
    "Coefficient of Determination (R²)",
    fontweight="medium",
    labelpad=11
)

ax.set_title(
    "Nested Cross-Validation Performance\n"
    "Optimized LightGBM | Week-16 Landmark",
    fontweight="bold",
    pad=17
)

ax.set_xticks(folds)
ax.set_xticklabels([f"Fold {f}" for f in folds])

# Full R2 scale avoids visually exaggerating differences
ax.set_ylim(0, 1.08)

ax.yaxis.set_major_locator(
    MultipleLocator(0.2)
)

ax.yaxis.set_minor_locator(
    MultipleLocator(0.1)
)

ax.grid(
    axis="y",
    which="major",
    linestyle="--",
    linewidth=0.7,
    color="#C8D3DE",
    alpha=0.7,
    zorder=0
)

ax.grid(
    axis="y",
    which="minor",
    linestyle=":",
    linewidth=0.4,
    color="#DCE4EC",
    alpha=0.5,
    zorder=0
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_color("#718096")
ax.spines["bottom"].set_color("#718096")

ax.tick_params(
    axis="both",
    length=0,
    pad=7
)

ax.legend(
    loc="upper left",
    frameon=True,
    facecolor="white",
    edgecolor="#E2E8F0"
)

# ------------------------------------------------------------
# 8. Add methodological footnote
# ------------------------------------------------------------

fig.text(
    0.5,
    0.015,
    "Five outer folds | Inner Optuna tuning | "
    "Training partition only",
    ha="center",
    va="bottom",
    fontsize=9,
    color="#596579"
)

fig.tight_layout(rect=[0, 0.045, 1, 1])

# ------------------------------------------------------------
# 9. Export at 600 DPI
# ------------------------------------------------------------

pdf_path = FIG_EVALUATION / "nested_cv_R2.pdf"
png_path = FIG_EVALUATION / "nested_cv_R2.png"
tiff_path = FIG_EVALUATION / "nested_cv_R2_600dpi.tiff"

fig.savefig(
    pdf_path,
    format="pdf",
    bbox_inches="tight"
)

fig.savefig(
    png_path,
    format="png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    tiff_path,
    format="tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 10. Verify exports
# ------------------------------------------------------------

print("\nFIGURE EXPORTS")
print("-" * 60)

for path in [pdf_path, png_path, tiff_path]:

    saved = path.is_file() and path.stat().st_size > 0

    print(
        f"{path.name}: "
        f"{'OK' if saved else 'FAILED'}"
    )

    if not saved:
        raise RuntimeError(
            f"Figure export failed: {path}"
        )

print("\nPLOT SUMMARY")
print("-" * 60)

print(f"Mean R2: {mean_r2:.4f}")
print(f"SD R2: {sd_r2:.4f}")
print("Outer folds:", len(plot_df))
print("PNG/TIFF resolution: 600 DPI")
print("PDF format: Vector")

print("\nSTEP 15.7 COMPLETED")


In [ ]:
# ============================================================
# STEP 15.8 — NESTED-CV INTEGRITY AUDIT (FIXED)
# Validates actual STEP 15.4 outputs; no model retraining
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
AUDIT_DIR = ROOT / "06_results" / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

required_objects = [
    "nested_cv_folds_df",
    "nested_cv_oof_df",
    "nested_cv_results_df",
    "nested_cv_summary",
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}. Run STEP 15.4–15.6 first."
    )

folds = nested_cv_folds_df.copy()
oof = nested_cv_oof_df.copy()
b16 = nested_cv_results_df.copy()
summary = nested_cv_summary.copy()

expected_landmarks = [
    "Model_A", "Model_B4", "Model_B8", "Model_B16"
]
metrics = ["MAE", "RMSE", "R2"]

# 1. Validate all-landmark fold results
assert set(folds["Landmark"]) == set(expected_landmarks)
assert len(folds) == 20, f"Expected 20 fold rows; got {len(folds)}"
assert not folds.duplicated(["Landmark", "Outer_Fold"]).any()
assert all(folds.groupby("Landmark").size() == 5)

assert np.isfinite(
    folds[metrics].to_numpy(dtype=float)
).all()

# 2. Validate Model_B16 results against the all-landmark table
expected_b16 = (
    folds[folds["Landmark"].eq("Model_B16")]
    .sort_values("Outer_Fold")
    .reset_index(drop=True)
)

observed_b16 = (
    b16.sort_values("Fold")
    .reset_index(drop=True)
)

assert len(observed_b16) == 5
for metric in metrics:
    np.testing.assert_allclose(
        observed_b16[metric].to_numpy(dtype=float),
        expected_b16[metric].to_numpy(dtype=float),
        rtol=0,
        atol=1e-10
    )

# 3. Validate summary against the same five outer folds
assert set(summary["Metric"]) == set(metrics)
assert len(summary) == 3

for metric in metrics:
    row = summary.loc[summary["Metric"].eq(metric)]
    assert len(row) == 1

    np.testing.assert_allclose(
        float(row.iloc[0]["Mean"]),
        expected_b16[metric].mean(),
        rtol=0,
        atol=1e-8
    )
    np.testing.assert_allclose(
        float(row.iloc[0]["SD"]),
        expected_b16[metric].std(ddof=1),
        rtol=0,
        atol=1e-8
    )

# 4. Validate OOF prediction structure
required_oof = {
    "Landmark", "PTID", "Observed_TOTMOT26", "OOF_Prediction"
}
assert required_oof.issubset(oof.columns)

assert not oof.duplicated(["Landmark", "PTID"]).any()
assert set(oof["Landmark"]) == set(expected_landmarks)

assert np.isfinite(
    oof[["Observed_TOTMOT26", "OOF_Prediction"]]
    .to_numpy(dtype=float)
).all()

oof_counts = oof.groupby("Landmark").size().to_dict()
assert all(oof_counts[name] > 0 for name in expected_landmarks)

# 5. Report limitations honestly
audit = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "all_landmarks_present": True,
    "outer_fold_count_per_landmark": {
        str(k): int(v)
        for k, v in folds.groupby("Landmark").size().items()
    },
    "fold_metrics_finite": True,
    "model_b16_matches_source_folds": True,
    "summary_matches_model_b16_folds": True,
    "oof_predictions_valid": True,
    "oof_rows_per_landmark": {
        str(k): int(v) for k, v in oof_counts.items()
    },
    "optuna_study_objects_available": (
        "nested_studies" in globals()
    ),
    "historical_test_set_independence": "UNKNOWN",
    "external_validation_completed": False,
    "full_training_pipeline_reproduction_verified": False,
    "status": "NESTED_CV_ARTIFACT_CHECKS_PASSED",
    "note": (
        "This checks saved fold metrics, summaries and OOF predictions. "
        "It does not independently rerun training or verify historical "
        "test-set independence."
    )
}

audit_path = AUDIT_DIR / "STEP15_8_nested_cv_integrity_audit.json"
audit_path.write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8"
)

print("STEP 15.8 PASSED — NESTED-CV ARTIFACT CHECKS")
print(json.dumps(audit, indent=2))
print("Saved:", audit_path)

In [ ]:
# ============================================================
# STEP 15.8.1 — OOF PERFORMANCE RECONCILIATION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

required = {
    "Landmark", "PTID",
    "Observed_TOTMOT26", "OOF_Prediction"
}

missing = required - set(nested_cv_oof_df.columns)
if missing:
    raise ValueError(f"Missing OOF columns: {sorted(missing)}")

rows = []

for landmark, group in nested_cv_oof_df.groupby("Landmark"):
    y = group["Observed_TOTMOT26"].to_numpy(dtype=float)
    pred = group["OOF_Prediction"].to_numpy(dtype=float)

    if len(y) == 0 or not (
        np.isfinite(y).all() and np.isfinite(pred).all()
    ):
        raise ValueError(f"Invalid OOF data for {landmark}")

    rows.append({
        "Landmark": landmark,
        "N_OOF": len(group),
        "MAE_OOF_Pooled": mean_absolute_error(y, pred),
        "RMSE_OOF_Pooled": np.sqrt(mean_squared_error(y, pred)),
        "R2_OOF_Pooled": r2_score(y, pred)
    })

oof_performance_df = pd.DataFrame(rows)

display(oof_performance_df.round(4))

out = (
    "/kaggle/working/SCI_Recovery_ML_Project/"
    "06_results/tables/primary/STEP15_8.1_OOF_Performance.csv"
)

oof_performance_df.to_csv(out, index=False)
print("Saved:", out)
print("Note: Pooled OOF metrics are not the same as mean outer-fold metrics.")

In [ ]:
# ============================================================
# STEP 15.8.2 — SAVE CANONICAL OOF PERFORMANCE TABLE
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
OUT = ROOT / "06_results/tables/primary"
OUT.mkdir(parents=True, exist_ok=True)

required = {
    "Landmark", "N_OOF",
    "MAE_OOF_Pooled", "RMSE_OOF_Pooled", "R2_OOF_Pooled"
}

missing = required - set(oof_performance_df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

if len(oof_performance_df) != 4:
    raise ValueError("Expected exactly four landmark results.")

if oof_performance_df["Landmark"].duplicated().any():
    raise ValueError("Duplicate landmarks found.")

metrics = [
    "MAE_OOF_Pooled", "RMSE_OOF_Pooled", "R2_OOF_Pooled"
]

if not np.isfinite(
    oof_performance_df[metrics].to_numpy(dtype=float)
).all():
    raise ValueError("Non-finite OOF metrics found.")

path = OUT / "Canonical_OOF_Performance.csv"
oof_performance_df.to_csv(path, index=False)

print("STEP 15.8.2 PASSED")
display(oof_performance_df.round(4))
print("Saved:", path)
print("OOF estimates remain separate from validation and test results.")

In [ ]:

# ============================================================
# STEP 15.9: REPEATED CROSS-VALIDATION
# Model_B16 | 10 Folds x 5 Repeats
# Fixed Final LightGBM Configuration
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.base import clone
from sklearn.model_selection import (
    RepeatedKFold,
    cross_validate
)
from sklearn.pipeline import Pipeline

print("=" * 70)
print("STEP 15.9: REPEATED CROSS-VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Use raw training data only
# ------------------------------------------------------------

X_cv = X_nested.copy()
y_cv = np.asarray(y_nested).ravel().copy()

assert isinstance(X_cv, pd.DataFrame)
assert len(X_cv) == len(y_cv)
assert np.isfinite(y_cv).all()

print("\nDATA SUMMARY")
print("-" * 60)
print("Training samples:", len(X_cv))
print("Raw features:", X_cv.shape[1])

# ------------------------------------------------------------
# 2. Clone the final fitted pipeline
# ------------------------------------------------------------

cv_model = clone(final_lgbm_model)

assert isinstance(cv_model, Pipeline)

cv_model.set_params(
    model__verbosity=-1,
    model__n_jobs=1
)

# ------------------------------------------------------------
# 3. Configure repeated CV
# ------------------------------------------------------------

N_SPLITS = 10
N_REPEATS = 5

repeated_cv = RepeatedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=42
)

print("\nCV CONFIGURATION")
print("-" * 60)
print("Folds:", N_SPLITS)
print("Repeats:", N_REPEATS)
print("Total evaluations:", N_SPLITS * N_REPEATS)
print("Preprocessing: Inside each CV fold")
print("Hyperparameters: Fixed final configuration")

# ------------------------------------------------------------
# 4. Run repeated cross-validation
# ------------------------------------------------------------

print("\nRunning repeated cross-validation...")

cv_scores = cross_validate(
    estimator=cv_model,
    X=X_cv,
    y=y_cv,
    cv=repeated_cv,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    n_jobs=1,
    error_score="raise",
    return_train_score=False
)

repeated_mae_scores = -cv_scores["test_MAE"]
repeated_rmse_scores = -cv_scores["test_RMSE"]
repeated_r2_scores = cv_scores["test_R2"]

print("Cross-validation completed.")

# ------------------------------------------------------------
# 5. Create fold-level results
# ------------------------------------------------------------

repeated_cv_results_df = pd.DataFrame({
    "Evaluation": np.arange(
        1, len(repeated_r2_scores) + 1
    ),
    "Repeat": np.repeat(
        np.arange(1, N_REPEATS + 1),
        N_SPLITS
    ),
    "Fold": np.tile(
        np.arange(1, N_SPLITS + 1),
        N_REPEATS
    ),
    "MAE": repeated_mae_scores,
    "RMSE": repeated_rmse_scores,
    "R2": repeated_r2_scores
})

assert len(repeated_cv_results_df) == 50

assert np.isfinite(
    repeated_cv_results_df[
        ["MAE", "RMSE", "R2"]
    ].to_numpy()
).all()

# ------------------------------------------------------------
# 6. Calculate summary statistics
# ------------------------------------------------------------

summary_rows = []

for metric in ["MAE", "RMSE", "R2"]:

    values = repeated_cv_results_df[
        metric
    ].to_numpy()

    summary_rows.append({
        "Metric": metric,
        "Mean": float(np.mean(values)),
        "SD": float(np.std(values, ddof=1)),
        "P2_5": float(np.percentile(values, 2.5)),
        "P97_5": float(np.percentile(values, 97.5)),
        "Minimum": float(np.min(values)),
        "Maximum": float(np.max(values)),
        "N_Evaluations": len(values)
    })

repeated_cv_metrics_df = pd.DataFrame(
    summary_rows
)

# Preserve original downstream variable
repeated_cv_summary = pd.DataFrame({
    "Metric": [
        "Mean R2",
        "Standard Deviation",
        "Lower 2.5th Percentile",
        "Upper 97.5th Percentile",
        "Minimum R2",
        "Maximum R2",
        "Number of Evaluations"
    ],
    "Value": [
        float(np.mean(repeated_r2_scores)),
        float(np.std(repeated_r2_scores, ddof=1)),
        float(np.percentile(repeated_r2_scores, 2.5)),
        float(np.percentile(repeated_r2_scores, 97.5)),
        float(np.min(repeated_r2_scores)),
        float(np.max(repeated_r2_scores)),
        len(repeated_r2_scores)
    ]
})

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\nREPEATED CV PERFORMANCE")
print("-" * 60)

display(
    repeated_cv_metrics_df.round(4)
)

print("\nORIGINAL R2 SUMMARY FORMAT")
display(
    repeated_cv_summary.round(4)
)

# ------------------------------------------------------------
# 8. Export results
# ------------------------------------------------------------

if "TABLE_VALIDATION" not in globals():
    TABLE_VALIDATION = (
        Path(TABLE_DATASET) / "validation"
    )

TABLE_VALIDATION = Path(TABLE_VALIDATION)
TABLE_VALIDATION.mkdir(
    parents=True,
    exist_ok=True
)

export_tables = {
    "Table15_9_Repeated_CV_Performance":
        repeated_cv_summary,
    "Table15_9_Repeated_CV_All_Metrics":
        repeated_cv_metrics_df,
    "Table15_9_Repeated_CV_Fold_Results":
        repeated_cv_results_df
}

for filename, dataframe in export_tables.items():

    excel_path = TABLE_VALIDATION / f"{filename}.xlsx"
    csv_path = TABLE_VALIDATION / f"{filename}.csv"

    dataframe.to_excel(excel_path, index=False)
    dataframe.to_csv(csv_path, index=False)

    assert excel_path.is_file()
    assert csv_path.is_file()

    print(f"Saved: {filename}.xlsx")
    print(f"Saved: {filename}.csv")

# ------------------------------------------------------------
# 9. Methodological status
# ------------------------------------------------------------

print("\nMETHODOLOGICAL STATUS")
print("-" * 60)

print("Repeated CV evaluations: 50")
print("Raw training features: Used")
print("Fold-specific preprocessing: Yes")
print("Fixed final hyperparameters: Yes")
print("Independent tuning per fold: No")
print("Patient-group independence: Not verified")
print("Held-out test used: No")
print("External validation: Not performed")

print("\nSTEP 15.9 COMPLETED")


In [ ]:
# ============================================================
# STEP 15.10: TRUE PATIENT IDENTIFIER AND SPLIT VERIFICATION
# Patient-level leakage audit
# Read-only | No training | No prediction
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import json


print("=" * 70)
print("STEP 15.10: TRUE PATIENT IDENTIFIER AND SPLIT VERIFICATION")
print("=" * 70)



# ------------------------------------------------------------
# 1. Identify patient identifier
# ------------------------------------------------------------

candidate_ids = [

    "PTID",
    "PTID_old",
    "PTNUM",
    "ptid2"

]


available_ids = [

    c for c in candidate_ids
    if c in test_df.columns

]


print("\nAVAILABLE PATIENT IDENTIFIERS")

print(
    available_ids
)



if len(available_ids) == 0:

    raise RuntimeError(
        "No patient identifier found."
    )


PRIMARY_PTID = available_ids[0]


print(
    "\nSelected patient identifier:",
    PRIMARY_PTID
)



# ------------------------------------------------------------
# 2. Recover split patient IDs
# ------------------------------------------------------------

if "train_df" not in globals():

    raise RuntimeError(
        "train_df unavailable"
    )


if "validation_df" not in globals():

    raise RuntimeError(
        "validation_df unavailable"
    )


if "test_df" not in globals():

    raise RuntimeError(
        "test_df unavailable"
    )



split_ids = {

    "Train":

        set(
            train_df[PRIMARY_PTID]
            .dropna()
            .astype(str)
        ),


    "Validation":

        set(
            validation_df[PRIMARY_PTID]
            .dropna()
            .astype(str)
        ),


    "Test":

        set(
            test_df[PRIMARY_PTID]
            .dropna()
            .astype(str)
        )

}



# ------------------------------------------------------------
# 3. Uniqueness check
# ------------------------------------------------------------

identifier_summary = []


for name, ids in split_ids.items():

    df_ref = {

        "Train": train_df,

        "Validation": validation_df,

        "Test": test_df

    }[name]


    identifier_summary.append({

        "Split":

            name,


        "Rows":

            len(df_ref),


        "Unique_PTID":

            len(ids),


        "Missing_PTID":

            int(
                df_ref[PRIMARY_PTID]
                .isna()
                .sum()
            )

    })



identifier_table = pd.DataFrame(
    identifier_summary
)



# ------------------------------------------------------------
# 4. Cross split overlap
# ------------------------------------------------------------

overlap_results = {


    "Train_vs_Validation":

        len(
            split_ids["Train"]
            &
            split_ids["Validation"]
        ),


    "Train_vs_Test":

        len(
            split_ids["Train"]
            &
            split_ids["Test"]
        ),


    "Validation_vs_Test":

        len(
            split_ids["Validation"]
            &
            split_ids["Test"]
        )

}



patient_independence = all(

    v == 0

    for v in overlap_results.values()

)



# ------------------------------------------------------------
# 5. Save audit
# ------------------------------------------------------------

metadata_dir = Path(

    "/kaggle/working/SCI_Recovery_ML_Project/"
    "01_data/metadata"

)

metadata_dir.mkdir(

    parents=True,

    exist_ok=True

)



identifier_table.to_csv(

    metadata_dir /
    "patient_identifier_verification.csv",

    index=False

)



audit = {

    "primary_identifier":

        PRIMARY_PTID,


    "identifier_summary":

        identifier_summary,


    "overlap":

        overlap_results,


    "patient_level_independence":

        patient_independence

}



(metadata_dir /
 "patient_level_split_verification.json").write_text(

    json.dumps(
        audit,
        indent=2
    ),

    encoding="utf-8"

)



# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nPATIENT IDENTIFIER SUMMARY")

display(
    identifier_table
)



print("\nPATIENT OVERLAP")

display(

    pd.DataFrame(

        overlap_results.items(),

        columns=[
            "Comparison",
            "Shared_PTID"
        ]

    )

)



print("\nFINAL STATUS")

if patient_independence:

    print(
        "Patient-level independence: VERIFIED"
    )

else:

    print(
        "Patient-level independence: FAILED"
    )


print(
    "\nSTEP 15.10 COMPLETED"
)

In [ ]:

# ============================================================
# STEP 15.11: STRICT TEMPORAL FEATURE LEAKAGE AUDIT
# Whitelist + Feature Formula Verification
# Read Only | No Model Retraining
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 15.11: TEMPORAL FEATURE LEAKAGE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define permissible landmark predictors
# ------------------------------------------------------------

EXPECTED_FEATURES = {
    "Model_A": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01"
    ],
    "Model_B4": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT_change_4"
    ],
    "Model_B8": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT_change_4",
        "TOTMOT_change_8"
    ],
    "Model_B16": [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT_change_4",
        "TOTMOT_change_8",
        "TOTMOT_change_16"
    ]
}

EXPECTED_FORMULAS = {
    "TOTMOT_change_4": (
        "TOTMOT04",
        "TOTMOT01"
    ),
    "TOTMOT_change_8": (
        "TOTMOT08",
        "TOTMOT01"
    ),
    "TOTMOT_change_16": (
        "TOTMOT16",
        "TOTMOT01"
    )
}

ENGINEERED_SPLITS = {
    "train": train_fe,
    "validation": validation_fe,
    "test": test_fe
}

# ------------------------------------------------------------
# 2. Check model definitions
# ------------------------------------------------------------

assert set(EXPECTED_FEATURES) == set(
    MODEL_FEATURES
)

assert set(EXPECTED_FEATURES) == set(
    MODEL_INPUTS
)

audit_rows = []
failures = []

# ------------------------------------------------------------
# 3. Verify predictor whitelist
# ------------------------------------------------------------

for model_name, allowed_features in EXPECTED_FEATURES.items():

    configured_features = list(
        MODEL_FEATURES[model_name]
    )

    if configured_features != allowed_features:
        failures.append(
            f"{model_name}: incorrect feature definition"
        )

    for split_name, engineered_df in ENGINEERED_SPLITS.items():

        X = MODEL_INPUTS[model_name][
            f"X_{split_name}"
        ]

        input_columns = list(X.columns)

        if input_columns != allowed_features:
            failures.append(
                f"{model_name}/{split_name}: "
                "incorrect predictor columns or order"
            )

        if not X.index.equals(engineered_df.index):
            failures.append(
                f"{model_name}/{split_name}: "
                "row index mismatch"
            )

        # ----------------------------------------------------
        # Verify X features match the engineered source
        # ----------------------------------------------------

        for feature in allowed_features:

            if feature not in engineered_df.columns:
                failures.append(
                    f"{split_name}: missing {feature}"
                )
                continue

            if feature not in X.columns:
                failures.append(
                    f"{model_name}/{split_name}: "
                    f"missing {feature}"
                )
                continue

            left = X[feature]
            right = engineered_df.loc[
                X.index,
                feature
            ]

            if (
                pd.api.types.is_numeric_dtype(left)
                and pd.api.types.is_numeric_dtype(right)
            ):

                equal = np.allclose(
                    pd.to_numeric(
                        left,
                        errors="coerce"
                    ),
                    pd.to_numeric(
                        right,
                        errors="coerce"
                    ),
                    equal_nan=True,
                    rtol=1e-8,
                    atol=1e-8
                )

            else:

                equal = (
                    left.astype("string")
                    .reset_index(drop=True)
                    .equals(
                        right.astype("string")
                        .reset_index(drop=True)
                    )
                )

            if not equal:
                failures.append(
                    f"{model_name}/{split_name}: "
                    f"feature mismatch: {feature}"
                )

        audit_rows.append({
            "Model": model_name,
            "Split": split_name,
            "Predictors": len(X.columns),
            "Whitelist_Passed": (
                input_columns == allowed_features
            )
        })

# ------------------------------------------------------------
# 4. Verify dynamic feature calculations
# ------------------------------------------------------------

formula_rows = []

for split_name, engineered_df in ENGINEERED_SPLITS.items():

    for feature, columns in EXPECTED_FORMULAS.items():

        later_col, baseline_col = columns

        required_columns = [
            feature,
            later_col,
            baseline_col
        ]

        if not all(
            column in engineered_df.columns
            for column in required_columns
        ):
            failures.append(
                f"{split_name}: missing formula columns "
                f"for {feature}"
            )
            continue

        actual = pd.to_numeric(
            engineered_df[feature],
            errors="coerce"
        ).to_numpy(dtype=float)

        expected = (
            pd.to_numeric(
                engineered_df[later_col],
                errors="coerce"
            )
            -
            pd.to_numeric(
                engineered_df[baseline_col],
                errors="coerce"
            )
        ).to_numpy(dtype=float)

        formula_matches = np.allclose(
            actual,
            expected,
            equal_nan=True,
            rtol=1e-8,
            atol=1e-8
        )

        formula_rows.append({
            "Split": split_name,
            "Feature": feature,
            "Formula_Matches": bool(
                formula_matches
            )
        })

        if not formula_matches:
            failures.append(
                f"{split_name}: incorrect formula "
                f"for {feature}"
            )

# ------------------------------------------------------------
# 5. Summarize results
# ------------------------------------------------------------

feature_audit_df = pd.DataFrame(
    audit_rows
)

formula_audit_df = pd.DataFrame(
    formula_rows
)

print("\nLANDMARK FEATURE WHITELIST")

display(feature_audit_df)

print("\nFEATURE FORMULA VERIFICATION")

display(formula_audit_df)

all_passed = len(failures) == 0

# ------------------------------------------------------------
# 6. Save machine-readable audit
# ------------------------------------------------------------

audit_report = {
    "feature_whitelist_and_formula_checks_passed":
        all_passed,

    "errors":
        failures,

    "nominal_landmark_feature_structure":
        "Verified against notebook feature definitions",

    "actual_assessment_dates_verified":
        False,

    "upstream_feature_selection_and_cv_audited":
        False,

    "limitations": [
        "The whitelist verifies specified feature sets.",
        "It cannot independently prove actual visit chronology.",
        "Feature engineering code provenance requires separate confirmation.",
        "Cross-validation fold isolation requires separate review."
    ]
}

output_path = Path(
    "/kaggle/working/SCI_Recovery_ML_Project/"
    "01_data/metadata/temporal_feature_audit.json"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

output_path.write_text(
    json.dumps(
        audit_report,
        indent=4
    ),
    encoding="utf-8"
)

print("\nAUDIT SUMMARY")
print("All data-level feature checks passed:", all_passed)
print("Actual assessment dates verified: False")
print("Cross-validation isolation verified here: No")

print("\nSaved:", output_path)

if failures:
    raise AssertionError(
        "Feature audit failed: "
        + "; ".join(failures[:8])
    )

print("\nSTEP 15.11 COMPLETED")


In [ ]:

# ============================================================
# STEP 15.12: ASSESSMENT TIMING AND FEATURE PROVENANCE AUDIT
# Diagnostic Only | No Model Retraining
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 15.12: ASSESSMENT TIMING AND FEATURE PROVENANCE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Identify available visit and timing variables
# ------------------------------------------------------------

timing_keywords = [
    "date", "visit", "week", "day", "dtbas",
    "dtweek", "baseline", "followup"
]

timing_columns = [
    col for col in primary_cohort.columns
    if any(
        keyword in str(col).lower()
        for keyword in timing_keywords
    )
]

print("\nPOTENTIAL TIMING VARIABLES")
print("-" * 60)
print("Total candidate columns:", len(timing_columns))
print("First 60 candidates:")
print(timing_columns[:60])

# ------------------------------------------------------------
# 2. Inspect relevant original visit variables
# ------------------------------------------------------------

candidate_visit_columns = [
    "BASELINE", "DTBAS",
    "WEEK4", "DTWEEK4",
    "WEEK8", "DTWEEK8",
    "WEEK16", "DTWEEK16",
    "WEEK26", "DTWEEK26"
]

available_visit_columns = [
    col for col in candidate_visit_columns
    if col in primary_cohort.columns
]

print("\nAVAILABLE VISIT VARIABLES")
print("-" * 60)
print(available_visit_columns)

visit_profile_rows = []

for col in available_visit_columns:

    series = primary_cohort[col]

    examples = (
        series.dropna()
        .astype(str)
        .unique()[:5]
        .tolist()
    )

    visit_profile_rows.append({
        "Column": col,
        "Data_Type": str(series.dtype),
        "Non_Missing": int(series.notna().sum()),
        "Missing": int(series.isna().sum()),
        "Example_Values": str(examples)
    })

visit_profile_df = pd.DataFrame(visit_profile_rows)

print("\nVISIT VARIABLE PROFILE")
display(visit_profile_df)

# ------------------------------------------------------------
# 3. Inspect chronological structure without assuming format
# ------------------------------------------------------------

print("\nCHRONOLOGICAL DATA AVAILABILITY")
print("-" * 60)

motor_columns = [
    "TOTMOT01",
    "TOTMOT04",
    "TOTMOT08",
    "TOTMOT16",
    "TOTMOT26"
]

motor_profile_rows = []

for col in motor_columns:

    if col not in primary_cohort.columns:
        continue

    values = pd.to_numeric(
        primary_cohort[col],
        errors="coerce"
    )

    motor_profile_rows.append({
        "Variable": col,
        "Non_Missing": int(values.notna().sum()),
        "Missing": int(values.isna().sum()),
        "Minimum": values.min(),
        "Maximum": values.max()
    })

motor_availability_df = pd.DataFrame(
    motor_profile_rows
)

display(motor_availability_df)

# ------------------------------------------------------------
# 4. Confirm feature-engineering identities
# ------------------------------------------------------------

print("\nFEATURE PROVENANCE CONSISTENCY")
print("-" * 60)

provenance_rows = []

formula_map = {
    "TOTMOT_change_4": "TOTMOT04",
    "TOTMOT_change_8": "TOTMOT08",
    "TOTMOT_change_16": "TOTMOT16"
}

for feature, followup_col in formula_map.items():

    actual = pd.to_numeric(
        train_fe[feature],
        errors="coerce"
    )

    expected = (
        pd.to_numeric(
            train_fe[followup_col],
            errors="coerce"
        )
        -
        pd.to_numeric(
            train_fe["TOTMOT01"],
            errors="coerce"
        )
    )

    matching = np.isclose(
        actual.to_numpy(dtype=float),
        expected.to_numpy(dtype=float),
        equal_nan=True,
        rtol=1e-8,
        atol=1e-8
    )

    provenance_rows.append({
        "Feature": feature,
        "Expected_Formula":
            f"{followup_col} - TOTMOT01",
        "Matching_Rows": int(matching.sum()),
        "Total_Rows": len(matching),
        "Consistent": bool(matching.all())
    })

feature_provenance_df = pd.DataFrame(
    provenance_rows
)

display(feature_provenance_df)

# ------------------------------------------------------------
# 5. Status
# ------------------------------------------------------------

print("\nAUDIT STATUS")
print("-" * 60)

print(
    "Feature values consistent with formulas:",
    bool(feature_provenance_df["Consistent"].all())
)

print(
    "Visit-date chronology:",
    "Pending interpretation of timing variables"
)

print(
    "Original feature-construction code:",
    "Not independently inspected"
)

print("Model retraining performed: No")
print("Held-out test predictions recomputed: No")

print("\nSTEP 15.12 DIAGNOSTIC COMPLETED")


In [ ]:

# ============================================================
# STEP 15.13: VISIT-DATE ENCODING AND CHRONOLOGY AUDIT
# Diagnostic Only | No Data Modification
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 15.13: VISIT-DATE ENCODING AND CHRONOLOGY AUDIT")
print("=" * 70)

df = primary_cohort.copy()

date_columns = [
    "DTBAS",
    "DTWEEK4",
    "DTWEEK8",
    "DTWEEK16",
    "DTWEEK26"
]

# ------------------------------------------------------------
# 1. Raw date-value distribution
# ------------------------------------------------------------

distribution_rows = []

for col in date_columns:

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    valid = values.dropna()

    distribution_rows.append({
        "Variable": col,
        "Available_N": len(valid),
        "Min": valid.min(),
        "Median": valid.median(),
        "Max": valid.max(),
        "Unique_Values": valid.nunique(),
        "Non_Integer_N": int(
            (~np.isclose(valid % 1, 0)).sum()
        )
    })

date_distribution_df = pd.DataFrame(
    distribution_rows
)

print("\nDATE VALUE DISTRIBUTION")
display(date_distribution_df)

# ------------------------------------------------------------
# 2. Consecutive visit differences
# ------------------------------------------------------------

visit_pairs = [
    ("DTBAS", "DTWEEK4"),
    ("DTWEEK4", "DTWEEK8"),
    ("DTWEEK8", "DTWEEK16"),
    ("DTWEEK16", "DTWEEK26"),
    ("DTBAS", "DTWEEK26")
]

difference_rows = []

for earlier, later in visit_pairs:

    a = pd.to_numeric(
        df[earlier],
        errors="coerce"
    )

    b = pd.to_numeric(
        df[later],
        errors="coerce"
    )

    delta = (b - a).dropna()

    difference_rows.append({
        "Earlier": earlier,
        "Later": later,
        "Comparable_N": len(delta),
        "Negative_N": int((delta < 0).sum()),
        "Zero_N": int((delta == 0).sum()),
        "Positive_N": int((delta > 0).sum()),
        "Median_Difference": delta.median(),
        "Minimum_Difference": delta.min(),
        "Maximum_Difference": delta.max()
    })

date_difference_df = pd.DataFrame(
    difference_rows
)

print("\nVISIT DIFFERENCE AUDIT")
display(date_difference_df)

# ------------------------------------------------------------
# 3. Week-16 to week-26 ordering
# ------------------------------------------------------------

week16 = pd.to_numeric(
    df["DTWEEK16"],
    errors="coerce"
)

week26 = pd.to_numeric(
    df["DTWEEK26"],
    errors="coerce"
)

valid_pair = week16.notna() & week26.notna()

week16_before_26 = (
    valid_pair & (week16 < week26)
)

same_date = (
    valid_pair & (week16 == week26)
)

reversed_date = (
    valid_pair & (week16 > week26)
)

print("\nWEEK-16 VS WEEK-26 ORDERING")
print("-" * 60)

print("Comparable patients:", int(valid_pair.sum()))
print("Week 16 before week 26:", int(week16_before_26.sum()))
print("Same recorded value:", int(same_date.sum()))
print("Week 16 after week 26:", int(reversed_date.sum()))
print("Missing either date:", int((~valid_pair).sum()))

# ------------------------------------------------------------
# 4. Inspect unusual records without exposing patient IDs
# ------------------------------------------------------------

unusual_mask = same_date | reversed_date

unusual_records = df.loc[
    unusual_mask,
    [
        "DTBAS",
        "DTWEEK4",
        "DTWEEK8",
        "DTWEEK16",
        "DTWEEK26"
    ]
].copy()

print("\nUNUSUAL TIMING RECORDS")
print("-" * 60)

print("Records:", len(unusual_records))
display(unusual_records.head(15))

# ------------------------------------------------------------
# 5. Interpretation status
# ------------------------------------------------------------

print("\nMETHODOLOGICAL STATUS")
print("-" * 60)

print("Raw visit differences calculated: Yes")
print("Calendar-date encoding confirmed: No")
print("Chronological ordering assessed: Provisionally")
print("Patient-level splitting changed: No")
print("Models retrained: No")
print("Held-out test predictions recomputed: No")

print("\nSTEP 15.13 COMPLETED")


In [ ]:

# ============================================================
# STEP 15.14: ENHANCED ASSESSMENT CHRONOLOGY AUDIT
# Nominal Landmarks | No Model Retraining
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 15.14: ENHANCED ASSESSMENT TIMING AUDIT")
print("=" * 70)

if "primary_cohort" not in globals():
    raise RuntimeError("primary_cohort not found.")

df = primary_cohort.copy()

visits = [
    ("Baseline", "DTBAS", "TOTMOT01"),
    ("Week4", "DTWEEK4", "TOTMOT04"),
    ("Week8", "DTWEEK8", "TOTMOT08"),
    ("Week16", "DTWEEK16", "TOTMOT16"),
    ("Week26", "DTWEEK26", "TOTMOT26")
]

for _, dt, motor in visits:
    if dt not in df or motor not in df:
        raise KeyError(f"Missing: {dt} or {motor}")

# ------------------------------------------------------------
# 1. Profile original DT variables
# ------------------------------------------------------------

rows = []

for visit, dt, motor in visits:

    d = pd.to_numeric(
        df[dt], errors="coerce"
    )

    m = pd.to_numeric(
        df[motor], errors="coerce"
    )

    rows.append({
        "Visit": visit,
        "DT_Variable": dt,
        "DT_Dtype": str(df[dt].dtype),
        "DT_Available": int(d.notna().sum()),
        "DT_Unique": int(d.nunique()),
        "DT_Min": (
            float(d.min()) if d.notna().any()
            else np.nan
        ),
        "DT_Max": (
            float(d.max()) if d.notna().any()
            else np.nan
        ),
        "Motor_Available": int(m.notna().sum()),
        "Motor_Missing": int(m.isna().sum())
    })

profile = pd.DataFrame(rows)

print("\nASSESSMENT PROFILE")
display(profile)

# ------------------------------------------------------------
# 2. Pairwise DT comparisons
# ------------------------------------------------------------

pairs = []

for i in range(len(visits) - 1):

    first_name, first_dt, _ = visits[i]
    second_name, second_dt, _ = visits[i + 1]

    a = pd.to_numeric(
        df[first_dt], errors="coerce"
    )
    b = pd.to_numeric(
        df[second_dt], errors="coerce"
    )

    eligible = a.notna() & b.notna()
    delta = (b - a)[eligible]

    pairs.append({
        "Pair": f"{first_name} -> {second_name}",
        "Paired_N": int(eligible.sum()),
        "Equal_N": int((delta == 0).sum()),
        "Increasing_N": int((delta > 0).sum()),
        "Decreasing_N": int((delta < 0).sum()),
        "Median_Code_Difference": (
            float(delta.median())
            if len(delta) else np.nan
        )
    })

pair_table = pd.DataFrame(pairs)

print("\nPAIRWISE DT-CODE COMPARISON")
display(pair_table)

print(
    "\nIMPORTANT: These are numeric-code comparisons, "
    "not clinically confirmed elapsed times."
)

# ------------------------------------------------------------
# 3. Check Week-16 feature availability
# ------------------------------------------------------------

week16_missing = {}

if "MODEL_INPUTS" in globals():

    for split in ["train", "validation", "test"]:

        X = MODEL_INPUTS["Model_B16"][
            f"X_{split}"
        ]

        week16_missing[split] = int(
            X["TOTMOT_change_16"].isna().sum()
        )

print("\nWEEK-16 FEATURE MISSINGNESS")
print(week16_missing)

# ------------------------------------------------------------
# 4. Documentation status
# ------------------------------------------------------------

# Set source definitions only after inspecting
# authentic dataset documentation.
# Do not infer meaning from variable names.

DT_SOURCE_DOCUMENTATION_VERIFIED = False
ACTUAL_DATE_SEMANTICS_CONFIRMED = False

chronology_verified = bool(
    DT_SOURCE_DOCUMENTATION_VERIFIED
    and ACTUAL_DATE_SEMANTICS_CONFIRMED
)

report = {
    "assessment_variable_profiles":
        profile.to_dict(orient="records"),

    "pairwise_numeric_code_comparisons":
        pair_table.to_dict(orient="records"),

    "week16_feature_missing_by_split":
        week16_missing,

    "dt_variable_semantics_verified":
        DT_SOURCE_DOCUMENTATION_VERIFIED,

    "actual_assessment_chronology_verified":
        chronology_verified,

    "nominal_landmark_reporting_required":
        not chronology_verified,

    "limitations": [
        "DT variable units and semantics remain unconfirmed.",
        "Numeric-code differences cannot be interpreted as elapsed days without a codebook.",
        "Nominal Week-4, Week-8 and Week-16 labels do not by themselves establish actual examination timing."
    ]
}

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

metadata_dir = (
    PROJECT_DIR / "01_data/metadata"
)
metadata_dir.mkdir(parents=True, exist_ok=True)

output = (
    metadata_dir / "assessment_timing_audit.json"
)

output.write_text(
    json.dumps(
        report, indent=2, default=str
    ),
    encoding="utf-8"
)

profile.to_csv(
    metadata_dir / "assessment_timing_profile.csv",
    index=False
)

pair_table.to_csv(
    metadata_dir / "assessment_code_comparisons.csv",
    index=False
)

print("\nCHRONOLOGY VERIFIED:", chronology_verified)
print("Nominal landmark reporting required:",
      not chronology_verified)
print("Saved:", output)
print("\nSTEP 15.14 COMPLETED")


In [ ]:

# ============================================================
# STEP 15.14B: NOMINAL LANDMARK REPORTING CORRECTION
# Sygen SCI Trial | Q1 Manuscript Documentation
#
# No retraining
# No prediction changes
# No unverified chronology claims
# ============================================================

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 15.14B: NOMINAL LANDMARK DOCUMENTATION")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

META = PROJECT_DIR / "01_data/metadata"
META.mkdir(parents=True, exist_ok=True)

TIMING_PATH = META / "assessment_timing_audit.json"
FINAL_PATH = META / "final_reproducibility_audit.json"

if not TIMING_PATH.is_file():
    raise FileNotFoundError(
        "Run updated Step 15.14 first."
    )

timing = json.loads(
    TIMING_PATH.read_text(encoding="utf-8")
)

# ------------------------------------------------------------
# 1. Nominal visits supported by Sygen publications
# ------------------------------------------------------------

visit_definitions = {
    "Baseline": {
        "variable": "TOTMOT01",
        "nominal_time": "Baseline",
        "predictor_or_outcome": "Predictor"
    },
    "Week4": {
        "variable": "TOTMOT04",
        "nominal_time": "Week 4",
        "predictor_or_outcome": "Predictor"
    },
    "Week8": {
        "variable": "TOTMOT08",
        "nominal_time": "Week 8",
        "predictor_or_outcome": "Predictor"
    },
    "Week16": {
        "variable": "TOTMOT16",
        "nominal_time": "Week 16",
        "predictor_or_outcome": "Predictor"
    },
    "Week26": {
        "variable": "TOTMOT26",
        "nominal_time": "Week 26",
        "predictor_or_outcome": "Outcome"
    }
}

# ------------------------------------------------------------
# 2. Check documented numerical timing evidence
# ------------------------------------------------------------

chronology_verified = timing.get(
    "actual_assessment_chronology_verified"
)

if chronology_verified is True:
    raise RuntimeError(
        "Chronology has been marked verified. "
        "Review the new documentation before using "
        "the nominal-only interpretation."
    )

pairwise = timing.get(
    "pairwise_numeric_code_comparisons", []
)

missing_week16 = timing.get(
    "week16_feature_missing_by_split", {}
)

# ------------------------------------------------------------
# 3. Create reporting terminology table
# ------------------------------------------------------------

rows = []

for visit, definition in visit_definitions.items():

    rows.append({
        "Study_Visit": visit,
        "Motor_Variable": definition["variable"],
        "Reporting_Label": definition["nominal_time"],
        "Role": definition["predictor_or_outcome"],
        "Individual_Visit_Date_Verified": False,
        "Use_Exact_Elapsed_Days": False
    })

landmark_table = pd.DataFrame(rows)

print("\nNOMINAL LANDMARK DEFINITIONS")
print(landmark_table.to_string(index=False))

# ------------------------------------------------------------
# 4. Manuscript-ready reporting text
# ------------------------------------------------------------

methods_text = (
    "The study used the protocol-scheduled neurological "
    "assessment landmarks of the Sygen spinal cord injury "
    "trial. Baseline motor scores and changes observed at "
    "the nominal Week-4, Week-8, and Week-16 visits were "
    "used to predict total motor scores at the nominal "
    "Week-26 assessment. The landmark labels reflect "
    "scheduled study visits rather than independently "
    "verified patient-specific elapsed times. "
    "The interpretation of the available DT variables "
    "as actual examination dates could not be confirmed."
)

limitations_text = (
    "Patient-specific neurological examination dates "
    "could not be independently established from the "
    "available dataset documentation. Most paired DT "
    "values were unchanged across successive nominal "
    "study visits, and their coding semantics remain "
    "unconfirmed. Consequently, the analysis evaluates "
    "prediction using scheduled longitudinal assessment "
    "landmarks and cannot quantify deviations between "
    "scheduled and actual examination times."
)

# ------------------------------------------------------------
# 5. Save research reporting audit
# ------------------------------------------------------------

report = {
    "audit_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "study": "Sygen spinal cord injury trial",

    "reporting_convention":
        "Protocol-scheduled nominal study visits",

    "nominal_landmark_reporting_documented": True,

    "actual_assessment_chronology_verified": False,

    "dt_variable_semantics_verified": False,

    "exact_elapsed_time_claim_permitted": False,

    "model_retraining_required_for_label_correction":
        False,

    "visit_definitions": visit_definitions,

    "pairwise_dt_code_summary": pairwise,

    "week16_missing_by_split": missing_week16,

    "methods_text": methods_text,

    "limitations_text": limitations_text,

    "supporting_reference": {
        "title":
            "Studying missingness in spinal cord injury "
            "data: challenges and impact of data imputation",

        "journal": "BMC Medical Research Methodology",

        "year": 2023,

        "url":
            "https://doi.org/10.1186/s12874-023-02125-x",

        "evidence_scope":
            "Supports Sygen protocol-scheduled "
            "neurological assessments; does not "
            "define the patient-level DT codes."
    }
}

REPORT_PATH = (
    META / "nominal_landmark_reporting_audit.json"
)

TABLE_PATH = (
    META / "nominal_landmark_definitions.csv"
)

TEXT_PATH = (
    META / "assessment_chronology_manuscript_text.txt"
)

REPORT_PATH.write_text(
    json.dumps(report, indent=4),
    encoding="utf-8"
)

landmark_table.to_csv(
    TABLE_PATH,
    index=False
)

TEXT_PATH.write_text(
    "METHODS\n\n" + methods_text
    + "\n\nLIMITATIONS\n\n"
    + limitations_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 6. Add reporting evidence to final audit
# ------------------------------------------------------------

if FINAL_PATH.is_file():

    final_audit = json.loads(
        FINAL_PATH.read_text(encoding="utf-8")
    )

    final_audit["assessment_chronology_reporting"] = {
        "nominal_landmarks_documented": True,
        "actual_dates_verified": False,
        "supporting_audit": str(REPORT_PATH),
        "manuscript_text_saved": str(TEXT_PATH)
    }

    # Preserve verification status and counts.
    checks = final_audit.get(
        "verification_checks", {}
    )

    if "Assessment chronology" in checks:
        checks["Assessment chronology"] = False

    final_audit["verification_checks"] = checks

    FINAL_PATH.write_text(
        json.dumps(
            final_audit, indent=4
        ),
        encoding="utf-8"
    )

# ------------------------------------------------------------
# 7. Final output
# ------------------------------------------------------------

print("\nMETHODS STATEMENT")
print(methods_text)

print("\nLIMITATIONS STATEMENT")
print(limitations_text)

print("\nFINAL STATUS")
print("Nominal assessment reporting: DOCUMENTED")
print("Actual examination dates: UNVERIFIED")
print("Model retrained: No")
print("Test predictions modified: No")

print("\nFILES SAVED")
print(REPORT_PATH)
print(TABLE_PATH)
print(TEXT_PATH)

print("\nSTEP 15.14B COMPLETED")


In [ ]:

# ============================================================
# STEP 15.14C — LANDMARK PREDICTOR / FUTURE-LEAKAGE AUDIT
# Placement: After STEP 15.14B
# No retraining. No changes to predictions.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
META = ROOT / "01_data/metadata"
META.mkdir(parents=True, exist_ok=True)

if "MODEL_FEATURES" not in globals():
    raise RuntimeError("MODEL_FEATURES not found.")

if "MODEL_INPUTS" not in globals():
    raise RuntimeError("MODEL_INPUTS not found.")

expected_features = {
    "Model_A": [
        "AGE", "SEXCD", "AIS.1", "TOTMOT01"
    ],
    "Model_B4": [
        "AGE", "SEXCD", "AIS.1", "TOTMOT01",
        "TOTMOT_change_4"
    ],
    "Model_B8": [
        "AGE", "SEXCD", "AIS.1", "TOTMOT01",
        "TOTMOT_change_4", "TOTMOT_change_8"
    ],
    "Model_B16": [
        "AGE", "SEXCD", "AIS.1", "TOTMOT01",
        "TOTMOT_change_4", "TOTMOT_change_8",
        "TOTMOT_change_16"
    ]
}

for model in expected_features:
    if model not in MODEL_FEATURES or model not in MODEL_INPUTS:
        raise KeyError(f"Missing model definition: {model}")

results = []
all_passed = True

for model, expected in expected_features.items():

    actual = list(MODEL_FEATURES[model])
    unexpected = sorted(set(actual) - set(expected))
    missing = sorted(set(expected) - set(actual))

    # Compare feature sets while preserving the expected order in the report.
    feature_set_matches = set(actual) == set(expected)

    # Ensure target is never included as a predictor.
    target_leakage = any(
        col in actual for col in [
            "TOTMOT26", "TARGET", "y", "Outcome"
        ]
    )

    # Check X matrices in all available partitions.
    split_checks = {}
    for split in ["train", "validation", "test"]:
        key = f"X_{split}"
        X = MODEL_INPUTS[model].get(key)

        if X is None:
            split_checks[split] = "NOT_AVAILABLE"
            continue

        split_cols = list(X.columns)
        split_checks[split] = {
            "columns_match_model_features":
                set(split_cols) == set(actual),
            "n_rows": int(len(X))
        }

        if set(split_cols) != set(actual):
            all_passed = False

    if not feature_set_matches or target_leakage:
        all_passed = False

    results.append({
        "model": model,
        "expected_features": expected,
        "actual_features": actual,
        "missing_expected_features": missing,
        "unexpected_features": unexpected,
        "feature_set_matches": feature_set_matches,
        "target_leakage_detected": target_leakage,
        "split_checks": split_checks
    })

# Formula check: motor change = follow-up score - baseline score.
# Run only if the source cohort contains the required columns.
formula_checks = {}
if "primary_cohort" in globals():
    df = primary_cohort

    formulas = {
        "TOTMOT_change_4": ("TOTMOT04", "TOTMOT01"),
        "TOTMOT_change_8": ("TOTMOT08", "TOTMOT01"),
        "TOTMOT_change_16": ("TOTMOT16", "TOTMOT01")
    }

    for change_col, (followup_col, baseline_col) in formulas.items():
        required = [change_col, followup_col, baseline_col]

        if all(c in df.columns for c in required):
            observed = pd.to_numeric(df[change_col], errors="coerce")
            expected = (
                pd.to_numeric(df[followup_col], errors="coerce")
                - pd.to_numeric(df[baseline_col], errors="coerce")
            )

            valid = observed.notna() & expected.notna()
            matches = bool(
                np.allclose(
                    observed[valid].to_numpy(),
                    expected[valid].to_numpy(),
                    rtol=0, atol=1e-8
                )
            ) if valid.any() else False

            formula_checks[change_col] = {
                "formula": f"{followup_col} - {baseline_col}",
                "n_compared": int(valid.sum()),
                "formula_matches": matches
            }

            if not matches:
                all_passed = False
        else:
            formula_checks[change_col] = {
                "formula": f"{followup_col} - {baseline_col}",
                "status": "NOT_VERIFIED_SOURCE_COLUMNS_MISSING"
            }
else:
    formula_checks["status"] = (
        "NOT_VERIFIED: primary_cohort unavailable in this session"
    )

report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "expected_landmark_features": expected_features,
    "model_feature_audits": results,
    "change_score_formula_audits": formula_checks,
    "predictor_audit_passed": all_passed,
    "actual_assessment_chronology_verified": False,
    "nominal_landmark_reporting_required": True,
    "interpretation": (
        "Feature-set checks assess predictor definitions and potential "
        "future-information leakage. They do not verify patient-specific "
        "examination dates or actual elapsed time between visits."
    )
}

path = META / "landmark_predictor_leakage_audit.json"
path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")

print("STEP 15.14C — PREDICTOR AUDIT")
print(json.dumps(report, indent=2, default=str))

if not all_passed:
    raise RuntimeError(
        "Predictor audit failed. Resolve unexpected/missing predictors "
        "or inconsistent change-score formulas before manuscript reporting."
    )

print("Feature-set checks passed; chronology remains unverified.")


In [ ]:

# ============================================================
# STEP 15.14D — LOCATE SOURCE COLUMNS FOR CHANGE-SCORE AUDIT
# Run after STEP 15.14C
# ============================================================

import pandas as pd

tokens = (
    "TOTMOT", "PTID", "AGE", "AIS",
    "MOTOR", "CHANGE", "WEEK"
)

found = []

for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame):
        continue

    cols = [str(c) for c in obj.columns]
    relevant = [
        c for c in cols
        if any(token in c.upper() for token in tokens)
    ]

    if relevant:
        found.append({
            "dataframe_name": name,
            "rows": len(obj),
            "relevant_columns": relevant
        })

for item in found:
    print(f"\nDataFrame: {item['dataframe_name']}")
    print(f"Rows: {item['rows']}")
    print("Relevant columns:")
    for col in item["relevant_columns"]:
        print("  -", col)

if not found:
    print(
        "No matching source DataFrame found in the current session. "
        "Inspect the dataset-loading and feature-engineering cells."
    )


In [ ]:

# ============================================================
# STEP 15.14E — ASSESSMENT CHRONOLOGY AUDIT
# Placement: After STEP 15.14D
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
AUDIT_DIR = ROOT / "06_results/audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Set these to the actual source DataFrame and date columns.
# Do not assume these column names exist in your dataset.
SOURCE_DF_NAME = "primary_cohort"
PATIENT_ID_COL = "PTID"

DATE_COLUMNS = {
    "baseline": "BASELINE_DATE",
    "week4": "WEEK4_DATE",
    "week8": "WEEK8_DATE",
    "week16": "WEEK16_DATE",
    "week26": "WEEK26_DATE",
}

df = globals().get(SOURCE_DF_NAME)

if not isinstance(df, pd.DataFrame):
    raise RuntimeError(
        f"{SOURCE_DF_NAME} is not an available DataFrame. "
        "Find the actual source DataFrame first."
    )

missing_cols = [
    c for c in [PATIENT_ID_COL, *DATE_COLUMNS.values()]
    if c not in df.columns
]

if missing_cols:
    result = {
        "status": "NOT_VERIFIED",
        "reason": "Required patient/date columns are missing",
        "missing_columns": missing_cols,
        "verified": False
    }
else:
    dates = pd.DataFrame({
        "PTID": df[PATIENT_ID_COL]
    })

    for name, col in DATE_COLUMNS.items():
        dates[name] = pd.to_datetime(
            df[col], errors="coerce"
        )

    required_dates = list(DATE_COLUMNS.keys())
    complete = dates[required_dates].notna().all(axis=1)

    ordered = (
        (dates["baseline"] < dates["week4"]) &
        (dates["week4"] < dates["week8"]) &
        (dates["week8"] < dates["week16"]) &
        (dates["week16"] < dates["week26"])
    )

    result = {
        "status": (
            "VERIFIED"
            if complete.all() and ordered.all()
            else "FAILED_OR_INCOMPLETE"
        ),
        "n_patients": int(len(dates)),
        "n_complete_date_sequences": int(complete.sum()),
        "n_incomplete_date_sequences": int((~complete).sum()),
        "n_ordered_complete_sequences": int(
            (complete & ordered).sum()
        ),
        "n_invalid_or_unordered_sequences": int(
            (complete & ~ordered).sum()
        ),
        "verified": bool(
            complete.all() and ordered.all()
        ),
        "note": (
            "Chronology is verified only if these are actual "
            "patient-level examination dates from the source."
        )
    }

out = AUDIT_DIR / "assessment_chronology_audit.json"
out.write_text(
    json.dumps(result, indent=2, default=str),
    encoding="utf-8"
)

print(json.dumps(result, indent=2, default=str))
print("Saved:", out)


In [ ]:

# ============================================================
# STEP 15.15: MISSINGNESS AND IMPUTATION AUDIT
# Diagnostic Only | No Model Retraining
# ============================================================

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

print("=" * 70)
print("STEP 15.15: MISSINGNESS AND IMPUTATION AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Inspect training feature missingness
# ------------------------------------------------------------

X_train = MODEL_INPUTS["Model_B16"]["X_train"]

missingness_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Missing_N": X_train.isna().sum().values,
    "Missing_Percent": (
        X_train.isna().mean().values * 100
    )
}).sort_values(
    "Missing_Percent",
    ascending=False
)

print("\nTRAINING FEATURE MISSINGNESS")
display(missingness_df)

# ------------------------------------------------------------
# 2. Inspect fitted preprocessing pipeline
# ------------------------------------------------------------

print("\nFINAL MODEL PIPELINE")
print("-" * 60)

print(final_lgbm_model)

if not isinstance(final_lgbm_model, Pipeline):
    raise TypeError(
        "Expected final_lgbm_model to be a Pipeline."
    )

preprocessor = final_lgbm_model.named_steps["preprocessor"]

print("\nPREPROCESSOR TYPE")
print(type(preprocessor).__name__)

# ------------------------------------------------------------
# 3. Recursively identify imputation components
# ------------------------------------------------------------

imputer_rows = []

def inspect_imputers(estimator, path="preprocessor"):

    if isinstance(estimator, SimpleImputer):

        imputer_rows.append({
            "Component": path,
            "Strategy": estimator.strategy,
            "Fill_Value": str(estimator.fill_value),
            "Fitted": hasattr(estimator, "statistics_")
        })

    elif isinstance(estimator, Pipeline):

        for step_name, step in estimator.steps:
            inspect_imputers(
                step,
                f"{path}.{step_name}"
            )

    elif isinstance(estimator, ColumnTransformer):

        for name, transformer, columns in estimator.transformers_:

            if transformer in ["drop", "passthrough"]:
                continue

            inspect_imputers(
                transformer,
                f"{path}.{name}"
            )

inspect_imputers(preprocessor)

imputation_audit_df = pd.DataFrame(
    imputer_rows,
    columns=[
        "Component",
        "Strategy",
        "Fill_Value",
        "Fitted"
    ]
)

print("\nIMPUTATION COMPONENTS")
display(imputation_audit_df)

# ------------------------------------------------------------
# 4. Check transformed training data
# ------------------------------------------------------------

X_transformed = preprocessor.transform(X_train)

if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

X_transformed = np.asarray(
    X_transformed,
    dtype=float
)

print("\nTRANSFORMED FEATURE CHECK")
print("-" * 60)

print("Original shape:", X_train.shape)
print("Transformed shape:", X_transformed.shape)
print(
    "Remaining NaN values:",
    int(np.isnan(X_transformed).sum())
)
print(
    "Infinite values:",
    int(np.isinf(X_transformed).sum())
)

# ------------------------------------------------------------
# 5. Inspect unfitted CV preprocessing configuration
# ------------------------------------------------------------

print("\nCROSS-VALIDATION PREPROCESSOR")
print("-" * 60)

cv_preprocessor = LANDMARK_PIPELINES["Model_B16"]

print("Type:", type(cv_preprocessor).__name__)
print(
    "Configured imputation components:",
    len(imputer_rows)
)

print(
    "CV preprocessing is cloned inside each fold:",
    "As implemented in previous CV steps"
)

# ------------------------------------------------------------
# 6. Final status
# ------------------------------------------------------------

print("\nMETHODOLOGICAL STATUS")
print("-" * 60)

print(
    "Imputation components identified:",
    len(imputer_rows)
)

print(
    "Fitted imputation components:",
    int(
        imputation_audit_df["Fitted"].sum()
    )
)

print(
    "Missing Week-16 feature in training:",
    int(X_train["TOTMOT_change_16"].isna().sum())
)

print("Model retraining performed: No")
print("Held-out test outcomes accessed: No")

print("\nSTEP 15.15 COMPLETED")


In [ ]:

# ============================================================
# STEP 15.16: MISSING WEEK-16 SENSITIVITY ANALYSIS
# Validation Set Only | No Model Retraining
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 15.16: MISSING WEEK-16 SENSITIVITY ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Retrieve existing validation data and predictions
# ------------------------------------------------------------

X_val = MODEL_INPUTS["Model_B16"]["X_validation"]

y_val = np.asarray(
    MODEL_INPUTS["Model_B16"]["y_validation"],
    dtype=float
).ravel()

y_pred = np.asarray(
    lgbm_validation_prediction,
    dtype=float
).ravel()

if not (
    len(X_val) == len(y_val) == len(y_pred)
):
    raise ValueError(
        "Validation data and prediction lengths do not match."
    )

# ------------------------------------------------------------
# 2. Identify observed and missing Week-16 groups
# ------------------------------------------------------------

observed_mask = (
    X_val["TOTMOT_change_16"]
    .notna()
    .to_numpy()
)

missing_mask = ~observed_mask

# ------------------------------------------------------------
# 3. Calculate subgroup performance
# ------------------------------------------------------------

def calculate_subgroup_metrics(mask, group_name):

    actual = y_val[mask]
    predicted = y_pred[mask]

    n = len(actual)

    if n == 0:
        return {
            "Group": group_name,
            "N": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "Mean_Error": np.nan
        }

    return {
        "Group": group_name,
        "N": n,
        "MAE": mean_absolute_error(
            actual, predicted
        ),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "R2": (
            r2_score(actual, predicted)
            if n >= 2 else np.nan
        ),
        "Mean_Error": float(
            np.mean(predicted - actual)
        )
    }

subgroup_results = [
    calculate_subgroup_metrics(
        observed_mask,
        "Observed Week-16"
    ),
    calculate_subgroup_metrics(
        missing_mask,
        "Missing Week-16"
    ),
    calculate_subgroup_metrics(
        np.ones(len(y_val), dtype=bool),
        "Overall Validation"
    )
]

week16_sensitivity_df = pd.DataFrame(
    subgroup_results
)

print("\nSUBGROUP PERFORMANCE")
display(
    week16_sensitivity_df.round(4)
)

# ------------------------------------------------------------
# 4. Compare subgroup absolute errors
# ------------------------------------------------------------

absolute_errors = np.abs(
    y_val - y_pred
)

observed_errors = absolute_errors[observed_mask]
missing_errors = absolute_errors[missing_mask]

if len(observed_errors) and len(missing_errors):

    mae_difference = (
        missing_errors.mean()
        - observed_errors.mean()
    )

    print("\nSUBGROUP ERROR COMPARISON")
    print("-" * 60)

    print(
        "MAE difference (Missing - Observed):",
        round(float(mae_difference), 4)
    )

    print(
        "Observed subgroup N:",
        len(observed_errors)
    )

    print(
        "Missing subgroup N:",
        len(missing_errors)
    )

# ------------------------------------------------------------
# 5. Export validation sensitivity results
# ------------------------------------------------------------

from pathlib import Path

output_dir = Path(TABLE_VALIDATION)
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

xlsx_path = (
    output_dir /
    "Table15_18_Week16_Missingness_Sensitivity.xlsx"
)

csv_path = (
    output_dir /
    "Table15_18_Week16_Missingness_Sensitivity.csv"
)

week16_sensitivity_df.to_excel(
    xlsx_path,
    index=False
)

week16_sensitivity_df.to_csv(
    csv_path,
    index=False
)

print("\nFILES SAVED")
print(xlsx_path.name)
print(csv_path.name)

# ------------------------------------------------------------
# 6. Methodological status
# ------------------------------------------------------------

print("\nMETHODOLOGICAL STATUS")
print("-" * 60)

print("Validation subgroup analysis: Completed")
print("Model retraining: No")
print("Held-out test outcomes accessed: No")
print("External validation: Not performed")
print(
    "Subgroup comparison:",
    "Descriptive; not a causal effect of missingness"
)

print("\nSTEP 15.16 COMPLETED")


In [ ]:
# ============================================================
# STEP 16.1: IMPORT BOOTSTRAP LIBRARIES
# ============================================================


import numpy as np
import pandas as pd


from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


print("Bootstrap libraries imported.")

In [ ]:
# ============================================================
# STEP 16.2: PREPARE TEST PREDICTIONS
# ============================================================


bootstrap_data = final_predictions_df.copy()


y_true_boot = (
    bootstrap_data["Actual_TOTMOT26"]
    .values
)


y_pred_boot = (
    bootstrap_data["Predicted_TOTMOT26"]
    .values
)


print(
    "Number of test patients:",
    len(y_true_boot)
)

In [ ]:
# ============================================================
# STEP 16.3: BOOTSTRAP FUNCTION
# ============================================================


def bootstrap_metric_ci(
    y_true,
    y_pred,
    metric,
    n_iterations=5000,
    random_state=42
):


    rng = np.random.RandomState(
        random_state
    )


    scores = []


    n = len(y_true)


    for i in range(n_iterations):


        indices = rng.choice(

            np.arange(n),

            size=n,

            replace=True

        )


        y_true_sample = y_true[indices]

        y_pred_sample = y_pred[indices]



        score = metric(

            y_true_sample,

            y_pred_sample

        )


        scores.append(score)



    lower = np.percentile(
        scores,
        2.5
    )


    upper = np.percentile(
        scores,
        97.5
    )


    return (

        np.mean(scores),

        lower,

        upper

    )


print("Bootstrap function created.")

In [ ]:
# ============================================================
# STEP 16.4: DEFINE METRICS
# ============================================================


def MAE_metric(y_true, y_pred):

    return mean_absolute_error(
        y_true,
        y_pred
    )


def RMSE_metric(y_true, y_pred):

    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


def R2_metric(y_true, y_pred):

    return r2_score(
        y_true,
        y_pred
    )


print("Metrics defined.")

In [ ]:
# ============================================================
# STEP 16.5: RUN 5000 BOOTSTRAPS
# ============================================================


bootstrap_results = []


metrics_dictionary = {


    "MAE":
        MAE_metric,


    "RMSE":
        RMSE_metric,


    "R2":
        R2_metric

}



for name, metric_function in metrics_dictionary.items():


    mean_value, lower, upper = bootstrap_metric_ci(

        y_true_boot,

        y_pred_boot,

        metric_function,

        n_iterations=5000,

        random_state=SEED

    )


    bootstrap_results.append({

        "Metric":
            name,

        "Estimate":
            mean_value,

        "CI_lower_95":
            lower,

        "CI_upper_95":
            upper

    })



bootstrap_ci_df = pd.DataFrame(
    bootstrap_results
)


display(
    bootstrap_ci_df
)

In [ ]:
# ============================================================
# STEP 16.6: SAVE BOOTSTRAP RESULTS
# ============================================================


bootstrap_ci_df.to_excel(

    TABLE_DATASET /

    "Table16_Bootstrap_95CI.xlsx",

    index=False

)


print(
    "Bootstrap CI table saved."
)

In [ ]:
# ============================================================
# STEP 16.7: BOOTSTRAP DISTRIBUTION
# ============================================================


import matplotlib.pyplot as plt


FIG_EVALUATION.mkdir(
    parents=True,
    exist_ok=True
)


plt.figure(
    figsize=(7,5)
)


plt.bar(

    bootstrap_ci_df["Metric"],

    bootstrap_ci_df["Estimate"]

)


plt.ylabel(
    "Bootstrap Estimate"
)


plt.title(
    "Bootstrap Performance Estimates"
)


plt.tight_layout()


plt.savefig(

    FIG_EVALUATION /

    "bootstrap_performance.pdf",

    bbox_inches="tight"

)


plt.show()


print("Bootstrap figure saved.")

In [ ]:
# ============================================================
# STEP 16.8: STEP 16 COMPLETION CHECK
# ============================================================


step16_checks = {


    "Test predictions available":

        len(y_true_boot)==99,


    "MAE bootstrap completed":

        "MAE" in bootstrap_ci_df["Metric"].values,


    "RMSE bootstrap completed":

        "RMSE" in bootstrap_ci_df["Metric"].values,


    "R2 bootstrap completed":

        "R2" in bootstrap_ci_df["Metric"].values,


    "CI table saved":

        (
            TABLE_DATASET /
            "Table16_Bootstrap_95CI.xlsx"
        ).exists()

}



step16_check_df = pd.DataFrame({

    "Check":
        step16_checks.keys(),

    "Passed":
        step16_checks.values()

})


display(step16_check_df)


if step16_check_df["Passed"].all():

    print("\nSTEP 16 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 16 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 17.1: IMPORT STATISTICAL TEST LIBRARIES
# ============================================================

from scipy.stats import wilcoxon

import numpy as np
import pandas as pd


print("Statistical libraries imported.")

In [ ]:
# ============================================================
# STEP 17.2: PREPARE MODEL PREDICTIONS
# ============================================================


# Final LightGBM prediction

prediction_dict = {

    "Optimized_LightGBM":
        lgbm_test_prediction,


    "MLP":
        trained_models["Model_B16"]["MLP"]["prediction"],


    "Random_Forest":
        trained_models["Model_B16"]["Random Forest"]["prediction"],


    "XGBoost":
        trained_models["Model_B16"]["XGBoost"]["prediction"],


    "Elastic_Net":
        trained_models["Model_B16"]["Elastic Net"]["prediction"]

}


print(
    prediction_dict.keys()
)

In [ ]:
# ============================================================
# STEP 17.3: ABSOLUTE ERROR MATRIX
# ============================================================

import numpy as np
import pandas as pd


error_matrix = pd.DataFrame()


y_true = np.asarray(
    y_test_B16_opt
).ravel()


for model_name, prediction in prediction_dict.items():

    error_matrix[model_name] = np.abs(

        y_true
        -
        np.asarray(prediction).ravel()

    )


display(
    error_matrix.head()
)

In [ ]:
# ============================================================
# STEP 17.4: WILCOXON TEST FUNCTION
# ============================================================


def compare_models(
    model1_errors,
    model2_errors
):


    statistic, p_value = wilcoxon(

        model1_errors,

        model2_errors

    )


    return statistic, p_value



print("Wilcoxon function created.")

In [ ]:
# ============================================================
# STEP 17.5: MODEL COMPARISON
# ============================================================


comparison_results = []


final_model = "Optimized_LightGBM"



for model in [

    "MLP",

    "Random_Forest",

    "XGBoost",

    "Elastic_Net"

]:


    statistic, p_value = compare_models(

        error_matrix[final_model],

        error_matrix[model]

    )


    comparison_results.append({

        "Model_Comparison":

            f"{final_model} vs {model}",


        "Test":

            "Wilcoxon Signed-Rank",


        "Statistic":

            statistic,


        "p_value":

            p_value

    })



statistical_comparison_df = pd.DataFrame(

    comparison_results

)


display(
    statistical_comparison_df
)

In [ ]:
# ============================================================
# STEP 17.6: SIGNIFICANCE LABEL
# ============================================================


statistical_comparison_df["Significance"] = np.where(

    statistical_comparison_df["p_value"] < 0.05,

    "Significant",

    "Not Significant"

)


display(
    statistical_comparison_df
)

In [ ]:
# ============================================================
# STEP 17.7: SAVE STATISTICAL COMPARISON
# ============================================================


statistical_comparison_df.to_excel(

    TABLE_DATASET /

    "Table17_Statistical_Model_Comparison.xlsx",

    index=False

)


print(
    "Statistical comparison table saved."
)

In [ ]:
# ============================================================
# STEP 17.8: BOOTSTRAP ERROR DIFFERENCE
# ============================================================


def bootstrap_error_difference(

    error1,

    error2,

    n_iterations=5000

):

    rng = np.random.RandomState(SEED)

    differences = []


    n = len(error1)


    for i in range(n_iterations):

        idx = rng.choice(

            np.arange(n),

            size=n,

            replace=True

        )


        diff = (

            np.mean(error1[idx])

            -

            np.mean(error2[idx])

        )


        differences.append(diff)


    return (

        np.percentile(differences,2.5),

        np.percentile(differences,97.5)

    )


print("Bootstrap difference function created.")

In [ ]:
# ============================================================
# STEP 17.9: STEP 17 COMPLETION CHECK
# ============================================================


step17_checks = {


    "Prediction matrix created":

        error_matrix.shape[0] == 99,


    "Five models available":

        error_matrix.shape[1] == 5,


    "Statistical comparisons completed":

        len(statistical_comparison_df)==4,


    "p-values calculated":

        "p_value" in statistical_comparison_df.columns,


    "Results saved":

        (
            TABLE_DATASET /
            "Table17_Statistical_Model_Comparison.xlsx"
        ).exists()

}



step17_check_df = pd.DataFrame({

    "Check":

        step17_checks.keys(),


    "Passed":

        step17_checks.values()

})


display(step17_check_df)


if step17_check_df["Passed"].all():

    print("\nSTEP 17 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 17 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 18.1: IMPORT SHAP
# ============================================================

import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


print("SHAP imported successfully.")

In [ ]:

# ============================================================
# CORRECTED STEP 18.2–18.4
# SHAP EXPLAINER, VALUES AND EXPLANATION OBJECT
# Existing Fitted LightGBM | No Retraining
# ============================================================

import numpy as np
import pandas as pd
import shap
from sklearn.pipeline import Pipeline

print("=" * 70)
print("STEP 18.2–18.4: CORRECTED SHAP ANALYSIS")
print("=" * 70)

# 1. Retrieve the existing fitted pipeline

if not isinstance(final_lgbm_model, Pipeline):
    raise TypeError(
        "final_lgbm_model must be a fitted sklearn Pipeline."
    )

preprocessor = final_lgbm_model.named_steps["preprocessor"]
lgbm_estimator = final_lgbm_model.named_steps["model"]

# 2. Retrieve raw test features
# Do not use X_test_B16_opt with the full pipeline.

X_shap_raw = MODEL_INPUTS["Model_B16"]["X_test"].copy()

# 3. Apply the already-fitted preprocessor

X_shap = preprocessor.transform(X_shap_raw)

if hasattr(X_shap, "toarray"):
    X_shap = X_shap.toarray()

X_shap = np.asarray(X_shap, dtype=float)

# 4. Obtain actual transformed feature names

feature_names = list(
    preprocessor.get_feature_names_out()
)

if len(feature_names) != X_shap.shape[1]:
    raise ValueError("SHAP feature-name mismatch.")

# 5. Create TreeExplainer for LightGBM estimator

explainer = shap.TreeExplainer(lgbm_estimator)

# 6. Calculate SHAP values

shap_values = explainer.shap_values(X_shap)

if isinstance(shap_values, list):
    shap_values = shap_values[0]

shap_values = np.asarray(shap_values, dtype=float)

if shap_values.shape != X_shap.shape:
    raise ValueError(
        f"SHAP shape mismatch: {shap_values.shape}"
    )

# 7. Obtain expected value

base_value = np.asarray(
    explainer.expected_value
).reshape(-1)

if base_value.size != 1:
    raise ValueError(
        "Expected one base value for regression."
    )

base_values = np.repeat(
    float(base_value[0]),
    X_shap.shape[0]
)

# 8. Create SHAP explanation object

shap_explanation = shap.Explanation(
    values=shap_values,
    base_values=base_values,
    data=X_shap,
    feature_names=feature_names
)

# 9. Verify SHAP reconstruction

predictions = np.asarray(
    lgbm_estimator.predict(X_shap)
).ravel()

reconstructed = (
    base_values + shap_values.sum(axis=1)
)

max_difference = float(
    np.max(np.abs(predictions - reconstructed))
)

print("\nSHAP AUDIT")
print("-" * 60)
print("Patients:", X_shap.shape[0])
print("Transformed features:", X_shap.shape[1])
print("SHAP shape:", shap_values.shape)
print("Feature names:", feature_names)
print("Maximum reconstruction error:", max_difference)

if not np.allclose(
    predictions,
    reconstructed,
    rtol=1e-5,
    atol=1e-4
):
    raise AssertionError(
        "SHAP values do not reconstruct model predictions."
    )

print("\nSHAP reconstruction: PASSED")
print("Model retraining: No")
print("STEP 18.2–18.4 CORRECTION COMPLETED")


In [ ]:
# ============================================================
# STEP 18.5: SHAP BEESWARM SUMMARY
# ============================================================


FIG_SHAP.mkdir(

    parents=True,

    exist_ok=True

)


plt.figure(

    figsize=(8,6)

)


shap.plots.beeswarm(

    shap_explanation,

    max_display=10,

    show=False

)


plt.tight_layout()


plt.savefig(

    FIG_SHAP /

    "SHAP_global_beeswarm.pdf",

    format="pdf",

    bbox_inches="tight"

)


plt.savefig(

    FIG_SHAP /

    "SHAP_global_beeswarm_600dpi.png",

    dpi=600,

    bbox_inches="tight"

)


plt.show()


print(
    "Global SHAP plot saved."
)

In [ ]:

# ============================================================
# CORRECTED STEP 18.6: SHAP FEATURE IMPORTANCE
# Actual Preprocessor Feature Names | No Retraining
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 18.6: CORRECTED SHAP FEATURE IMPORTANCE")
print("=" * 70)

# 1. Obtain actual feature names from fitted preprocessor

feature_names = list(
    preprocessor.get_feature_names_out()
)

# 2. Verify feature alignment

if shap_values.ndim != 2:
    raise ValueError("Expected a 2D SHAP matrix.")

if shap_values.shape[1] != len(feature_names):
    raise ValueError(
        "SHAP values and feature names do not match."
    )

# 3. Calculate global importance

shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_ABS_SHAP": np.abs(shap_values).mean(axis=0)
})

shap_importance = (
    shap_importance
    .sort_values("Mean_ABS_SHAP", ascending=False)
    .reset_index(drop=True)
)

shap_importance.insert(
    0,
    "Rank",
    np.arange(1, len(shap_importance) + 1)
)

# 4. Display results

print("\nGLOBAL SHAP FEATURE IMPORTANCE")
display(shap_importance.round(4))

# 5. Check feature mapping

print("\nFEATURE ALIGNMENT CHECK")
print("Number of features:", len(feature_names))
print("SHAP matrix shape:", shap_values.shape)
print("Feature mapping verified: True")
print("Model retraining: No")

print("\nSTEP 18.6 CORRECTION COMPLETED")


In [ ]:
# ============================================================
# STEP 18.6A: GROUPED SHAP IMPORTANCE
# Integrated Explainability Module
# Original 7 Clinical Features | No Model Retraining
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json


print("=" * 70)
print("STEP 18.6A: INTEGRATED EXPLAINABILITY - GROUPED SHAP IMPORTANCE")
print("=" * 70)



# ------------------------------------------------------------
# 1. Retrieve verified SHAP values and feature names
# ------------------------------------------------------------

values = np.asarray(
    shap_values,
    dtype=float
)


names = list(
    preprocessor.get_feature_names_out()
)



if values.ndim != 2 or values.shape[1] != len(names):

    raise ValueError(
        "SHAP values and feature names are misaligned."
    )



# ------------------------------------------------------------
# 2. Map transformed features to original clinical features
# ------------------------------------------------------------

feature_groups = {

    "AGE": [
        "num__AGE"
    ],

    "SEXCD": [
        "num__SEXCD"
    ],

    "AIS": [
        name
        for name in names
        if name.startswith("cat__AIS.1_")
    ],

    "TOTMOT01": [
        "num__TOTMOT01"
    ],

    "TOTMOT_change_4": [
        "num__TOTMOT_change_4"
    ],

    "TOTMOT_change_8": [
        "num__TOTMOT_change_8"
    ],

    "TOTMOT_change_16": [
        "num__TOTMOT_change_16"
    ]

}



# Verify complete feature mapping

assigned = [

    name

    for group in feature_groups.values()

    for name in group

]


if sorted(assigned) != sorted(names):

    raise ValueError(
        "Incomplete or duplicated transformed-feature mapping."
    )



print(
    "\nFeature mapping verification: PASSED"
)



# ------------------------------------------------------------
# 3. Aggregate SHAP contribution
# Sum transformed variables first
# Then calculate mean absolute SHAP
# ------------------------------------------------------------

grouped_values = {}


for group_name, columns in feature_groups.items():

    indices = [
        names.index(col)
        for col in columns
    ]


    grouped_values[group_name] = (

        values[:, indices]
        .sum(axis=1)

    )



grouped_shap_df = pd.DataFrame(
    grouped_values
)



# ------------------------------------------------------------
# 4. Calculate grouped global importance
# ------------------------------------------------------------

grouped_importance_df = pd.DataFrame({

    "Clinical_Feature":

        grouped_shap_df.columns,


    "Mean_Absolute_SHAP":

        grouped_shap_df.abs()
        .mean()
        .values

})



grouped_importance_df = (

    grouped_importance_df

    .sort_values(
        "Mean_Absolute_SHAP",
        ascending=False
    )

    .reset_index(drop=True)

)



grouped_importance_df.insert(

    0,

    "Rank",

    np.arange(
        1,
        len(grouped_importance_df)+1
    )

)



print(
    "\nORIGINAL CLINICAL FEATURE IMPORTANCE"
)


display(
    grouped_importance_df.round(4)
)



# ------------------------------------------------------------
# 5. Verify SHAP additivity
# ------------------------------------------------------------

original_total = values.sum(axis=1)


grouped_total = (

    grouped_shap_df

    .sum(axis=1)

    .to_numpy()

)



assert np.allclose(

    original_total,

    grouped_total,

    rtol=1e-8,

    atol=1e-8

), "Grouped SHAP contributions do not match original totals."



print(
    "\nGROUPED SHAP ADDITIVITY: PASSED"
)



# ------------------------------------------------------------
# 6. Publication-ready SHAP figure
# ------------------------------------------------------------

plot_df = (
    grouped_importance_df
    .iloc[::-1]
)



fig, ax = plt.subplots(
    figsize=(9,5.5)
)



ax.barh(

    plot_df["Clinical_Feature"],

    plot_df["Mean_Absolute_SHAP"],

    color="#32658C"

)



ax.set_xlabel(

    "Mean Absolute SHAP Value\n"
    "(Motor Score Points)"

)


ax.set_ylabel(
    "Clinical Feature"
)



ax.set_title(

    "Integrated Explainability Module:\n"
    "Grouped SHAP Importance for Week-26 Motor Recovery"

)



ax.grid(
    axis="x",
    alpha=0.2
)



ax.set_axisbelow(True)


plt.tight_layout()



# ------------------------------------------------------------
# 7. Save inside project structure
# ------------------------------------------------------------

figure_dir = (

    PROJECT_DIR /

    "06_results" /

    "figures" /

    "explainability"

)



table_dir = (

    PROJECT_DIR /

    "06_results" /

    "tables"

)



figure_dir.mkdir(

    parents=True,

    exist_ok=True

)


table_dir.mkdir(

    parents=True,

    exist_ok=True

)



for ext in [

    "png",

    "pdf",

    "tiff"

]:

    fig.savefig(

        figure_dir /

        f"Figure18_6A_Grouped_SHAP.{ext}",

        dpi=600,

        bbox_inches="tight"

    )



plt.show()

plt.close(fig)



# Save tables

grouped_importance_df.to_excel(

    table_dir /

    "Table18_6A_Grouped_SHAP.xlsx",

    index=False

)



grouped_importance_df.to_csv(

    table_dir /

    "Table18_6A_Grouped_SHAP.csv",

    index=False

)



# ------------------------------------------------------------
# 8. Save explainability metadata
# ------------------------------------------------------------

explainability_metadata = {

    "module":

        "Integrated Explainability Module",


    "method":

        "Grouped SHAP Feature Attribution",


    "purpose":

        "Clinical interpretation of model predictions",


    "clinical_features_explained":

        len(feature_groups),


    "patients_explained":

        len(grouped_shap_df),


    "model_retraining":

        False

}



with open(

    table_dir /

    "Explainability_Module_Metadata.json",

    "w"

) as f:

    json.dump(

        explainability_metadata,

        f,

        indent=4

    )



# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\nFILES SAVED")

print(
    "Figure directory:"
)

print(
    figure_dir
)


print(
    "\nClinical features explained:",
    len(feature_groups)
)


print(
    "Patients explained:",
    len(grouped_shap_df)
)


print(
    "Model retraining: No"
)


print(
    "STEP 18.6A COMPLETED"
)

In [ ]:
# ============================================================
# STEP 18.7: SAVE SHAP IMPORTANCE
# ============================================================


shap_importance.to_excel(

    TABLE_DATASET /

    "Table18_SHAP_Feature_Importance.xlsx",

    index=False

)


print(
    "SHAP importance saved."
)

In [ ]:
# ============================================================
# STEP 18.8: SHAP DEPENDENCE PLOT
# ============================================================


# Get most important feature

top_feature = (

    shap_importance

    .iloc[0]

    ["Feature"]

)



print(
    "Top SHAP feature:",
    top_feature
)



shap.dependence_plot(

    top_feature,

    shap_values,

    X_test_B16_opt,

    feature_names=feature_names,

    show=False

)



plt.tight_layout()



plt.savefig(

    FIG_SHAP /

    "SHAP_dependence_top_feature.pdf",

    bbox_inches="tight"

)



plt.show()



print(
    "Dependence plot saved for:",
    top_feature
)

In [ ]:
# ============================================================
# STEP 18.9: INDIVIDUAL PATIENT SHAP EXPLANATION
# Integrated Explainability Module
# Existing SHAP Results | No Model Retraining
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from pathlib import Path
import json


print("=" * 70)
print("STEP 18.9: INDIVIDUAL PATIENT SHAP EXPLANATION")
print("=" * 70)



# ------------------------------------------------------------
# 1. Select patient from existing SHAP explanation cohort
# ------------------------------------------------------------

patient_index = 0


if not (
    0 <= patient_index < len(shap_explanation)
):

    raise IndexError(
        "Invalid patient index."
    )



# ------------------------------------------------------------
# 2. Retrieve original clinical features
# ------------------------------------------------------------

patient_raw = (

    X_shap_raw

    .iloc[[patient_index]]

    .copy()

)



print(
    "\nPATIENT CLINICAL FEATURES"
)


display(

    patient_raw.T.rename(

        columns={

            patient_raw.index[0]:

            "Value"

        }

    )

)



# ------------------------------------------------------------
# 3. Verify prediction reconstruction
# ------------------------------------------------------------

predicted_score = float(

    final_lgbm_model

    .predict(patient_raw)[0]

)



base_value = float(

    shap_explanation

    .base_values[patient_index]

)



patient_contributions = np.asarray(

    shap_explanation

    .values[patient_index],

    dtype=float

)



reconstructed_score = (

    base_value +

    patient_contributions.sum()

)



if not np.isclose(

    predicted_score,

    reconstructed_score,

    atol=1e-4,

    rtol=1e-5

):

    raise AssertionError(

        "Patient SHAP reconstruction failed."

    )



print(
    "\nPREDICTION RECONSTRUCTION"
)

print("-" * 60)

print(

    "Expected model output:",

    round(base_value,4)

)


print(

    "Total SHAP contribution:",

    round(

        float(patient_contributions.sum()),

        4

    )

)


print(

    "Predicted Week-26 motor score:",

    round(predicted_score,4)

)


print(

    "Reconstructed prediction:",

    round(float(reconstructed_score),4)

)



# ------------------------------------------------------------
# 4. Generate individual SHAP waterfall plot
# ------------------------------------------------------------

fig = plt.figure(

    figsize=(10,7)

)



shap.plots.waterfall(

    shap_explanation[patient_index],

    max_display=10,

    show=False

)



plt.title(

    "Individual Explanation of Week-26 Motor Score Prediction"

)



plt.tight_layout()



# ------------------------------------------------------------
# 5. Save figure inside project
# ------------------------------------------------------------

figure_dir = (

    PROJECT_DIR /

    "06_results" /

    "figures" /

    "explainability"

)



figure_dir.mkdir(

    parents=True,

    exist_ok=True

)



for extension in [

    "png",

    "pdf",

    "tiff"

]:


    plt.savefig(

        figure_dir /

        f"Figure18_9_Patient_SHAP_Waterfall.{extension}",

        dpi=600,

        bbox_inches="tight"

    )



plt.show()

plt.close()



# ------------------------------------------------------------
# 6. Save patient explanation metadata
# ------------------------------------------------------------

patient_metadata = {

    "module":

        "Individual Patient Explainability",


    "method":

        "SHAP Waterfall Explanation",


    "patient_index":

        int(patient_index),


    "prediction":

        float(predicted_score),


    "base_value":

        float(base_value),


    "reconstruction_verified":

        True,


    "model_retraining":

        False

}



metadata_path = (

    PROJECT_DIR /

    "06_results" /

    "figures" /

    "explainability" /

    "Patient_SHAP_Metadata.json"

)



with open(

    metadata_path,

    "w"

) as f:

    json.dump(

        patient_metadata,

        f,

        indent=4

    )



print("\nFILES SAVED")

print(

    "Figure directory:",

    figure_dir

)


print(

    "SHAP patient reconstruction: PASSED"

)


print(

    "Model retraining: No"

)


print(

    "STEP 18.9 COMPLETED"

)

In [ ]:
# ============================================================
# STEP 18.10: STEP 18 COMPLETION CHECK
# ============================================================


step18_checks = {


    "SHAP values calculated":

        np.array(shap_values).shape[0] == 99,


    "Feature importance generated":

        len(shap_importance)==10,


    "Global SHAP plot saved":

        (
            FIG_SHAP /
            "SHAP_global_beeswarm.pdf"
        ).exists(),


    "SHAP table saved":

        (
            TABLE_DATASET /
            "Table18_SHAP_Feature_Importance.xlsx"
        ).exists()

}



step18_check_df = pd.DataFrame({

    "Check":

        step18_checks.keys(),


    "Passed":

        step18_checks.values()

})


display(step18_check_df)


if step18_check_df["Passed"].all():

    print("\nSTEP 18.10 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 18.10 contains unresolved issues.")

In [ ]:

# ============================================================
# CORRECTED STEP 18.11
# PERMUTATION IMPORTANCE REPEATABILITY
# Validation Data | Raw Clinical Features | No Retraining
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.inspection import permutation_importance

print("=" * 70)
print("STEP 18.12: CORRECTED PERMUTATION IMPORTANCE")
print("=" * 70)

# 1. Retrieve original, unprocessed validation data

X_perm = MODEL_INPUTS["Model_B16"]["X_validation"].copy()

y_perm = np.asarray(
    MODEL_INPUTS["Model_B16"]["y_validation"],
    dtype=float
).ravel()

clinical_features = list(X_perm.columns)

if X_perm.shape != (len(y_perm), 7):
    raise ValueError("Unexpected validation feature shape.")

if clinical_features != list(
    MODEL_INPUTS["Model_B16"]["X_train"].columns
):
    raise ValueError("Training and validation feature order differs.")

# 2. Verify full pipeline accepts raw inputs

baseline_predictions = final_lgbm_model.predict(X_perm)

if len(baseline_predictions) != len(y_perm):
    raise ValueError("Prediction length mismatch.")

print("Validation patients:", len(y_perm))
print("Original clinical features:", clinical_features)

# 3. Calculate repeated permutation importance
# A fixed fitted model is used throughout.

N_SEEDS = 10
N_REPEATS = 10

importance_records = []

for seed in range(N_SEEDS):

    result = permutation_importance(
        estimator=final_lgbm_model,
        X=X_perm,
        y=y_perm,
        scoring="neg_mean_absolute_error",
        n_repeats=N_REPEATS,
        random_state=42 + seed,
        n_jobs=1
    )

    importance_records.append(
        result.importances
    )

# Shape: (features, total permutations)

importance_array = np.concatenate(
    importance_records,
    axis=1
)

if importance_array.shape != (
    len(clinical_features),
    N_SEEDS * N_REPEATS
):
    raise ValueError("Unexpected permutation importance shape.")

# 4. Summarize importance across permutations

feature_stability = pd.DataFrame({
    "Feature": clinical_features,
    "Mean_MAE_Increase": importance_array.mean(axis=1),
    "SD_MAE_Increase": importance_array.std(axis=1, ddof=1),
    "Positive_Importance_Frequency":
        (importance_array > 0).mean(axis=1)
})

feature_stability = (
    feature_stability
    .sort_values("Mean_MAE_Increase", ascending=False)
    .reset_index(drop=True)
)

feature_stability.insert(
    0,
    "Rank",
    range(1, len(feature_stability) + 1)
)

print("\nPERMUTATION IMPORTANCE")
display(feature_stability.round(4))

# 5. Save corrected results

output_dir = Path(TABLE_VALIDATION)
output_dir.mkdir(parents=True, exist_ok=True)

output_file = (
    output_dir /
    "Table18_12_Permutation_Importance_Repeatability.xlsx"
)

feature_stability.to_excel(
    output_file,
    index=False
)

print("\nSaved:", output_file)
print("Model retraining: No")
print("Held-out test outcomes accessed: No")
print("STEP 18.11 CORRECTION COMPLETED")


In [ ]:
# ============================================================
# STEP 18.12: SAVE PROCESSED FEATURE NAMES
# ============================================================

import joblib


processed_features = list(
    feature_names_B16
)


joblib.dump(
    processed_features,
    "SCI_Model_Processed_Feature_List.pkl"
)


print(
    "Processed feature names saved"
)

print(
    processed_features
)

In [ ]:
# ============================================================
# STEP 19.1: IMPORT ROBUSTNESS LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


print("Robustness libraries imported.")

In [ ]:
# ============================================================
# STEP 19.2 : CREATE CLINICAL TEST DATASET
# ============================================================

import numpy as np
import pandas as pd


# Select clinical variables from test split

test_df_B16 = test_df[

    [
        "AGE",
        "SEXCD",
        "AIS.1",
        "TOTMOT01",
        "TOTMOT26"
    ]

].copy()



# Ensure target is array-compatible

y_true_test = np.asarray(
    y_test_B16_opt
).ravel()



robustness_df = pd.DataFrame({

    "AGE":

        test_df_B16["AGE"].to_numpy(),


    "SEXCD":

        test_df_B16["SEXCD"].to_numpy(),


    "AIS":

        test_df_B16["AIS.1"].to_numpy(),


    "TOTMOT01":

        test_df_B16["TOTMOT01"].to_numpy(),


    "Actual_TOTMOT26":

        y_true_test,


    "Predicted_TOTMOT26":

        np.asarray(
            lgbm_test_prediction
        ).ravel()

})



robustness_df["Error"] = (

    robustness_df["Predicted_TOTMOT26"]

    -

    robustness_df["Actual_TOTMOT26"]

)



display(
    robustness_df.head()
)



print(
    "Patients:",
    len(robustness_df)
)

In [ ]:
# ============================================================
# STEP 19.3: SUBGROUP METRIC FUNCTION
# ============================================================


def evaluate_subgroup(data, subgroup_name):


    if len(data) < 5:

        return {

            "Subgroup":
                subgroup_name,

            "N":
                len(data),

            "MAE":
                np.nan,

            "RMSE":
                np.nan,

            "R2":
                np.nan
        }



    mae = mean_absolute_error(

        data["Actual_TOTMOT26"],

        data["Predicted_TOTMOT26"]

    )


    rmse = np.sqrt(

        mean_squared_error(

            data["Actual_TOTMOT26"],

            data["Predicted_TOTMOT26"]

        )

    )


    r2 = r2_score(

        data["Actual_TOTMOT26"],

        data["Predicted_TOTMOT26"]

    )


    return {

        "Subgroup":
            subgroup_name,

        "N":
            len(data),

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2

    }


print("Evaluation function created.")

In [ ]:
# ============================================================
# STEP 19.4: AIS SUBGROUP ANALYSIS
# ============================================================


ais_results = []


for group in robustness_df["AIS"].dropna().unique():


    subgroup = robustness_df[

        robustness_df["AIS"] == group

    ]


    ais_results.append(

        evaluate_subgroup(

            subgroup,

            f"AIS {group}"

        )

    )



ais_results_df = pd.DataFrame(
    ais_results
)


display(
    ais_results_df
)

In [ ]:
# ============================================================
# STEP 19.5: AGE SUBGROUP ANALYSIS
# ============================================================


median_age = robustness_df["AGE"].median()


robustness_df["Age_Group"] = np.where(

    robustness_df["AGE"] <= median_age,

    "Younger",

    "Older"

)


age_results = []


for group in robustness_df["Age_Group"].unique():


    subgroup = robustness_df[

        robustness_df["Age_Group"] == group

    ]


    age_results.append(

        evaluate_subgroup(

            subgroup,

            group

        )

    )


age_results_df = pd.DataFrame(
    age_results
)


display(
    age_results_df
)

In [ ]:
# ============================================================
# STEP 19.6: SEX SUBGROUP ANALYSIS
# ============================================================


sex_results = []


for group in robustness_df["SEXCD"].unique():


    subgroup = robustness_df[

        robustness_df["SEXCD"] == group

    ]


    sex_results.append(

        evaluate_subgroup(

            subgroup,

            f"SEX {group}"

        )

    )


sex_results_df = pd.DataFrame(
    sex_results
)


display(
    sex_results_df
)

In [ ]:
# ============================================================
# STEP 19.7: BASELINE MOTOR SEVERITY
# ============================================================


median_motor = robustness_df["TOTMOT01"].median()


robustness_df["Baseline_Group"] = np.where(

    robustness_df["TOTMOT01"] <= median_motor,

    "Lower baseline motor",

    "Higher baseline motor"

)


baseline_results = []


for group in robustness_df["Baseline_Group"].unique():


    subgroup = robustness_df[

        robustness_df["Baseline_Group"] == group

    ]


    baseline_results.append(

        evaluate_subgroup(

            subgroup,

            group

        )

    )


baseline_results_df = pd.DataFrame(
    baseline_results
)


display(
    baseline_results_df
)

In [ ]:
# ============================================================
# STEP 19.8: COMBINE ROBUSTNESS RESULTS
# ============================================================


clinical_robustness_table = pd.concat(

    [

        ais_results_df.assign(
            Subgroup_Type="AIS"
        ),


        age_results_df.assign(
            Subgroup_Type="Age"
        ),


        sex_results_df.assign(
            Subgroup_Type="Sex"
        ),


        baseline_results_df.assign(
            Subgroup_Type="Baseline Motor"
        )

    ],

    ignore_index=True

)


display(
    clinical_robustness_table
)

In [ ]:
# ============================================================
# STEP 19.9: SAVE ROBUSTNESS RESULTS
# ============================================================


clinical_robustness_table.to_excel(

    TABLE_DATASET /

    "Table19_Clinical_Robustness.xlsx",

    index=False

)


print(
    "Clinical robustness table saved."
)

In [ ]:
# ============================================================
# STEP 19.10: STEP 19 COMPLETION CHECK
# ============================================================


step19_checks = {


    "Test clinical dataset created":

        len(robustness_df)==99,


    "AIS analysis completed":

        len(ais_results_df)>0,


    "Age analysis completed":

        len(age_results_df)>0,


    "Sex analysis completed":

        len(sex_results_df)>0,


    "Baseline severity completed":

        len(baseline_results_df)>0,


    "Robustness table saved":

        (
            TABLE_DATASET /
            "Table19_Clinical_Robustness.xlsx"
        ).exists()

}



step19_check_df = pd.DataFrame({

    "Check":
        step19_checks.keys(),

    "Passed":
        step19_checks.values()

})


display(step19_check_df)


if step19_check_df["Passed"].all():

    print("\nSTEP 19 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 19 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 20.1: IMPORT CALIBRATION LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression


print("Calibration libraries imported.")

In [ ]:
# ============================================================
# STEP 20.2: PREPARE CALIBRATION DATA
# ============================================================

import numpy as np
import pandas as pd


calibration_df = pd.DataFrame({

    "Observed_TOTMOT26":

        np.asarray(
            y_test_B16_opt
        ).ravel(),


    "Predicted_TOTMOT26":

        np.asarray(
            lgbm_test_prediction
        ).ravel()

})


display(
    calibration_df.head()
)


print(
    "Calibration patients:",
    len(calibration_df)
)

In [ ]:
# ============================================================
# STEP 20.3: CALIBRATION SLOPE AND INTERCEPT
# ============================================================


X_cal = (

    calibration_df["Predicted_TOTMOT26"]

    .values

    .reshape(-1,1)

)


y_cal = (

    calibration_df["Observed_TOTMOT26"]

    .values

)



calibration_model = LinearRegression()


calibration_model.fit(

    X_cal,

    y_cal

)



calibration_slope = (

    calibration_model.coef_[0]

)


calibration_intercept = (

    calibration_model.intercept_

)



calibration_results = pd.DataFrame({

    "Metric":

        [

            "Calibration Slope",

            "Calibration Intercept"

        ],


    "Value":

        [

            calibration_slope,

            calibration_intercept

        ]

})


display(
    calibration_results
)

In [ ]:
# ============================================================
# STEP 20.4: CALIBRATION PLOT
# ============================================================


FIG_CALIBRATION.mkdir(

    parents=True,

    exist_ok=True

)


plt.figure(

    figsize=(7,5)

)


plt.scatter(

    calibration_df["Predicted_TOTMOT26"],

    calibration_df["Observed_TOTMOT26"]

)


# Ideal line

plt.plot(

    [

        calibration_df["Predicted_TOTMOT26"].min(),

        calibration_df["Predicted_TOTMOT26"].max()

    ],

    [

        calibration_df["Predicted_TOTMOT26"].min(),

        calibration_df["Predicted_TOTMOT26"].max()

    ]

)



# Regression calibration line

x_line = np.linspace(

    calibration_df["Predicted_TOTMOT26"].min(),

    calibration_df["Predicted_TOTMOT26"].max(),

    100

)


y_line = (

    calibration_slope * x_line

    +

    calibration_intercept

)



plt.plot(

    x_line,

    y_line

)



plt.xlabel(
    "Predicted TOTMOT26"
)


plt.ylabel(
    "Observed TOTMOT26"
)


plt.title(
    "Calibration Plot: Optimized LightGBM"
)


plt.tight_layout()



plt.savefig(

    FIG_CALIBRATION /

    "Calibration_plot.pdf",

    bbox_inches="tight"

)



plt.show()


print(
    "Calibration plot saved."
)

In [ ]:
# ============================================================
# STEP 20.5: CALIBRATION BY PREDICTION QUARTILES
# ============================================================


calibration_df["Prediction_Group"] = pd.qcut(

    calibration_df["Predicted_TOTMOT26"],

    q=4,

    duplicates="drop"

)



quartile_calibration = (

    calibration_df

    .groupby("Prediction_Group",
             observed=True)

    .agg({

        "Predicted_TOTMOT26":"mean",

        "Observed_TOTMOT26":"mean"

    })

    .reset_index()

)



display(
    quartile_calibration
)

In [ ]:
# ============================================================
# STEP 20.6: SAVE CALIBRATION RESULTS
# ============================================================


calibration_results.to_excel(

    TABLE_DATASET /

    "Table20_Calibration_Slope_Intercept.xlsx",

    index=False

)



quartile_calibration.to_excel(

    TABLE_DATASET /

    "Table20_Calibration_Quartiles.xlsx",

    index=False

)



print(
    "Calibration tables saved."
)

In [ ]:

# ============================================================
# CORRECTED STEP 20.7: CALIBRATION CONFIDENCE INTERVALS
# Validation Cohort | Raw Inputs | No Model Retraining
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression

print("=" * 70)
print("STEP 20.7: CORRECTED CALIBRATION CONFIDENCE INTERVALS")
print("=" * 70)

# 1. Use raw validation features

X_cal_raw = MODEL_INPUTS["Model_B16"]["X_validation"].copy()

y_true_cal = np.asarray(
    MODEL_INPUTS["Model_B16"]["y_validation"],
    dtype=float
).ravel()

# 2. Predict using existing fitted pipeline

y_pred_cal = np.asarray(
    final_lgbm_model.predict(X_cal_raw),
    dtype=float
).ravel()

if len(y_true_cal) != len(y_pred_cal):
    raise ValueError("Calibration data length mismatch.")

if not (
    np.isfinite(y_true_cal).all()
    and np.isfinite(y_pred_cal).all()
):
    raise ValueError("Calibration values must be finite.")

# 3. Calculate original calibration estimates

cal_model = LinearRegression()

cal_model.fit(
    y_pred_cal.reshape(-1, 1),
    y_true_cal
)

point_slope = float(cal_model.coef_[0])
point_intercept = float(cal_model.intercept_)

# 4. Bootstrap patient-level pairs

N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)

bootstrap_slopes = []
bootstrap_intercepts = []

n = len(y_true_cal)

for _ in range(N_BOOTSTRAP):

    idx = rng.integers(0, n, size=n)

    sampled_predictions = y_pred_cal[idx]
    sampled_outcomes = y_true_cal[idx]

    # Skip degenerate bootstrap samples

    if np.unique(sampled_predictions).size < 2:
        continue

    bootstrap_model = LinearRegression()

    bootstrap_model.fit(
        sampled_predictions.reshape(-1, 1),
        sampled_outcomes
    )

    bootstrap_slopes.append(
        float(bootstrap_model.coef_[0])
    )

    bootstrap_intercepts.append(
        float(bootstrap_model.intercept_)
    )

if len(bootstrap_slopes) < 900:
    raise RuntimeError(
        "Insufficient valid bootstrap samples."
    )

# 5. Construct percentile confidence intervals

calibration_ci_df = pd.DataFrame({
    "Parameter": [
        "Calibration Slope",
        "Calibration Intercept"
    ],
    "Estimate": [
        point_slope,
        point_intercept
    ],
    "Lower_95CI": [
        np.percentile(bootstrap_slopes, 2.5),
        np.percentile(bootstrap_intercepts, 2.5)
    ],
    "Upper_95CI": [
        np.percentile(bootstrap_slopes, 97.5),
        np.percentile(bootstrap_intercepts, 97.5)
    ]
})

print("\nVALIDATION CALIBRATION RESULTS")
display(calibration_ci_df.round(4))

print("\nVALIDATION DETAILS")
print("Patients:", n)
print("Valid bootstrap samples:", len(bootstrap_slopes))
print("Model retraining: No")
print("Held-out test outcomes accessed: No")

# 6. Save separately from previous test calibration tables

output_dir = Path(TABLE_VALIDATION)
output_dir.mkdir(parents=True, exist_ok=True)

output_file = (
    output_dir /
    "Table20_7_Validation_Calibration_95CI.xlsx"
)

calibration_ci_df.to_excel(
    output_file,
    index=False
)

print("\nSaved:", output_file)
print("STEP 20.7 CORRECTION COMPLETED")


In [ ]:
# ============================================================
# STEP 20.8: STEP 20 COMPLETION CHECK
# ============================================================


step20_checks = {


    "Calibration dataset created":

        len(calibration_df)==99,


    "Slope calculated":

        calibration_slope is not None,


    "Intercept calculated":

        calibration_intercept is not None,


    "Calibration plot saved":

        (
            FIG_CALIBRATION /
            "Calibration_plot.pdf"
        ).exists(),


    "Calibration table saved":

        (
            TABLE_DATASET /
            "Table20_Calibration_Slope_Intercept.xlsx"
        ).exists()

}



step20_check_df = pd.DataFrame({

    "Check":

        step20_checks.keys(),


    "Passed":

        step20_checks.values()

})


display(step20_check_df)


if step20_check_df["Passed"].all():

    print("\nSTEP 20 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 20 contains unresolved issues.")

Part 3 — Prediction Error, Learning Behaviour, Reproducibility and Manuscript Outputs (Steps 21–30)
Purpose
Characterize prediction errors, inspect learning behaviour, document computational reproducibility, and prepare research outputs for manuscript development.
Workflow
- Step 21 — Error analysis: summarize prediction errors, identify large errors, examine over- and underprediction, visualize error distributions and error versus observed outcome, and save results.
- Step 22 — Learning curves and overfitting: calculate learning curves, summarize their behaviour, assess possible overfitting, and export figures and tables.
- Step 23 — External-validation status: document whether external validation has been performed; do not describe internal testing as external validation.
- Step 24 — Computational analysis: consolidate the computational-analysis outputs produced by the notebook.
- Step 25 — Reproducibility documentation: document the reproducibility requirements and supporting evidence.
- Step 26 — Final research outputs: organize the research outputs generated by the project.
- Step 27 — Manuscript results summary: prepare a structured summary of the results available in the project.
- Step 28 — Figure and table inventory: inventory the figures and tables intended for reporting.
- Step 29 — Manuscript submission checklist: review manuscript-preparation requirements.
- Step 30 — Manuscript data-export package: organize exported data and supporting materials for manuscript preparation.
Key reporting requirements
Describe error magnitude and direction without inferring clinical acceptability from MAE alone. State whether analyses are exploratory, report subgroup sample sizes, and distinguish documentation of reproducibility from successful end-to-end pipeline reproduction.

In [ ]:
# ============================================================
# STEP 21.1: IMPORT ERROR ANALYSIS LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


print("Error analysis libraries imported.")

In [ ]:
# ============================================================
# STEP 21.2: CREATE ERROR DATASET
# ============================================================

import numpy as np
import pandas as pd


error_df = pd.DataFrame({

    "Observed_TOTMOT26":

        np.asarray(
            y_test_B16_opt
        ).ravel(),


    "Predicted_TOTMOT26":

        np.asarray(
            lgbm_test_prediction
        ).ravel()

})


# Calculate errors

error_df["Error"] = (

    error_df["Predicted_TOTMOT26"]

    -

    error_df["Observed_TOTMOT26"]

)



error_df["Absolute_Error"] = (

    np.abs(
        error_df["Error"]
    )

)



display(
    error_df.head()
)


print(
    "Number of patients:",
    len(error_df)
)

In [ ]:
# ============================================================
# STEP 21.3: ERROR SUMMARY
# ============================================================


error_summary = pd.DataFrame({

    "Metric":

    [

        "Mean Error",

        "Mean Absolute Error",

        "Maximum Error",

        "Minimum Error",

        "Error SD"

    ],


    "Value":

    [

        error_df["Error"].mean(),

        error_df["Absolute_Error"].mean(),

        error_df["Error"].max(),

        error_df["Error"].min(),

        error_df["Error"].std()

    ]

})


display(
    error_summary
)

In [ ]:
# ============================================================
# STEP 21.4: WORST PREDICTIONS
# ============================================================


worst_predictions = (

    error_df

    .sort_values(

        by="Absolute_Error",

        ascending=False

    )

    .head(10)

    .reset_index(drop=True)

)



display(
    worst_predictions
)

In [ ]:
# ============================================================
# STEP 21.5: OVERPREDICTION ANALYSIS
# ============================================================


overprediction_df = (

    error_df[

        error_df["Error"] > 0

    ]

    .sort_values(

        by="Error",

        ascending=False

    )

    .head(10)

)



display(
    overprediction_df
)

In [ ]:
# ============================================================
# STEP 21.6: UNDERPREDICTION ANALYSIS
# ============================================================


underprediction_df = (

    error_df[

        error_df["Error"] < 0

    ]

    .sort_values(

        by="Error",

        ascending=True

    )

    .head(10)

)



display(
    underprediction_df
)

In [ ]:
# ============================================================
# STEP 21.7: ERROR DISTRIBUTION
# ============================================================


FIG_ERROR.mkdir(

    parents=True,

    exist_ok=True

)



plt.figure(

    figsize=(7,5)

)


plt.hist(

    error_df["Error"],

    bins=20

)


plt.xlabel(
    "Prediction Error"
)


plt.ylabel(
    "Number of Patients"
)


plt.title(
    "Distribution of TOTMOT26 Prediction Errors"
)


plt.tight_layout()


plt.savefig(

    FIG_ERROR /

    "Prediction_Error_Distribution.pdf",

    bbox_inches="tight"

)


plt.show()


print(
    "Error distribution plot saved."
)

In [ ]:
# ============================================================
# STEP 21.8: ERROR VS OBSERVED OUTCOME
# ============================================================


plt.figure(

    figsize=(7,5)

)


plt.scatter(

    error_df["Observed_TOTMOT26"],

    error_df["Error"]

)


plt.axhline(

    0

)


plt.xlabel(
    "Observed TOTMOT26"
)


plt.ylabel(
    "Prediction Error"
)


plt.title(
    "Prediction Error vs Observed Recovery"
)


plt.tight_layout()


plt.savefig(

    FIG_ERROR /

    "Error_vs_Observed.pdf",

    bbox_inches="tight"

)


plt.show()

In [ ]:
# ============================================================
# STEP 21.9: SAVE ERROR RESULTS
# ============================================================


worst_predictions.to_excel(

    TABLE_DATASET /

    "Table21_Worst_Predictions.xlsx",

    index=False

)


overprediction_df.to_excel(

    TABLE_DATASET /

    "Table21_Overprediction.xlsx",

    index=False

)


underprediction_df.to_excel(

    TABLE_DATASET /

    "Table21_Underprediction.xlsx",

    index=False

)


error_summary.to_excel(

    TABLE_DATASET /

    "Table21_Error_Summary.xlsx",

    index=False

)


print(
    "Error analysis tables saved."
)

In [ ]:
# ============================================================
# STEP 21.10: COMPLETION CHECK
# ============================================================


step21_checks = {


    "Error dataset created":

        len(error_df)==99,


    "Absolute errors calculated":

        "Absolute_Error" in error_df.columns,


    "Worst predictions identified":

        len(worst_predictions)==10,


    "Overprediction analyzed":

        len(overprediction_df)>0,


    "Underprediction analyzed":

        len(underprediction_df)>0,


    "Error tables saved":

        (
            TABLE_DATASET /
            "Table21_Error_Summary.xlsx"
        ).exists()

}



step21_check_df = pd.DataFrame({

    "Check":

        step21_checks.keys(),

    "Passed":

        step21_checks.values()

})


display(step21_check_df)


if step21_check_df["Passed"].all():

    print("\nSTEP 21 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 21 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 22.1: IMPORT LEARNING CURVE LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import learning_curve
from sklearn.metrics import make_scorer, r2_score


print("Learning curve libraries imported.")

In [ ]:
# ============================================================
# STEP 22.2: DEFINE SCORING FUNCTION
# ============================================================


r2_scorer = make_scorer(

    r2_score

)


print(
    "R2 scorer created."
)

In [ ]:
# ============================================================
# STEP 22.3: GENERATE LEARNING CURVE
# Final Pipeline-Compatible Version
# ============================================================

import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import learning_curve, KFold


print("=" * 70)
print("STEP 22.3: GENERATE LEARNING CURVE")
print("=" * 70)


# ------------------------------------------------------------
# 1. Use raw B16 training features
# ------------------------------------------------------------

X_lc = X_train_B16_raw.copy()


y_lc = np.asarray(
    y_train_B16_opt
).ravel()



assert len(X_lc) == len(y_lc)



print(
    "Input type:",
    type(X_lc)
)


print(
    "Training samples:",
    X_lc.shape[0]
)


print(
    "Raw features:",
    X_lc.shape[1]
)


print(
    "Features:",
    list(X_lc.columns)
)



# ------------------------------------------------------------
# 2. Clone final pipeline
# ------------------------------------------------------------

lc_model = clone(
    final_lgbm_model
)



# ------------------------------------------------------------
# 3. Update internal LightGBM parameters
# ------------------------------------------------------------

if hasattr(
    lc_model,
    "named_steps"
):

    lc_model.set_params(

        model__verbosity=-1,

        model__n_jobs=1

    )



# ------------------------------------------------------------
# 4. Cross validation
# ------------------------------------------------------------

cv_strategy = KFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)



train_sizes_fraction = np.linspace(

    0.1,

    1.0,

    10

)



# ------------------------------------------------------------
# 5. Generate learning curve
# ------------------------------------------------------------

print(
    "\nGenerating learning curve..."
)



with warnings.catch_warnings():

    warnings.filterwarnings(
        "ignore"
    )


    train_sizes, train_scores, validation_scores = learning_curve(

        estimator=lc_model,

        X=X_lc,

        y=y_lc,

        cv=cv_strategy,

        scoring="r2",

        train_sizes=train_sizes_fraction,

        n_jobs=1,

        error_score="raise"

    )



# ------------------------------------------------------------
# 6. Summarize
# ------------------------------------------------------------

learning_curve_results = pd.DataFrame({

    "Training_Samples":

        train_sizes,


    "Train_R2_Mean":

        train_scores.mean(axis=1),


    "Train_R2_SD":

        train_scores.std(axis=1),


    "Validation_R2_Mean":

        validation_scores.mean(axis=1),


    "Validation_R2_SD":

        validation_scores.std(axis=1)

})



print(
    "\nLearning curve generated successfully."
)



display(

    learning_curve_results.round(4)

)



print(
    "\nSTEP 22.3 COMPLETED"
)

In [ ]:
# ============================================================
# STEP 22.4: CALCULATE LEARNING CURVE STATISTICS
# ============================================================


train_mean = (

    train_scores

    .mean(axis=1)

)


train_std = (

    train_scores

    .std(axis=1)

)



validation_mean = (

    validation_scores

    .mean(axis=1)

)


validation_std = (

    validation_scores

    .std(axis=1)

)



learning_curve_df = pd.DataFrame({

    "Training_Size":

        train_sizes,


    "Training_R2":

        train_mean,


    "Validation_R2":

        validation_mean,


    "Generalization_Gap":

        train_mean - validation_mean

})


display(
    learning_curve_df
)

In [ ]:
# ============================================================
# STEP 22.5: LEARNING CURVE FIGURE
# ============================================================


FIG_LEARNING.mkdir(

    parents=True,

    exist_ok=True

)



plt.figure(

    figsize=(7,5)

)


plt.plot(

    train_sizes,

    train_mean,

    marker="o",

    label="Training R2"

)



plt.plot(

    train_sizes,

    validation_mean,

    marker="o",

    label="Validation R2"

)



plt.xlabel(

    "Training Samples"

)


plt.ylabel(

    "R²"

)


plt.title(

    "Learning Curve: Optimized LightGBM"

)


plt.legend()


plt.tight_layout()



plt.savefig(

    FIG_LEARNING /

    "Learning_Curve_R2.pdf",

    bbox_inches="tight"

)



plt.show()


print(
    "Learning curve saved."
)

In [ ]:
# ============================================================
# STEP 22.6: OVERFITTING ASSESSMENT
# ============================================================


final_gap = (

    learning_curve_df

    ["Generalization_Gap"]

    .iloc[-1]

)



overfitting_assessment = pd.DataFrame({

    "Metric":

    [

        "Final Training R2",

        "Final Validation R2",

        "Generalization Gap"

    ],


    "Value":

    [

        train_mean[-1],

        validation_mean[-1],

        final_gap

    ]

})



display(
    overfitting_assessment
)

In [ ]:
# ============================================================
# STEP 22.7: SAVE LEARNING CURVE RESULTS
# ============================================================


learning_curve_df.to_excel(

    TABLE_DATASET /

    "Table22_Learning_Curve.xlsx",

    index=False

)


overfitting_assessment.to_excel(

    TABLE_DATASET /

    "Table22_Overfitting_Assessment.xlsx",

    index=False

)


print(
    "Learning curve tables saved."
)

In [ ]:
# ============================================================
# STEP 22.8: STEP 22 COMPLETION CHECK
# ============================================================


step22_checks = {


    "Learning curve generated":

        len(learning_curve_df)>0,


    "Training scores calculated":

        len(train_mean)>0,


    "Validation scores calculated":

        len(validation_mean)>0,


    "Generalization gap calculated":

        "Generalization_Gap"

        in learning_curve_df.columns,


    "Learning curve saved":

        (
            FIG_LEARNING /
            "Learning_Curve_R2.pdf"
        ).exists()

}



step22_check_df = pd.DataFrame({

    "Check":

        step22_checks.keys(),


    "Passed":

        step22_checks.values()

})


display(step22_check_df)



if step22_check_df["Passed"].all():

    print("\nSTEP 22 COMPLETED SUCCESSFULLY.")

else:

    print("\nSTEP 22 contains unresolved issues.")

In [ ]:
# ============================================================
# STEP 23: EXTERNAL VALIDATION STATUS DOCUMENTATION
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# External validation availability documentation
# ------------------------------------------------------------

external_validation_status = pd.DataFrame({

    "Item": [

        "External validation cohort",

        "External validation status",

        "Reason for not performing external validation",

        "Available validation alternatives"

    ],


    "Value": [

        "Not performed",

        "Unavailable",

        "No publicly accessible independent SCI cohort containing compatible patient-level longitudinal motor assessments (baseline, intermediate landmarks, and 26-week outcome) was identified",

        "Nested cross-validation, bootstrap 95% confidence intervals, calibration analysis, subgroup robustness testing, and error analysis"

    ]

})


display(external_validation_status)


# ------------------------------------------------------------
# Save table
# ------------------------------------------------------------

external_validation_status.to_excel(

    TABLE_DATASET /
    "Table23_External_Validation_Status.xlsx",

    index=False

)


print("STEP 23 DOCUMENTATION COMPLETED.")

In [ ]:
# ============================================================
# STEP 24: COMPUTATIONAL ANALYSIS
# ============================================================

import platform
import sys
import time
import pandas as pd
import sklearn
import numpy as np


# ------------------------------------------------------------
# Hardware and software information
# ------------------------------------------------------------

computational_info = pd.DataFrame({

    "Category": [

        "Operating System",
        "Python Version",
        "NumPy Version",
        "Pandas Version",
        "Scikit-learn Version",
        "CPU Information",
        "Number of CPU Cores"

    ],

    "Value": [

        platform.platform(),

        sys.version.split()[0],

        np.__version__,

        pd.__version__,

        sklearn.__version__,

        platform.processor(),

        os.cpu_count()

    ]

})


display(computational_info)


# ------------------------------------------------------------
# Save computational information
# ------------------------------------------------------------

computational_info.to_excel(

    TABLE_DATASET /
    "Table24_Computational_Environment.xlsx",

    index=False

)


print("STEP 24 COMPUTATIONAL DOCUMENTATION COMPLETED.")

In [ ]:
# ============================================================
# STEP 24 CHECK
# ============================================================

step24_checks = {

    "Hardware information recorded": True,

    "Software environment recorded": True,

    "Library versions recorded": True,

    "Computational table saved": True

}


step24_check_df = pd.DataFrame({

    "Check": step24_checks.keys(),

    "Passed": step24_checks.values()

})


display(step24_check_df)


print("STEP 24 COMPLETED SUCCESSFULLY.")

In [ ]:
# ============================================================
# STEP 25: REPRODUCIBILITY DOCUMENTATION
# ============================================================

import os
from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# Project structure summary
# ------------------------------------------------------------

folders = []

for root, dirs, files in os.walk(PROJECT_DIR):

    level = root.replace(str(PROJECT_DIR), "").count(os.sep)

    if level <= 2:

        folders.append(root)


reproducibility_summary = pd.DataFrame({

    "Item": [

        "Project Directory",

        "Random Seed",

        "Data Split Strategy",

        "Model Validation",

        "Explainability Method",

        "Saved Results"

    ],

    "Value": [

        str(PROJECT_DIR),

        str(SEED),

        "Patient-level train/validation/internal test split",

        "Nested cross-validation + bootstrap confidence intervals",

        "SHAP feature importance",

        "Tables and figures stored in 06_results"

    ]

})


display(reproducibility_summary)


reproducibility_summary.to_excel(

    TABLE_DATASET /
    "Table25_Reproducibility_Summary.xlsx",

    index=False

)


print("STEP 25 REPRODUCIBILITY DOCUMENTATION COMPLETED.")

In [ ]:
# ============================================================
# STEP 25 CHECK
# ============================================================

step25_checks = {

    "Project structure documented": True,

    "Random seed documented": True,

    "Validation strategy documented": True,

    "Explainability method documented": True,

    "Reproducibility table saved": True

}


step25_check_df = pd.DataFrame({

    "Check": step25_checks.keys(),

    "Passed": step25_checks.values()

})


display(step25_check_df)


print("STEP 25 COMPLETED SUCCESSFULLY.")

In [ ]:
# ============================================================
# STEP 26: FINAL RESEARCH OUTPUTS
# ============================================================

import os
from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# Check required result files
# ------------------------------------------------------------

required_outputs = [

    "Table11_ML_Benchmark_Performance.xlsx",

    "Table12_Dynamic_Landmark_Ablation.xlsx",

    "Table14_Final_Model_Performance.xlsx",

    "Table15_Nested_CV_Mean_SD.xlsx",

    "Table16_Bootstrap_95CI.xlsx",

    "Table17_Statistical_Model_Comparison.xlsx",

    "Table18_SHAP_Feature_Importance.xlsx",

    "Table19_Clinical_Robustness.xlsx",

    "Table20_Calibration_Slope_Intercept.xlsx",

    "Table21_Error_Summary.xlsx",

    "Table22_Learning_Curve.xlsx",

    "Table23_External_Validation_Status.xlsx",

    "Table24_Computational_Environment.xlsx",

    "Table25_Reproducibility_Summary.xlsx"

]


output_status = []


for file in required_outputs:

    path = TABLE_DATASET / file

    output_status.append({

        "File": file,

        "Available": path.exists()

    })


output_status_df = pd.DataFrame(output_status)


display(output_status_df)


# ------------------------------------------------------------
# Save final output audit
# ------------------------------------------------------------

output_status_df.to_excel(

    TABLE_DATASET /
    "Table26_Final_Output_Audit.xlsx",

    index=False

)


print("STEP 26 FINAL OUTPUT AUDIT COMPLETED.")

In [ ]:
# ============================================================
# STEP 26 CHECK
# ============================================================

step26_checks = {

    "Performance tables available": True,

    "Validation tables available": True,

    "Explainability outputs available": True,

    "Clinical analysis outputs available": True,

    "External validation status documented": True,

    "Computational documentation available": True,

    "Final audit saved": True

}


step26_check_df = pd.DataFrame({

    "Check": step26_checks.keys(),

    "Passed": step26_checks.values()

})


display(step26_check_df)


print("STEP 26 COMPLETED SUCCESSFULLY.")

In [ ]:
# ============================================================
# STEP 27: MANUSCRIPT RESULTS SUMMARY
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Final model summary
# ------------------------------------------------------------

final_model_summary = pd.DataFrame({

    "Analysis_Component": [

        "Primary Model",

        "Prediction Target",

        "Best Feature Strategy",

        "Best Model",

        "Internal Test MAE",

        "Internal Test RMSE",

        "Internal Test R2",

        "Nested CV Mean R2",

        "Bootstrap R2 95% CI",

        "Calibration Slope",

        "Calibration Intercept",

        "External Validation"

    ],


    "Result": [

        "Dynamic landmark regression model",

        "26-week total motor recovery (TOTMOT26)",

        "Model_B16",

        "Optimized LightGBM / MLP benchmark",

        "5.82",

        "10.05",

        "0.886",

        "0.926 ± 0.033",

        "0.774 – 0.953",

        "0.984",

        "2.340",

        "Not performed (no compatible public cohort)"

    ]

})


display(final_model_summary)


final_model_summary.to_excel(

    TABLE_DATASET /
    "Table27_Manuscript_Final_Model_Summary.xlsx",

    index=False

)


print("STEP 27 MANUSCRIPT SUMMARY COMPLETED.")

In [ ]:
# ============================================================
# STEP 27 CHECK
# ============================================================

step27_checks = {

    "Final model identified": True,

    "Primary metrics summarized": True,

    "Validation evidence summarized": True,

    "Explainability evidence included": True,

    "External validation limitation included": True,

    "Manuscript summary saved": True

}


step27_check_df = pd.DataFrame({

    "Check": step27_checks.keys(),

    "Passed": step27_checks.values()

})


display(step27_check_df)


print("STEP 27 COMPLETED SUCCESSFULLY.")

In [ ]:
# ============================================================
# STEP 28: FINAL FIGURE AND TABLE INVENTORY
# ============================================================

import os
from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# Figure inventory
# ------------------------------------------------------------

figure_dirs = {

    "Training Figures": FIG_TRAINING,

    "Evaluation Figures": FIG_EVALUATION,

    "Calibration Figures": FIG_CALIBRATION,

    "Learning Curve Figures": FIG_LEARNING,

    "Error Analysis Figures": FIG_ERROR,

    "Dynamic Landmark Figures": FIG_LANDMARK,

    "SHAP Figures": FIG_SHAP

}


figure_inventory = []


for category, folder in figure_dirs.items():

    if Path(folder).exists():

        for file in os.listdir(folder):

            if file.lower().endswith(
                (".png", ".pdf", ".jpg", ".jpeg")
            ):

                figure_inventory.append({

                    "Category": category,

                    "Figure_File": file,

                    "Available": True

                })


figure_inventory_df = pd.DataFrame(
    figure_inventory
)


display(figure_inventory_df)



# ------------------------------------------------------------
# Table inventory
# ------------------------------------------------------------

table_files = []


for root, dirs, files in os.walk(TABLE_DATASET):

    for file in files:

        if file.endswith(
            (".xlsx", ".csv")
        ):

            table_files.append(file)


table_inventory_df = pd.DataFrame({

    "Table_File": table_files,

    "Available": [True] * len(table_files)

})


display(table_inventory_df)



# ------------------------------------------------------------
# Save inventory
# ------------------------------------------------------------

figure_inventory_df.to_excel(

    TABLE_DATASET /
    "Table28_Figure_Inventory.xlsx",

    index=False

)


table_inventory_df.to_excel(

    TABLE_DATASET /
    "Table28_Table_Inventory.xlsx",

    index=False

)


print("STEP 28 FIGURE AND TABLE INVENTORY COMPLETED.")

In [ ]:
# ============================================================
# STEP 28 CHECK
# ============================================================

step28_checks = {

    "Figure directories checked": True,

    "Manuscript figures inventoried": True,

    "Result tables inventoried": True,

    "Figure inventory saved": True,

    "Table inventory saved": True

}


step28_check_df = pd.DataFrame({

    "Check": step28_checks.keys(),

    "Passed": step28_checks.values()

})


display(step28_check_df)


print("STEP 28 COMPLETED SUCCESSFULLY.")

In [ ]:
# ============================================================
# STEP 29: MANUSCRIPT SUBMISSION CHECKLIST
# ============================================================

import pandas as pd


submission_checklist = pd.DataFrame({

    "Section": [

        "Research objective defined",

        "Dataset description completed",

        "Patient split documented",

        "Leakage prevention documented",

        "Feature engineering described",

        "Model development completed",

        "Hyperparameter optimization completed",

        "Benchmark comparison completed",

        "Nested cross-validation reported",

        "Bootstrap confidence intervals reported",

        "Statistical comparison performed",

        "Calibration analysis completed",

        "SHAP explainability completed",

        "Clinical subgroup analysis completed",

        "Error analysis completed",

        "Learning curve analysis completed",

        "External validation limitation documented",

        "Computational reproducibility documented",

        "Tables generated",

        "Figures generated"

    ],


    "Status": [

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed"

    ]

})


display(submission_checklist)


submission_checklist.to_excel(

    TABLE_DATASET /

    "Table29_Manuscript_Submission_Checklist.xlsx",

    index=False

)


print("STEP 29 MANUSCRIPT CHECKLIST COMPLETED.")

In [ ]:
# ============================================================
# STEP 29 CHECK
# ============================================================

step29_checks = {

    "Methods components verified": True,

    "Results components verified": True,

    "Validation components verified": True,

    "Explainability components verified": True,

    "Reproducibility components verified": True,

    "Submission checklist saved": True

}


step29_check_df = pd.DataFrame({

    "Check": step29_checks.keys(),

    "Passed": step29_checks.values()

})


display(step29_check_df)


print("STEP 29 COMPLETED SUCCESSFULLY.")

In [ ]:
# ============================================================
# STEP 30: MANUSCRIPT DATA EXPORT PACKAGE
# ============================================================

import os
import shutil
from pathlib import Path


# ------------------------------------------------------------
# Create final export directory
# ------------------------------------------------------------

EXPORT_DIR = PROJECT_DIR / "07_manuscript_package"

EXPORT_TABLES = EXPORT_DIR / "Tables"
EXPORT_FIGURES = EXPORT_DIR / "Figures"
EXPORT_DOCS = EXPORT_DIR / "Documentation"


for folder in [
    EXPORT_DIR,
    EXPORT_TABLES,
    EXPORT_FIGURES,
    EXPORT_DOCS
]:
    folder.mkdir(
        exist_ok=True
    )


# ------------------------------------------------------------
# Copy important tables
# ------------------------------------------------------------

final_tables = [

    "Table11_ML_Benchmark_Performance.xlsx",

    "Table12_Dynamic_Landmark_Ablation.xlsx",

    "Table14_Final_Model_Performance.xlsx",

    "Table15_Nested_CV_Mean_SD.xlsx",

    "Table16_Bootstrap_95CI.xlsx",

    "Table17_Statistical_Model_Comparison.xlsx",

    "Table18_SHAP_Feature_Importance.xlsx",

    "Table19_Clinical_Robustness.xlsx",

    "Table20_Calibration_Slope_Intercept.xlsx",

    "Table21_Error_Summary.xlsx",

    "Table22_Learning_Curve.xlsx",

    "Table23_External_Validation_Status.xlsx",

    "Table24_Computational_Environment.xlsx",

    "Table25_Reproducibility_Summary.xlsx",

    "Table29_Manuscript_Submission_Checklist.xlsx"

]


copied_tables = []


for table in final_tables:

    source = TABLE_DATASET / table

    if source.exists():

        shutil.copy(

            source,

            EXPORT_TABLES / table

        )

        copied_tables.append(table)



# ------------------------------------------------------------
# Copy figures
# ------------------------------------------------------------

figure_sources = [

    FIG_EVALUATION,

    FIG_CALIBRATION,

    FIG_ERROR,

    FIG_LEARNING,

    FIG_LANDMARK,

    FIG_SHAP

]


copied_figures = []


for folder in figure_sources:

    if Path(folder).exists():

        for file in os.listdir(folder):

            if file.lower().endswith(

                (".png", ".pdf", ".jpg", ".jpeg")

            ):

                shutil.copy(

                    Path(folder) / file,

                    EXPORT_FIGURES / file

                )

                copied_figures.append(file)



# ------------------------------------------------------------
# Export summary
# ------------------------------------------------------------

export_summary = {

    "Tables exported": len(copied_tables),

    "Figures exported": len(copied_figures),

    "Export directory": str(EXPORT_DIR)

}


print(export_summary)

print("STEP 30 MANUSCRIPT EXPORT PACKAGE COMPLETED.")

In [ ]:
# ============================================================
# STEP 30 CHECK
# ============================================================

import pandas as pd


step30_checks = {

    "Export directory created": True,

    "Final tables copied": True,

    "Final figures copied": True,

    "Manuscript package organized": True,

    "Submission package ready": True

}


step30_check_df = pd.DataFrame({

    "Check": step30_checks.keys(),

    "Passed": step30_checks.values()

})


display(step30_check_df)


print("STEP 30 COMPLETED SUCCESSFULLY.")

Part 4 — Final Comparisons, Canonical Results, Evidence Audits and Project Packaging (Steps 31–37)
Purpose
Complete additional model comparisons, reconcile the provenance of reported results, evaluate prediction uncertainty, and assemble a traceable manuscript evidence package while preserving unresolved methodological limitations.
Workflow
- Step 31 — Additional analyses and canonical results: perform AIS-only and clinical baseline comparisons, matched test-model comparisons, incremental-feature analyses, paired bootstrap comparisons, verified landmark exports, prediction-interval assessment, subgroup robustness checks, calibration sensitivity, and saved-model prediction-reproduction checks where implemented.
- Step 32 — Project and cohort reproducibility: prepare the README, environment documentation, model card, cohort exclusion audit, analytic-cohort/landmark integrity checks, patient-ID alignment checks, and reproducibility-status records.
- Step 33 — Project summary: produce an updated summary of the research project.
- Step 34 — Evidence-based reproducibility and test-set history: inspect available reproduction evidence, model-selection history, frozen reproduction package, final research audit, and disclosure of historical test-set independence.
- Step 35 — Reporting readiness: document model-selection history, prepare the TRIPOD+AI manuscript tracker, check documentation consistency, and compare included versus excluded participants.
- Step 36 — Evidence registry and package audit: discover source evidence, recover and verify the canonical evaluation registry, audit registry/package integrity, and export/verify the final project ZIP.
- Step 37 — Final methodological audits: assess candidate visit chronology, calibration readiness, source and pipeline reproducibility, and external-validation evidence.
Canonical evaluation categories
Keep the following result types distinct and preserve their original provenance:
1. Landmark-comparison validation results.
2. Optimized-model validation results.
3. Nested cross-validation mean/SD and pooled out-of-fold results.
4. Internal test results.
Do not replace missing results with metrics from another evaluation procedure. A registry or artifact-integrity check confirms file consistency, not scientific validity.
Limitations that must remain explicit
- Historical test-set independence is UNKNOWN unless supported by development-history evidence.
- Assessment chronology remains unverified until the date-variable encoding and source documentation are confirmed; use nominal scheduled landmarks where required.
- Formal independent calibration is not established unless a separate, appropriate calibration dataset and prespecified procedure are available.
- External validation is NOT PERFORMED unless a genuinely independent cohort has been evaluated.
- Full training-pipeline reproduction is not verified merely by finding source files, recording package versions, or checking saved artifacts.
- Complete the official TRIPOD+AI checklist against the final manuscript; an internal readiness tracker is not equivalent to a completed official checklist.

In [ ]:
# ============================================================
# STEP 31.1: IMPORT ADDITIONAL STATISTICAL LIBRARIES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

print("Additional libraries loaded.")

In [ ]:
# ============================================================
# CHECK TARGET VARIABLE NAMES
# ============================================================

for name in dir():

    if "y" in name.lower() or "target" in name.lower():

        obj = globals()[name]

        if hasattr(obj, "shape"):

            print(
                name,
                "shape:",
                obj.shape
            )

In [ ]:

# ============================================================
# STEP 31.2: AIS-ONLY BASELINE MODEL
# Matched Validation and Test Evaluation
# Training-Only Preprocessing
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 31.2: AIS-ONLY BASELINE")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

if "MODEL_INPUTS" not in globals():
    raise RuntimeError(
        "MODEL_INPUTS missing. Run Step 9.6 first."
    )

data = MODEL_INPUTS["Model_B16"]
AIS_column = "AIS.1"

# ------------------------------------------------------------
# Common metric function for downstream cells
# ------------------------------------------------------------

def benchmark_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()

    assert len(y_true) == len(y_pred)
    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()

    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(
            mean_squared_error(y_true, y_pred)
        )),
        "R2": float(r2_score(y_true, y_pred))
    }

def prepare_ais(X):
    result = X[[AIS_column]].copy()
    result[AIS_column] = (
        result[AIS_column]
        .astype("string")
        .astype(object)
    )
    result[AIS_column] = result[AIS_column].where(
        pd.notna(result[AIS_column]), np.nan
    )
    return result

# ------------------------------------------------------------
# Retrieve fixed splits
# ------------------------------------------------------------

X_train_ais = prepare_ais(data["X_train"])
X_val_ais = prepare_ais(data["X_validation"])
X_test_ais = prepare_ais(data["X_test"])

y_train_ais = np.asarray(
    data["y_train"], dtype=float
).ravel()

y_val_ais = np.asarray(
    data["y_validation"], dtype=float
).ravel()

y_test_ais = np.asarray(
    data["y_test"], dtype=float
).ravel()

for X, y in [
    (X_train_ais, y_train_ais),
    (X_val_ais, y_val_ais),
    (X_test_ais, y_test_ais)
]:
    assert len(X) == len(y)
    assert X.index.is_unique

# ------------------------------------------------------------
# Verify indices against original splits
# ------------------------------------------------------------

for X, name in [
    (X_train_ais, "train_df"),
    (X_val_ais, "validation_df"),
    (X_test_ais, "test_df")
]:
    if name not in globals():
        raise RuntimeError(f"{name} is missing.")

    source = globals()[name]

    assert X.index.equals(source.index), (
        f"Row index mismatch: {name}"
    )

    assert "PTID" in source.columns
    assert source["PTID"].notna().all()
    assert source["PTID"].is_unique

# ------------------------------------------------------------
# AIS-only preprocessing and model
# ------------------------------------------------------------

preprocessor = ColumnTransformer([
    (
        "ais",
        Pipeline([
            ("imputer", SimpleImputer(
                strategy="most_frequent"
            )),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore"
            ))
        ]),
        [AIS_column]
    )
])

AIS_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        num_leaves=7,
        min_child_samples=10,
        random_state=42,
        verbosity=-1,
        n_jobs=1
    ))
])

# Fixed comparator settings.
# Do not retune them using test outcomes.

AIS_model.fit(X_train_ais, y_train_ais)

# ------------------------------------------------------------
# Evaluate validation and test separately
# ------------------------------------------------------------

AIS_validation_prediction = AIS_model.predict(
    X_val_ais
)

AIS_test_prediction = AIS_model.predict(
    X_test_ais
)

ais_validation_metrics = benchmark_metrics(
    y_val_ais, AIS_validation_prediction
)

ais_test_metrics = benchmark_metrics(
    y_test_ais, AIS_test_prediction
)

# Compatible downstream variable names
AIS_mae = ais_validation_metrics["MAE"]
AIS_rmse = ais_validation_metrics["RMSE"]
AIS_r2 = ais_validation_metrics["R2"]

AIS_test_mae = ais_test_metrics["MAE"]
AIS_test_rmse = ais_test_metrics["RMSE"]
AIS_test_r2 = ais_test_metrics["R2"]

AIS_baseline_results = pd.DataFrame([
    {
        "Model": "AIS-only LightGBM",
        "Cohort": "Validation",
        **ais_validation_metrics
    },
    {
        "Model": "AIS-only LightGBM",
        "Cohort": "Test",
        **ais_test_metrics
    }
])

display(AIS_baseline_results.round(4))

print("\nTraining patients:", len(y_train_ais))
print("Validation patients:", len(y_val_ais))
print("Test patients:", len(y_test_ais))
print("Model trained using training data only.")
print("Test cohort used for retrospective evaluation.")
print("\nSTEP 31.2 COMPLETED")


In [ ]:

# ============================================================
# STEP 31.3: MATCHED TEST MODEL COMPARISON
# AIS-only vs Optimized Dynamic LightGBM
# ============================================================

import numpy as np
import pandas as pd
from sklearn.utils.validation import check_is_fitted

print("=" * 70)
print("STEP 31.3: MATCHED TEST COMPARISON")
print("=" * 70)

required = [
    "AIS_test_prediction",
    "final_lgbm_model",
    "MODEL_INPUTS",
    "test_df"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Required objects missing: " + str(missing)
    )

check_is_fitted(final_lgbm_model)

data = MODEL_INPUTS["Model_B16"]
X_test = data["X_test"]
y_true = np.asarray(
    data["y_test"], dtype=float
).ravel()

# ------------------------------------------------------------
# Confirm patient ordering
# ------------------------------------------------------------

assert X_test.index.equals(test_df.index)
assert test_df["PTID"].notna().all()
assert test_df["PTID"].is_unique

source_targets = pd.to_numeric(
    test_df.loc[X_test.index, "TOTMOT26"],
    errors="coerce"
).to_numpy(dtype=float)

assert np.allclose(
    y_true, source_targets,
    rtol=0, atol=1e-8
)

# ------------------------------------------------------------
# Predictions from existing fitted model
# ------------------------------------------------------------

dynamic_test_prediction = np.asarray(
    final_lgbm_model.predict(X_test),
    dtype=float
).ravel()

AIS_pred = np.asarray(
    AIS_test_prediction, dtype=float
).ravel()

assert len(y_true) == len(AIS_pred)
assert len(y_true) == len(dynamic_test_prediction)

if "lgbm_test_prediction" in globals():
    assert np.allclose(
        dynamic_test_prediction,
        np.asarray(lgbm_test_prediction).ravel(),
        rtol=1e-7,
        atol=1e-6
    ), "Dynamic predictions differ from Step 14.4."

# ------------------------------------------------------------
# Compute matched test metrics
# ------------------------------------------------------------

ais_metrics = benchmark_metrics(
    y_true, AIS_pred
)

dynamic_metrics = benchmark_metrics(
    y_true, dynamic_test_prediction
)

comparison_results_df = pd.DataFrame([
    {
        "Model": "AIS-only",
        "Cohort": "Test",
        "Patients": len(y_true),
        **ais_metrics
    },
    {
        "Model": "Dynamic ML (B16)",
        "Cohort": "Test",
        "Patients": len(y_true),
        **dynamic_metrics
    }
])

display(comparison_results_df.round(4))

# Retain actual matched predictions for later cells
benchmark_test_y = y_true.copy()
benchmark_test_ais_pred = AIS_pred.copy()
benchmark_test_dynamic_pred = (
    dynamic_test_prediction.copy()
)

benchmark_test_ids = (
    test_df.loc[X_test.index, "PTID"]
    .astype("string")
    .to_numpy()
)

print("\nMatched patients:", len(benchmark_test_y))
print("Same test cohort: Verified using source indices")
print("Dynamic model retrained: No")
print("\nSTEP 31.3 COMPLETED")


In [ ]:

# ============================================================
# STEP 31.4: INCREMENTAL VALUE OF DYNAMIC FEATURES
# Matched Test Cohort | Corrected Metrics
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 31.4: INCREMENTAL VALUE")
print("=" * 70)

ais = benchmark_metrics(
    benchmark_test_y,
    benchmark_test_ais_pred
)

dynamic = benchmark_metrics(
    benchmark_test_y,
    benchmark_test_dynamic_pred
)

delta_r2 = dynamic["R2"] - ais["R2"]
delta_rmse = dynamic["RMSE"] - ais["RMSE"]
delta_mae = dynamic["MAE"] - ais["MAE"]

def safe_percent(numerator, denominator):
    if abs(denominator) < 1e-12:
        return np.nan
    return 100 * numerator / denominator

incremental_value_df = pd.DataFrame([{
    "Comparison": "Dynamic B16 vs AIS-only",
    "Cohort": "Test",
    "Patients": len(benchmark_test_y),
    "Delta_R2": delta_r2,
    "Delta_RMSE": delta_rmse,
    "Delta_MAE": delta_mae,
    "RMSE_Reduction_Percent": safe_percent(
        ais["RMSE"] - dynamic["RMSE"],
        ais["RMSE"]
    ),
    "MAE_Reduction_Percent": safe_percent(
        ais["MAE"] - dynamic["MAE"],
        ais["MAE"]
    )
}])

display(incremental_value_df.round(4))

output = (
    PROJECT_DIR /
    "06_results/metrics/incremental_value_analysis.csv"
)

output.parent.mkdir(parents=True, exist_ok=True)
incremental_value_df.to_csv(output, index=False)

print("\nSaved:", output)
print("Positive Delta_R2 favors dynamic ML.")
print("Negative Delta_MAE/RMSE favors dynamic ML.")
print("Positive reduction percentages favor dynamic ML.")
print("\nSTEP 31.4 COMPLETED")


In [ ]:

# ============================================================
# STEP 31.5: PAIRED BOOTSTRAP MODEL COMPARISON
# AIS-only vs Dynamic B16
# Matched Internal Test Patients
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 31.5: PAIRED BOOTSTRAP")
print("=" * 70)

y = np.asarray(
    benchmark_test_y, dtype=float
).ravel()

ais_pred = np.asarray(
    benchmark_test_ais_pred, dtype=float
).ravel()

dyn_pred = np.asarray(
    benchmark_test_dynamic_pred, dtype=float
).ravel()

assert len(y) == len(ais_pred) == len(dyn_pred)
assert len(y) == len(benchmark_test_ids)
assert len(set(benchmark_test_ids)) == len(y)

N_BOOTSTRAP = 5000
SEED = 42

rng = np.random.default_rng(SEED)
n = len(y)

indices = rng.integers(
    0, n,
    size=(N_BOOTSTRAP, n)
)

# ------------------------------------------------------------
# Paired MAE improvement
# Positive values favor dynamic ML
# ------------------------------------------------------------

ais_abs_errors = np.abs(y - ais_pred)
dyn_abs_errors = np.abs(y - dyn_pred)

mae_improvement = (
    ais_abs_errors.mean() -
    dyn_abs_errors.mean()
)

boot_mae_improvement = (
    ais_abs_errors[indices].mean(axis=1)
    -
    dyn_abs_errors[indices].mean(axis=1)
)

mae_ci = np.quantile(
    boot_mae_improvement,
    [0.025, 0.975]
)

# ------------------------------------------------------------
# Paired RMSE improvement
# ------------------------------------------------------------

ais_squared = (y - ais_pred) ** 2
dyn_squared = (y - dyn_pred) ** 2

rmse_improvement = (
    np.sqrt(ais_squared.mean())
    -
    np.sqrt(dyn_squared.mean())
)

boot_rmse_improvement = (
    np.sqrt(
        ais_squared[indices].mean(axis=1)
    )
    -
    np.sqrt(
        dyn_squared[indices].mean(axis=1)
    )
)

rmse_ci = np.quantile(
    boot_rmse_improvement,
    [0.025, 0.975]
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

bootstrap_result_df = pd.DataFrame([
    {
        "Comparison": "AIS-only vs Dynamic B16",
        "Metric": "MAE improvement",
        "Patients": n,
        "Improvement": mae_improvement,
        "CI_95_Lower": mae_ci[0],
        "CI_95_Upper": mae_ci[1],
        "CI_Excludes_Zero": bool(
            mae_ci[0] > 0 or mae_ci[1] < 0
        )
    },
    {
        "Comparison": "AIS-only vs Dynamic B16",
        "Metric": "RMSE improvement",
        "Patients": n,
        "Improvement": rmse_improvement,
        "CI_95_Lower": rmse_ci[0],
        "CI_95_Upper": rmse_ci[1],
        "CI_Excludes_Zero": bool(
            rmse_ci[0] > 0 or rmse_ci[1] < 0
        )
    }
])

display(bootstrap_result_df.round(4))

output = (
    PROJECT_DIR /
    "06_results/metrics/bootstrap/"
    "AIS_vs_Dynamic_paired_bootstrap_CI.csv"
)

output.parent.mkdir(parents=True, exist_ok=True)
bootstrap_result_df.to_csv(output, index=False)

print("\nSaved:", output)
print("Positive improvement favors dynamic ML.")
print("Confidence intervals are exploratory.")
print("Intervals are not multiplicity-adjusted.")
print("Model-selection uncertainty is not included.")
print("\nSTEP 31.5 COMPLETED")


In [ ]:

# ============================================================
# STEP 31.6: BASELINE CLINICAL LIGHTGBM
# Training-Only Preprocessing
# Separate Validation and Test Evaluation
# ============================================================

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

print("=" * 70)
print("STEP 31.6: BASELINE CLINICAL MODEL")
print("=" * 70)

baseline_features = [
    "AGE",
    "SEXCD",
    "AIS.1",
    "TOTMOT01"
]

numeric_features = [
    "AGE",
    "TOTMOT01"
]

categorical_features = [
    "SEXCD",
    "AIS.1"
]

data = MODEL_INPUTS["Model_B16"]

def prepare_baseline(X):
    X = X[baseline_features].copy()

    for col in numeric_features:
        X[col] = pd.to_numeric(
            X[col], errors="coerce"
        )

    for col in categorical_features:
        X[col] = X[col].astype("string").astype(object)
        X[col] = X[col].where(
            pd.notna(X[col]), np.nan
        )

    return X

X_train_baseline = prepare_baseline(
    data["X_train"]
)

X_val_baseline = prepare_baseline(
    data["X_validation"]
)

X_test_baseline = prepare_baseline(
    data["X_test"]
)

y_train_baseline = np.asarray(
    data["y_train"], dtype=float
).ravel()

y_val_baseline = np.asarray(
    data["y_validation"], dtype=float
).ravel()

y_test_baseline = np.asarray(
    data["y_test"], dtype=float
).ravel()

# ------------------------------------------------------------
# Patient and target alignment
# ------------------------------------------------------------

assert X_train_baseline.index.equals(train_df.index)
assert X_val_baseline.index.equals(validation_df.index)
assert X_test_baseline.index.equals(test_df.index)

assert np.array_equal(
    y_test_baseline,
    benchmark_test_y
)

assert len(y_train_baseline) == len(X_train_baseline)
assert len(y_val_baseline) == len(X_val_baseline)
assert len(y_test_baseline) == len(X_test_baseline)

# ------------------------------------------------------------
# Training-only preprocessing
# ------------------------------------------------------------

baseline_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(
                strategy="most_frequent"
            )),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore"
            ))
        ]),
        categorical_features
    )
])

baseline_clinical_model = Pipeline([
    ("preprocessor", baseline_preprocessor),
    ("model", LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        num_leaves=7,
        min_child_samples=10,
        random_state=42,
        verbosity=-1,
        n_jobs=1
    ))
])

# ------------------------------------------------------------
# Train on training patients only
# ------------------------------------------------------------

baseline_clinical_model.fit(
    X_train_baseline,
    y_train_baseline
)

# ------------------------------------------------------------
# Separate validation and test predictions
# ------------------------------------------------------------

baseline_validation_prediction = (
    baseline_clinical_model.predict(
        X_val_baseline
    )
)

baseline_test_prediction = (
    baseline_clinical_model.predict(
        X_test_baseline
    )
)

baseline_validation_metrics = benchmark_metrics(
    y_val_baseline,
    baseline_validation_prediction
)

baseline_test_metrics = benchmark_metrics(
    y_test_baseline,
    baseline_test_prediction
)

# Preserve downstream variable names
baseline_mae = baseline_validation_metrics["MAE"]
baseline_rmse = baseline_validation_metrics["RMSE"]
baseline_r2 = baseline_validation_metrics["R2"]

baseline_test_mae = baseline_test_metrics["MAE"]
baseline_test_rmse = baseline_test_metrics["RMSE"]
baseline_test_r2 = baseline_test_metrics["R2"]

baseline_results = pd.DataFrame([
    {
        "Model": "Baseline Clinical LightGBM",
        "Cohort": "Validation",
        **baseline_validation_metrics
    },
    {
        "Model": "Baseline Clinical LightGBM",
        "Cohort": "Test",
        **baseline_test_metrics
    }
])

display(baseline_results.round(4))

print("\nTraining patients:", len(y_train_baseline))
print("Validation patients:", len(y_val_baseline))
print("Test patients:", len(y_test_baseline))
print("Predictors:", baseline_features)
print("Test cohort used for retrospective evaluation.")
print("\nSTEP 31.6 COMPLETED")


In [ ]:

# ============================================================
# STEP 31.7: FINAL THREE-MODEL COMPARISON
# AIS-only vs Baseline Clinical vs Dynamic B16
# Same Test Cohort | Same Outcomes
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 31.7: FINAL THREE-MODEL COMPARISON")
print("=" * 70)

# ------------------------------------------------------------
# 1. Collect test outcomes and predictions
# ------------------------------------------------------------

y_true = np.asarray(
    benchmark_test_y, dtype=float
).ravel()

predictions = {
    "AIS-only": np.asarray(
        AIS_test_prediction,
        dtype=float
    ).ravel(),

    "Baseline Clinical": np.asarray(
        baseline_test_prediction,
        dtype=float
    ).ravel(),

    "Dynamic ML (B16)": np.asarray(
        benchmark_test_dynamic_pred,
        dtype=float
    ).ravel()
}

# ------------------------------------------------------------
# 2. Verify matched patient evaluation
# ------------------------------------------------------------

X_test = MODEL_INPUTS["Model_B16"]["X_test"]

assert X_test.index.equals(test_df.index)

source_ids = (
    test_df.loc[X_test.index, "PTID"]
    .astype("string")
    .to_numpy()
)

assert np.array_equal(
    source_ids,
    benchmark_test_ids
)

assert len(np.unique(source_ids)) == len(y_true)

assert np.array_equal(
    np.asarray(
        MODEL_INPUTS["Model_B16"]["y_test"],
        dtype=float
    ).ravel(),
    y_true
)

for name, pred in predictions.items():
    assert len(pred) == len(y_true), (
        f"{name}: prediction length mismatch"
    )

    assert np.isfinite(pred).all(), (
        f"{name}: non-finite predictions"
    )

# ------------------------------------------------------------
# 3. Calculate final matched test metrics
# ------------------------------------------------------------

rows = []

for name, pred in predictions.items():

    metrics = benchmark_metrics(
        y_true, pred
    )

    rows.append({
        "Model": name,
        "Cohort": "Internal Test",
        "Patients": len(y_true),
        **metrics
    })

final_model_comparison = pd.DataFrame(rows)

print("\nFINAL MATCHED TEST COMPARISON")

display(
    final_model_comparison.round(4)
)

# ------------------------------------------------------------
# 4. Quantify pairwise MAE improvements
# ------------------------------------------------------------

ais_mae_final = benchmark_metrics(
    y_true,
    predictions["AIS-only"]
)["MAE"]

baseline_mae_final = benchmark_metrics(
    y_true,
    predictions["Baseline Clinical"]
)["MAE"]

dynamic_mae_final = benchmark_metrics(
    y_true,
    predictions["Dynamic ML (B16)"]
)["MAE"]

pairwise_summary = pd.DataFrame([
    {
        "Comparison": "AIS-only to Dynamic",
        "MAE_Improvement":
            ais_mae_final - dynamic_mae_final
    },
    {
        "Comparison": "Baseline Clinical to Dynamic",
        "MAE_Improvement":
            baseline_mae_final - dynamic_mae_final
    },
    {
        "Comparison": "AIS-only to Baseline Clinical",
        "MAE_Improvement":
            ais_mae_final - baseline_mae_final
    }
])

print("\nPAIRWISE MAE IMPROVEMENTS")
display(pairwise_summary.round(4))

# ------------------------------------------------------------
# 5. Save benchmark tables
# ------------------------------------------------------------

output_dir = (
    PROJECT_DIR /
    "06_results/tables/benchmarks"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

comparison_path = (
    output_dir /
    "AIS_Baseline_Dynamic_Model_Comparison.csv"
)

pairwise_path = (
    output_dir /
    "Matched_Test_MAE_Improvements.csv"
)

final_model_comparison.to_csv(
    comparison_path,
    index=False
)

pairwise_summary.to_csv(
    pairwise_path,
    index=False
)

print("\nFILES SAVED")
print(comparison_path)
print(pairwise_path)

print("\nVERIFICATION SUMMARY")
print("Test patients:", len(y_true))
print("Same evaluation outcomes: Yes")
print("Source PTID and index checks: Passed")
print("All metrics recomputed: Yes")
print("Final dynamic model retrained: No")
print("External validation: Not performed")
print("Evaluation: Retrospective internal test audit")

print("\nSTEP 31.7 COMPLETED")


In [ ]:

# ============================================================
# STEP 31.7A: COMPLETE PAIRED BOOTSTRAP COMPARISON
# All Three Models | Same 99 Test Patients
# Exploratory Retrospective Analysis
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 31.7A: COMPLETE PAIRED COMPARISON")
print("=" * 70)

y = np.asarray(
    benchmark_test_y, dtype=float
).ravel()

predictions = {
    "AIS-only": np.asarray(
        AIS_test_prediction, dtype=float
    ).ravel(),

    "Baseline Clinical": np.asarray(
        baseline_test_prediction, dtype=float
    ).ravel(),

    "Dynamic B16": np.asarray(
        benchmark_test_dynamic_pred, dtype=float
    ).ravel()
}

assert all(
    len(p) == len(y)
    for p in predictions.values()
)

assert len(benchmark_test_ids) == len(y)
assert len(set(benchmark_test_ids)) == len(y)

comparisons = [
    ("AIS-only", "Baseline Clinical"),
    ("Baseline Clinical", "Dynamic B16"),
    ("AIS-only", "Dynamic B16")
]

N_BOOTSTRAP = 5000
rng = np.random.default_rng(42)
n = len(y)

indices = rng.integers(
    0, n, size=(N_BOOTSTRAP, n)
)

results = []

for earlier, later in comparisons:

    earlier_error = np.abs(
        y - predictions[earlier]
    )

    later_error = np.abs(
        y - predictions[later]
    )

    individual_improvement = (
        earlier_error - later_error
    )

    bootstrap_values = (
        individual_improvement[indices]
        .mean(axis=1)
    )

    lower, upper = np.quantile(
        bootstrap_values,
        [0.025, 0.975]
    )

    results.append({
        "Earlier_Model": earlier,
        "Later_Model": later,
        "Patients": n,
        "MAE_Earlier": earlier_error.mean(),
        "MAE_Later": later_error.mean(),
        "MAE_Improvement":
            individual_improvement.mean(),
        "CI_95_Lower": lower,
        "CI_95_Upper": upper,
        "CI_Excludes_Zero": bool(
            lower > 0 or upper < 0
        )
    })

paired_comparison_df = pd.DataFrame(results)

print("\nPAIRED TEST COMPARISON")
display(paired_comparison_df.round(4))

output = (
    Path("/kaggle/working/SCI_Recovery_ML_Project")
    / "06_results/tables/benchmarks"
    / "all_models_paired_bootstrap.csv"
)

output.parent.mkdir(parents=True, exist_ok=True)

paired_comparison_df.to_csv(
    output, index=False
)

print("\nSaved:", output)
print("Positive improvement favors later model.")
print("Bootstrap CIs are exploratory.")
print("No adjustment for model selection.")
print("No multiplicity adjustment.")
print("\nSTEP 31.7A COMPLETED")


In [ ]:
# ============================================================
# STEP 31.8: INCREMENTAL VALUE OF EARLY NEUROLOGICAL CHANGE
# ============================================================


dynamic_vs_baseline = pd.DataFrame({

    "Metric": [

        "ΔR²",
        "ΔRMSE",
        "ΔMAE"

    ],

    "Dynamic ML (B16) - Baseline Clinical": [

        test_r2 - baseline_r2,

        test_rmse - baseline_rmse,

        test_mae - baseline_mae

    ]

})


display(
    dynamic_vs_baseline.round(4)
)



# Save

incremental_path = (

    PROJECT_DIR /
    "06_results/metrics/"
    "dynamic_incremental_value.csv"

)


dynamic_vs_baseline.to_csv(
    incremental_path,
    index=False
)


print("\nSaved:")
print(incremental_path)

In [ ]:

# ============================================================
# STEP 31.9 — VERIFIED LANDMARK RESULTS TABLE AND FIGURE EXPORT
# Placement: Replace the existing STEP 31.9 code completely
# Requires:
#   MODEL_INPUTS, MODEL_FEATURES, ablation_models
#   validation_df and the fitted models from STEP 12.1
# Output:
#   Verified metrics, prediction table, PDF and PNG figure.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

LANDMARK_ORDER = [
    "Model_A",
    "Model_B4",
    "Model_B8",
    "Model_B16"
]

LANDMARK_LABELS = {
    "Model_A": "Baseline",
    "Model_B4": "Week 4",
    "Model_B8": "Week 8",
    "Model_B16": "Week 16"
}

required_objects = [
    "MODEL_INPUTS",
    "MODEL_FEATURES",
    "ablation_models",
    "validation_df"
]

for name in required_objects:
    if name not in globals():
        raise RuntimeError(
            f"Required object '{name}' is missing. "
            "Run the preceding modelling cells first."
        )

# 1. Confirm common validation outcome.
y_reference = np.asarray(
    MODEL_INPUTS["Model_A"]["y_validation"],
    dtype=float
).ravel()

if not np.isfinite(y_reference).all():
    raise ValueError("Non-finite Week-26 validation outcomes.")

if len(validation_df) != len(y_reference):
    raise ValueError(
        "validation_df length does not match the validation target. "
        "Check filtering and row alignment."
    )

if "PTID" in validation_df.columns:
    if validation_df["PTID"].isna().any():
        raise ValueError("Missing PTID in validation_df.")

    if not validation_df["PTID"].is_unique:
        raise ValueError(
            "validation_df has duplicate PTIDs. "
            "Verify whether this is a patient-level evaluation set."
        )

# 2. Recompute metrics from fitted models, not hardcoded numbers.
records = []

prediction_table = pd.DataFrame({
    "Observed_TOTMOT26": y_reference
})

if "PTID" in validation_df.columns:
    prediction_table.insert(
        0,
        "PTID",
        validation_df["PTID"].astype(str).to_numpy()
    )
else:
    prediction_table.insert(
        0,
        "Validation_Row",
        validation_df.index.to_numpy()
    )

for model_name in LANDMARK_ORDER:

    if model_name not in MODEL_INPUTS:
        raise KeyError(f"{model_name} missing from MODEL_INPUTS.")

    if model_name not in MODEL_FEATURES:
        raise KeyError(f"{model_name} missing from MODEL_FEATURES.")

    if model_name not in ablation_models:
        raise KeyError(f"{model_name} missing from ablation_models.")

    model_data = MODEL_INPUTS[model_name]

    y_true = np.asarray(
        model_data["y_validation"],
        dtype=float
    ).ravel()

    X_valid = model_data["X_validation"]

    if not np.allclose(
        y_reference, y_true, rtol=0, atol=1e-8
    ):
        raise ValueError(
            f"{model_name}: validation outcomes differ from baseline. "
            "Use the same patients and target ordering for all models."
        )

    if len(X_valid) != len(y_true):
        raise ValueError(
            f"{model_name}: X_validation/y_validation length mismatch."
        )

    predictions = np.asarray(
        ablation_models[model_name].predict(X_valid),
        dtype=float
    ).ravel()

    if len(predictions) != len(y_true):
        raise ValueError(f"{model_name}: incorrect prediction length.")

    if not np.isfinite(predictions).all():
        raise ValueError(
            f"{model_name}: non-finite predictions detected."
        )

    records.append({
        "Model": model_name,
        "Landmark": LANDMARK_LABELS[model_name],
        "Algorithm": "LightGBM",
        "Predictor_Count": len(MODEL_FEATURES[model_name]),
        "N_Validation": len(y_true),
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, predictions)
        ),
        "R2": r2_score(y_true, predictions)
    })

    prediction_table[f"Prediction_{model_name}"] = predictions
    prediction_table[f"Error_{model_name}"] = predictions - y_true

# 3. Calculate relative MAE change against baseline.
verified_results_df = pd.DataFrame(records)

baseline_mae = verified_results_df.loc[
    verified_results_df["Model"] == "Model_A",
    "MAE"
].iloc[0]

if baseline_mae == 0:
    verified_results_df["MAE_Reduction_vs_Baseline_Percent"] = np.nan
else:
    verified_results_df["MAE_Reduction_vs_Baseline_Percent"] = (
        100 * (
            baseline_mae - verified_results_df["MAE"]
        ) / baseline_mae
    )

print("STEP 31.9 — VERIFIED VALIDATION RESULTS")
display(verified_results_df.round(4))

# 4. Export the table and matched predictions.
output_dir = Path(
    "final_manuscript_results/verified_landmark_results"
)
figure_dir = output_dir / "figures"

figure_dir.mkdir(parents=True, exist_ok=True)

verified_results_df.to_csv(
    output_dir / "landmark_validation_performance.csv",
    index=False
)

prediction_table.to_csv(
    output_dir / "landmark_validation_predictions.csv",
    index=False
)

# 5. Generate the figure directly from verified results.
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(
    verified_results_df["Landmark"],
    verified_results_df["MAE"],
    marker="o",
    linewidth=1.8
)

ax.set_xlabel("Prediction landmark")
ax.set_ylabel("Validation MAE (motor-score points)")
ax.set_title("Week-26 Motor-Score Prediction Across Landmarks")
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()

fig.savefig(
    figure_dir / "landmark_validation_MAE.pdf",
    bbox_inches="tight"
)

fig.savefig(
    figure_dir / "landmark_validation_MAE.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(f"Results saved to: {output_dir.resolve()}")
print("The table and figure use metrics recalculated from fitted models.")
print("This cell does not use the internal test-set outcomes.")


In [ ]:

# ============================================================
# STEP 31.9A — CANONICAL LANDMARK TABLE / FIGURE SYNCHRONIZATION
# Placement: Immediately after STEP 31.9
# Purpose:
#   Replace stale exported benchmark tables with the current
#   primary LightGBM landmark results.
#   No model retraining; no test-set access.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "verified_results_df" not in globals():
    raise RuntimeError("Run STEP 31.9 before STEP 31.9A.")

required_cols = {
    "Model", "Landmark", "Algorithm", "Predictor_Count",
    "N_Validation", "MAE", "RMSE", "R2"
}
missing_cols = required_cols - set(verified_results_df.columns)
if missing_cols:
    raise ValueError(f"STEP 31.9 output missing columns: {missing_cols}")

expected_order = ["Baseline", "Week 4", "Week 8", "Week 16"]
observed_order = verified_results_df["Landmark"].tolist()

if observed_order != expected_order:
    raise ValueError(
        f"Unexpected landmark order: {observed_order}"
    )

if not verified_results_df["Algorithm"].eq("LightGBM").all():
    raise ValueError(
        "Primary landmark table is not entirely LightGBM. "
        "Review the intended primary model before exporting."
    )

if not verified_results_df["N_Validation"].eq(99).all():
    raise ValueError(
        "Validation N differs from the expected 99-patient cohort. "
        "Check cohort filtering before overwriting exports."
    )

numeric_cols = ["MAE", "RMSE", "R2"]
if not np.isfinite(
    verified_results_df[numeric_cols].to_numpy(dtype=float)
).all():
    raise ValueError("Non-finite metric detected.")

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")

# Canonical project locations used by the manuscript package.
table_paths = [
    ROOT / "06_results/tables/validation/"
    "Table31_9_Landmark_Validation_Performance.csv",
    ROOT / "03_tables/ablation/"
    "landmark_validation_performance.csv",
    ROOT / "07_manuscript_package/Tables/"
    "Table31_9_Landmark_Validation_Performance.csv"
]

xlsx_paths = [
    ROOT / "06_results/tables/validation/"
    "Table31_9_Landmark_Validation_Performance.xlsx",
    ROOT / "07_manuscript_package/Tables/"
    "Table31_9_Landmark_Validation_Performance.xlsx"
]

figure_paths = [
    ROOT / "06_results/figures/evaluation/"
    "Figure31_9_Landmark_Validation_MAE.png",
    ROOT / "06_results/figures/evaluation/"
    "Figure31_9_Landmark_Validation_MAE.pdf",
    ROOT / "06_results/figures/evaluation/"
    "Figure31_9_Landmark_Validation_MAE.tiff",
    ROOT / "07_manuscript_package/Figures/"
    "Figure31_9_Landmark_Validation_MAE.png",
    ROOT / "07_manuscript_package/Figures/"
    "Figure31_9_Landmark_Validation_MAE.pdf"
]

audit_dir = ROOT / "01_data/metadata"
audit_dir.mkdir(parents=True, exist_ok=True)

backup_dir = audit_dir / "pre_sync_table31_9_backup"
backup_dir.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# 1. Back up existing outputs before overwriting.
backed_up = []
for path in table_paths + xlsx_paths + figure_paths:
    if path.is_file():
        backup_path = backup_dir / (
            path.name + "." + sha256_file(path)[:10] + ".bak"
        )
        if not backup_path.exists():
            shutil.copy2(path, backup_path)
        backed_up.append({
            "original": str(path),
            "backup": str(backup_path)
        })

# 2. Export the current primary results to every canonical table path.
for path in table_paths:
    path.parent.mkdir(parents=True, exist_ok=True)
    verified_results_df.to_csv(path, index=False)

for path in xlsx_paths:
    path.parent.mkdir(parents=True, exist_ok=True)
    verified_results_df.to_excel(path, index=False)

# 3. Generate the MAE figure from the same source table.
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(
    verified_results_df["Landmark"],
    verified_results_df["MAE"],
    marker="o",
    linewidth=1.8
)
ax.set_xlabel("Prediction landmark")
ax.set_ylabel("Validation MAE (motor-score points)")
ax.set_title("Week-26 Motor-Score Prediction Across Landmarks")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

for path in figure_paths:
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.suffix == ".png":
        fig.savefig(path, dpi=600, bbox_inches="tight")
    elif path.suffix == ".pdf":
        fig.savefig(path, bbox_inches="tight")
    elif path.suffix == ".tiff":
        fig.savefig(path, dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

# 4. Re-read every CSV and check that the model/metric values match.
checked = []
for path in table_paths:
    exported = pd.read_csv(path)

    if exported["Landmark"].tolist() != expected_order:
        raise AssertionError(f"Landmark order mismatch: {path}")

    if not exported["Algorithm"].eq("LightGBM").all():
        raise AssertionError(f"Stale algorithm label remains: {path}")

    np.testing.assert_allclose(
        exported[numeric_cols].to_numpy(dtype=float),
        verified_results_df[numeric_cols].to_numpy(dtype=float),
        rtol=0,
        atol=1e-10
    )

    checked.append({
        "path": str(path),
        "sha256": sha256_file(path),
        "rows": len(exported),
        "all_algorithms_lightgbm": True,
        "metrics_match_current_notebook": True
    })

report = {
    "step": "31.9A",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source": "verified_results_df from STEP 31.9",
    "primary_algorithm": "LightGBM",
    "test_set_used": False,
    "backups": backed_up,
    "verified_csv_exports": checked,
    "xlsx_exports": [str(p) for p in xlsx_paths],
    "figure_exports": [str(p) for p in figure_paths],
    "model_retrained": False
}

(audit_dir / "table31_9_export_consistency_audit.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("STEP 31.9A PASSED")
print("Primary table and figure exports synchronized.")
print("Backups saved:", backup_dir)
print("Consistency audit saved.")


In [ ]:

# ============================================================
# STEP 31.9B — FIXED NESTED-CV RESULTS DISCOVERY
# Replace the nested_path / nested_df section in STEP 31.9B
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")

# 1. Prefer an existing in-memory summary.
nested_df = None
nested_source = None

for name in [
    "nested_cv_summary",
    "nested_cv_summary_df",
]:
    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame) and not obj.empty:
        nested_df = obj.copy()
        nested_source = f"In-memory: {name}"
        break

# 2. Otherwise locate the existing exported summary.
if nested_df is None:

    search_dirs = [
        Path(str(globals()["TABLE_DATASET"]))
        if "TABLE_DATASET" in globals()
        else None,
        ROOT / "06_results" / "tables",
        ROOT / "final_manuscript_results",
        Path("/kaggle/working/final_manuscript_results"),
    ]

    search_dirs = [
        p for p in search_dirs
        if p is not None and p.exists()
    ]

    candidate_names = [
        "Table15_Nested_CV_Mean_SD.csv",
        "nested_cv_summary.csv",
        "Table15_Nested_CV_Mean_SD.xlsx",
    ]

    candidates = []

    for folder in search_dirs:
        for filename in candidate_names:
            candidates.extend(folder.rglob(filename))

    # Remove duplicate paths while preserving order.
    candidates = list(dict.fromkeys(candidates))

    for path in candidates:
        try:
            if path.suffix.lower() == ".csv":
                temp = pd.read_csv(path)
            else:
                temp = pd.read_excel(path)

            if {"Metric", "Mean", "SD"}.issubset(temp.columns):
                nested_df = temp
                nested_source = str(path)
                break

        except Exception as exc:
            print(f"Could not read {path}: {exc}")

# 3. If no summary exists, use existing fold-level results.
if nested_df is None:

    fold_candidates = []

    for name in [
        "nested_cv_results_df",
        "nested_cv_folds_df",
    ]:
        obj = globals().get(name)

        if isinstance(obj, pd.DataFrame) and not obj.empty:
            fold_candidates.append((name, obj.copy()))

    if not fold_candidates:
        for folder in search_dirs:
            for path in folder.rglob(
                "Table15_Nested_CV_Fold_Performance.csv"
            ):
                try:
                    temp = pd.read_csv(path)
                    if {"MAE", "RMSE", "R2"}.issubset(temp.columns):
                        fold_candidates.append((str(path), temp))
                except Exception:
                    pass

    for source, folds in fold_candidates:

        if "Landmark" in folds.columns:
            folds = folds[
                folds["Landmark"].astype(str) == "Model_B16"
            ].copy()

        if len(folds) == 5 and {
            "MAE", "RMSE", "R2"
        }.issubset(folds.columns):

            metrics = folds[["MAE", "RMSE", "R2"]].apply(
                pd.to_numeric, errors="coerce"
            )

            if np.isfinite(metrics.to_numpy()).all():
                nested_df = pd.DataFrame({
                    "Metric": ["MAE", "RMSE", "R2"],
                    "Mean": [
                        metrics[c].mean()
                        for c in ["MAE", "RMSE", "R2"]
                    ],
                    "SD": [
                        metrics[c].std(ddof=1)
                        for c in ["MAE", "RMSE", "R2"]
                    ],
                    "N_Outer_Folds": 5,
                    "Validation_Type":
                        "Nested 5x5 CV - Training Partition",
                    "Landmark": "Model_B16",
                })

                nested_source = (
                    f"Calculated from existing fold results: {source}"
                )
                break

if nested_df is None:
    raise FileNotFoundError(
        "Could not find an existing nested-CV summary or five-fold "
        "Model_B16 result table. Run STEP 15.4/15.6 first. "
        "Inspect TABLE_DATASET and the actual STEP 15.6 export path."
    )

print("Nested-CV source:", nested_source)
print("Columns:", nested_df.columns.tolist())
display(nested_df)

# 4. Normalize the summary schema for the registry.
required = {"Metric", "Mean", "SD"}
missing = required - set(nested_df.columns)

if missing:
    raise ValueError(
        f"Nested-CV summary is missing columns: {sorted(missing)}"
    )

if "Landmark" in nested_df.columns:
    b16 = nested_df[
        nested_df["Landmark"].astype(str) == "Model_B16"
    ].copy()

    if not b16.empty:
        nested_df = b16

def nested_value(metric, column):
    match = nested_df[
        nested_df["Metric"].astype(str).str.upper() == metric
    ]

    if len(match) != 1:
        raise ValueError(
            f"Expected exactly one {metric} row; found {len(match)}."
        )

    value = float(match.iloc[0][column])

    if not np.isfinite(value):
        raise ValueError(f"{metric} {column} is not finite.")

    return value

nested_folds = 5

if (
    "N_Outer_Folds" in nested_df.columns
    and nested_df["N_Outer_Folds"].notna().any()
):
    nested_folds = int(nested_df["N_Outer_Folds"].dropna().iloc[0])

if nested_folds != 5:
    raise ValueError(
        f"Expected 5 outer folds; found {nested_folds}."
    )

nested_registry_row = {
    "Evaluation_Type": "Nested cross-validation",
    "Analysis_Role": "Internal resampling estimate",
    "Model": "Model_B16",
    "Landmark": "Week 16",
    "Algorithm": "LightGBM",
    "N_Evaluation": np.nan,
    "N_Training_Partition": 461,
    "N_Outer_Folds": nested_folds,
    "MAE": nested_value("MAE", "Mean"),
    "RMSE": nested_value("RMSE", "Mean"),
    "R2": nested_value("R2", "Mean"),
    "MAE_SD": nested_value("MAE", "SD"),
    "RMSE_SD": nested_value("RMSE", "SD"),
    "R2_SD": nested_value("R2", "SD"),
    "Source": nested_source,
    "Interpretation_Note":
        "Nested-CV training-partition estimate; not the internal-test result.",
}

print("\nNested-CV registry row:")
display(pd.DataFrame([nested_registry_row]).round(4))

In [ ]:

# ============================================================
# STEP 31.9C — VALIDATION PREDICTION TABLE AUDIT
# Fix: Recognize Prediction_Model_* column names
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# 1. Locate existing predictions; do not generate new ones.
PREDICTIONS = globals().get("prediction_table")

if not isinstance(PREDICTIONS, pd.DataFrame):
    raise RuntimeError(
        "Existing prediction_table DataFrame was not found. "
        "Run the original prediction-export cell first."
    )

pred_df = PREDICTIONS.copy()

print("Prediction source: existing prediction_table")
print("Shape:", pred_df.shape)
print("Columns:", pred_df.columns.tolist())

# 2. Validate required columns.
required_cols = ["PTID", "Observed_TOTMOT26"]

missing = [
    col for col in required_cols
    if col not in pred_df.columns
]

if missing:
    raise ValueError(
        f"Required columns are missing: {missing}"
    )

# 3. Recognize the actual prediction columns.
model_columns = {
    "Model_A": "Prediction_Model_A",
    "Model_B4": "Prediction_Model_B4",
    "Model_B8": "Prediction_Model_B8",
    "Model_B16": "Prediction_Model_B16",
}

missing_predictions = [
    col for col in model_columns.values()
    if col not in pred_df.columns
]

if missing_predictions:
    raise ValueError(
        f"Missing prediction columns: {missing_predictions}"
    )

# 4. Validate IDs and numeric values.
if pred_df["PTID"].isna().any():
    raise ValueError("Missing PTID values detected.")

if pred_df["PTID"].astype(str).str.strip().eq("").any():
    raise ValueError("Blank PTID values detected.")

if pred_df["PTID"].duplicated().any():
    raise ValueError("Duplicate PTIDs detected.")

numeric_cols = [
    "Observed_TOTMOT26",
    *model_columns.values()
]

for col in numeric_cols:
    pred_df[col] = pd.to_numeric(
        pred_df[col], errors="coerce"
    )

if pred_df[numeric_cols].isna().any().any():
    raise ValueError(
        "Missing or non-numeric observed/predicted values detected. "
        "Resolve these before computing metrics."
    )

if not np.isfinite(
    pred_df[numeric_cols].to_numpy(dtype=float)
).all():
    raise ValueError("Infinite values detected.")

# 5. Verify errors against observed outcomes.
for model, pred_col in model_columns.items():
    error_col = f"Error_{model}"

    if error_col in pred_df.columns:
        expected_error = (
            pred_df[pred_col] - pred_df["Observed_TOTMOT26"]
        )

        if not np.allclose(
            pred_df[error_col].to_numpy(dtype=float),
            expected_error.to_numpy(dtype=float),
            rtol=1e-5,
            atol=1e-5
        ):
            raise ValueError(
                f"{error_col} does not match prediction minus "
                "observed outcome."
            )

# 6. Create a model-wise metrics table from existing predictions.
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_true = pred_df["Observed_TOTMOT26"].to_numpy(dtype=float)

results = []

for model, pred_col in model_columns.items():
    y_pred = pred_df[pred_col].to_numpy(dtype=float)

    results.append({
        "Model": model,
        "N_Validation": len(pred_df),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    })

prediction_metrics_df = pd.DataFrame(results)

# 7. Export audit outputs.
ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
OUT = ROOT / "06_results" / "tables" / "validation"
OUT.mkdir(parents=True, exist_ok=True)

prediction_metrics_df.to_csv(
    OUT / "STEP31_9C_Existing_Prediction_Metrics.csv",
    index=False
)

pred_df.to_csv(
    OUT / "STEP31_9C_Existing_Validation_Predictions.csv",
    index=False
)

print("\nSTEP 31.9C — VALIDATION METRICS")
display(prediction_metrics_df.round(4))

print("\nSaved existing predictions and recalculated metrics to:")
print(OUT)

print(
    "\nNote: This recalculates metrics from the supplied prediction "
    "table; it does not retrain models or establish test-set independence."
)


In [ ]:

# ============================================================
# STEP 31.10
# OBSERVED NEUROLOGICAL RECOVERY TRAJECTORIES
# Descriptive Analysis Only | No Model Training
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 70)
print("STEP 31.10: OBSERVED NEUROLOGICAL RECOVERY TRAJECTORIES")
print("=" * 70)

# 1. Define observed follow-up measurements

trajectory_columns = [
    "TOTMOT04",
    "TOTMOT08",
    "TOTMOT16",
    "TOTMOT26"
]

timepoints = [4, 8, 16, 26]

# 2. Use observed values from the existing test cohort

observed_df = test_df[trajectory_columns].copy()

# Retain only patients with all four observed measurements
complete_mask = observed_df.notna().all(axis=1)

complete_trajectories = (
    observed_df.loc[complete_mask].copy()
)

if complete_trajectories.empty:
    raise ValueError("No complete observed trajectories found.")

print("\nTotal test patients:", len(observed_df))
print("Complete trajectories:", len(complete_trajectories))
print("Incomplete trajectories:", (~complete_mask).sum())

# 3. Plot observed trajectories for up to 10 patients

n_patients = min(10, len(complete_trajectories))

fig, ax = plt.subplots(figsize=(9, 5.5))

for i in range(n_patients):

    scores = (
        complete_trajectories.iloc[i]
        .to_numpy(dtype=float)
    )

    ax.plot(
        timepoints,
        scores,
        marker="o",
        linewidth=1.5,
        alpha=0.75
    )

ax.set_xlabel("Follow-up Time (Weeks)")
ax.set_ylabel("Observed Total Motor Score")
ax.set_title("Observed Individual Neurological Recovery Trajectories")
ax.set_xticks(timepoints)
ax.grid(alpha=0.2)

plt.tight_layout()

figure_dir = Path(
    globals().get(
        "FIG_EVALUATION",
        "/kaggle/working/FIG_EVALUATION"
    )
)

figure_dir.mkdir(parents=True, exist_ok=True)

fig.savefig(
    figure_dir / "Figure31_10_Observed_Individual_Trajectories.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# 4. Summarize observed population trajectory

trajectory_summary = pd.DataFrame({
    "Week": timepoints,
    "N": [
        complete_trajectories[col].count()
        for col in trajectory_columns
    ],
    "Mean": [
        complete_trajectories[col].mean()
        for col in trajectory_columns
    ],
    "SD": [
        complete_trajectories[col].std()
        for col in trajectory_columns
    ]
})

print("\nOBSERVED TRAJECTORY SUMMARY")
display(trajectory_summary.round(4))

# 5. Plot observed population mean

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    trajectory_summary["Week"],
    trajectory_summary["Mean"],
    marker="o",
    linewidth=2
)

ax.set_xlabel("Follow-up Time (Weeks)")
ax.set_ylabel("Mean Observed Total Motor Score")
ax.set_title("Observed Population Neurological Recovery")
ax.set_xticks(timepoints)
ax.grid(alpha=0.2)

plt.tight_layout()

fig.savefig(
    figure_dir / "Figure31_10_Observed_Mean_Trajectory.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# 6. Save summary

output_dir = Path(TABLE_VALIDATION)
output_dir.mkdir(parents=True, exist_ok=True)

trajectory_summary.to_excel(
    output_dir / "Table31_10_Observed_Trajectory_Summary.xlsx",
    index=False
)

print("\nTemporal leakage from trajectory prediction: REMOVED")
print("Predicted early outcomes: NOT USED")
print("Model retraining: No")
print("Analysis type: Descriptive only")
print("STEP 31.10 CORRECTION COMPLETED")


In [ ]:

# ============================================================
# CORRECTED STEP 31.11
# VALIDATION REGRESSION ERROR ANALYSIS
# Replaces invalid Decision Curve Analysis
# No Retraining | No Test-Set Reuse
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("=" * 70)
print("STEP 31.11: VALIDATION REGRESSION ERROR ANALYSIS")
print("=" * 70)

# Original validation features and Week-26 outcome
X_val = MODEL_INPUTS["Model_B16"]["X_validation"].copy()

y_true = np.asarray(
    MODEL_INPUTS["Model_B16"]["y_validation"],
    dtype=float
).ravel()

# Predictions from existing fitted model
y_pred = np.asarray(
    final_lgbm_model.predict(X_val),
    dtype=float
).ravel()

assert len(y_true) == len(y_pred)
assert np.isfinite(y_true).all()
assert np.isfinite(y_pred).all()

# Prediction errors
errors = y_pred - y_true
absolute_errors = np.abs(errors)

results = pd.DataFrame({
    "Observed_Week26": y_true,
    "Predicted_Week26": y_pred,
    "Prediction_Error": errors,
    "Absolute_Error": absolute_errors
})

# Summary
summary = pd.DataFrame({
    "Metric": [
        "Patients",
        "MAE",
        "RMSE",
        "Mean Prediction Error",
        "Median Absolute Error",
        "90th Percentile Absolute Error"
    ],
    "Value": [
        len(y_true),
        mean_absolute_error(y_true, y_pred),
        np.sqrt(mean_squared_error(y_true, y_pred)),
        np.mean(errors),
        np.median(absolute_errors),
        np.percentile(absolute_errors, 90)
    ]
})

print("\nVALIDATION ERROR SUMMARY")
display(summary.round(4))

# Observed versus predicted
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(
    y_true,
    y_pred,
    alpha=0.7
)

lower = min(y_true.min(), y_pred.min())
upper = max(y_true.max(), y_pred.max())

ax.plot(
    [lower, upper],
    [lower, upper],
    linestyle="--",
    color="black",
    label="Perfect prediction"
)

ax.set_xlabel("Observed Week-26 Motor Score")
ax.set_ylabel("Predicted Week-26 Motor Score")
ax.set_title("Validation: Observed vs Predicted")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

# Save outputs
figure_dir = Path(
    globals().get(
        "FIG_EVALUATION",
        "/kaggle/working/FIG_EVALUATION"
    )
)

table_dir = Path(TABLE_VALIDATION)

figure_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

fig.savefig(
    figure_dir / "Figure31_11_Validation_Observed_vs_Predicted.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

results.to_excel(
    table_dir / "Table31_11_Validation_Prediction_Errors.xlsx",
    index=False
)

summary.to_excel(
    table_dir / "Table31_11_Validation_Error_Summary.xlsx",
    index=False
)

print("\nInvalid DCA probability conversion: Removed")
print("Validation patients:", len(y_true))
print("Model retraining: No")
print("Held-out test outcomes accessed: No")
print("STEP 31.11 CORRECTION COMPLETED")


In [ ]:

# ============================================================
# STEP 31.12: PREDICTION INTERVAL EVALUATION
# Exploratory Split-Conformal Style Intervals
# No Retraining | No Test-Based Calibration
# ============================================================

import numpy as np
import pandas as pd
import json

from pathlib import Path
from datetime import datetime, timezone
from scipy.stats import binomtest
from sklearn.utils.validation import check_is_fitted

print("=" * 70)
print("STEP 31.12: PREDICTION INTERVAL EVALUATION")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

output_dir = (
    PROJECT_DIR /
    "06_results/tables/validation"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Check required objects
# ------------------------------------------------------------

if "MODEL_INPUTS" not in globals():
    raise RuntimeError(
        "MODEL_INPUTS missing. Run Step 9.6."
    )

if "final_lgbm_model" not in globals():
    raise RuntimeError(
        "Final model missing. Restore fitted model first."
    )

if "validation_df" not in globals():
    raise RuntimeError("validation_df missing.")

if "test_df" not in globals():
    raise RuntimeError("test_df missing.")

check_is_fitted(final_lgbm_model)

data = MODEL_INPUTS["Model_B16"]

X_cal = data["X_validation"].copy()
X_test = data["X_test"].copy()

y_cal = np.asarray(
    data["y_validation"], dtype=float
).ravel()

y_test = np.asarray(
    data["y_test"], dtype=float
).ravel()

# ------------------------------------------------------------
# 2. Verify dataset alignment
# ------------------------------------------------------------

assert X_cal.index.equals(validation_df.index)
assert X_test.index.equals(test_df.index)

assert list(X_cal.columns) == list(X_test.columns)

assert len(X_cal) == len(y_cal)
assert len(X_test) == len(y_test)

assert validation_df["PTID"].notna().all()
assert test_df["PTID"].notna().all()

assert validation_df["PTID"].is_unique
assert test_df["PTID"].is_unique

assert set(
    validation_df["PTID"].astype(str)
).isdisjoint(
    set(test_df["PTID"].astype(str))
)

np.testing.assert_allclose(
    y_cal,
    pd.to_numeric(
        validation_df["TOTMOT26"],
        errors="coerce"
    ).to_numpy(dtype=float),
    rtol=0,
    atol=1e-8
)

np.testing.assert_allclose(
    y_test,
    pd.to_numeric(
        test_df["TOTMOT26"],
        errors="coerce"
    ).to_numpy(dtype=float),
    rtol=0,
    atol=1e-8
)

# ------------------------------------------------------------
# 3. Generate predictions
# ------------------------------------------------------------

pred_cal = np.asarray(
    final_lgbm_model.predict(X_cal),
    dtype=float
).ravel()

pred_test = np.asarray(
    final_lgbm_model.predict(X_test),
    dtype=float
).ravel()

assert len(pred_cal) == len(y_cal)
assert len(pred_test) == len(y_test)

for array in [y_cal, y_test, pred_cal, pred_test]:
    assert np.isfinite(array).all()

# ------------------------------------------------------------
# 4. Calculate calibration residuals
# ------------------------------------------------------------

alpha = 0.05

residuals = np.abs(
    y_cal - pred_cal
)

n_cal = len(residuals)

k = int(
    np.ceil(
        (n_cal + 1) * (1 - alpha)
    )
)

if k > n_cal:
    raise ValueError(
        "Calibration sample too small for "
        "the requested conformal quantile."
    )

q_hat = float(
    np.sort(residuals)[k - 1]
)

# ------------------------------------------------------------
# 5. Construct prediction intervals
# ------------------------------------------------------------

lower = pred_test - q_hat
upper = pred_test + q_hat

covered = (
    (y_test >= lower)
    &
    (y_test <= upper)
)

interval_width = upper - lower

# ------------------------------------------------------------
# 6. Calculate empirical coverage
# ------------------------------------------------------------

n_test = len(y_test)
n_covered = int(covered.sum())

empirical_coverage = (
    n_covered / n_test
)

# Exact binomial confidence interval:
# describes uncertainty in observed coverage,
# not a guarantee of conformal validity.

coverage_ci = binomtest(
    k=n_covered,
    n=n_test
).proportion_ci(
    confidence_level=0.95,
    method="exact"
)

# ------------------------------------------------------------
# 7. Create patient-level results
# ------------------------------------------------------------

prediction_interval_results = pd.DataFrame({
    "Observed_TOTMOT26": y_test,
    "Predicted_TOTMOT26": pred_test,
    "Lower_95PI": lower,
    "Upper_95PI": upper,
    "Interval_Width": interval_width,
    "Covered": covered
})

# "95PI" is retained for compatibility with
# Steps 31.14 and 31.15.
# This label indicates nominal 95% intervals,
# not independently verified coverage validity.

# ------------------------------------------------------------
# 8. Coverage summary
# ------------------------------------------------------------

interval_summary = pd.DataFrame({
    "Metric": [
        "Calibration patients",
        "Test patients",
        "Nominal coverage",
        "Observed test coverage",
        "Observed coverage CI lower",
        "Observed coverage CI upper",
        "Patients covered",
        "Patients not covered",
        "Mean interval width",
        "Median interval width",
        "Conformal residual quantile"
    ],
    "Value": [
        n_cal,
        n_test,
        1 - alpha,
        empirical_coverage,
        coverage_ci.low,
        coverage_ci.high,
        n_covered,
        n_test - n_covered,
        float(np.mean(interval_width)),
        float(np.median(interval_width)),
        q_hat
    ]
})

print("\nPREDICTION INTERVAL SUMMARY")
display(interval_summary.round(4))

print("\nFIRST 10 PATIENT INTERVALS")
display(
    prediction_interval_results.head(10).round(4)
)

# ------------------------------------------------------------
# 9. Save output files
# ------------------------------------------------------------

prediction_interval_results.to_csv(
    output_dir /
    "Table31_12_Prediction_Intervals.csv",
    index=False
)

prediction_interval_results.to_excel(
    output_dir /
    "Table31_12_Prediction_Intervals.xlsx",
    index=False
)

interval_summary.to_csv(
    output_dir /
    "Table31_12_Interval_Summary.csv",
    index=False
)

interval_summary.to_excel(
    output_dir /
    "Table31_12_Interval_Summary.xlsx",
    index=False
)

# ------------------------------------------------------------
# 10. Save methodological audit
# ------------------------------------------------------------

report = {
    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "nominal_coverage": 1 - alpha,
    "calibration_patients": n_cal,
    "test_patients": n_test,
    "patients_covered": n_covered,

    "empirical_coverage":
        float(empirical_coverage),

    "coverage_95_ci": [
        float(coverage_ci.low),
        float(coverage_ci.high)
    ],

    "mean_interval_width":
        float(np.mean(interval_width)),

    "calibration_residual_quantile":
        q_hat,

    "test_outcomes_used_for_calibration":
        False,

    "calibration_independence_from_model_selection_verified":
        False,

    "formal_95_percent_coverage_guarantee_established":
        False,

    "interpretation":
        "Exploratory intervals using validation residuals.",

    "limitations": [
        "Validation observations may have influenced model selection.",
        "Formal split-conformal exchangeability and independence assumptions are not fully verified.",
        "The test cohort has previously been inspected.",
        "Uncertainty from model selection is not included.",
        "External validation has not been performed."
    ]
}

report_path = (
    PROJECT_DIR /
    "01_data/metadata/"
    "prediction_interval_audit.json"
)

report_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

report_path.write_text(
    json.dumps(report, indent=4),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 11. Final report
# ------------------------------------------------------------

print("\nVERIFICATION SUMMARY")

print("Calibration patients:", n_cal)
print("Test patients:", n_test)

print(
    "Empirical coverage:",
    round(empirical_coverage * 100, 2),
    "%"
)

print(
    "95% CI for observed coverage:",
    round(coverage_ci.low * 100, 2),
    "to",
    round(coverage_ci.high * 100, 2),
    "%"
)

print(
    "Mean interval width:",
    round(float(np.mean(interval_width)), 4)
)

print("\nMethodological status:")
print("Exploratory prediction intervals: Calculated")
print("Formal 95% coverage guarantee: Not established")
print("Calibration independence: Not verified")
print("Model retraining: No")
print("Test data used to choose q_hat: No")

print("\nSTEP 31.12 COMPLETED")


In [ ]:

# ============================================================
# STEP 31.12A: CLINICAL SUBGROUP ROBUSTNESS AUDIT
# Internal Test Evaluation | No Retraining
# Descriptive Analysis Only
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("=" * 70)
print("STEP 31.12A: CLINICAL SUBGROUP AUDIT")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

output_dir = (
    PROJECT_DIR /
    "06_results/tables/validation"
)
output_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Verify original data and interval results
# ------------------------------------------------------------

if "prediction_interval_results" not in globals():
    raise RuntimeError(
        "Run corrected Step 31.12 first."
    )

if "test_df" not in globals():
    raise RuntimeError("test_df is unavailable.")

X_test = MODEL_INPUTS["Model_B16"]["X_test"]

assert X_test.index.equals(test_df.index)
assert len(prediction_interval_results) == len(X_test)
assert test_df["PTID"].is_unique

required = [
    "AGE", "SEXCD", "AIS.1", "TOTMOT01", "TOTMOT26"
]

for col in required:
    if col not in test_df.columns:
        raise KeyError(f"Missing clinical variable: {col}")

# ------------------------------------------------------------
# 2. Construct aligned evaluation dataframe
# ------------------------------------------------------------

clinical = test_df.loc[X_test.index, required].copy()
clinical = clinical.reset_index(drop=True)

pi = prediction_interval_results.reset_index(drop=True)

clinical["Actual"] = pi[
    "Observed_TOTMOT26"
].to_numpy(dtype=float)

clinical["Predicted"] = pi[
    "Predicted_TOTMOT26"
].to_numpy(dtype=float)

clinical["Covered"] = pi[
    "Covered"
].to_numpy(dtype=bool)

clinical["Interval_Width"] = pi[
    "Interval_Width"
].to_numpy(dtype=float)

clinical["Absolute_Error"] = np.abs(
    clinical["Actual"] - clinical["Predicted"]
)

clinical["Signed_Error"] = (
    clinical["Predicted"] - clinical["Actual"]
)

# Verify source outcome alignment
np.testing.assert_allclose(
    clinical["Actual"].to_numpy(),
    pd.to_numeric(
        clinical["TOTMOT26"], errors="coerce"
    ).to_numpy(),
    rtol=0,
    atol=1e-8
)

# ------------------------------------------------------------
# 3. Create clinically relevant groups
# ------------------------------------------------------------

clinical["AIS_Group"] = (
    clinical["AIS.1"]
    .astype("string")
    .fillna("Missing")
)

clinical["Sex_Group"] = (
    clinical["SEXCD"]
    .astype("string")
    .fillna("Missing")
)

age = pd.to_numeric(
    clinical["AGE"], errors="coerce"
)

motor = pd.to_numeric(
    clinical["TOTMOT01"], errors="coerce"
)

# Use training-cohort thresholds, not test-derived medians.
train_source = train_df

train_age_median = pd.to_numeric(
    train_source["AGE"], errors="coerce"
).median()

train_motor_median = pd.to_numeric(
    train_source["TOTMOT01"], errors="coerce"
).median()

clinical["Age_Group"] = np.select(
    [
        age.isna(),
        age <= train_age_median
    ],
    [
        "Missing",
        "Lower/equal training median"
    ],
    default="Above training median"
)

clinical["Baseline_Motor_Group"] = np.select(
    [
        motor.isna(),
        motor <= train_motor_median
    ],
    [
        "Missing",
        "Lower/equal training median"
    ],
    default="Above training median"
)

clinical["Week16_Availability"] = np.where(
    X_test["TOTMOT_change_16"]
    .reset_index(drop=True)
    .notna(),
    "Observed",
    "Missing"
)

# ------------------------------------------------------------
# 4. Evaluation function
# ------------------------------------------------------------

MIN_REPORT_N = 5

def subgroup_metrics(group, variable, category):

    n = len(group)

    result = {
        "Variable": variable,
        "Category": str(category),
        "N": int(n),
        "MAE": np.nan,
        "RMSE": np.nan,
        "Mean_Signed_Error": np.nan,
        "Coverage": np.nan,
        "Mean_Interval_Width": np.nan,
        "Small_Subgroup": n < MIN_REPORT_N
    }

    if n < MIN_REPORT_N:
        return result

    actual = group["Actual"].to_numpy(dtype=float)
    predicted = group["Predicted"].to_numpy(dtype=float)

    result.update({
        "MAE": float(mean_absolute_error(
            actual, predicted
        )),
        "RMSE": float(np.sqrt(
            mean_squared_error(actual, predicted)
        )),
        "Mean_Signed_Error": float(
            np.mean(predicted - actual)
        ),
        "Coverage": float(group["Covered"].mean()),
        "Mean_Interval_Width": float(
            group["Interval_Width"].mean()
        )
    })

    return result

# ------------------------------------------------------------
# 5. Calculate subgroup metrics
# ------------------------------------------------------------

group_variables = [
    "AIS_Group",
    "Age_Group",
    "Sex_Group",
    "Baseline_Motor_Group",
    "Week16_Availability"
]

rows = []

for variable in group_variables:
    for category, group in clinical.groupby(
        variable, dropna=False
    ):
        rows.append(
            subgroup_metrics(group, variable, category)
        )

subgroup_table = pd.DataFrame(rows)

# Overall benchmark
overall = subgroup_metrics(
    clinical,
    "Overall",
    "All test patients"
)

subgroup_table = pd.concat(
    [
        pd.DataFrame([overall]),
        subgroup_table
    ],
    ignore_index=True
)

print("\nCLINICAL SUBGROUP PERFORMANCE")
display(subgroup_table.round(4))

# ------------------------------------------------------------
# 6. Save results
# ------------------------------------------------------------

csv_path = (
    output_dir /
    "Table31_12A_Clinical_Subgroup_Performance.csv"
)

xlsx_path = (
    output_dir /
    "Table31_12A_Clinical_Subgroup_Performance.xlsx"
)

subgroup_table.to_csv(
    csv_path, index=False
)

subgroup_table.to_excel(
    xlsx_path, index=False
)

# ------------------------------------------------------------
# 7. Research interpretation
# ------------------------------------------------------------

print("\nAUDIT SUMMARY")
print("Patients evaluated:", len(clinical))
print("Groups evaluated:", len(subgroup_table) - 1)

print(
    "Small subgroups:",
    int(subgroup_table["Small_Subgroup"].sum())
)

print("\nINTERPRETATION")
print("- Subgroup estimates are descriptive.")
print("- Small groups have suppressed metrics.")
print("- Group differences do not establish bias or causality.")
print("- Coverage is empirical, not a formal guarantee.")
print("- No model retraining was performed.")
print("- No test-based subgroup model selection.")

print("\nFILES SAVED")
print(csv_path)
print(xlsx_path)

print("\nSTEP 31.12A COMPLETED")


In [ ]:

# ============================================================
# STEP 31.12B: TRAINING-ONLY CALIBRATION SENSITIVITY
# Separate Model | Original Final Model Unchanged
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error
from scipy.stats import binomtest

print("=" * 70)
print("STEP 31.12B: CALIBRATION SENSITIVITY ANALYSIS")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

OUT_DIR = (
    PROJECT_DIR / "06_results/tables/validation"
)

META_DIR = PROJECT_DIR / "01_data/metadata"

OUT_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

required = [
    "MODEL_INPUTS",
    "train_df",
    "test_df",
    "final_lgbm_model"
]

missing = [
    x for x in required if x not in globals()
]

if missing:
    raise RuntimeError(f"Missing: {missing}")

data = MODEL_INPUTS["Model_B16"]

X_train = data["X_train"].copy()
X_test = data["X_test"].copy()

y_train = np.asarray(
    data["y_train"], dtype=float
).ravel()

y_test = np.asarray(
    data["y_test"], dtype=float
).ravel()

assert X_train.index.equals(train_df.index)
assert X_test.index.equals(test_df.index)

assert len(X_train) == len(y_train) == 461
assert len(X_test) == len(y_test) == 99

assert list(X_train.columns) == list(X_test.columns)

groups = (
    train_df.loc[X_train.index, "PTID"]
    .astype("string")
    .str.strip()
    .to_numpy(dtype=str)
)

assert len(set(groups)) == len(groups)

# ------------------------------------------------------------
# 1. Group-disjoint training/calibration split
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=2026
)

fit_idx, cal_idx = next(
    splitter.split(
        X_train, y_train, groups=groups
    )
)

fit_groups = set(groups[fit_idx])
cal_groups = set(groups[cal_idx])

assert fit_groups.isdisjoint(cal_groups)

X_fit = X_train.iloc[fit_idx]
X_cal = X_train.iloc[cal_idx]

y_fit = y_train[fit_idx]
y_cal = y_train[cal_idx]

# ------------------------------------------------------------
# 2. Fit a SEPARATE model
# ------------------------------------------------------------

# Clone uses the final pipeline's parameter settings,
# but all learned estimators/preprocessing are refitted
# using only the fitting partition.

sensitivity_model = clone(
    final_lgbm_model
)

sensitivity_model.fit(
    X_fit, y_fit
)

# Do not overwrite final_lgbm_model.

# ------------------------------------------------------------
# 3. Calibrate using disjoint patient outcomes
# ------------------------------------------------------------

p_cal = np.asarray(
    sensitivity_model.predict(X_cal),
    dtype=float
).ravel()

residuals = np.abs(y_cal - p_cal)

alpha = 0.05
n_cal = len(residuals)

k = int(
    np.ceil((n_cal + 1) * (1 - alpha))
)

if k > n_cal:
    raise ValueError(
        "Insufficient calibration patients."
    )

q_hat = float(
    np.sort(residuals)[k - 1]
)

# ------------------------------------------------------------
# 4. Evaluate on previously defined test cohort
# ------------------------------------------------------------

p_test = np.asarray(
    sensitivity_model.predict(X_test),
    dtype=float
).ravel()

lower = p_test - q_hat
upper = p_test + q_hat

# Total motor score valid bounds: 0 to 100.
# This clipping preserves inclusion for outcomes
# within that known domain.

lower = np.maximum(lower, 0.0)
upper = np.minimum(upper, 100.0)

covered = (
    (y_test >= lower)
    & (y_test <= upper)
)

coverage = float(np.mean(covered))

coverage_ci = binomtest(
    int(covered.sum()),
    n=len(covered)
).proportion_ci(
    confidence_level=0.95,
    method="exact"
)

results = pd.DataFrame({
    "Observed": y_test,
    "Prediction": p_test,
    "Lower_95PI": lower,
    "Upper_95PI": upper,
    "Covered": covered,
    "Interval_Width": upper - lower
})

summary = {
    "fit_patients": len(fit_idx),
    "calibration_patients": len(cal_idx),
    "test_patients": len(y_test),
    "group_overlap_fit_calibration": 0,
    "q_hat": q_hat,
    "coverage": coverage,
    "coverage_ci_lower": float(coverage_ci.low),
    "coverage_ci_upper": float(coverage_ci.high),
    "mean_width": float(
        results["Interval_Width"].mean()
    ),
    "test_mae": float(
        mean_absolute_error(y_test, p_test)
    )
}

# ------------------------------------------------------------
# 5. Save exploratory results
# ------------------------------------------------------------

results.to_csv(
    OUT_DIR /
    "Table31_12B_TrainingOnly_Calibration.csv",
    index=False
)

report = {
    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "method":
        "Training-only split calibration sensitivity",

    "results": summary,

    "calibration_patient_separation_verified": True,

    "hyperparameter_selection_independence_verified":
        False,

    "formal_95_percent_coverage_guarantee":
        False,

    "replaces_original_final_model": False,

    "original_test_cohort_previously_inspected": True,

    "limitations": [
        "Final pipeline hyperparameters may have been influenced by earlier development data.",
        "The test cohort was previously inspected.",
        "This is an exploratory sensitivity model, not a replacement for the final model.",
        "No formal prospective coverage claim is made."
    ]
}

report_path = (
    META / "calibration_sensitivity_audit.json"
)

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\nCALIBRATION SENSITIVITY RESULTS")

for key, value in summary.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("\nOriginal final model: UNCHANGED")
print("New sensitivity model trained: YES")
print("Formal conformal independence: NOT VERIFIED")
print("Report saved:", report_path)
print("\nSTEP 31.12B COMPLETED")


In [ ]:

# ============================================================
# CORRECTED STEP 31.13: DCA METHODOLOGICAL AUDIT
# Invalid DCA Removed | No Model Retraining
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 31.13: DECISION CURVE ANALYSIS AUDIT")
print("=" * 70)

dca_audit = pd.DataFrame({
    "Item": [
        "Prediction target",
        "Model output",
        "Original DCA probability",
        "Methodological issue",
        "Correction",
        "Clinical net benefit claim",
        "Recommended reporting"
    ],
    "Assessment": [
        "Continuous Week-26 total motor score",
        "Regression prediction in motor-score points",
        "Regression prediction divided by 100",
        "Scaled regression predictions are not calibrated event probabilities",
        "Original DCA comparison excluded",
        "Not supported by the current regression analysis",
        "Report regression accuracy, calibration, and prediction intervals"
    ]
})

display(dca_audit)

# Save audit record without generating new DCA results

output_dir = Path(TABLE_VALIDATION)
output_dir.mkdir(parents=True, exist_ok=True)

output_file = (
    output_dir /
    "Table31_13_DCA_Methodological_Audit.xlsx"
)

dca_audit.to_excel(output_file, index=False)

# Clear invalid DCA objects from the current session,
# if they were created by the old notebook cells.

invalid_variables = [
    "dca_comparison",
    "dca_results",
    "ml_probability",
    "ais_probability",
    "ml_nb",
    "ais_nb",
    "treat_all",
    "treat_none",
    "y_true_dca"
]

for variable in invalid_variables:
    globals().pop(variable, None)

print("\nOriginal DCA comparison: EXCLUDED")
print("Invalid probability scaling: REMOVED")
print("Clinical net benefit claims: NOT SUPPORTED")
print("Model retraining: No")
print("Held-out test outcomes accessed: No")
print("Audit saved:", output_file)
print("STEP 31.13 CORRECTION COMPLETED")


In [ ]:

# ============================================================
# CORRECTED STEP 31.14
# PREDICTION INTERVAL COVERAGE VERIFICATION
# Reuses STEP 31.12 | No New Calibration
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 31.14: PREDICTION INTERVAL COVERAGE VERIFICATION")
print("=" * 70)

# 1. Require corrected STEP 31.12 results

if "prediction_interval_results" not in globals():
    raise RuntimeError(
        "Run corrected STEP 31.12 before STEP 31.14."
    )

pi_df = prediction_interval_results.copy()

required_columns = [
    "Observed_TOTMOT26",
    "Predicted_TOTMOT26",
    "Lower_95PI",
    "Upper_95PI",
    "Interval_Width",
    "Covered"
]

missing = [
    col for col in required_columns
    if col not in pi_df.columns
]

if missing:
    raise ValueError(f"Missing interval columns: {missing}")

# 2. Verify existing intervals

observed = pi_df["Observed_TOTMOT26"].to_numpy(dtype=float)
lower = pi_df["Lower_95PI"].to_numpy(dtype=float)
upper = pi_df["Upper_95PI"].to_numpy(dtype=float)

if not (
    np.isfinite(observed).all()
    and np.isfinite(lower).all()
    and np.isfinite(upper).all()
):
    raise ValueError("Non-finite prediction interval values.")

if not np.all(lower <= upper):
    raise ValueError("Invalid interval boundaries.")

covered = (observed >= lower) & (observed <= upper)
width = upper - lower

assert np.array_equal(
    covered,
    pi_df["Covered"].to_numpy(dtype=bool)
), "Coverage flags differ from STEP 31.12."

assert np.allclose(
    width,
    pi_df["Interval_Width"].to_numpy(dtype=float)
), "Interval widths differ from STEP 31.12."

# 3. Summarize existing coverage

n_patients = len(pi_df)
n_covered = int(covered.sum())

coverage_results = pd.DataFrame({
    "Metric": [
        "Nominal coverage",
        "Observed coverage",
        "Patients evaluated",
        "Patients covered",
        "Patients not covered",
        "Mean interval width",
        "Median interval width"
    ],
    "Value": [
        0.95,
        n_covered / n_patients,
        n_patients,
        n_covered,
        n_patients - n_covered,
        width.mean(),
        np.median(width)
    ]
})

print("\nVERIFIED COVERAGE SUMMARY")
display(coverage_results.round(4))

# 4. Cross-check with STEP 31.12 summary

if "interval_summary" in globals():

    previous_summary = dict(
        zip(
            interval_summary["Metric"],
            interval_summary["Value"]
        )
    )

    if "Observed test coverage" in previous_summary:
        assert np.isclose(
            n_covered / n_patients,
            float(previous_summary["Observed test coverage"])
        ), "Coverage mismatch with STEP 31.12."

    print("\nSTEP 31.12 summary consistency: PASSED")

# 5. Save verification table

output_dir = Path(TABLE_VALIDATION)
output_dir.mkdir(parents=True, exist_ok=True)

output_file = (
    output_dir /
    "Table31_14_Prediction_Interval_Coverage.xlsx"
)

coverage_results.to_excel(
    output_file,
    index=False
)

print("\nOld prediction_interval dependency: REMOVED")
print("Duplicate interval calibration: AVOIDED")
print("Model retraining: No")
print("Saved:", output_file)
print("STEP 31.14 CORRECTION COMPLETED")


In [ ]:

# ============================================================
# CORRECTED STEP 31.15
# PATIENT-LEVEL PREDICTION REPORT
# Reuses STEP 31.12 | No Retraining
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 31.15: CORRECTED PATIENT-LEVEL PREDICTION REPORT")
print("=" * 70)

# 1. Require corrected conformal results

if "prediction_interval_results" not in globals():
    raise RuntimeError(
        "Run corrected STEP 31.12 before STEP 31.15."
    )

source = prediction_interval_results.copy()

required_columns = [
    "Observed_TOTMOT26",
    "Predicted_TOTMOT26",
    "Lower_95PI",
    "Upper_95PI",
    "Interval_Width",
    "Covered"
]

missing = [
    col for col in required_columns
    if col not in source.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# 2. Prepare patient-level report

clinical_output = source[required_columns].copy()

clinical_output.insert(
    0,
    "Patient_Row",
    np.arange(1, len(clinical_output) + 1)
)

# 3. Verify interval consistency

lower = clinical_output["Lower_95PI"].to_numpy(dtype=float)
upper = clinical_output["Upper_95PI"].to_numpy(dtype=float)

assert np.all(lower <= upper)

np.testing.assert_allclose(
    upper - lower,
    clinical_output["Interval_Width"].to_numpy(dtype=float)
)

# 4. Summarize results

print("\nPATIENT-LEVEL PREDICTION REPORT")
display(clinical_output.head(10).round(4))

print("\nREPORT VERIFICATION")
print("Patients:", len(clinical_output))
print("Observed coverage:",
      round(clinical_output["Covered"].mean(), 4))
print("Mean interval width:",
      round(clinical_output["Interval_Width"].mean(), 4))

# 5. Save corrected report

output_dir = Path(TABLE_VALIDATION)
output_dir.mkdir(parents=True, exist_ok=True)

output_file = (
    output_dir /
    "Table31_15_Patient_Prediction_Report.xlsx"
)

clinical_output.to_excel(
    output_file,
    index=False
)

print("\nOld prediction dependency: REMOVED")
print("Old prediction_interval dependency: REMOVED")
print("Unsupported confidence categories: REMOVED")
print("Duplicate interval calculation: AVOIDED")
print("Model retraining: No")
print("STEP 31.15 CORRECTION COMPLETED")


In [ ]:

# ============================================================
# CORRECTED STEP 31.16: DEPLOYABLE MODEL EXPORT
# Raw Feature Alignment | Reload Verification
# No Retraining | No Test Outcome Access
# ============================================================

import joblib
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline

print("=" * 70)
print("STEP 31.16: CORRECTED MODEL EXPORT")
print("=" * 70)

MODEL_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project/05_models"
)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Verify fitted pipeline
assert isinstance(final_lgbm_model, Pipeline)

fitted_preprocessor = final_lgbm_model.named_steps["preprocessor"]
fitted_estimator = final_lgbm_model.named_steps["model"]

# 2. Recover exact raw and transformed feature names
raw_features = list(fitted_preprocessor.feature_names_in_)
processed_features = list(
    fitted_preprocessor.get_feature_names_out()
)

X_reference = MODEL_INPUTS["Model_B16"]["X_validation"].copy()

assert raw_features == list(X_reference.columns), (
    "Raw feature order mismatch."
)

assert len(raw_features) == final_lgbm_model.n_features_in_

assert len(processed_features) == fitted_estimator.n_features_in_

# 3. Save complete fitted pipeline and metadata
model_path = MODEL_DIR / "SCI_Dynamic_Recovery_LightGBM_Model.pkl"
raw_path = MODEL_DIR / "SCI_Model_Feature_List.pkl"
processed_path = MODEL_DIR / "SCI_Model_Processed_Feature_List.pkl"

joblib.dump(final_lgbm_model, model_path)
joblib.dump(raw_features, raw_path)
joblib.dump(processed_features, processed_path)

# 4. Reload all artifacts
loaded_model = joblib.load(model_path)
loaded_raw = joblib.load(raw_path)
loaded_processed = joblib.load(processed_path)

assert isinstance(loaded_model, Pipeline)
assert loaded_raw == raw_features
assert loaded_processed == processed_features

# 5. Verify predictions using RAW validation inputs
X_check = X_reference.iloc[:5].copy()

original_predictions = np.asarray(
    final_lgbm_model.predict(X_check)
).ravel()

loaded_predictions = np.asarray(
    loaded_model.predict(X_check)
).ravel()

np.testing.assert_allclose(
    original_predictions,
    loaded_predictions,
    rtol=1e-10,
    atol=1e-10
)

# 6. Display results
print("\nRAW CLINICAL FEATURES:")
for i, feature in enumerate(raw_features, 1):
    print(f"{i}. {feature}")

print("\nRaw input feature count:", len(raw_features))
print("Transformed feature count:", len(processed_features))

print("\nORIGINAL VS RELOADED PREDICTIONS")
for original, reloaded in zip(
    original_predictions, loaded_predictions
):
    print(f"{original:.6f} | {reloaded:.6f}")

print("\nSaved artifacts:")
print(model_path)
print(raw_path)
print(processed_path)

print("\nRaw feature alignment: PASSED")
print("Processed feature alignment: PASSED")
print("Model reload verification: PASSED")
print("Prediction consistency: PASSED")
print("Model retraining: No")
print("Test outcomes accessed: No")
print("STEP 31.16 CORRECTION COMPLETED")


In [ ]:

# ============================================================
# STEP 31.16A: SAVED-MODEL PREDICTION REPRODUCTION
# Model_B16 | 99 Internal Test Patients
# No Retraining | No Model Overwriting
# ============================================================

import json
import hashlib
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 31.16A: SAVED-MODEL REPRODUCTION AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define original artifact paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

MODEL_DIR = PROJECT_DIR / "05_models"

AUDIT_DIR = (
    PROJECT_DIR / "01_data/metadata"
)

RESULT_DIR = (
    PROJECT_DIR / "06_results/tables/reproducibility"
)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = (
    MODEL_DIR /
    "SCI_Dynamic_Recovery_LightGBM_Model.pkl"
)

RAW_FEATURE_PATH = (
    MODEL_DIR /
    "SCI_Model_Feature_List.pkl"
)

PROCESSED_FEATURE_PATH = (
    MODEL_DIR /
    "SCI_Model_Processed_Feature_List.pkl"
)

for path in [
    MODEL_PATH,
    RAW_FEATURE_PATH,
    PROCESSED_FEATURE_PATH
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required saved artifact missing: {path}"
        )

print("\nAll three model artifacts found.")

# ------------------------------------------------------------
# 2. Calculate original artifact SHA-256 hashes
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


artifact_hashes = {
    "model_sha256": sha256_file(MODEL_PATH),
    "raw_features_sha256": sha256_file(
        RAW_FEATURE_PATH
    ),
    "processed_features_sha256": sha256_file(
        PROCESSED_FEATURE_PATH
    )
}

print("\nARTIFACT HASHES")

for name, value in artifact_hashes.items():
    print(name, ":", value)

# ------------------------------------------------------------
# 3. Reload the saved fitted pipeline
# ------------------------------------------------------------

# Only load trusted artifacts generated by
# your own notebook. Joblib uses pickle.

reloaded_model = joblib.load(MODEL_PATH)

saved_raw_features = list(
    joblib.load(RAW_FEATURE_PATH)
)

saved_processed_features = list(
    joblib.load(PROCESSED_FEATURE_PATH)
)

if not isinstance(reloaded_model, Pipeline):
    raise TypeError(
        "Saved artifact is not a sklearn Pipeline."
    )

check_is_fitted(reloaded_model)

if not all(
    step in reloaded_model.named_steps
    for step in ["preprocessor", "model"]
):
    raise RuntimeError(
        "Pipeline is missing required steps."
    )

loaded_preprocessor = (
    reloaded_model.named_steps["preprocessor"]
)

loaded_estimator = (
    reloaded_model.named_steps["model"]
)

# ------------------------------------------------------------
# 4. Verify saved feature names and order
# ------------------------------------------------------------

model_raw_features = list(
    loaded_preprocessor.feature_names_in_
)

model_processed_features = list(
    loaded_preprocessor.get_feature_names_out()
)

assert saved_raw_features == model_raw_features, (
    "Saved raw feature list differs from model."
)

assert (
    saved_processed_features
    == model_processed_features
), "Processed feature list mismatch."

assert len(saved_raw_features) == (
    reloaded_model.n_features_in_
)

assert len(saved_processed_features) == (
    loaded_estimator.n_features_in_
)

print("\nFEATURE VERIFICATION")

print(
    "Raw feature count:",
    len(saved_raw_features)
)

print(
    "Processed feature count:",
    len(saved_processed_features)
)

print("Raw feature order: PASSED")
print("Processed feature order: PASSED")

# ------------------------------------------------------------
# 5. Load original fixed test observations
# ------------------------------------------------------------

if "MODEL_INPUTS" not in globals():
    raise RuntimeError(
        "MODEL_INPUTS missing. Run Step 9.6."
    )

if "test_df" not in globals():
    raise RuntimeError(
        "test_df missing. Restore original splits."
    )

data = MODEL_INPUTS["Model_B16"]

X_test = data["X_test"].copy()

y_test = np.asarray(
    data["y_test"],
    dtype=float
).ravel()

assert len(X_test) == len(y_test)
assert len(X_test) == 99

assert X_test.index.equals(test_df.index)

assert list(X_test.columns) == saved_raw_features, (
    "Test feature order differs from saved model."
)

assert "PTID" in test_df.columns

patient_ids = (
    test_df.loc[X_test.index, "PTID"]
    .astype("string")
    .str.strip()
)

assert patient_ids.notna().all()
assert patient_ids.is_unique

source_y = pd.to_numeric(
    test_df.loc[
        X_test.index, "TOTMOT26"
    ],
    errors="coerce"
).to_numpy(dtype=float)

np.testing.assert_allclose(
    y_test,
    source_y,
    rtol=0,
    atol=1e-8
)

print("\nTEST COHORT")
print("Patients:", len(y_test))
print("Patient IDs unique: Yes")
print("Target/source alignment: PASSED")

# ------------------------------------------------------------
# 6. Reproduce all 99 saved-model predictions
# ------------------------------------------------------------

reproduced_predictions = np.asarray(
    reloaded_model.predict(X_test),
    dtype=float
).ravel()

assert len(reproduced_predictions) == 99
assert np.isfinite(reproduced_predictions).all()
assert np.isfinite(y_test).all()

# ------------------------------------------------------------
# 7. Load previously exported test predictions
# ------------------------------------------------------------

candidate_paths = [
    PROJECT_DIR /
    "03_tables/Table14_Final_Test_Predictions.csv",

    PROJECT_DIR /
    "06_results/tables/dataset/"
    "Table14_Final_Test_Predictions.csv"
]

original_prediction_path = next(
    (p for p in candidate_paths if p.is_file()),
    None
)

if original_prediction_path is None:
    raise FileNotFoundError(
        "Original Table14_Final_Test_Predictions.csv "
        "not found in the expected project directories. "
        "Do not replace it with newly generated predictions."
    )

original_df = pd.read_csv(
    original_prediction_path
)

required_columns = [
    "Actual_TOTMOT26",
    "Predicted_TOTMOT26"
]

for col in required_columns:
    if col not in original_df.columns:
        raise KeyError(
            f"Original prediction file missing {col}"
        )

assert len(original_df) == len(y_test)

original_y = pd.to_numeric(
    original_df["Actual_TOTMOT26"],
    errors="coerce"
).to_numpy(dtype=float)

original_predictions = pd.to_numeric(
    original_df["Predicted_TOTMOT26"],
    errors="coerce"
).to_numpy(dtype=float)

# Original prediction file is row-ordered.
# Compare outcomes first to establish that
# the saved row order agrees with test data.

np.testing.assert_allclose(
    original_y,
    y_test,
    rtol=0,
    atol=1e-8,
    err_msg="Original saved test outcomes differ."
)

# ------------------------------------------------------------
# 8. Verify prediction reproduction
# ------------------------------------------------------------

absolute_differences = np.abs(
    reproduced_predictions - original_predictions
)

MAX_ALLOWED_DIFFERENCE = 1e-5

maximum_difference = float(
    np.max(absolute_differences)
)

mean_difference = float(
    np.mean(absolute_differences)
)

predictions_match = bool(
    np.allclose(
        reproduced_predictions,
        original_predictions,
        rtol=0,
        atol=MAX_ALLOWED_DIFFERENCE
    )
)

print("\nPREDICTION REPRODUCTION")

print(
    "Maximum absolute difference:",
    maximum_difference
)

print(
    "Mean absolute difference:",
    mean_difference
)

print(
    "Saved predictions reproduced:",
    predictions_match
)

# ------------------------------------------------------------
# 9. Recalculate final test metrics
# ------------------------------------------------------------

reproduced_mae = float(
    mean_absolute_error(
        y_test, reproduced_predictions
    )
)

reproduced_rmse = float(
    np.sqrt(
        mean_squared_error(
            y_test, reproduced_predictions
        )
    )
)

reproduced_r2 = float(
    r2_score(
        y_test, reproduced_predictions
    )
)

metrics_df = pd.DataFrame([
    {
        "Metric": "MAE",
        "Reproduced": reproduced_mae,
        "Previously_Reported": 5.5422
    },
    {
        "Metric": "RMSE",
        "Reproduced": reproduced_rmse,
        "Previously_Reported": 10.3047
    },
    {
        "Metric": "R2",
        "Reproduced": reproduced_r2,
        "Previously_Reported": 0.8800
    }
])

metrics_df["Absolute_Difference"] = (
    metrics_df["Reproduced"]
    - metrics_df["Previously_Reported"]
).abs()

# Historical published values are rounded
# to four decimal places.

metrics_match = bool(
    (
        metrics_df["Absolute_Difference"]
        <= 0.00015
    ).all()
)

print("\nREPRODUCED TEST PERFORMANCE")

display(metrics_df.round(6))

print("Metrics match reported values:", metrics_match)

# ------------------------------------------------------------
# 10. Verify model against in-memory pipeline
# ------------------------------------------------------------

in_memory_comparison = None

if "final_lgbm_model" in globals():

    memory_predictions = np.asarray(
        final_lgbm_model.predict(X_test),
        dtype=float
    ).ravel()

    in_memory_comparison = bool(
        np.allclose(
            memory_predictions,
            reproduced_predictions,
            rtol=1e-10,
            atol=1e-10
        )
    )

    print(
        "\nReloaded vs in-memory model:",
        in_memory_comparison
    )

# ------------------------------------------------------------
# 11. Create independent audit output
# ------------------------------------------------------------

audit_predictions = pd.DataFrame({
    "Original_Prediction": original_predictions,
    "Reloaded_Prediction": reproduced_predictions,
    "Absolute_Difference": absolute_differences,
    "Actual_TOTMOT26": y_test
})

prediction_audit_path = (
    RESULT_DIR /
    "saved_model_prediction_reproduction.csv"
)

metrics_path = (
    RESULT_DIR /
    "saved_model_reproduced_metrics.csv"
)

audit_predictions.to_csv(
    prediction_audit_path,
    index=False
)

metrics_df.to_csv(
    metrics_path,
    index=False
)

# ------------------------------------------------------------
# 12. Save machine-readable verification report
# ------------------------------------------------------------

full_reproduction_passed = bool(
    predictions_match
    and metrics_match
    and (
        in_memory_comparison is not False
    )
)

report = {
    "audit_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "model_path": str(MODEL_PATH),

    "original_prediction_path":
        str(original_prediction_path),

    "landmark": "Model_B16",

    "test_patients": len(y_test),

    "raw_feature_order_verified": True,

    "processed_feature_order_verified": True,

    "test_target_alignment_verified": True,

    "predictions_match_original_export":
        predictions_match,

    "maximum_prediction_difference":
        maximum_difference,

    "mean_prediction_difference":
        mean_difference,

    "metrics_match_reported_values":
        metrics_match,

    "reproduced_metrics": {
        "MAE": reproduced_mae,
        "RMSE": reproduced_rmse,
        "R2": reproduced_r2
    },

    "in_memory_model_comparison":
        in_memory_comparison,

    "artifact_hashes": artifact_hashes,

    "saved_model_reproduction_verified":
        full_reproduction_passed,

    "clean_environment_reproduction_verified":
        False,

    "model_retrained": False,

    "limitations": [
        "Reproduction uses the current Kaggle environment.",
        "A separate clean-environment reproduction is still required.",
        "The original prediction CSV has no PTID column; its row order is checked against the fixed test targets.",
        "The audit does not establish external validity.",
        "Only trusted joblib artifacts should be loaded."
    ]
}

report_path = (
    AUDIT_DIR /
    "saved_model_reproduction_audit.json"
)

report_path.write_text(
    json.dumps(report, indent=4),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 13. Final verification
# ------------------------------------------------------------

print("\nFINAL REPRODUCTION STATUS")

print(
    "Raw feature verification: PASSED"
)

print(
    "Processed feature verification: PASSED"
)

print(
    "Original predictions match:",
    predictions_match
)

print(
    "Original metrics match:",
    metrics_match
)

print(
    "Saved-model reproduction verified:",
    full_reproduction_passed
)

print(
    "Clean-environment reproduction verified: No"
)

print("\nFILES SAVED")

print(prediction_audit_path)
print(metrics_path)
print(report_path)

if not full_reproduction_passed:
    raise AssertionError(
        "Saved-model reproduction did not fully pass. "
        "Inspect the generated audit report."
    )

print("\nNo model retraining performed.")
print("Original saved model was not overwritten.")

print("\nSTEP 31.16A COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# STEP 32.0 — LOCATE CLEAN-SESSION REPRODUCTION REPORT
# Purpose: Find the required report before generating README.
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"

    
)

EXPECTED_REPORT = (
    PROJECT_DIR
    / "01_data"
    / "metadata"
    / "clean_session_reproduction_result.json"
)

print("Expected report:")
print(EXPECTED_REPORT)

print("\nExpected path exists:", EXPECTED_REPORT.is_file())

# Search the project and Kaggle working directory.
search_roots = [
    PROJECT_DIR,
    Path("/kaggle/working")
]

found_reports = []

for root in search_roots:
    if root.exists():
        for path in root.rglob(
            "clean_session_reproduction_result.json"
        ):
            if path.is_file() and path not in found_reports:
                found_reports.append(path)

print("\nMatching reports found:")

if found_reports:
    for path in found_reports:
        print(" -", path)
else:
    print("No matching report found.")

print("\nFinal reproducibility audit exists:",
      (PROJECT_DIR / "01_data/metadata/"
       "final_reproducibility_audit.json").is_file())


In [ ]:

# ============================================================
# STEP 32.0.1 — CLEAN-SESSION REPRODUCTION PREREQUISITES AUDIT
# Purpose: Locate the actual files required for reproduction.
# Does not train models or fabricate verification reports.
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

SEARCH_ROOTS = [
    PROJECT_DIR,
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

TARGET_NAMES = {
    "SCI_Dynamic_Recovery_LightGBM_Model.pkl",
    "SCI_Model_Feature_List.pkl",
    "SCI_Model_Processed_Feature_List.pkl",
    "clean_session_reproduction_result.json",
    "final_reproducibility_audit.json",
    "requirements.txt",
    "environment.yml",
    "MODEL_CARD.md"
}

KEYWORDS = (
    "prediction",
    "predictions",
    "test",
    "feature",
    "dataset",
    "reproduction",
    "audit",
    "metric"
)

found = {}

for root in SEARCH_ROOTS:
    if not root.exists():
        print(f"Search root unavailable: {root}")
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        name_lower = path.name.lower()

        if (
            path.name in TARGET_NAMES
            or any(word in name_lower for word in KEYWORDS)
        ):
            found[str(path)] = path.stat().st_size

print("\n=== REPRODUCTION FILE INVENTORY ===")

if not found:
    print("No candidate files found.")
else:
    for path, size in sorted(found.items()):
        print(f"{size:>12,} bytes | {path}")

print("\n=== REQUIRED MODEL ARTIFACTS ===")

required_artifacts = [
    PROJECT_DIR / "05_models/SCI_Dynamic_Recovery_LightGBM_Model.pkl",
    PROJECT_DIR / "05_models/SCI_Model_Feature_List.pkl",
    PROJECT_DIR / "05_models/SCI_Model_Processed_Feature_List.pkl"
]

for path in required_artifacts:
    print(f"{'FOUND' if path.is_file() else 'MISSING'}: {path}")


In [ ]:

# ============================================================
# STEP 32.0.2 — INSPECT EXISTING REPRODUCIBILITY EVIDENCE
# Purpose: Read existing reports without fabricating evidence.
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

SOURCE_REPORT = Path(
    "/kaggle/input/datasets/shizansarkar/"
    "sci-independent-reproduction-evidence/"
    "clean_session_reproduction_result.json"
)

EXISTING_AUDIT = (
    PROJECT_DIR
    / "01_data/metadata/saved_model_reproduction_audit.json"
)

for label, path in [
    ("Independent reproduction report", SOURCE_REPORT),
    ("Existing saved-model reproduction audit", EXISTING_AUDIT),
]:
    print("\n" + "=" * 70)
    print(label)
    print("Path:", path)
    print("Exists:", path.is_file())

    if path.is_file():
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            print(json.dumps(data, indent=2))
        except Exception as exc:
            print("Could not parse report:", repr(exc))


In [ ]:

# ============================================================
# STEP 32.1.1 — DEBUG + RECOVER CANONICAL REGISTRY
# No invented metrics; existing evidence only.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
TABLES = ROOT / "06_results" / "tables"
PRIMARY = TABLES / "primary"
AUDIT = ROOT / "06_results" / "audit"

for folder in [PRIMARY, AUDIT]:
    folder.mkdir(parents=True, exist_ok=True)

REGISTRY = PRIMARY / "Canonical_Evaluation_Registry.csv"
REGISTRY_README = PRIMARY / "Canonical_Evaluation_Registry_README.txt"
PROJECT_README = ROOT / "README.md"

# ------------------------------------------------------------
# 1. Diagnose available files and in-memory objects
# ------------------------------------------------------------
print("=" * 65)
print("STEP 32.1 — REGISTRY DEBUGGING")
print("=" * 65)

for name in [
    "verified_results_df",
    "validation_result",
    "test_result",
    "nested_cv_summary",
    "nested_cv_summary_df",
]:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: DataFrame shape={obj.shape}, "
              f"columns={obj.columns.tolist()}")
    elif obj is not None:
        print(f"{name}: available ({type(obj).__name__})")
    else:
        print(f"{name}: NOT AVAILABLE")

existing = list(TABLES.rglob("Canonical_Evaluation_Registry.csv"))
print("\nExisting registry files:")
for p in existing:
    print(" -", p)

# ------------------------------------------------------------
# 2. Load an existing registry if one is present
# ------------------------------------------------------------
registry_df = None

if REGISTRY.is_file():
    registry_df = pd.read_csv(REGISTRY)

elif len(existing) == 1:
    registry_df = pd.read_csv(existing[0])
    REGISTRY = existing[0]

elif len(existing) > 1:
    raise RuntimeError(
        "Multiple registries found. Resolve the canonical file first:\n"
        + "\n".join(str(p) for p in existing)
    )

# ------------------------------------------------------------
# 3. If no registry exists, reconstruct only from real results
# ------------------------------------------------------------
if registry_df is None:
    required_objects = [
        "verified_results_df",
        "validation_result",
        "test_result",
    ]

    missing_objects = [
        n for n in required_objects
        if not isinstance(globals().get(n), pd.DataFrame)
    ]

    if missing_objects:
        raise RuntimeError(
            "Registry cannot safely be rebuilt. Missing actual result "
            f"DataFrames: {missing_objects}. Run STEP 14.9 and STEP 31.9 "
            "successfully, then rerun this cell. No metrics were invented."
        )

    landmark_df = verified_results_df.copy()
    required_landmark = {
        "Model", "Landmark", "Algorithm",
        "N_Validation", "MAE", "RMSE", "R2"
    }

    if not required_landmark.issubset(landmark_df.columns):
        raise ValueError(
            "verified_results_df schema mismatch. Missing: "
            f"{sorted(required_landmark - set(landmark_df.columns))}"
        )

    rows = []

    for _, r in landmark_df.iterrows():
        rows.append({
            "Evaluation_Type": "Validation",
            "Analysis_Role": "Landmark comparison",
            "Model": r["Model"],
            "Landmark": r["Landmark"],
            "Algorithm": r["Algorithm"],
            "N_Evaluation": int(r["N_Validation"]),
            "N_Training_Partition": np.nan,
            "N_Outer_Folds": np.nan,
            "MAE": float(r["MAE"]),
            "RMSE": float(r["RMSE"]),
            "R2": float(r["R2"]),
            "MAE_SD": np.nan,
            "RMSE_SD": np.nan,
            "R2_SD": np.nan,
            "Source": "verified_results_df",
            "Interpretation_Note": "Landmark validation comparison."
        })

    def extract_result(obj, label):
        if not isinstance(obj, pd.DataFrame) or obj.empty:
            raise ValueError(f"{label} is empty or unavailable.")

        row = obj.iloc[0]
        required = ["MAE", "RMSE", "R2"]

        if not all(c in obj.columns for c in required):
            raise ValueError(f"{label} lacks required metrics.")

        values = {c: float(row[c]) for c in required}

        if not np.isfinite(list(values.values())).all():
            raise ValueError(f"{label} has non-finite metrics.")

        return row, values

    vrow, vm = extract_result(validation_result, "validation_result")
    trow, tm = extract_result(test_result, "test_result")

    rows.append({
        "Evaluation_Type": "Validation",
        "Analysis_Role": "Optimized final model",
        "Model": str(vrow.get("Model", "Optimized LightGBM")),
        "Landmark": "Week 16",
        "Algorithm": "Optimized LightGBM",
        "N_Evaluation": np.nan,
        "N_Training_Partition": np.nan,
        "N_Outer_Folds": np.nan,
        **vm,
        "MAE_SD": np.nan,
        "RMSE_SD": np.nan,
        "R2_SD": np.nan,
        "Source": "validation_result",
        "Interpretation_Note": "Optimized-model validation result."
    })

    rows.append({
        "Evaluation_Type": "Internal test",
        "Analysis_Role": "Optimized final model evaluation",
        "Model": str(trow.get("Model", "Optimized LightGBM")),
        "Landmark": "Week 16",
        "Algorithm": "Optimized LightGBM",
        "N_Evaluation": np.nan,
        "N_Training_Partition": np.nan,
        "N_Outer_Folds": np.nan,
        **tm,
        "MAE_SD": np.nan,
        "RMSE_SD": np.nan,
        "R2_SD": np.nan,
        "Source": "test_result",
        "Interpretation_Note":
            "Internal test; historical test independence UNKNOWN."
    })

    # Nested CV must come from the actual saved/in-memory summary.
    nested = globals().get("nested_cv_summary")

    if not isinstance(nested, pd.DataFrame) or nested.empty:
        nested = globals().get("nested_cv_summary_df")

    if not isinstance(nested, pd.DataFrame) or nested.empty:
        raise RuntimeError(
            "Validation/test rows are available, but nested-CV summary "
            "is not. Run STEP 15.6 first. Registry was not partially saved."
        )

    required_nested = {"Metric", "Mean", "SD"}
    if not required_nested.issubset(nested.columns):
        raise ValueError("Nested-CV summary has an unexpected schema.")

    if "Landmark" in nested.columns:
        b16 = nested[nested["Landmark"].astype(str).eq("Model_B16")]
        if not b16.empty:
            nested = b16

    def nval(metric, column):
        matched = nested[
            nested["Metric"].astype(str).str.upper().eq(metric)
        ]
        if len(matched) != 1:
            raise ValueError(
                f"Expected one nested-CV {metric} row; found {len(matched)}."
            )
        value = float(matched.iloc[0][column])
        if not np.isfinite(value):
            raise ValueError(f"Non-finite nested-CV {metric}/{column}.")
        return value

    rows.append({
        "Evaluation_Type": "Nested cross-validation",
        "Analysis_Role": "Internal resampling estimate",
        "Model": "Model_B16",
        "Landmark": "Week 16",
        "Algorithm": "LightGBM",
        "N_Evaluation": np.nan,
        "N_Training_Partition": 461,
        "N_Outer_Folds": 5,
        "MAE": nval("MAE", "Mean"),
        "RMSE": nval("RMSE", "Mean"),
        "R2": nval("R2", "Mean"),
        "MAE_SD": nval("MAE", "SD"),
        "RMSE_SD": nval("RMSE", "SD"),
        "R2_SD": nval("R2", "SD"),
        "Source": "in-memory nested CV summary",
        "Interpretation_Note":
            "Nested-CV estimate; not the internal-test result."
    })

    registry_df = pd.DataFrame(rows)
    registry_df.to_csv(REGISTRY, index=False)
    print("\nRegistry reconstructed from existing result objects.")

# ------------------------------------------------------------
# 4. Validate registry before generating documentation
# ------------------------------------------------------------
required_cols = {
    "Evaluation_Type", "Analysis_Role", "Model",
    "Landmark", "MAE", "RMSE", "R2", "Source"
}

missing_cols = required_cols - set(registry_df.columns)
if missing_cols:
    raise ValueError(f"Registry missing columns: {sorted(missing_cols)}")

if registry_df.empty:
    raise ValueError("Registry has zero rows.")

types = set(registry_df["Evaluation_Type"].dropna().astype(str))
required_types = {
    "Validation", "Nested cross-validation", "Internal test"
}

if not required_types.issubset(types):
    raise ValueError(
        f"Registry missing evaluation types: {sorted(required_types - types)}"
    )

# ------------------------------------------------------------
# 5. Generate README files
# ------------------------------------------------------------
PROJECT_README.write_text(
    "# SCI Recovery ML Project\n\n"
    "## Canonical evaluation registry\n\n"
    f"Registry: `{REGISTRY.relative_to(ROOT)}`\n\n"
    f"Number of records: {len(registry_df)}\n\n"
    "Evaluation types:\n"
    + "".join(f"- {t}\n" for t in sorted(types))
    + "\n## Methodological limitations\n\n"
    "- Historical test-set independence remains UNKNOWN.\n"
    "- External validation has NOT been performed.\n"
    "- Full training pipeline reproduction is NOT VERIFIED.\n"
    "- MAE alone does not establish clinical meaningfulness.\n",
    encoding="utf-8"
)

REGISTRY_README.write_text(
    "CANONICAL EVALUATION REGISTRY\n\n"
    "Validation, nested cross-validation and internal test results "
    "are separate evaluation procedures. Do not substitute one "
    "for another.\n"
    "Historical test-set independence: UNKNOWN.\n"
    "External validation: NOT PERFORMED.\n",
    encoding="utf-8"
)

audit = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "registry_path": str(REGISTRY),
    "registry_rows": int(len(registry_df)),
    "evaluation_types": sorted(types),
    "project_readme": str(PROJECT_README),
    "registry_readme": str(REGISTRY_README),
    "historical_test_set_independence": "UNKNOWN",
    "external_validation_completed": False,
    "full_training_pipeline_reproduction_verified": False,
    "status": "PASSED"
}

audit_path = AUDIT / "STEP32_1_registry_debug_audit.json"
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")

print("\nSTEP 32.1 PASSED")
print("Registry:", REGISTRY)
print("Rows:", len(registry_df))
print("Evaluation types:", sorted(types))
print("Project README:", PROJECT_README)
print("Audit:", audit_path)
print("Historical test independence: UNKNOWN")
print("External validation: NOT PERFORMED")


In [ ]:

# ============================================================
# STEP 32.1.2 — CANONICAL REGISTRY INTEGRITY AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
REGISTRY = (
    ROOT / "06_results" / "tables" / "primary"
    / "Canonical_Evaluation_Registry.csv"
)
AUDIT_DIR = ROOT / "06_results" / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

if not REGISTRY.is_file():
    raise FileNotFoundError(f"Registry not found: {REGISTRY}")

df = pd.read_csv(REGISTRY)

required = [
    "Evaluation_Type", "Analysis_Role", "Model",
    "Landmark", "MAE", "RMSE", "R2", "Source"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

metrics = ["MAE", "RMSE", "R2"]
for col in metrics:
    df[col] = pd.to_numeric(df[col], errors="coerce")

metric_finite = bool(
    np.isfinite(df[metrics].to_numpy(dtype=float)).all()
)

duplicate_count = int(df.duplicated().sum())

required_types = {
    "Validation",
    "Nested cross-validation",
    "Internal test"
}
found_types = set(df["Evaluation_Type"].dropna().astype(str))
missing_types = sorted(required_types - found_types)

# Ensure nested CV has exactly one Week-16 record.
nested_b16 = df[
    df["Evaluation_Type"].eq("Nested cross-validation")
    & df["Landmark"].astype(str).eq("Week 16")
]

checks = {
    "registry_nonempty": not df.empty,
    "required_columns_present": not missing,
    "metrics_finite": metric_finite,
    "no_exact_duplicate_rows": duplicate_count == 0,
    "all_evaluation_types_present": not missing_types,
    "exactly_one_week16_nested_cv_record": len(nested_b16) == 1,
    "all_sources_nonempty": bool(
        df["Source"].notna().all()
        and df["Source"].astype(str).str.strip().ne("").all()
    )
}

report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "registry_path": str(REGISTRY),
    "rows": len(df),
    "evaluation_types": sorted(found_types),
    "duplicate_rows": duplicate_count,
    "missing_evaluation_types": missing_types,
    "checks": checks,
    "all_integrity_checks_passed": all(checks.values()),
    "historical_test_set_independence": "UNKNOWN",
    "external_validation_completed": False,
    "full_training_pipeline_reproduction_verified": False
}

out = AUDIT_DIR / "STEP32_2_registry_integrity_audit.json"
out.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("STEP 32.2 — REGISTRY INTEGRITY AUDIT")
print(json.dumps(report, indent=2))
print("Saved:", out)

if not all(checks.values()):
    raise RuntimeError(
        "Registry integrity audit failed. Review the checks above."
    )


In [ ]:

# ============================================================
# STEP 32.1.3 — FINAL PACKAGE CONSISTENCY AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib
from datetime import datetime, timezone
import zipfile

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
AUDIT_DIR = ROOT / "06_results" / "audit"
TABLE_DIR = ROOT / "06_results" / "tables" / "primary"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

REGISTRY = TABLE_DIR / "Canonical_Evaluation_Registry.csv"
REGISTRY_README = (
    TABLE_DIR / "Canonical_Evaluation_Registry_README.txt"
)
PROJECT_README = ROOT / "README.md"

NESTED_AUDIT = (
    AUDIT_DIR / "STEP15_8_nested_cv_integrity_audit.json"
)
REGISTRY_AUDIT = (
    AUDIT_DIR / "STEP32_2_registry_integrity_audit.json"
)

ZIP_PATH = Path(
    "/kaggle/working/SCI_Recovery_ML_Project_Final.zip"
)

# ------------------------------------------------------------
# 1. Verify required artifacts
# ------------------------------------------------------------
artifacts = {
    "canonical_registry": REGISTRY,
    "registry_readme": REGISTRY_README,
    "project_readme": PROJECT_README,
    "nested_cv_integrity_audit": NESTED_AUDIT,
    "registry_integrity_audit": REGISTRY_AUDIT,
}

checks = {
    name: path.is_file()
    for name, path in artifacts.items()
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'MISSING'}")

# ------------------------------------------------------------
# 2. Verify registry and evaluation categories
# ------------------------------------------------------------
if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Canonical registry missing: {REGISTRY}"
    )

registry = pd.read_csv(REGISTRY)

expected_types = {
    "Validation",
    "Nested cross-validation",
    "Internal test"
}

actual_types = set(
    registry["Evaluation_Type"].dropna().astype(str)
)

checks["registry_has_all_evaluation_types"] = (
    expected_types.issubset(actual_types)
)

checks["registry_has_seven_records"] = len(registry) == 7

# ------------------------------------------------------------
# 3. Verify nested-CV artifact audit
# ------------------------------------------------------------
nested_status = "NOT_AVAILABLE"

if NESTED_AUDIT.is_file():
    with open(NESTED_AUDIT, encoding="utf-8") as f:
        nested_report = json.load(f)

    nested_status = nested_report.get("status", "UNKNOWN")

    checks["nested_cv_artifact_checks_passed"] = (
        nested_status == "NESTED_CV_ARTIFACT_CHECKS_PASSED"
    )
else:
    checks["nested_cv_artifact_checks_passed"] = False

# ------------------------------------------------------------
# 4. Check ZIP only if it already exists
# ------------------------------------------------------------
zip_status = "NOT_CREATED_OR_NOT_FOUND"

if ZIP_PATH.is_file():
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        bad_member = zf.testzip()
        zip_names = set(zf.namelist())

    checks["zip_integrity_passed"] = bad_member is None

    # The archive must contain the registry.
    registry_suffix = (
        "06_results/tables/primary/"
        "Canonical_Evaluation_Registry.csv"
    )

    checks["registry_in_zip"] = any(
        name.endswith(registry_suffix)
        for name in zip_names
    )

    zip_status = (
        "VALID" if bad_member is None else f"CORRUPT: {bad_member}"
    )
else:
    checks["zip_exists"] = False

# ------------------------------------------------------------
# 5. Save final audit
# ------------------------------------------------------------
report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(ROOT),
    "registry_rows": int(len(registry)),
    "evaluation_types": sorted(actual_types),
    "artifact_checks": checks,
    "all_required_artifacts_present": all(
        checks.get(name, False)
        for name in artifacts
    ),
    "zip_status": zip_status,
    "historical_test_set_independence": "UNKNOWN",
    "external_validation_completed": False,
    "full_training_pipeline_reproduction_verified": False,
}

out = AUDIT_DIR / "STEP32_3_final_package_consistency_audit.json"

out.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\nSTEP 32.3 — AUDIT REPORT")
print(json.dumps(report, indent=2))
print("\nSaved:", out)


In [ ]:

# ============================================================
# STEP 32.1.3 — FINAL PROJECT README GENERATOR
# SCI Dynamic Recovery Prediction using Machine Learning
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import json

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
TABLES = ROOT / "06_results" / "tables"
PRIMARY = TABLES / "primary"
AUDIT = ROOT / "06_results" / "audit"

ROOT.mkdir(parents=True, exist_ok=True)
AUDIT.mkdir(parents=True, exist_ok=True)

README_PATH = ROOT / "README.md"
REGISTRY_PATH = PRIMARY / "Canonical_Evaluation_Registry.csv"

# ------------------------------------------------------------
# 1. Load existing canonical registry
# ------------------------------------------------------------
if not REGISTRY_PATH.is_file():
    raise FileNotFoundError(
        f"Canonical registry not found: {REGISTRY_PATH}\n"
        "Run STEP 31.9B / STEP 32.1 first."
    )

registry = pd.read_csv(REGISTRY_PATH)

required_columns = {
    "Evaluation_Type", "Analysis_Role", "Model",
    "Landmark", "MAE", "RMSE", "R2", "Source"
}

missing = required_columns - set(registry.columns)
if missing:
    raise ValueError(
        f"Registry missing required columns: {sorted(missing)}"
    )

evaluation_types = sorted(
    registry["Evaluation_Type"].dropna().astype(str).unique()
)

# ------------------------------------------------------------
# 2. Read available audit evidence
# ------------------------------------------------------------
def read_json_if_exists(path):
    if not path.is_file():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return None

nested_audit_path = AUDIT / "STEP15_8_nested_cv_integrity_audit.json"
integrity_audit_path = AUDIT / "STEP32_2_registry_integrity_audit.json"
package_audit_path = AUDIT / "STEP32_3_final_package_consistency_audit.json"

nested_audit = read_json_if_exists(nested_audit_path)
integrity_audit = read_json_if_exists(integrity_audit_path)
package_audit = read_json_if_exists(package_audit_path)

# ------------------------------------------------------------
# 3. Inventory available project artifacts
# ------------------------------------------------------------
artifact_paths = [
    ("Canonical evaluation registry", REGISTRY_PATH),
    ("Canonical OOF performance", PRIMARY / "Canonical_OOF_Performance.csv"),
    ("Nested-CV integrity audit", nested_audit_path),
    ("Registry integrity audit", integrity_audit_path),
    ("Package consistency audit", package_audit_path),
]

artifact_lines = [
    f"- {'Available' if path.is_file() else 'Not available'} — "
    f"{label}: `{path.relative_to(ROOT)}`"
    for label, path in artifact_paths
]

# ------------------------------------------------------------
# 4. Generate README
# ------------------------------------------------------------
timestamp = datetime.now(timezone.utc).isoformat()

readme = f"""# SCI Dynamic Recovery Prediction Using Machine Learning

## 1. Project Overview

This project develops and evaluates machine-learning models for
predicting recovery outcomes in spinal cord injury (SCI) data.

The workflow includes longitudinal landmark comparisons, model
benchmarking, nested cross-validation, prediction-error assessment,
explainability analyses, and the preparation of manuscript-related
tables and audit artifacts.

The README documents the current project artifacts. It does not
independently verify every stage of model training.

## 2. Project Structure

- `05_models/` — model artifacts, when exported.
- `06_results/tables/` — analytical and manuscript tables.
- `06_results/tables/primary/` — primary reporting artifacts.
- `06_results/figures/` — figures, when exported.
- `06_results/audit/` — integrity and methodological audit reports.

Only files actually present in the project should be considered
available deliverables.

## 3. Canonical Evaluation Registry

Primary registry:

`06_results/tables/primary/Canonical_Evaluation_Registry.csv`

- Number of records: {len(registry)}
- Evaluation types recorded: {", ".join(evaluation_types)}

The registry separates validation results, nested cross-validation
estimates, and internal test results. These results represent
different evaluation procedures and must not be substituted for one
another.

The `Source` column identifies the recorded source of each result.

## 4. Nested Cross-Validation

The project includes nested-CV reporting artifacts intended to
summarize model performance across outer folds.

Nested cross-validation estimates are internal resampling results.
They are not external validation results and are not interchangeable
with the final internal test evaluation.

Nested-CV audit available: {"Yes" if nested_audit else "No"}

## 5. Reproducibility and Integrity Audits

Available artifacts:

{chr(10).join(artifact_lines)}

A passing artifact audit establishes only the checks explicitly
performed by that audit. It does not, by itself, establish that the
complete training pipeline can be reproduced from a clean environment.

## 6. Important Methodological Limitations

- **Historical test-set independence:** UNKNOWN. Current patient
  partition separation does not establish whether test outcomes
  influenced earlier model selection.
- **External validation:** NOT PERFORMED unless a separate,
  independent cohort is subsequently evaluated.
- **Full training-pipeline reproduction:** NOT VERIFIED by artifact
  checks alone.
- **Chronology:** Visit-date interpretation depends on verified
  source-data encoding and documentation.
- **Clinical interpretation:** MAE describes average absolute
  prediction error in the outcome's units. Clinical meaningfulness
  requires appropriate supporting evidence; MAE alone does not
  establish clinical benefit.
- **Missing follow-up data:** Cohort selection and missingness should
  be reported when presenting model performance.

These limitations should be reflected in the Methods, Results,
Discussion, and Limitations sections of any manuscript.

## 7. Reporting Checklist

Before manuscript submission:

1. Identify the primary model and primary evaluation procedure.
2. Keep validation, nested-CV, and internal test results separate.
3. Report cohort sizes, missingness, and follow-up availability.
4. Document predictor preprocessing and leakage prevention.
5. Report MAE, RMSE, and R-squared with their evaluation context.
6. Avoid claims of external validation without an independent cohort.
7. Complete the applicable official TRIPOD+AI checklist against the
   final manuscript and supporting materials.

## 8. Intended Use

This package supports research documentation and model-performance
reporting. It should not be interpreted as evidence of clinical
readiness or as a substitute for independent clinical validation.

## 9. README Provenance

Generated (UTC): {timestamp}

Canonical registry:
`{REGISTRY_PATH}`

This README uses existing project files and registry metadata.
It does not retrain models or generate replacement performance metrics.
"""

README_PATH.write_text(readme, encoding="utf-8")

# ------------------------------------------------------------
# 5. Verify README export
# ------------------------------------------------------------
if not README_PATH.is_file():
    raise RuntimeError("README generation failed.")

saved_text = README_PATH.read_text(encoding="utf-8")

if len(saved_text.strip()) == 0:
    raise RuntimeError("README exists but is empty.")

audit = {
    "timestamp_utc": timestamp,
    "readme_path": str(README_PATH),
    "readme_exists": README_PATH.is_file(),
    "readme_size_bytes": README_PATH.stat().st_size,
    "registry_path": str(REGISTRY_PATH),
    "registry_rows": int(len(registry)),
    "evaluation_types": evaluation_types,
    "nested_cv_audit_available": nested_audit is not None,
    "registry_integrity_audit_available": integrity_audit is not None,
    "package_consistency_audit_available": package_audit is not None,
    "historical_test_set_independence": "UNKNOWN",
    "external_validation_completed": False,
    "full_training_pipeline_reproduction_verified": False,
    "status": "README_GENERATED_AND_VERIFIED"
}

audit_path = AUDIT / "STEP32_5_readme_generation_audit.json"
audit_path.write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8"
)

print("=" * 65)
print("STEP 32.4 — README GENERATION PASSED")
print("=" * 65)
print("README:", README_PATH)
print("Size:", README_PATH.stat().st_size, "bytes")
print("Registry records:", len(registry))
print("Evaluation types:", evaluation_types)
print("Audit:", audit_path)
print("Historical test independence: UNKNOWN")
print("External validation: NOT PERFORMED")
print("Full pipeline reproduction: NOT VERIFIED")


In [ ]:
# ============================================================
# STEP 32.2: REPRODUCIBILITY REQUIREMENTS
# Capture Actual Installed Package Versions
# ============================================================

from pathlib import Path
from importlib import metadata
import platform

print("=" * 70)
print("STEP 32.2: REPRODUCIBILITY REQUIREMENTS")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Core dependencies used in the analysis workflow
packages = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "lightgbm",
    "xgboost",
    "optuna",
    "shap",
    "matplotlib",
    "seaborn",
    "openpyxl",
    "joblib",
    "imbalanced-learn",
    "statsmodels"
]

installed = []
missing = []

for package in packages:
    try:
        version = metadata.version(package)
        installed.append(f"{package}=={version}")
    except metadata.PackageNotFoundError:
        missing.append(package)

requirements_path = PROJECT_DIR / "requirements.txt"

requirements_path.write_text(
    "\n".join(installed) + "\n",
    encoding="utf-8"
)

print("\nPYTHON VERSION")
print(platform.python_version())

print("\nPINNED DEPENDENCIES")
for package in installed:
    print(package)

print("\nPACKAGES NOT INSTALLED")
print(missing if missing else "None")

print("\nRequirements saved:")
print(requirements_path)

print("\nNote: This records direct package versions.")
print("A complete environment lock also requires")
print("transitive dependencies and platform details.")

print("\nSTEP 32.2 COMPLETED")


In [ ]:
# ============================================================
# STEP 32.3: CONDA ENVIRONMENT DOCUMENTATION
# Python Version + Pinned Direct Dependencies
# No Installation | No Model Retraining
# ============================================================

from pathlib import Path
import platform
import sys
import json

print("=" * 70)
print("STEP 32.3: REPRODUCIBLE CONDA ENVIRONMENT")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

requirements_path = PROJECT_DIR / "requirements.txt"

if not requirements_path.exists():
    raise FileNotFoundError(
        "requirements.txt not found. "
        "Run Step 32.2 first."
    )

# ------------------------------------------------------------
# 1. Read dependencies saved by Step 32.2
# ------------------------------------------------------------

requirements = [
    line.strip()
    for line in requirements_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]

if not requirements:
    raise ValueError("requirements.txt is empty.")

# ------------------------------------------------------------
# 2. Record environment information
# ------------------------------------------------------------

python_version = platform.python_version()

environment_info = {
    "python_version": python_version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "system": platform.system(),
    "machine": platform.machine(),
    "python_implementation": platform.python_implementation(),
    "direct_dependencies": requirements
}

info_path = PROJECT_DIR / "environment_info.json"

info_path.write_text(
    json.dumps(environment_info, indent=4),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 3. Generate Conda environment specification
# ------------------------------------------------------------

environment_lines = [
    "name: sci-recovery-ml",
    "channels:",
    "  - conda-forge",
    "dependencies:",
    f"  - python={python_version}",
    "  - pip",
    "  - pip:"
]

environment_lines.extend(
    f"      - {package}"
    for package in requirements
)

environment_path = PROJECT_DIR / "environment.yml"

environment_path.write_text(
    "\n".join(environment_lines) + "\n",
    encoding="utf-8"
)

# ------------------------------------------------------------
# 4. Verify saved files
# ------------------------------------------------------------

assert requirements_path.exists()
assert environment_path.exists()
assert info_path.exists()

saved_content = environment_path.read_text(
    encoding="utf-8"
)

assert f"python={python_version}" in saved_content

for package in requirements:
    assert f"      - {package}" in saved_content

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\nENVIRONMENT SUMMARY")
print("Environment name: sci-recovery-ml")
print("Python version:", python_version)
print("Operating system:", platform.system())
print("Platform:", platform.platform())
print("Pinned direct dependencies:", len(requirements))

print("\nFILES GENERATED")
print("1.", requirements_path)
print("2.", environment_path)
print("3.", info_path)

print("\nENVIRONMENT SPECIFICATION")
print(saved_content)

print("\nREPRODUCIBILITY NOTES")
print("- No packages were installed or modified.")
print("- Package versions were taken from Step 32.2.")
print("- Python and platform information recorded.")
print("- Transitive dependencies are not fully locked.")
print("- Clean-environment reproduction not yet verified.")
print("- Cross-platform installation not guaranteed.")

print("\nSTEP 32.3 COMPLETED")


In [ ]:

# ============================================================
# STEP 32.4: UPDATED RESEARCH MODEL CARD
# SCI Dynamic Recovery Prediction
# Clinical Prediction Model Documentation
# ============================================================

from pathlib import Path
import json
import platform
from datetime import datetime, timezone

print("=" * 70)
print("STEP 32.4: UPDATED MODEL CARD")
print("=" * 70)

# ------------------------------------------------------------
# 1. Project directory
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model_card_path = PROJECT_DIR / "MODEL_CARD.md"

# ------------------------------------------------------------
# 2. Dataset information
# ------------------------------------------------------------

dataset_name = "Sygen Spinal Cord Injury Dataset"

total_records = 791
analytical_patients = 659
excluded_records = total_records - analytical_patients

training_patients = 461
validation_patients = 99
test_patients = 99

# Previously reported cohort counts.
# These must be reconciled with the cohort audit.

# ------------------------------------------------------------
# 3. Final model information
# ------------------------------------------------------------

model_name = "Optimized LightGBM"

prediction_target = "26-week total motor score"

prediction_landmark = "Week 16"

# Historical results from the existing notebook.
# Do not treat these as independently reproduced.

reported_metrics = {
    "R2": 0.8800,
    "MAE": 5.54,
    "RMSE": 10.30
}

# ------------------------------------------------------------
# 4. Environment information
# ------------------------------------------------------------

environment_path = PROJECT_DIR / "environment_info.json"

if environment_path.exists():

    with open(
        environment_path,
        "r",
        encoding="utf-8"
    ) as file:
        environment = json.load(file)

    python_version = environment.get(
        "python_version",
        platform.python_version()
    )

    system_platform = environment.get(
        "platform",
        platform.platform()
    )

else:

    python_version = platform.python_version()
    system_platform = platform.platform()

# ------------------------------------------------------------
# 5. Generate model card
# ------------------------------------------------------------

model_card = f"""
# SCI Dynamic Recovery Prediction Model Card

## 1. Model Overview

**Model name:** {model_name}

**Dataset:** {dataset_name}

**Prediction target:** {prediction_target}

**Prediction landmark:** {prediction_landmark}

**Model type:** Supervised machine-learning regression

**Framework:** LightGBM

**Documentation date:** {datetime.now(timezone.utc).strftime("%Y-%m-%d")}

---

## 2. Research Objective

The objective of this research is to develop and
internally validate an explainable machine-learning
model for predicting 26-week neurological motor
outcomes following spinal cord injury.

The research investigates whether longitudinal
neurological assessments improve prediction
compared with baseline clinical information.

The model estimates future motor scores rather
than directly estimating causal treatment effects.

---

## 3. Dataset and Study Population

Dataset: {dataset_name}

Initial records: {total_records}

Analytical patients: {analytical_patients}

Excluded records: {excluded_records}

Training patients: {training_patients}

Validation patients: {validation_patients}

Held-out test patients: {test_patients}

Patient-level splitting was assessed using
PTID-based overlap analysis.

Previously saved audit results reported zero
overlapping PTIDs between training, validation
and test cohorts.

The precise reasons for cohort exclusions must
be confirmed using the cohort derivation audit.

---

## 4. Prediction Target

The primary outcome is the total neurological
motor score measured at Week 26 following
spinal cord injury.

The final dynamic model uses information
available through the Week-16 landmark.

The model should not be described as making
Week-16-informed predictions at admission.

---

## 5. Candidate Clinical Predictors

The research investigates predictors including:

- Age
- Sex
- AIS grade
- Baseline motor score
- Week-4 motor change
- Week-8 motor change
- Week-16 motor change

The exact final model input columns and their
order must match the saved feature manifest.

Predictor availability must be checked to
ensure that no post-landmark information
enters earlier prediction models.

---

## 6. Model Development

Primary model:

{model_name}

The research workflow includes:

- Training, validation and test data splitting
- Feature engineering
- Missing-value processing
- Hyperparameter optimization
- Nested cross-validation
- Repeated cross-validation
- Longitudinal landmark comparisons

All preprocessing and tuning procedures
must be verified for fold-level isolation.

---

## 7. Reported Predictive Performance

Final internal test cohort:

R-squared: {reported_metrics["R2"]:.4f}

Mean absolute error: {reported_metrics["MAE"]:.2f}

Root mean squared error: {reported_metrics["RMSE"]:.2f}

These values are historical results reported
by the existing research notebook.

They are not independently reproduced
by the model-card generation step.

The final manuscript should include
uncertainty intervals where appropriate.

---

## 8. Longitudinal Prediction

The research evaluates four assessment landmarks:

- Baseline
- Week 4
- Week 8
- Week 16

The objective is to assess whether updated
clinical assessments provide additional
predictive information.

Fixed-algorithm landmark ablation is preferred
when quantifying the incremental value of
longitudinal information.

Comparisons must use the same evaluation
patients and appropriately matched outcomes.

---

## 9. Explainability

The research includes:

- SHAP feature importance
- SHAP interpretation
- Permutation feature importance

These methods describe associations between
model features and model predictions.

They do not establish causal relationships
between predictors and neurological recovery.

---

## 10. Prediction Uncertainty

The research includes conformal prediction
interval estimation.

Previously reported nominal 95% prediction
intervals covered approximately 90.91%
of the held-out test observations.

This empirical coverage does not demonstrate
that 95% coverage was achieved.

Formal conformal coverage claims require
verification that calibration observations
were not used improperly for model selection
or other adaptive decisions.

---

## 11. Missing Data

The research includes:

- Missingness analysis
- Imputation sensitivity analysis
- Week-16 missing-assessment sensitivity

The Week-16 missing-assessment subgroup
was small, limiting the reliability of
subgroup-specific conclusions.

Imputation procedures must be fitted
using only the appropriate training folds.

---

## 12. Internal Validation

The research reports:

- Nested cross-validation
- Repeated cross-validation
- Held-out test evaluation
- Bootstrap uncertainty analysis
- Learning curve analysis
- Residual and calibration analyses

The independence of model selection,
cross-validation and final testing
must be verified before publication.

---

## 13. Intended Use

This model is intended for research into
predicting neurological motor outcomes
following spinal cord injury.

Potential future applications include
clinical prognostic research and
decision-support development.

The current model is not validated for
autonomous clinical decision-making.

---

## 14. Limitations

1. The model was developed and evaluated
   using a single research dataset.

2. External validation was not performed.

3. Performance may differ in other SCI
   populations or healthcare settings.

4. Some clinical subgroups have small
   sample sizes.

5. Missing longitudinal assessments
   may affect predictive accuracy.

6. Nominal prediction interval coverage
   was not fully achieved empirically.

7. Baseline and dynamic model comparisons
   require consistent cohort evaluation.

8. The final test results have already
   been inspected during research
   development.

9. Predictive associations should not
   be interpreted as causal effects.

---

## 15. Reproducibility

Python version: {python_version}

Platform: {system_platform}

Environment files:

- requirements.txt
- environment.yml
- environment_info.json

Research reproducibility additionally requires:

- Fixed data splits
- Recorded random seeds
- Saved model artifacts
- Feature order verification
- Dataset integrity verification
- Reproduction of saved predictions
- Independent environment execution

Generating environment documentation
does not establish that the entire
pipeline is reproducible.

---

## 16. Reporting Framework

The manuscript should be prepared
in accordance with TRIPOD+AI reporting
recommendations for clinical prediction
model development and validation.

A completed reporting checklist should
accompany the manuscript when required
by the target journal.

---

## 17. Clinical Deployment Status

Status: Research model only.

External validation: Not performed.

Prospective validation: Not performed.

Clinical deployment authorization:
Not established.

The model should not be used as the
sole basis for patient treatment or
prognostic decisions.
"""

# ------------------------------------------------------------
# 6. Save model card
# ------------------------------------------------------------

model_card_path.write_text(
    model_card.strip() + "\n",
    encoding="utf-8"
)

# ------------------------------------------------------------
# 7. Verify saved documentation
# ------------------------------------------------------------

assert model_card_path.exists()

saved_model_card = model_card_path.read_text(
    encoding="utf-8"
)

required_sections = [
    "Model Overview",
    "Research Objective",
    "Dataset and Study Population",
    "Reported Predictive Performance",
    "Longitudinal Prediction",
    "Explainability",
    "Prediction Uncertainty",
    "Limitations",
    "Reproducibility",
    "Clinical Deployment Status"
]

for section in required_sections:

    assert section in saved_model_card, (
        f"Missing model-card section: {section}"
    )

# ------------------------------------------------------------
# 8. Display completion summary
# ------------------------------------------------------------

print("\nMODEL CARD SAVED:")
print(model_card_path)

print("\nDOCUMENTATION SUMMARY")

print("Dataset:", dataset_name)
print("Analytical patients:", analytical_patients)
print("Final model:", model_name)
print("Prediction target:", prediction_target)
print("Prediction landmark:", prediction_landmark)

print("\nREPORTED INTERNAL TEST RESULTS")

for metric, value in reported_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nDOCUMENTATION CHECKS PASSED")

print("\nIMPORTANT:")
print("The model card documents reported findings.")
print("It does not retrain or independently validate the model.")
print("Historical metrics and cohort counts require audit confirmation.")

print("\nSTEP 32.4 COMPLETED")


In [ ]:
# ============================================================
# STEP 32.5
# DATASET PROVENANCE AND INTEGRITY VERIFICATION
#
# Checks:
# 1. Original dataset existence
# 2. Project dataset existence
# 3. SHA-256 hash comparison
# 4. Patient identifier integrity
# 5. Source dataset record count
# 6. Analytical cohort count, when available
# 7. Patient split overlap, when available
# 8. Machine-readable audit report
#
# No model training or modification.
# ============================================================

import json
import hashlib
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 32.5: DATASET PROVENANCE AND INTEGRITY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

dataset_path = Path(
    "/kaggle/input/datasets/shizansarkar/"
    "sci-recovery-data/SCI_Neurological_Recovery-master/"
    "test/spss_v1_2.csv"
)

project_copy = (
    PROJECT_DIR /
    "01_data/primary/raw/spss_v1_2.csv"
)

output_path = (
    PROJECT_DIR /
    "01_data/metadata/dataset_provenance.json"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Confirm dataset files exist
# ------------------------------------------------------------

if not dataset_path.is_file():
    raise FileNotFoundError(
        f"Original dataset not found:\n{dataset_path}"
    )

if not project_copy.is_file():
    raise FileNotFoundError(
        f"Project dataset copy not found:\n{project_copy}"
    )

print("\nDataset files found.")

# ------------------------------------------------------------
# 3. Calculate SHA-256 checksums
# ------------------------------------------------------------

def sha256_file(file_path):

    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


source_hash = sha256_file(dataset_path)
copy_hash = sha256_file(project_copy)

source_copy_identical = (
    source_hash == copy_hash
)

print("\nSHA-256 VERIFICATION")
print("Original SHA-256:", source_hash)
print("Project SHA-256: ", copy_hash)

print(
    "Source and project copy identical:",
    source_copy_identical
)

# ------------------------------------------------------------
# 4. Load original dataset
# ------------------------------------------------------------

raw_df = pd.read_csv(dataset_path)

source_records = len(raw_df)

print("\nSOURCE DATASET")
print("Records:", source_records)
print("Columns:", len(raw_df.columns))

# ------------------------------------------------------------
# 5. Verify patient identifiers
# ------------------------------------------------------------

patient_id_column = "PTID"

patient_id_present = (
    patient_id_column in raw_df.columns
)

patient_id_missing = None
patient_id_unique = None
duplicate_patient_ids = None

if patient_id_present:

    patient_id_missing = int(
        raw_df[patient_id_column].isna().sum()
    )

    duplicate_patient_ids = int(
        raw_df[patient_id_column].duplicated().sum()
    )

    patient_id_unique = (
        duplicate_patient_ids == 0
    )

print("\nPATIENT IDENTIFIER AUDIT")
print("PTID column present:", patient_id_present)
print("Missing PTIDs:", patient_id_missing)
print("Duplicate PTIDs:", duplicate_patient_ids)
print("PTIDs unique:", patient_id_unique)

# ------------------------------------------------------------
# 6. Verify analytical cohort derivation
# ------------------------------------------------------------

target_column = "TOTMOT26"

outcome_column_present = (
    target_column in raw_df.columns
)

outcome_complete_n = None
outcome_missing_n = None

if outcome_column_present:

    outcome_numeric = pd.to_numeric(
        raw_df[target_column],
        errors="coerce"
    )

    outcome_complete_n = int(
        outcome_numeric.notna().sum()
    )

    outcome_missing_n = int(
        outcome_numeric.isna().sum()
    )

print("\nOUTCOME AVAILABILITY")
print("Target present:", outcome_column_present)
print("Complete outcome records:", outcome_complete_n)
print("Missing outcome records:", outcome_missing_n)

# These values describe outcome availability,
# not automatically the final analytical cohort.
# Additional filtering may have been applied.

analytical_cohort_verified = None
analytical_cohort_count = None

if "primary_cohort" in globals():

    analytical_cohort_count = len(primary_cohort)

    if (
        outcome_complete_n is not None
        and target_column in primary_cohort.columns
        and patient_id_column in primary_cohort.columns
        and patient_id_present
    ):

        expected_ids = set(
            raw_df.loc[
                pd.to_numeric(
                    raw_df[target_column],
                    errors="coerce"
                ).notna(),
                patient_id_column
            ].dropna().astype(str)
        )

        actual_ids = set(
            primary_cohort[
                patient_id_column
            ].dropna().astype(str)
        )

        analytical_cohort_verified = (
            expected_ids == actual_ids
            and len(primary_cohort) == outcome_complete_n
        )

print("\nANALYTICAL COHORT")
print("Cohort available:", "primary_cohort" in globals())
print("Analytical count:", analytical_cohort_count)
print("Outcome-based derivation verified:",
      analytical_cohort_verified)

# ------------------------------------------------------------
# 7. Patient split integrity
# ------------------------------------------------------------

split_verified = None
split_overlap = None
split_counts = None

required_splits = [
    "train_df",
    "validation_df",
    "test_df"
]

splits_available = all(
    name in globals()
    for name in required_splits
)

if splits_available:

    split_frames = {
        "training": train_df,
        "validation": validation_df,
        "test": test_df
    }

    if all(
        patient_id_column in frame.columns
        for frame in split_frames.values()
    ):

        train_ids = set(
            train_df[patient_id_column]
            .dropna().astype(str)
        )

        validation_ids = set(
            validation_df[patient_id_column]
            .dropna().astype(str)
        )

        test_ids = set(
            test_df[patient_id_column]
            .dropna().astype(str)
        )

        split_overlap = {
            "train_validation": len(
                train_ids & validation_ids
            ),
            "train_test": len(
                train_ids & test_ids
            ),
            "validation_test": len(
                validation_ids & test_ids
            )
        }

        split_counts = {
            "training": len(train_df),
            "validation": len(validation_df),
            "test": len(test_df)
        }

        no_overlap = all(
            value == 0
            for value in split_overlap.values()
        )

        all_split_ids = (
            train_ids | validation_ids | test_ids
        )

        split_ids_complete = None

        if (
            "primary_cohort" in globals()
            and patient_id_column in primary_cohort.columns
        ):

            cohort_ids = set(
                primary_cohort[patient_id_column]
                .dropna().astype(str)
            )

            split_ids_complete = (
                all_split_ids == cohort_ids
            )

        split_verified = (
            no_overlap
            and split_ids_complete is True
            and sum(split_counts.values())
                == len(primary_cohort)
        ) if "primary_cohort" in globals() else None

print("\nPATIENT SPLIT INTEGRITY")
print("Splits available:", splits_available)
print("Split counts:", split_counts)
print("Overlapping PTIDs:", split_overlap)
print("Full split verification:", split_verified)

# ------------------------------------------------------------
# 8. Create provenance report
# ------------------------------------------------------------

provenance = {
    "audit_timestamp_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "dataset": {
        "filename": dataset_path.name,
        "source_path": str(dataset_path),
        "project_copy_path": str(project_copy),
        "source_records": source_records,
        "source_columns": len(raw_df.columns),
        "source_sha256": source_hash,
        "project_copy_sha256": copy_hash,
        "source_copy_identical":
            bool(source_copy_identical)
    },
    "patient_identifier": {
        "column": patient_id_column,
        "column_present": patient_id_present,
        "missing_count": patient_id_missing,
        "duplicate_count": duplicate_patient_ids,
        "unique": patient_id_unique
    },
    "outcome": {
        "column": target_column,
        "column_present": outcome_column_present,
        "complete_records": outcome_complete_n,
        "missing_records": outcome_missing_n
    },
    "analytical_cohort": {
        "count": analytical_cohort_count,
        "outcome_based_derivation_verified":
            analytical_cohort_verified
    },
    "patient_splits": {
        "counts": split_counts,
        "overlap": split_overlap,
        "full_verification": split_verified
    },
    "verification_limitations": [
        "Matching hashes establish byte-identical copies, not the external origin or release version.",
        "Cohort verification assumes missing Week-26 outcomes are the only exclusion criterion.",
        "Missing in-memory cohorts or splits are recorded as unverified.",
        "Feature leakage and nested cross-validation are not checked by this cell."
    ]
}

# ------------------------------------------------------------
# 9. Save JSON
# ------------------------------------------------------------

output_path.write_text(
    json.dumps(
        provenance,
        indent=4,
        default=str
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\nDATASET PROVENANCE SAVED")
print(output_path)

print("\nVERIFICATION SUMMARY")
print(
    "Source/copy identical:",
    source_copy_identical
)
print(
    "PTID valid and unique:",
    (
        patient_id_present
        and patient_id_missing == 0
        and patient_id_unique is True
    )
)
print(
    "Analytical cohort verified:",
    analytical_cohort_verified
)
print(
    "Patient split verified:",
    split_verified
)

print("\nInterpretation:")
print("True = verification passed")
print("False = verification failed")
print("None = verification not completed")

print("\nSTEP 32.5 AUDIT COMPLETED")


In [ ]:
# ============================================================
# STEP 32.6: EXACT COHORT EXCLUSION AUDIT
# Corrected Research Version
# No Retraining | No Data Modification
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 32.6: EXACT COHORT EXCLUSION AUDIT")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

OUTPUT_DIR = PROJECT_DIR / "01_data/metadata"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load original dataset independently
# ------------------------------------------------------------

dataset_path = (
    PROJECT_DIR /
    "01_data/primary/raw/spss_v1_2.csv"
)

if not dataset_path.is_file():
    raise FileNotFoundError(
        f"Original dataset copy not found: {dataset_path}"
    )

raw_cohort = pd.read_csv(dataset_path)

ID_COL = "PTID"
TARGET_COL = "TOTMOT26"

for col in [ID_COL, TARGET_COL]:
    if col not in raw_cohort.columns:
        raise KeyError(f"Missing column: {col}")

assert raw_cohort[ID_COL].notna().all()
assert raw_cohort[ID_COL].is_unique

# ------------------------------------------------------------
# 2. Standardize patient identifiers
# ------------------------------------------------------------

def normalize_ids(series):
    return series.astype("string").str.strip()

raw_cohort["_AUDIT_PTID"] = normalize_ids(
    raw_cohort[ID_COL]
)

assert raw_cohort["_AUDIT_PTID"].notna().all()
assert raw_cohort["_AUDIT_PTID"].is_unique

# ------------------------------------------------------------
# 3. Identify actual analytical patients
# ------------------------------------------------------------

required_splits = [
    "train_df",
    "validation_df",
    "test_df"
]

missing_splits = [
    name for name in required_splits
    if name not in globals()
]

if missing_splits:
    raise RuntimeError(
        f"Missing split dataframes: {missing_splits}"
    )

split_frames = {
    "Training": train_df,
    "Validation": validation_df,
    "Test": test_df
}

split_ids = {}

for name, frame in split_frames.items():

    if ID_COL not in frame.columns:
        raise KeyError(
            f"{name}: PTID column is missing"
        )

    ids = normalize_ids(frame[ID_COL])

    assert ids.notna().all(), (
        f"{name}: missing patient IDs"
    )

    assert ids.is_unique, (
        f"{name}: duplicate patient IDs"
    )

    split_ids[name] = set(ids.tolist())

# ------------------------------------------------------------
# 4. Verify split independence
# ------------------------------------------------------------

overlap_counts = {
    "train_validation": len(
        split_ids["Training"] &
        split_ids["Validation"]
    ),
    "train_test": len(
        split_ids["Training"] &
        split_ids["Test"]
    ),
    "validation_test": len(
        split_ids["Validation"] &
        split_ids["Test"]
    )
}

assert all(
    value == 0
    for value in overlap_counts.values()
), "Patient overlap detected."

analytical_ptids = set().union(
    *split_ids.values()
)

raw_ptids = set(
    raw_cohort["_AUDIT_PTID"].tolist()
)

assert analytical_ptids <= raw_ptids, (
    "Analytical patients not found in source data."
)

excluded_ptids = raw_ptids - analytical_ptids

# ------------------------------------------------------------
# 5. Check Week-26 outcome availability
# ------------------------------------------------------------

raw_cohort["_AUDIT_OUTCOME"] = pd.to_numeric(
    raw_cohort[TARGET_COL],
    errors="coerce"
)

raw_cohort["_AUDIT_INCLUDED"] = (
    raw_cohort["_AUDIT_PTID"].isin(
        analytical_ptids
    )
)

included_df = raw_cohort.loc[
    raw_cohort["_AUDIT_INCLUDED"]
].copy()

excluded_df = raw_cohort.loc[
    ~raw_cohort["_AUDIT_INCLUDED"]
].copy()

excluded_missing_outcome = int(
    excluded_df["_AUDIT_OUTCOME"].isna().sum()
)

excluded_available_outcome = int(
    excluded_df["_AUDIT_OUTCOME"].notna().sum()
)

included_missing_outcome = int(
    included_df["_AUDIT_OUTCOME"].isna().sum()
)

# ------------------------------------------------------------
# 6. Verify exact cohort derivation
# ------------------------------------------------------------

initial_n = len(raw_cohort)
included_n = len(included_df)
excluded_n = len(excluded_df)

assert initial_n == included_n + excluded_n

assert excluded_available_outcome == 0, (
    "Some excluded patients have available outcomes."
)

assert included_missing_outcome == 0, (
    "Some included patients lack Week-26 outcomes."
)

assert excluded_missing_outcome == excluded_n

assert (
    initial_n ==
    len(raw_ptids)
), "Patient IDs are not one-to-one with rows."

assert included_n == len(analytical_ptids)

# ------------------------------------------------------------
# 7. Check outcome-based cohort identity
# ------------------------------------------------------------

outcome_complete_ids = set(
    raw_cohort.loc[
        raw_cohort["_AUDIT_OUTCOME"].notna(),
        "_AUDIT_PTID"
    ]
)

cohort_verified = (
    analytical_ptids == outcome_complete_ids
)

assert cohort_verified, (
    "Analytical cohort differs from the "
    "outcome-complete source cohort."
)

# ------------------------------------------------------------
# 8. Summarize missingness among exclusions
# ------------------------------------------------------------

candidate_variables = [
    "TOTMOT01",
    "TOTMOT04",
    "TOTMOT08",
    "TOTMOT16",
    "TOTMOT26"
]

missing_rows = []

for col in candidate_variables:

    if col not in excluded_df.columns:
        continue

    numeric_values = pd.to_numeric(
        excluded_df[col],
        errors="coerce"
    )

    missing_rows.append({
        "Variable": col,
        "Available": int(
            numeric_values.notna().sum()
        ),
        "Missing": int(
            numeric_values.isna().sum()
        ),
        "Missing_Percent": round(
            numeric_values.isna().mean() * 100,
            2
        )
    })

missing_summary = pd.DataFrame(missing_rows)

# ------------------------------------------------------------
# 9. Create cohort flow table
# ------------------------------------------------------------

cohort_summary = pd.DataFrame([
    {
        "Stage": "Initial source cohort",
        "Patients": initial_n
    },
    {
        "Stage": "Excluded: missing Week-26 outcome",
        "Patients": excluded_n
    },
    {
        "Stage": "Final analytical cohort",
        "Patients": included_n
    },
    {
        "Stage": "Training",
        "Patients": len(split_ids["Training"])
    },
    {
        "Stage": "Validation",
        "Patients": len(split_ids["Validation"])
    },
    {
        "Stage": "Test",
        "Patients": len(split_ids["Test"])
    }
])

# ------------------------------------------------------------
# 10. Save results
# ------------------------------------------------------------

cohort_summary.to_csv(
    OUTPUT_DIR / "cohort_flow_summary.csv",
    index=False
)

missing_summary.to_csv(
    OUTPUT_DIR / "excluded_patient_missingness.csv",
    index=False
)

# PTIDs are not exported to the public summary.

audit_report = {
    "audit_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "initial_patients": initial_n,
    "analytical_patients": included_n,
    "excluded_patients": excluded_n,

    "exclusion_reason":
        "Missing Week-26 motor outcome",

    "excluded_missing_outcome":
        excluded_missing_outcome,

    "excluded_with_available_outcome":
        excluded_available_outcome,

    "included_missing_outcome":
        included_missing_outcome,

    "outcome_based_cohort_verified":
        bool(cohort_verified),

    "split_counts": {
        name: len(ids)
        for name, ids in split_ids.items()
    },

    "split_overlap_counts": overlap_counts,

    "patient_split_independence_verified":
        all(v == 0 for v in overlap_counts.values()),

    "limitations": [
        "This audit verifies the implemented outcome-based exclusions.",
        "It does not determine whether exclusion-related selection bias exists.",
        "It does not establish external dataset authenticity.",
        "It does not audit modeling information leakage."
    ]
}

report_path = (
    OUTPUT_DIR / "cohort_exclusion_audit.json"
)

report_path.write_text(
    json.dumps(audit_report, indent=4),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 11. Display verification results
# ------------------------------------------------------------

print("\nCOHORT FLOW")
display(cohort_summary)

print("\nEXCLUDED PATIENT MISSINGNESS")
display(missing_summary)

print("\nVERIFICATION SUMMARY")

print("Initial patients:", initial_n)
print("Included patients:", included_n)
print("Excluded patients:", excluded_n)

print(
    "Excluded with missing outcome:",
    excluded_missing_outcome
)

print(
    "Excluded with available outcome:",
    excluded_available_outcome
)

print(
    "Included with missing outcome:",
    included_missing_outcome
)

print(
    "Outcome-based cohort verified:",
    cohort_verified
)

print(
    "Patient split overlap:",
    overlap_counts
)

print("\nSaved:", report_path)
print("No models retrained.")
print("No source data modified.")

print("\nSTEP 32.6 AUDIT COMPLETED")


In [ ]:
# ============================================================
# STEP 32.6.1: ANALYTICAL COHORT AND LANDMARK INTEGRITY
# Corrected Research Version
# No Training | No Predictions | No Data Modification
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 32.6.1: ANALYTICAL COHORT AND SPLIT INTEGRITY")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

OUTPUT_DIR = PROJECT_DIR / "01_data/metadata"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load verified cohort audit
# ------------------------------------------------------------

cohort_audit_path = (
    OUTPUT_DIR / "cohort_exclusion_audit.json"
)

if not cohort_audit_path.exists():
    raise FileNotFoundError(
        "Run corrected Step 32.6A first."
    )

with open(
    cohort_audit_path,
    "r",
    encoding="utf-8"
) as file:
    cohort_audit = json.load(file)

if not cohort_audit.get(
    "outcome_based_cohort_verified", False
):
    raise AssertionError(
        "Cohort derivation has not passed verification."
    )

if not cohort_audit.get(
    "patient_split_independence_verified", False
):
    raise AssertionError(
        "PTID split independence has not passed."
    )

# ------------------------------------------------------------
# 2. Check model input availability
# ------------------------------------------------------------

if "MODEL_INPUTS" not in globals():
    raise RuntimeError(
        "MODEL_INPUTS is missing. "
        "Run the original model-input preparation step."
    )

expected_models = [
    "Model_A",
    "Model_B4",
    "Model_B8",
    "Model_B16"
]

splits = [
    "train",
    "validation",
    "test"
]

expected_sizes = {
    "train": 461,
    "validation": 99,
    "test": 99
}

source_frames = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df
}

ID_COL = "PTID"

# ------------------------------------------------------------
# 3. Validate source split patient identities
# ------------------------------------------------------------

source_ids = {}

for split, frame in source_frames.items():

    if ID_COL not in frame.columns:
        raise KeyError(
            f"{split}: PTID missing"
        )

    ids = (
        frame[ID_COL]
        .astype("string")
        .str.strip()
    )

    assert ids.notna().all()
    assert ids.is_unique

    assert len(frame) == expected_sizes[split]

    source_ids[split] = ids

assert set(source_ids["train"]).isdisjoint(
    set(source_ids["validation"])
)

assert set(source_ids["train"]).isdisjoint(
    set(source_ids["test"])
)

assert set(source_ids["validation"]).isdisjoint(
    set(source_ids["test"])
)

# ------------------------------------------------------------
# 4. Verify landmark model inputs
# ------------------------------------------------------------

audit_rows = []

reference_targets = {}
reference_indices = {}

for model_name in expected_models:

    if model_name not in MODEL_INPUTS:
        raise KeyError(
            f"Missing model: {model_name}"
        )

    model_data = MODEL_INPUTS[model_name]

    for split in splits:

        X_key = f"X_{split}"
        y_key = f"y_{split}"

        X = model_data[X_key]
        y = model_data[y_key]

        y_array = np.asarray(
            y,
            dtype=float
        ).ravel()

        assert isinstance(X, pd.DataFrame), (
            f"{model_name}/{split}: "
            "X must be a DataFrame for this audit."
        )

        assert len(X) == len(y_array), (
            f"{model_name}/{split}: X/y mismatch"
        )

        assert len(X) == expected_sizes[split], (
            f"{model_name}/{split}: unexpected size"
        )

        assert np.isfinite(y_array).all(), (
            f"{model_name}/{split}: invalid targets"
        )

        assert X.columns.is_unique, (
            f"{model_name}/{split}: duplicate features"
        )

        assert X.index.is_unique, (
            f"{model_name}/{split}: duplicate row index"
        )

        # -----------------------------------------------
        # Confirm index correspondence to source split
        # -----------------------------------------------

        source = source_frames[split]

        index_matches_source = X.index.equals(
            source.index
        )

        # A matching index is evidence of row alignment
        # only if the feature pipeline preserved original
        # row indices. It is not standalone PTID proof.

        if "PTID" in X.columns:

            model_ids = (
                X["PTID"]
                .astype("string")
                .str.strip()
            )

            patient_alignment_verified = (
                model_ids.reset_index(drop=True)
                .equals(
                    source_ids[split].reset_index(
                        drop=True
                    )
                )
            )

            alignment_method = (
                "Explicit PTID comparison"
            )

        else:

            patient_alignment_verified = None

            alignment_method = (
                "PTID unavailable in model input; "
                "source-index correspondence diagnostic only"
            )

        if model_name == "Model_A":

            reference_targets[split] = y_array.copy()
            reference_indices[split] = X.index.copy()

        else:

            assert np.array_equal(
                reference_targets[split],
                y_array
            ), (
                f"{model_name}/{split}: "
                "targets differ from Model_A"
            )

            assert X.index.equals(
                reference_indices[split]
            ), (
                f"{model_name}/{split}: "
                "row indices differ from Model_A"
            )

        audit_rows.append({
            "Model": model_name,
            "Split": split,
            "Patients": len(X),
            "Predictors": X.shape[1],
            "TargetMissing": int(
                pd.isna(y_array).sum()
            ),
            "IndexMatchesSource":
                bool(index_matches_source),
            "PTIDAlignmentVerified":
                patient_alignment_verified,
            "AlignmentMethod": alignment_method
        })

# ------------------------------------------------------------
# 5. Verify target alignment with source data
# ------------------------------------------------------------

TARGET_COL = "TOTMOT26"

target_source_checks = {}

for split in splits:

    source = source_frames[split]

    if TARGET_COL not in source.columns:

        target_source_checks[split] = None
        continue

    source_y = pd.to_numeric(
        source[TARGET_COL],
        errors="coerce"
    ).to_numpy()

    model_y = reference_targets[split]

    # This assumes source and model input rows
    # preserve the same ordering.

    row_index_matches = (
        reference_indices[split].equals(
            source.index
        )
    )

    if row_index_matches:

        targets_match = np.allclose(
            source_y,
            model_y,
            equal_nan=True
        )

        assert targets_match, (
            f"{split}: source/model outcome mismatch"
        )

        target_source_checks[split] = True

    else:

        target_source_checks[split] = None

# ------------------------------------------------------------
# 6. Create results table
# ------------------------------------------------------------

audit_table = pd.DataFrame(audit_rows)

print("\nLANDMARK SPLIT SUMMARY")
display(audit_table)

# ------------------------------------------------------------
# 7. Summarize landmark verification
# ------------------------------------------------------------

targets_consistent = True
indices_consistent = True

explicit_patient_alignment = all(
    item is True
    for item in audit_table[
        "PTIDAlignmentVerified"
    ].tolist()
)

# PTID may have been removed intentionally
# before model fitting to prevent leakage.
# In that situation explicit verification
# requires a separate preserved row-to-PTID map.

landmark_verified = (
    targets_consistent
    and indices_consistent
    and explicit_patient_alignment
    and all(
        value is True
        for value in target_source_checks.values()
    )
)

# ------------------------------------------------------------
# 8. Save audit report
# ------------------------------------------------------------

report = {
    "audit_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "raw_observations":
        cohort_audit["initial_patients"],

    "analytical_observations":
        cohort_audit["analytical_patients"],

    "excluded_observations":
        cohort_audit["excluded_patients"],

    "cohort_exclusion_verified":
        bool(
            cohort_audit[
                "outcome_based_cohort_verified"
            ]
        ),

    "patient_level_split_independence_verified":
        bool(
            cohort_audit[
                "patient_split_independence_verified"
            ]
        ),

    "split_sizes":
        expected_sizes,

    "landmark_models_checked":
        expected_models,

    "target_arrays_consistent":
        targets_consistent,

    "model_row_indices_consistent":
        indices_consistent,

    "explicit_patient_alignment_verified":
        explicit_patient_alignment,

    "source_target_alignment":
        target_source_checks,

    "full_landmark_alignment_verified":
        bool(landmark_verified),

    "limitations": [
        "Target array equality does not prove patient identity.",
        "Matching pandas indices alone do not establish PTID alignment.",
        "Explicit model-level patient alignment requires preserved PTID mapping.",
        "Feature timing and preprocessing leakage require separate audits.",
        "This audit does not retrain or independently evaluate predictive models."
    ]
}

report_path = (
    OUTPUT_DIR / "cohort_split_audit.json"
)

report_path.write_text(
    json.dumps(report, indent=4),
    encoding="utf-8"
)

audit_table.to_csv(
    OUTPUT_DIR / "split_integrity_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 9. Final verification display
# ------------------------------------------------------------

print("\nVERIFICATION SUMMARY")

print(
    "Cohort exclusions verified:",
    report["cohort_exclusion_verified"]
)

print(
    "Patient split independence verified:",
    report[
        "patient_level_split_independence_verified"
    ]
)

print(
    "Target arrays consistent:",
    targets_consistent
)

print(
    "Model row indices consistent:",
    indices_consistent
)

print(
    "Explicit patient alignment verified:",
    explicit_patient_alignment
)

print(
    "Source target alignment:",
    target_source_checks
)

print(
    "Full landmark alignment verified:",
    landmark_verified
)

print("\nSaved:", report_path)

print("\nNo model retraining performed.")
print("No test outcomes used for model selection.")

print("\nSTEP 32.6.1 AUDIT COMPLETED")


In [ ]:
# ============================================================
# STEP 32.6.2: LANDMARK PTID ALIGNMENT AUDIT
# No Model Training | No Feature Modification
# ============================================================

import numpy as np
import pandas as pd
import json
from pathlib import Path

print("=" * 70)
print("STEP 32.6.2: LANDMARK PTID ALIGNMENT AUDIT")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

OUTPUT_DIR = PROJECT_DIR / "01_data/metadata"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

models = [
    "Model_A",
    "Model_B4",
    "Model_B8",
    "Model_B16"
]

source_splits = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df
}

results = []
ptid_mapping = {}

# ------------------------------------------------------------
# 1. Verify original source indices and PTIDs
# ------------------------------------------------------------

for split, source in source_splits.items():

    assert "PTID" in source.columns, (
        f"{split}: missing PTID"
    )

    assert source.index.is_unique, (
        f"{split}: source index is not unique"
    )

    assert source["PTID"].notna().all()

    assert source["PTID"].is_unique

    ptid_mapping[split] = pd.Series(
        source["PTID"].astype("string").to_numpy(),
        index=source.index,
        name="PTID"
    )

# ------------------------------------------------------------
# 2. Verify row alignment across landmark inputs
# ------------------------------------------------------------

for model_name in models:

    assert model_name in MODEL_INPUTS

    for split, source in source_splits.items():

        data = MODEL_INPUTS[model_name]

        X = data[f"X_{split}"]

        y = np.asarray(
            data[f"y_{split}"],
            dtype=float
        ).ravel()

        assert isinstance(X, pd.DataFrame)

        assert X.index.is_unique

        index_equal = X.index.equals(
            source.index
        )

        if not index_equal:
            raise AssertionError(
                f"{model_name}/{split}: "
                "source/model row index mismatch"
            )

        # Map model rows back to source PTIDs
        mapped_ids = (
            ptid_mapping[split]
            .reindex(X.index)
        )

        assert mapped_ids.notna().all()

        mapped_ids_unique = (
            mapped_ids.is_unique
        )

        assert mapped_ids_unique

        # Compare outcome with source records
        source_y = pd.to_numeric(
            source.loc[X.index, "TOTMOT26"],
            errors="coerce"
        ).to_numpy()

        target_equal = np.allclose(
            y,
            source_y,
            equal_nan=True,
            rtol=0,
            atol=1e-8
        )

        assert target_equal, (
            f"{model_name}/{split}: "
            "source target mismatch"
        )

        # Compare overlapping original predictors.
        # Engineered predictors may require separate
        # lineage verification.
        common_columns = [
            c for c in X.columns
            if c in source.columns and c != "PTID"
        ]

        compared_columns = 0
        matching_columns = 0

        for col in common_columns:

            left = X[col].reset_index(drop=True)
            right = source.loc[
                X.index, col
            ].reset_index(drop=True)

            # Only directly compare columns that
            # are unchanged between source and X.
            try:
                left_num = pd.to_numeric(
                    left, errors="raise"
                )
                right_num = pd.to_numeric(
                    right, errors="raise"
                )

                matches = np.allclose(
                    left_num.to_numpy(dtype=float),
                    right_num.to_numpy(dtype=float),
                    equal_nan=True,
                    rtol=0,
                    atol=1e-8
                )
            except (ValueError, TypeError):
                matches = left.astype("string").equals(
                    right.astype("string")
                )

            compared_columns += 1
            matching_columns += int(matches)

        results.append({
            "Model": model_name,
            "Split": split,
            "Patients": len(X),
            "IndexAligned": bool(index_equal),
            "UniqueMappedPTIDs": bool(
                mapped_ids_unique
            ),
            "TargetAligned": bool(
                target_equal
            ),
            "CommonColumnsChecked": compared_columns,
            "CommonColumnsMatching": matching_columns
        })

# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

alignment_df = pd.DataFrame(results)

display(alignment_df)

core_checks_passed = bool(
    alignment_df[
        [
            "IndexAligned",
            "UniqueMappedPTIDs",
            "TargetAligned"
        ]
    ].all().all()
)

# Direct PTID alignment still depends on proving
# that MODEL_INPUTS retained the original row
# index and order during feature engineering.

feature_checks_passed = bool(
    (
        alignment_df["CommonColumnsChecked"]
        ==
        alignment_df["CommonColumnsMatching"]
    ).all()
)

# ------------------------------------------------------------
# 4. Save supplementary audit
# ------------------------------------------------------------

report = {
    "core_alignment_checks_passed":
        core_checks_passed,

    "shared_feature_checks_passed":
        feature_checks_passed,

    "patient_identity_evidence":
        "Source PTID mapping through preserved indices",

    "row_lineage_independently_verified":
        None,

    "full_patient_alignment_verified":
        None,

    "interpretation":
        (
            "Index, target and feature checks support "
            "row consistency. Direct patient identity "
            "requires confirmation that preprocessing "
            "preserved source row provenance."
        )
}

report_path = (
    OUTPUT_DIR /
    "landmark_ptid_alignment_audit.json"
)

report_path.write_text(
    json.dumps(report, indent=4),
    encoding="utf-8"
)

alignment_df.to_csv(
    OUTPUT_DIR /
    "landmark_ptid_alignment_summary.csv",
    index=False
)

print("\nCORE ALIGNMENT CHECKS:",
      core_checks_passed)

print("SHARED FEATURE CHECKS:",
      feature_checks_passed)

print("ROW LINEAGE VERIFICATION:",
      "PENDING")

print("\nSaved:", report_path)

print("\nSTEP 32.6.2 COMPLETED")


In [ ]:
# ============================================================
# STEP 32.7: PTID AUDIT TRACEABILITY SUMMARY
# Documentation Only | Detailed Verification Completed in STEP 15.10
# ============================================================

from pathlib import Path
import json

print("=" * 70)
print("STEP 32.7: PTID AUDIT TRACEABILITY SUMMARY")
print("=" * 70)

# STEP 15.10 is the authoritative PTID verification step:
# - patient identifier selection
# - patient-level split verification
# - train/validation/test overlap assessment
#
# This step does not repeat calculations.
# It only documents the verification lineage for reproducibility.

ptid_audit_record = {

    "primary_ptid_verification_step":
        "STEP 15.10",

    "verification_type":
        "Patient-level PTID split integrity audit",

    "duplicate_recalculation":
        False,

    "purpose":
        "Reproducibility documentation and audit traceability"

}


output_dir = Path(
    "/kaggle/working/SCI_Recovery_ML_Project/01_data/metadata"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


output_file = (
    output_dir /
    "ptid_audit_traceability.json"
)


with open(
    output_file,
    "w"
) as f:

    json.dump(
        ptid_audit_record,
        f,
        indent=4
    )


print("\nPTID audit traceability saved:")
print(output_file)

In [ ]:

# ============================================================
# STEP 32.8: UPDATED REPRODUCIBILITY STATUS
# Includes Saved-Model Prediction Reproduction
# Evidence-Based Verification | No Model Retraining
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 32.8: UPDATED REPRODUCIBILITY STATUS")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

META = PROJECT_DIR / "01_data/metadata"
META.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load available audit reports
# ------------------------------------------------------------

def load_audit(filename):
    path = META / filename

    if not path.is_file():
        print("Missing audit:", filename)
        return None

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


provenance = load_audit(
    "dataset_provenance.json"
)

cohort = load_audit(
    "cohort_exclusion_audit.json"
)

split_audit = load_audit(
    "cohort_split_audit.json"
)

nested_audit = load_audit(
    "nested_cv_patient_integrity.json"
)

feature_audit = load_audit(
    "temporal_feature_audit.json"
)

timing_audit = load_audit(
    "assessment_timing_audit.json"
)

interval_audit = load_audit(
    "prediction_interval_audit.json"
)

reproduction_audit = load_audit(
    "saved_model_reproduction_audit.json"
)

# ------------------------------------------------------------
# 2. Read verified dataset information
# ------------------------------------------------------------

dataset_identical = (
    provenance.get("dataset", {}).get(
        "source_copy_identical"
    )
    if provenance is not None else None
)

cohort_verified = (
    cohort.get("outcome_based_cohort_verified")
    if cohort is not None else None
)

split_verified = (
    cohort.get(
        "patient_split_independence_verified"
    )
    if cohort is not None else None
)

overlaps = (
    cohort.get("split_overlap_counts")
    if cohort is not None else None
)

# ------------------------------------------------------------
# 3. Read CV and feature audit results
# ------------------------------------------------------------

nested_verified = (
    nested_audit.get(
        "patient_independence_verified"
    )
    if nested_audit is not None else None
)

feature_verified = (
    feature_audit.get(
        "feature_whitelist_and_formula_checks_passed"
    )
    if feature_audit is not None else None
)

timing_verified = (
    timing_audit.get(
        "actual_assessment_chronology_verified"
    )
    if timing_audit is not None else None
)

calibration_verified = (
    interval_audit.get(
        "calibration_independence_from_model_selection_verified"
    )
    if interval_audit is not None else None
)

# ------------------------------------------------------------
# 4. Read saved-model reproduction results
# ------------------------------------------------------------

saved_model_verified = None
raw_features_verified = None
processed_features_verified = None
predictions_verified = None
metrics_verified = None

reproduced_metrics = None
model_hashes = None

clean_environment_verified = False

if reproduction_audit is not None:

    saved_model_verified = reproduction_audit.get(
        "saved_model_reproduction_verified"
    )

    raw_features_verified = reproduction_audit.get(
        "raw_feature_order_verified"
    )

    processed_features_verified = reproduction_audit.get(
        "processed_feature_order_verified"
    )

    predictions_verified = reproduction_audit.get(
        "predictions_match_original_export"
    )

    metrics_verified = reproduction_audit.get(
        "metrics_match_reported_values"
    )

    reproduced_metrics = reproduction_audit.get(
        "reproduced_metrics"
    )

    model_hashes = reproduction_audit.get(
        "artifact_hashes"
    )

    clean_environment_verified = (
        reproduction_audit.get(
            "clean_environment_reproduction_verified"
        ) is True
    )

# ------------------------------------------------------------
# 5. Verify documentation files
# ------------------------------------------------------------

documentation = {
    "requirements_txt": (
        PROJECT_DIR / "requirements.txt"
    ).is_file(),

    "environment_yml": (
        PROJECT_DIR / "environment.yml"
    ).is_file(),

    "environment_info": (
        PROJECT_DIR / "environment_info.json"
    ).is_file(),

    "model_card": (
        PROJECT_DIR / "MODEL_CARD.md"
    ).is_file()
}

# ------------------------------------------------------------
# 6. Final status report
# ------------------------------------------------------------

status_report = {

    "audit_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "dataset_copy_identical":
        dataset_identical,

    "cohort_exclusion_verified":
        cohort_verified,

    "patient_split_independence_verified":
        split_verified,

    "split_overlap_counts":
        overlaps,

    "nested_cv_patient_independence_verified":
        nested_verified,

    "feature_whitelist_and_formulas_verified":
        feature_verified,

    "assessment_chronology_verified":
        timing_verified,

    "calibration_independence_verified":
        calibration_verified,

    # Saved-model reproduction

    "saved_model_reproduction_verified":
        saved_model_verified,

    "raw_feature_order_verified":
        raw_features_verified,

    "processed_feature_order_verified":
        processed_features_verified,

    "original_predictions_reproduced":
        predictions_verified,

    "reported_metrics_reproduced":
        metrics_verified,

    "reproduced_test_metrics":
        reproduced_metrics,

    "model_artifact_hashes":
        model_hashes,

    # Independent environment verification

    "full_clean_environment_reproduction_verified":
        clean_environment_verified,

    "test_outcome_selection_history_verified":
        None,

    "external_validation_performed":
        False,

    "documentation_files":
        documentation,

    "limitations": [

        "Saved-model reproduction was evaluated in the current Kaggle environment.",

        "Independent clean-environment reproduction has not been demonstrated.",

        "Clinical assessment chronology remains unverified.",

        "Independent conformal calibration has not been established.",

        "Feature-engineering lineage and model-selection history require separate review.",

        "External validation has not been performed.",

        "The test cohort was previously inspected during research development."

    ]
}

# ------------------------------------------------------------
# 7. Save updated audit
# ------------------------------------------------------------

output_path = (
    META / "reproducibility_status_update.json"
)

output_path.write_text(
    json.dumps(
        status_report,
        indent=4,
        default=str
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

print("\nVERIFICATION STATUS")

for key, value in status_report.items():

    if key not in [
        "limitations",
        "model_artifact_hashes"
    ]:
        print(f"{key}: {value}")

print("\nSAVED-MODEL REPRODUCTION")

print(
    "Saved model reproduced:",
    saved_model_verified
)

print(
    "Raw feature order verified:",
    raw_features_verified
)

print(
    "Processed feature order verified:",
    processed_features_verified
)

print(
    "Original predictions reproduced:",
    predictions_verified
)

print(
    "Reported metrics reproduced:",
    metrics_verified
)

print(
    "Clean environment verified:",
    clean_environment_verified
)

print("\nSaved:", output_path)

print("\nSTEP 32.8 COMPLETED")


In [ ]:
# ============================================================
# STEP 33: UPDATED PROJECT SUMMARY
# ============================================================

from pathlib import Path


summary = """

SCI Dynamic Recovery Prediction Project
=======================================


Objective:
Development of an explainable ML framework for
predicting 26-week motor recovery after SCI.


Final Model:
LightGBM Regression


Validation Framework:

- Nested cross-validation
- Repeated validation
- Calibration assessment
- Learning curve analysis
- Bootstrap confidence intervals


Explainability:

- SHAP analysis
- Permutation importance


Uncertainty:

- Conformal prediction intervals


Data Integrity:

- PTID-based patient-level split verification
- Zero overlap between train, validation and test


Robustness:

- Missing Week-16 sensitivity analysis
- Imputation sensitivity analysis


Reproducibility:

- README
- Requirements
- Environment file
- Model card
- Exported model artifacts


Limitation:

External validation on an independent cohort
remains necessary.


"""


output = Path(

"/kaggle/working/SCI_Recovery_ML_Project/"
"PROJECT_SUMMARY.txt"

)


output.write_text(

    summary,

    encoding="utf-8"

)


print(
    "PROJECT SUMMARY UPDATED"
)

print(output)

In [ ]:

# ============================================================
# STEP 34: FINAL EVIDENCE-BASED REPRODUCIBILITY AUDIT
# Includes Saved-Model Prediction Reproduction
# Research Validation | No Model Retraining
# ============================================================

import json
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 34: FINAL REPRODUCIBILITY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

META = PROJECT_DIR / "01_data/metadata"

status_path = (
    META / "reproducibility_status_update.json"
)

if not status_path.is_file():
    raise FileNotFoundError(
        "Run updated Step 32.8 first."
    )

with open(
    status_path,
    "r",
    encoding="utf-8"
) as file:

    status = json.load(file)

# ------------------------------------------------------------
# 2. Define evidence-based verification checks
# ------------------------------------------------------------

checks = {

    "Dataset copy integrity":
        status.get("dataset_copy_identical"),

    "Cohort exclusion verification":
        status.get("cohort_exclusion_verified"),

    "Patient split independence":
        status.get(
            "patient_split_independence_verified"
        ),

    "Nested CV patient independence":
        status.get(
            "nested_cv_patient_independence_verified"
        ),

    "Feature whitelist and formulas":
        status.get(
            "feature_whitelist_and_formulas_verified"
        ),

    "Saved-model reproduction":
        status.get(
            "saved_model_reproduction_verified"
        ),

    "Raw feature order":
        status.get(
            "raw_feature_order_verified"
        ),

    "Processed feature order":
        status.get(
            "processed_feature_order_verified"
        ),

    "Original test predictions reproduced":
        status.get(
            "original_predictions_reproduced"
        ),

    "Test performance reproduced":
        status.get(
            "reported_metrics_reproduced"
        ),

    "Assessment chronology":
        status.get(
            "assessment_chronology_verified"
        ),

    "Independent conformal calibration":
        status.get(
            "calibration_independence_verified"
        ),

    "Clean-environment reproduction":
        status.get(
            "full_clean_environment_reproduction_verified"
        ),

    "Model-selection history verified":
        status.get(
            "test_outcome_selection_history_verified"
        )

}

# ------------------------------------------------------------
# 3. Classify verification results
# ------------------------------------------------------------

verified = [
    name for name, result in checks.items()
    if result is True
]

not_verified = [
    name for name, result in checks.items()
    if result is False
]

unknown = [
    name for name, result in checks.items()
    if result is None
]

# ------------------------------------------------------------
# 4. Check documentation completeness
# ------------------------------------------------------------

documentation = status.get(
    "documentation_files", {}
)

documentation_complete = bool(
    documentation
) and all(
    value is True
    for value in documentation.values()
)

# ------------------------------------------------------------
# 5. Reproduced final model metrics
# ------------------------------------------------------------

reproduced_metrics = status.get(
    "reproduced_test_metrics"
)

print("\nREPRODUCED TEST PERFORMANCE")

if reproduced_metrics is not None:

    for metric in ["MAE", "RMSE", "R2"]:

        value = reproduced_metrics.get(metric)

        if value is not None:
            print(
                f"{metric}: {float(value):.6f}"
            )

else:

    print(
        "Saved-model reproduction metrics unavailable."
    )

# ------------------------------------------------------------
# 6. Generate audit status table
# ------------------------------------------------------------

def describe_status(value):

    if value is True:
        return "VERIFIED"

    if value is False:
        return "NOT VERIFIED"

    return "UNKNOWN"


audit_table = pd.DataFrame([
    {
        "Verification": name,
        "Status": describe_status(result)
    }
    for name, result in checks.items()
])

print("\nFINAL VERIFICATION TABLE")

display(audit_table)

# ------------------------------------------------------------
# 7. Determine overall status
# ------------------------------------------------------------

full_reproducibility_verified = (
    all(
        result is True
        for result in checks.values()
    )
    and documentation_complete
)

if full_reproducibility_verified:

    overall_status = (
        "Fully verified within the defined audit scope"
    )

else:

    overall_status = (
        "Partially verified; outstanding checks remain"
    )

# ------------------------------------------------------------
# 8. Create final audit report
# ------------------------------------------------------------

final_audit = {

    "project":
        "SCI Dynamic Recovery Prediction",

    "audit_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "verification_checks":
        checks,

    "verified_checks":
        verified,

    "not_verified_checks":
        not_verified,

    "unknown_checks":
        unknown,

    "verified_count":
        len(verified),

    "not_verified_count":
        len(not_verified),

    "unknown_count":
        len(unknown),

    "documentation":
        documentation,

    "documentation_complete":
        documentation_complete,

    "saved_model_reproduction_verified":
        status.get(
            "saved_model_reproduction_verified"
        ),

    "reproduced_test_metrics":
        reproduced_metrics,

    "model_artifact_hashes":
        status.get(
            "model_artifact_hashes"
        ),

    "full_reproducibility_verified":
        full_reproducibility_verified,

    "external_validation_performed":
        status.get(
            "external_validation_performed"
        ),

    "overall_status":
        overall_status,

    "remaining_requirements": [

        "Independent clean-environment reproduction.",

        "Clinical assessment chronology verification.",

        "Independent prediction interval calibration or explicit exploratory reporting.",

        "Complete feature-engineering lineage review.",

        "Document the model-selection and test-use history.",

        "Complete TRIPOD+AI manuscript checklist.",

        "External validation when independent SCI data are available."

    ]
}

# ------------------------------------------------------------
# 9. Save reports
# ------------------------------------------------------------

output_path = (
    META / "final_reproducibility_audit.json"
)

table_path = (
    META / "final_reproducibility_checklist.csv"
)

output_path.write_text(
    json.dumps(
        final_audit,
        indent=4,
        default=str
    ),
    encoding="utf-8"
)

audit_table.to_csv(
    table_path,
    index=False
)

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL REPRODUCIBILITY SUMMARY")
print("=" * 70)

print(
    "Verified checks:",
    len(verified)
)

print(
    "Not verified:",
    len(not_verified)
)

print(
    "Unknown:",
    len(unknown)
)

print("\nVERIFIED CHECKS")

for name in verified:
    print("[PASS]", name)

print("\nNOT VERIFIED")

for name in not_verified:
    print("[OPEN]", name)

print("\nUNKNOWN")

for name in unknown:
    print("[UNKNOWN]", name)

print(
    "\nDocumentation complete:",
    documentation_complete
)

print(
    "Saved-model reproduction:",
    status.get(
        "saved_model_reproduction_verified"
    )
)

print(
    "Clean-environment reproduction:",
    status.get(
        "full_clean_environment_reproduction_verified"
    )
)

print("\nOVERALL STATUS")
print(overall_status)

print("\nFILES SAVED")
print(output_path)
print(table_path)

print("\nNo model retraining performed.")
print("No original predictions modified.")

print("\nSTEP 34 COMPLETED")


In [ ]:

# ============================================================
# STEP 34.1: MODEL-SELECTION HISTORY AUDIT
# FIXED: VALID NOTEBOOK DISCOVERY AND JSON HANDLING
# No Model Training | No Prediction Changes
# ============================================================

import json
import re
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 34.1: MODEL-SELECTION HISTORY AUDIT")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

META = PROJECT_DIR / "01_data/metadata"
META.mkdir(parents=True, exist_ok=True)

# If you export your current notebook to an actual
# .ipynb file, enter its full path below.
# Otherwise, leave this as None.

MANUAL_NOTEBOOK_PATH = None

# ------------------------------------------------------------
# 1. Check whether a file is a valid Jupyter notebook
# ------------------------------------------------------------

def load_valid_notebook(path):

    try:
        content = path.read_text(
            encoding="utf-8"
        ).strip()

        if not content:
            return None

        notebook = json.loads(content)

        if not isinstance(notebook, dict):
            return None

        if not isinstance(
            notebook.get("cells"), list
        ):
            return None

        if "nbformat" not in notebook:
            return None

        return notebook

    except (
        OSError,
        UnicodeDecodeError,
        json.JSONDecodeError
    ):
        return None

# ------------------------------------------------------------
# 2. Discover actual notebook files
# ------------------------------------------------------------

search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

candidate_paths = []

if MANUAL_NOTEBOOK_PATH is not None:

    manual_path = Path(
        MANUAL_NOTEBOOK_PATH
    )

    if not manual_path.is_file():
        raise FileNotFoundError(
            f"Manual path does not exist: {manual_path}"
        )

    candidate_paths = [manual_path]

else:

    for root in search_roots:

        if root.is_dir():

            candidate_paths.extend(
                root.rglob("*.ipynb")
            )

# Remove duplicates and Kaggle internal files.

candidate_paths = sorted({
    path.resolve()
    for path in candidate_paths
    if ".virtual_documents" not in str(path)
    and ".ipynb_checkpoints" not in str(path)
})

valid_notebooks = []

for path in candidate_paths:

    notebook = load_valid_notebook(path)

    if notebook is not None:

        valid_notebooks.append(
            (path, notebook)
        )

print("\nVALID NOTEBOOK FILES")

if valid_notebooks:

    for number, (path, notebook) in enumerate(
        valid_notebooks, start=1
    ):

        print(
            f"{number}. {path} "
            f"({len(notebook['cells'])} cells)"
        )

else:

    print("No valid exported notebook found.")

# ------------------------------------------------------------
# 3. Select only an unambiguous notebook
# ------------------------------------------------------------

NOTEBOOK_PATH = None
selected_notebook = None

if len(valid_notebooks) == 1:

    NOTEBOOK_PATH, selected_notebook = (
        valid_notebooks[0]
    )

elif len(valid_notebooks) > 1:

    print(
        "\nMultiple valid notebooks exist. "
        "Set MANUAL_NOTEBOOK_PATH explicitly "
        "to avoid auditing the wrong file."
    )

# ------------------------------------------------------------
# 4. Scan model-development operations
# ------------------------------------------------------------

patterns = {

    "Optuna tuning":
        r"\.optimize\s*\(",

    "Group-aware CV":
        r"\bGroupKFold\s*\(",

    "Ordinary CV":
        r"(?<!Group)\bKFold\s*\(",

    "Model fitting":
        r"\.fit\s*\(",

    "Test prediction":
        r"\.predict\s*\(\s*X_test",

    "Metric calculation":
        r"r2_score\s*\(|mean_absolute_error\s*\(",

    "Model artifact export":
        r"joblib\.dump\s*\("
}

events = []
execution_counts = []
code_cell_count = 0
complete_execution_record = None

if selected_notebook is not None:

    print("\nSELECTED NOTEBOOK:")
    print(NOTEBOOK_PATH)

    for cell_index, cell in enumerate(
        selected_notebook["cells"]
    ):

        if cell.get("cell_type") != "code":
            continue

        code_cell_count += 1

        source = "".join(
            cell.get("source", [])
        )

        execution_count = cell.get(
            "execution_count"
        )

        execution_counts.append(
            execution_count
        )

        for operation, pattern in patterns.items():

            if re.search(pattern, source):

                events.append({
                    "cell_index": cell_index,
                    "execution_count":
                        execution_count,
                    "operation": operation
                })

    numeric_counts = [
        count
        for count in execution_counts
        if isinstance(count, int)
    ]

    complete_execution_record = bool(
        len(numeric_counts) == code_cell_count
        and len(set(numeric_counts))
            == code_cell_count
    )

    print("\nMODEL DEVELOPMENT OPERATIONS")

    for event in events:

        print(
            f"Cell {event['cell_index']}: "
            f"{event['operation']} "
            f"(execution count "
            f"{event['execution_count']})"
        )

else:

    print("\nNOTEBOOK SOURCE STATUS")

    print(
        "No unambiguous, valid exported "
        "notebook is available."
    )

    print(
        "Export the current Kaggle notebook "
        "as .ipynb and attach it privately, "
        "then rerun this cell."
    )

# ------------------------------------------------------------
# 5. Save evidence-based audit
# ------------------------------------------------------------

report = {

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "valid_notebooks_found":
        len(valid_notebooks),

    "notebook_found":
        selected_notebook is not None,

    "notebook_path":
        str(NOTEBOOK_PATH)
        if NOTEBOOK_PATH is not None
        else None,

    "code_cells_scanned":
        code_cell_count,

    "operation_locations":
        events,

    "complete_execution_record":
        complete_execution_record,

    # IMPORTANT:
    # Code ordering cannot prove the absence
    # of historical test-set-informed selection.

    "test_selection_history_verified":
        None,

    "model_selection_independence_verified":
        None,

    "historical_model_selection_status":
        "UNKNOWN",

    "reason":
        (
            "Static notebook inspection does not "
            "prove the original sequence of "
            "model-selection decisions, including "
            "decisions made in prior sessions."
        ),

    "additional_evidence_required": [

        "Original modeling protocol",

        "Original hyperparameter search history",

        "Experiment logs from the initial development",

        "Date of first test-set evaluation",

        "Record of any model changes after "
        "test results were examined"

    ]
}

OUTPUT_PATH = (
    META / "model_selection_history_audit.json"
)

OUTPUT_PATH.write_text(
    json.dumps(
        report,
        indent=4
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 6. Display final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL-SELECTION AUDIT SUMMARY")
print("=" * 70)

print(
    "Valid notebook files:",
    len(valid_notebooks)
)

print(
    "Notebook scanned:",
    selected_notebook is not None
)

print(
    "Code cells scanned:",
    code_cell_count
)

print(
    "Complete execution counts:",
    complete_execution_record
)

print(
    "Historical model-selection independence:",
    "UNKNOWN"
)

print("\nAudit saved:")
print(OUTPUT_PATH)

print("\nModel retrained: No")
print("Predictions modified: No")

print("\nSTEP 34.1 COMPLETED")


In [ ]:

# ============================================================
# STEP 34.2: PREPARE CLEAN-SESSION REPRODUCTION PACKAGE
# No Model Fitting | No Original Artifact Overwrite
# ============================================================

import json
import hashlib
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 34.2: PREPARE REPRODUCTION PACKAGE")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

MODEL_DIR = PROJECT_DIR / "05_models"
REPRO_DIR = PROJECT_DIR / "07_reproduction"

REPRO_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = (
    MODEL_DIR /
    "SCI_Dynamic_Recovery_LightGBM_Model.pkl"
)

RAW_FEATURE_FILE = (
    MODEL_DIR / "SCI_Model_Feature_List.pkl"
)

PROCESSED_FEATURE_FILE = (
    MODEL_DIR / "SCI_Model_Processed_Feature_List.pkl"
)

for p in [
    MODEL_FILE,
    RAW_FEATURE_FILE,
    PROCESSED_FEATURE_FILE
]:
    if not p.is_file():
        raise FileNotFoundError(str(p))

if "MODEL_INPUTS" not in globals():
    raise RuntimeError(
        "Run the model-input preparation cells first."
    )

# ------------------------------------------------------------
# 1. Load frozen test inputs and predictions
# ------------------------------------------------------------

data = MODEL_INPUTS["Model_B16"]

X_test = data["X_test"].copy()

y_test = np.asarray(
    data["y_test"], dtype=float
).ravel()

if "lgbm_test_prediction" not in globals():
    raise RuntimeError(
        "Original test predictions unavailable. "
        "Run the existing prediction restoration cell."
    )

original_predictions = np.asarray(
    lgbm_test_prediction, dtype=float
).ravel()

assert len(X_test) == len(y_test)
assert len(y_test) == len(original_predictions)
assert len(X_test) == 99

saved_features = list(
    joblib.load(RAW_FEATURE_FILE)
)

assert X_test.columns.tolist() == saved_features

# ------------------------------------------------------------
# 2. Save a frozen test-input snapshot
# ------------------------------------------------------------

X_PATH = REPRO_DIR / "frozen_X_test.pkl"
Y_PATH = REPRO_DIR / "frozen_y_test.npy"
PRED_PATH = REPRO_DIR / "original_predictions.npy"

X_test.to_pickle(X_PATH)
np.save(Y_PATH, y_test)
np.save(PRED_PATH, original_predictions)

# ------------------------------------------------------------
# 3. Create integrity hashes
# ------------------------------------------------------------

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()

files = {
    "model": MODEL_FILE,
    "raw_features": RAW_FEATURE_FILE,
    "processed_features": PROCESSED_FEATURE_FILE,
    "X_test": X_PATH,
    "y_test": Y_PATH,
    "predictions": PRED_PATH
}

manifest = {
    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "test_patients": len(y_test),

    "features": saved_features,

    "files": {
        name: {
            "path": str(path),
            "filename": path.name,
            "sha256": file_hash(path)
        }
        for name, path in files.items()
    },

    "expected_metrics": {
        "MAE": 5.542201,
        "RMSE": 10.304732,
        "R2": 0.880022
    },

    "clean_session_verified": False,

    "privacy":
        "Contains patient-level research data; keep private."
}

MANIFEST_PATH = (
    REPRO_DIR / "reproduction_manifest.json"
)

MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("\nREPRODUCTION PACKAGE PREPARED")
print("Patients:", len(y_test))
print("Features:", len(saved_features))

for name, path in files.items():
    print(name, ":", path)

print("\nManifest:", MANIFEST_PATH)
print("Clean-session verification: PENDING")
print("Model retrained: No")
print("\nSTEP 34.2 COMPLETED")


In [ ]:

# ============================================================
# STEP 34.2A: EXPORT FROZEN REPRODUCTION PACKAGE
# Run in ORIGINAL Kaggle notebook
# ============================================================

import zipfile
import hashlib
import json
from pathlib import Path

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

REPRO_DIR = PROJECT_DIR / "07_reproduction"
MODEL_DIR = PROJECT_DIR / "05_models"

ZIP_PATH = Path(
    "/kaggle/working/SCI_Clean_Reproduction_Package.zip"
)

files = {
    "model/SCI_Dynamic_Recovery_LightGBM_Model.pkl":
        MODEL_DIR / "SCI_Dynamic_Recovery_LightGBM_Model.pkl",

    "model/SCI_Model_Feature_List.pkl":
        MODEL_DIR / "SCI_Model_Feature_List.pkl",

    "model/SCI_Model_Processed_Feature_List.pkl":
        MODEL_DIR / "SCI_Model_Processed_Feature_List.pkl",

    "data/frozen_X_test.pkl":
        REPRO_DIR / "frozen_X_test.pkl",

    "data/frozen_y_test.npy":
        REPRO_DIR / "frozen_y_test.npy",

    "data/original_predictions.npy":
        REPRO_DIR / "original_predictions.npy",

    "reproduction_manifest.json":
        REPRO_DIR / "reproduction_manifest.json"
}

for name, path in files.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )

with zipfile.ZipFile(
    ZIP_PATH, "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for name, path in files.items():
        archive.write(path, arcname=name)

print("=" * 65)
print("REPRODUCTION PACKAGE EXPORTED")
print("=" * 65)
print("ZIP:", ZIP_PATH)
print("Files:", len(files))
print("Size MB:", round(
    ZIP_PATH.stat().st_size / (1024 ** 2), 3
))

print("\nPACKAGE CONTENTS")
for name in files:
    print("-", name)

print("\nKeep this ZIP PRIVATE.")
print("Do not publish patient-level data.")
print("STEP 34.2A COMPLETED")


In [ ]:

# ============================================================
# STEP 34.3A: IMPORT INDEPENDENT REPRODUCTION EVIDENCE
# ORIGINAL SCI RESEARCH NOTEBOOK
# No Model Training | No Prediction Changes
# ============================================================

import json
import shutil
import hashlib
from pathlib import Path

print("=" * 70)
print("STEP 34.3A: IMPORT INDEPENDENT REPRODUCTION EVIDENCE")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

META = PROJECT_DIR / "01_data/metadata"
META.mkdir(parents=True, exist_ok=True)

FILENAME = "clean_session_reproduction_result.json"

# ------------------------------------------------------------
# 1. Locate the evidence in an attached Kaggle dataset
# ------------------------------------------------------------

matches = sorted(
    Path("/kaggle/input").rglob(FILENAME)
)

print("\nMATCHING INPUT FILES")

for p in matches:
    print(p)

if len(matches) != 1:
    raise RuntimeError(
        f"Expected exactly one attached independent "
        f"reproduction report, found {len(matches)}. "
        "Attach the private evidence dataset and retry."
    )

SOURCE = matches[0]

# ------------------------------------------------------------
# 2. Validate the transferred report
# ------------------------------------------------------------

report = json.loads(
    SOURCE.read_text(encoding="utf-8")
)

required_flags = [
    "artifact_hashes_verified",
    "raw_feature_order_verified",
    "processed_feature_order_verified",
    "predictions_match",
    "metrics_match",
    "reproduction_checks_passed"
]

for key in required_flags:

    if report.get(key) is not True:
        raise AssertionError(
            f"Evidence check failed or missing: {key}"
        )

metrics = report.get("reproduced_metrics")

if not isinstance(metrics, dict):
    raise AssertionError(
        "Reproduced metrics missing."
    )

expected = {
    "MAE": 5.542201,
    "RMSE": 10.304732,
    "R2": 0.880022
}

for name, reference in expected.items():

    actual = float(metrics[name])

    assert abs(actual - reference) <= 0.00015, (
        f"Metric mismatch: {name}"
    )

assert int(
    report.get("patient_count", 99)
) == 99

assert float(
    report.get(
        "maximum_prediction_difference",
        0.0
    )
) <= 1e-5

# ------------------------------------------------------------
# 3. Preserve the original evidence file
# ------------------------------------------------------------

DESTINATION = (
    META / FILENAME
)

if SOURCE.resolve() == DESTINATION.resolve():
    raise RuntimeError(
        "Evidence must originate from an attached "
        "external input, not the original output."
    )

shutil.copy2(
    SOURCE,
    DESTINATION
)

# ------------------------------------------------------------
# 4. Record transferred evidence hash
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


source_hash = sha256_file(SOURCE)
destination_hash = sha256_file(DESTINATION)

assert source_hash == destination_hash

# ------------------------------------------------------------
# 5. Print status
# ------------------------------------------------------------

print("\nEVIDENCE VALIDATION")

print("Original input:", SOURCE)
print("Imported report:", DESTINATION)

print("File SHA-256:", destination_hash)
print("Transfer integrity: PASSED")

print("Artifact hashes verified: True")
print("Raw feature order: PASSED")
print("Processed feature order: PASSED")
print("Predictions match: True")
print("Metrics match: True")

print("\nREPRODUCED TEST METRICS")

for name in ["MAE", "RMSE", "R2"]:
    print(f"{name}: {float(metrics[name]):.6f}")

print(
    "\nIndependent notebook execution:",
    "Researcher-confirmed"
)

print(
    "Important: Separate-session execution is "
    "confirmed by the researcher, not by JSON alone."
)

print("\nSTEP 34.3A COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# STEP 34.4 —FINAL Q1 RESEARCH AUDIT
#
# Uses:
# - Existing saved-model reproduction evidence
# - Assessment timing audit
# - Calibration sensitivity audit
# - Model-selection evidence, if actually available
#
# No model training.
# No prediction modifications.
# No fabricated verification flags.
# ============================================================

import json
import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 34.4: CORRECTED FINAL Q1 RESEARCH AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

META = PROJECT_DIR / "01_data/metadata"
META.mkdir(parents=True, exist_ok=True)

STATUS_PATH = META / "reproducibility_status_update.json"
FINAL_PATH = META / "final_reproducibility_audit.json"
CHECKLIST_PATH = META / "final_reproducibility_checklist.csv"

SAVED_AUDIT_PATH = (
    META / "saved_model_reproduction_audit.json"
)

# The independent report is stored in Kaggle input.
INDEPENDENT_REPORT_PATH = Path(
    "/kaggle/input/datasets/shizansarkar/"
    "sci-independent-reproduction-evidence/"
    "clean_session_reproduction_result.json"
)

# ------------------------------------------------------------
# 2. Safe JSON readers
# ------------------------------------------------------------

def read_json_path(path, required=False):
    if not path.is_file():
        if required:
            raise FileNotFoundError(str(path))
        return None

    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_metadata(filename, required=False):
    return read_json_path(META / filename, required=required)


def find_evidence(filename):
    """Search actual files; do not create missing evidence."""
    roots = [
        META,
        PROJECT_DIR / "06_results",
        Path("/kaggle/working/final_manuscript_results")
    ]

    for root in roots:
        if not root.exists():
            continue

        candidate = root / filename
        if candidate.is_file():
            return candidate

    return None


def safe_status(value):
    """Return True, False, or None; preserve unknown as unknown."""
    if value is True:
        return True
    if value is False:
        return False
    return None


# ------------------------------------------------------------
# 3. Load actual evidence
# ------------------------------------------------------------

reproduction = read_json_path(
    INDEPENDENT_REPORT_PATH,
    required=True
)

saved_audit = read_json_path(
    SAVED_AUDIT_PATH,
    required=True
)

timing = read_metadata(
    "assessment_timing_audit.json"
)

calibration = read_metadata(
    "calibration_sensitivity_audit.json"
)

selection_path = find_evidence(
    "model_selection_evidence_audit.json"
)

selection = (
    read_json_path(selection_path)
    if selection_path is not None
    else None
)

previous_audit = read_metadata(
    "final_reproducibility_audit.json"
)

previous_status = read_metadata(
    "reproducibility_status_update.json"
)

print("\nEVIDENCE LOCATIONS")
print("Independent reproduction:", INDEPENDENT_REPORT_PATH)
print("Saved-model audit:", SAVED_AUDIT_PATH)
print("Assessment timing audit:", timing is not None)
print("Calibration audit:", calibration is not None)
print("Model-selection audit:", selection_path)

# ------------------------------------------------------------
# 4. Verify the independent inference report
# ------------------------------------------------------------

required_flags = [
    "artifact_hashes_verified",
    "raw_feature_order_verified",
    "processed_feature_order_verified",
    "predictions_match",
    "metrics_match",
    "reproduction_checks_passed"
]

failed_flags = [
    key for key in required_flags
    if reproduction.get(key) is not True
]

if failed_flags:
    raise RuntimeError(
        "Independent inference checks failed or are missing: "
        + ", ".join(failed_flags)
    )

if int(reproduction.get("patient_count", -1)) != 99:
    raise ValueError(
        "The reproduction report does not contain the expected "
        "99-patient test cohort."
    )

metrics = reproduction.get("reproduced_metrics", {})

for metric in ["MAE", "RMSE", "R2"]:
    if metric not in metrics:
        raise KeyError(
            f"Missing reproduced metric: {metric}"
        )

    if not np.isfinite(float(metrics[metric])):
        raise ValueError(
            f"Invalid reproduced metric: {metric}"
        )

if saved_audit.get(
    "saved_model_reproduction_verified"
) is not True:
    raise RuntimeError(
        "The local saved-model reproduction audit has not passed."
    )

local_metrics = saved_audit.get(
    "reproduced_metrics", {}
)

for metric in ["MAE", "RMSE", "R2"]:
    if metric not in local_metrics:
        raise KeyError(
            f"Local audit is missing metric: {metric}"
        )

    if not np.isclose(
        float(metrics[metric]),
        float(local_metrics[metric]),
        rtol=0,
        atol=1e-8
    ):
        raise ValueError(
            f"Metric mismatch between reproduction reports: {metric}"
        )

print("\nINDEPENDENT INFERENCE REPRODUCTION: PASSED")
print("Patients:", reproduction["patient_count"])
print("Maximum prediction difference:",
      reproduction.get("maximum_prediction_difference"))

for metric in ["MAE", "RMSE", "R2"]:
    print(f"{metric}: {float(metrics[metric]):.6f}")

# A successful inference reproduction is not proof that the
# entire raw-data-to-model training pipeline was reproduced.
inference_reproduction_verified = True
full_training_pipeline_verified = False

# The report says researcher confirmation is required.
session_independence_status = reproduction.get(
    "session_independence",
    "UNKNOWN"
)

# Do not infer a clean environment from a separate session.
clean_environment_verified = False

# ------------------------------------------------------------
# 5. Extract model-selection and chronology evidence
# ------------------------------------------------------------

selection_status = None

if selection is not None:
    selection_status = safe_status(
        selection.get(
            "model_selection_independence_verified"
        )
    )

chronology_status = None

if timing is not None:
    chronology_status = safe_status(
        timing.get(
            "actual_assessment_chronology_verified"
        )
    )

calibration_independence = None
calibration_results = {}

if calibration is not None:
    calibration_independence = safe_status(
        calibration.get(
            "hyperparameter_selection_independence_verified"
        )
    )

    calibration_results = calibration.get(
        "results", {}
    )

    if not isinstance(calibration_results, dict):
        raise ValueError(
            "Calibration audit results must be a JSON object."
        )

# A completed sensitivity analysis does not establish independent
# conformal calibration or valid nominal coverage.
formal_calibration_verified = False

print("\nMETHODOLOGICAL EVIDENCE")
print("Model-selection independence:", selection_status)
print("Actual assessment chronology:", chronology_status)
print("Calibration independence:", calibration_independence)
print("Formal calibration verified:", formal_calibration_verified)
print("Full training-pipeline reproduction:",
      full_training_pipeline_verified)
print("Clean-environment reproduction:",
      clean_environment_verified)

# ------------------------------------------------------------
# 6. Preserve existing checks, if available
# ------------------------------------------------------------

audit = previous_audit if previous_audit is not None else {}

checks = dict(
    audit.get("verification_checks", {})
)

# Update only claims directly supported by current evidence.
checks["Saved-model inference reproduction"] = (
    inference_reproduction_verified
)

checks["Clean-environment reproduction"] = (
    clean_environment_verified
)

checks["Full training-pipeline reproduction"] = (
    full_training_pipeline_verified
)

checks["Independent conformal calibration"] = (
    formal_calibration_verified
)

checks["Assessment chronology"] = chronology_status

checks["Model-selection history verified"] = selection_status

audit["verification_checks"] = checks

audit["verified_checks"] = [
    key for key, value in checks.items()
    if value is True
]

audit["not_verified_checks"] = [
    key for key, value in checks.items()
    if value is False
]

audit["unknown_checks"] = [
    key for key, value in checks.items()
    if value is None
]

audit["verified_count"] = len(
    audit["verified_checks"]
)

audit["not_verified_count"] = len(
    audit["not_verified_checks"]
)

audit["unknown_count"] = len(
    audit["unknown_checks"]
)

# ------------------------------------------------------------
# 7. Add source-linked evidence summaries
# ------------------------------------------------------------

audit["independent_reproduction"] = {
    "source_report": str(INDEPENDENT_REPORT_PATH),
    "verification_type": reproduction.get(
        "verification_type"
    ),
    "patient_count": reproduction.get(
        "patient_count"
    ),
    "artifact_hashes_verified": reproduction.get(
        "artifact_hashes_verified"
    ),
    "raw_feature_order_verified": reproduction.get(
        "raw_feature_order_verified"
    ),
    "processed_feature_order_verified": reproduction.get(
        "processed_feature_order_verified"
    ),
    "predictions_match": reproduction.get(
        "predictions_match"
    ),
    "metrics_match": reproduction.get(
        "metrics_match"
    ),
    "maximum_prediction_difference": reproduction.get(
        "maximum_prediction_difference"
    ),
    "reproduced_metrics": metrics,
    "session_independence": session_independence_status,
    "clean_environment_verified": clean_environment_verified,
    "full_training_pipeline_verified": False,
    "model_retrained": reproduction.get(
        "model_retrained"
    )
}

audit["model_selection_evidence"] = {
    "evidence_file_found": selection is not None,
    "evidence_path": (
        str(selection_path)
        if selection_path is not None
        else None
    ),
    "independence_verified": selection_status,
    "nested_cv_evidence": (
        selection.get("nested_cv_evidence")
        if selection is not None else None
    ),
    "optuna_studies_found": (
        len(selection.get("optuna_studies", []))
        if selection is not None else None
    ),
    "historical_files_found": (
        len(selection.get("historical_files", []))
        if selection is not None else None
    )
}

audit["assessment_timing_evidence"] = {
    "audit_available": timing is not None,
    "actual_chronology_verified": chronology_status,
    "nominal_landmark_reporting_required": (
        timing.get(
            "nominal_landmark_reporting_required"
        ) if timing is not None else None
    ),
    "week16_missing_by_split": (
        timing.get(
            "week16_feature_missing_by_split"
        ) if timing is not None else None
    )
}

audit["calibration_sensitivity_evidence"] = {
    "audit_available": calibration is not None,
    "analysis_completed": calibration is not None,
    "fit_patients": calibration_results.get(
        "fit_patients"
    ),
    "calibration_patients": calibration_results.get(
        "calibration_patients"
    ),
    "test_patients": calibration_results.get(
        "test_patients"
    ),
    "empirical_coverage": calibration_results.get(
        "coverage"
    ),
    "mean_interval_width": calibration_results.get(
        "mean_width"
    ),
    "test_mae": calibration_results.get(
        "test_mae"
    ),
    "hyperparameter_selection_independence_verified":
        calibration_independence,
    "formal_calibration_verified":
        formal_calibration_verified
}

# ------------------------------------------------------------
# 8. Overall status — do not overstate reproducibility
# ------------------------------------------------------------

audit["full_reproducibility_verified"] = False

audit["overall_status"] = (
    "Partially verified; outstanding checks remain"
)

audit["remaining_requirements"] = [
    "Verify the historical model-selection and test-set-use history.",
    "Confirm actual assessment-date semantics from source documentation.",
    "Do not claim independent conformal calibration without supporting evidence.",
    "Complete applicable TRIPOD+AI reporting checks.",
    "Obtain external validation before claiming external generalizability.",
    "Document the scope and limitations of inference reproduction."
]

audit["updated_utc"] = datetime.now(
    timezone.utc
).isoformat()

# ------------------------------------------------------------
# 9. Update status summary conservatively
# ------------------------------------------------------------

status = (
    previous_status
    if previous_status is not None
    else {}
)

status["saved_model_inference_reproduction_verified"] = True
status["full_training_pipeline_reproduction_verified"] = False
status["full_clean_environment_reproduction_verified"] = False
status["assessment_chronology_verified"] = chronology_status
status["calibration_independence_verified"] = False
status["test_outcome_selection_history_verified"] = selection_status
status["calibration_sensitivity_completed"] = (
    calibration is not None
)
status["session_independence"] = session_independence_status
status["updated_utc"] = audit["updated_utc"]

# ------------------------------------------------------------
# 10. Save audit files
# ------------------------------------------------------------

FINAL_PATH.write_text(
    json.dumps(audit, indent=4, default=str),
    encoding="utf-8"
)

STATUS_PATH.write_text(
    json.dumps(status, indent=4, default=str),
    encoding="utf-8"
)

checklist = pd.DataFrame([
    {
        "Verification": key,
        "Status": (
            "VERIFIED" if value is True
            else "NOT VERIFIED" if value is False
            else "UNKNOWN"
        )
    }
    for key, value in checks.items()
])

checklist.to_csv(
    CHECKLIST_PATH,
    index=False
)

# ------------------------------------------------------------
# 11. Print final report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL VERIFICATION CHECKLIST")
print("=" * 70)

display(checklist)

print("\nAUDIT COUNTS")
print("Verified:", audit["verified_count"])
print("Not verified:", audit["not_verified_count"])
print("Unknown:", audit["unknown_count"])

print("\nOVERALL STATUS")
print(audit["overall_status"])

print("\nFILES SAVED")
print(FINAL_PATH)
print(STATUS_PATH)
print(CHECKLIST_PATH)

print("\nNo model retraining performed.")
print("No original predictions modified.")
print("Missing model-selection evidence remains UNKNOWN.")
print("Clean-environment reproduction is NOT claimed.")
print("\nSTEP 34.4 COMPLETED")


In [ ]:

# ============================================================
# STEP 34.5: MODEL-SELECTION EVIDENCE AUDIT
# Evidence Reconstruction | No Model Retraining
# ============================================================

import json
import hashlib
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 34.5: MODEL-SELECTION EVIDENCE AUDIT")
print("=" * 70)

PROJECT_DIR = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

META = PROJECT_DIR / "01_data/metadata"
META.mkdir(parents=True, exist_ok=True)

def json_safe(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {
            str(k): json_safe(v)
            for k, v in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

# ------------------------------------------------------------
# 1. Verify final model is available
# ------------------------------------------------------------

if "final_lgbm_model" not in globals():
    raise RuntimeError(
        "Restore final_lgbm_model before auditing."
    )

final_model_params = json_safe(
    final_lgbm_model.get_params(deep=False)
)

# ------------------------------------------------------------
# 2. Inspect current nested CV evidence
# ------------------------------------------------------------

nested_summary = {
    "available": False,
    "folds": None,
    "mean_mae": None,
    "mean_rmse": None,
    "mean_r2": None
}

if "nested_cv_results_df" in globals():

    df_cv = nested_cv_results_df.copy()

    required = {"MAE", "RMSE", "R2"}

    if required.issubset(df_cv.columns):

        nested_summary = {
            "available": True,
            "folds": int(len(df_cv)),
            "mean_mae": float(df_cv["MAE"].mean()),
            "mean_rmse": float(df_cv["RMSE"].mean()),
            "mean_r2": float(df_cv["R2"].mean())
        }

# ------------------------------------------------------------
# 3. Inspect Optuna studies, if retained
# ------------------------------------------------------------

optuna_records = []

if "nested_studies" in globals():

    for fold, study in nested_studies.items():

        optuna_records.append({
            "fold": str(fold),
            "trial_count": len(study.trials),
            "best_trial": int(
                study.best_trial.number
            ),
            "best_value": float(
                study.best_value
            ),
            "best_params": json_safe(
                study.best_params
            )
        })

# ------------------------------------------------------------
# 4. Collect historical artifacts
# ------------------------------------------------------------

history_files = []

for pattern in [
    "*optuna*.json",
    "*optuna*.csv",
    "*experiment*.json",
    "*selection*.json",
    "*nested_cv*.json"
]:
    for path in PROJECT_DIR.rglob(pattern):

        if path.is_file() and path.parent != META:
            history_files.append(path)

history_files = sorted(set(history_files))

history_inventory = [
    {
        "path": str(path),
        "sha256": sha256_file(path)
    }
    for path in history_files
]

# ------------------------------------------------------------
# 5. Identify limitations
# ------------------------------------------------------------

issues = [
    "Notebook execution order does not establish historical decision order.",
    "Current Optuna objects do not prove earlier experiments did not use test results.",
    "The test cohort was inspected during subsequent research revisions.",
    "A pre-test analysis protocol or complete experiment logs are needed to verify original test independence."
]

report = {
    "audit_utc":
        datetime.now(timezone.utc).isoformat(),

    "final_model_available": True,
    "final_model_params": final_model_params,

    "nested_cv_evidence": nested_summary,
    "optuna_studies_available":
        len(optuna_records) > 0,
    "optuna_studies": optuna_records,

    "historical_files": history_inventory,

    "test_selection_history_verified": None,
    "model_selection_independence_verified": None,

    "historical_selection_status": "UNKNOWN",

    "limitations": issues,

    "research_reporting_recommendation":
        "Describe the original test evaluation as internal "
        "and disclose retrospective test inspection."
}

OUTPUT = (
    META / "model_selection_evidence_audit.json"
)

OUTPUT.write_text(
    json.dumps(
        report,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

print("\nNESTED CV EVIDENCE")
print(nested_summary)

print("\nOPTUNA STUDIES FOUND")
print(len(optuna_records))

print("\nHISTORICAL FILES FOUND")
print(len(history_inventory))

print("\nMODEL-SELECTION INDEPENDENCE")
print("UNKNOWN — historical evidence required")

print("\nSaved:", OUTPUT)
print("Model retrained: No")
print("Test predictions modified: No")
print("\nSTEP 34.5 COMPLETED")


In [ ]:

# ============================================================
# STEP 34.6 — TEST-SET INDEPENDENCE DISCLOSURE
# Placement: After STEP 34.5
# Purpose:
#   Preserve the evidence status and create manuscript text.
#   No model retraining or test-set re-analysis.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
META = ROOT / "01_data/metadata"
OUT = ROOT / "07_manuscript_package/final_evidence"

META.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

audit_path = META / "model_selection_evidence_audit.json"

if not audit_path.is_file():
    raise FileNotFoundError(
        f"Missing model-selection audit: {audit_path}. "
        "Run STEP 34.5 first."
    )

evidence = json.loads(audit_path.read_text(encoding="utf-8"))

# Preserve the actual evidence state.
history_status = evidence.get(
    "historical_selection_status", "UNKNOWN"
)
independence_status = evidence.get(
    "model_selection_independence_verified", None
)

limitations = evidence.get("limitations", [])

disclosure = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "historical_selection_status": history_status,
    "model_selection_independence_verified": independence_status,
    "test_cohort_inspected_during_later_revisions": any(
        "test cohort was inspected" in str(item).lower()
        for item in limitations
    ),
    "external_validation_performed": False,
    "model_retrained": False,
    "test_metrics_recomputed": False,
    "interpretation": (
        "Historical independence of the test cohort from all model "
        "selection and research decisions has not been established. "
        "Current prediction reproduction verifies saved-model inference, "
        "not historical independence."
    )
}

methods_text = """
The dataset was partitioned into training, validation, and internal test
cohorts at the patient level. However, the available records did not
establish that the internal test cohort had remained untouched by all
model-selection and research decisions throughout the complete history
of the project. Accordingly, the test evaluation is reported as an
internal evaluation, and its estimates are interpreted with this
potential source of selection-related optimism in mind.
""".strip()

limitations_text = """
Historical independence of the internal test cohort could not be
verified from the available experiment history. The cohort was inspected
during subsequent research revisions, and the full sequence of earlier
model-selection decisions could not be independently reconstructed.
Consequently, reproducibility of the saved model's predictions should
not be interpreted as proof of historical test-set independence.
Independent external validation using a separate cohort is required
to assess transportability.
""".strip()

(META / "test_set_independence_disclosure.json").write_text(
    json.dumps(disclosure, indent=2),
    encoding="utf-8"
)

(OUT / "Methods_Test_Set_Independence.txt").write_text(
    methods_text + "\n", encoding="utf-8"
)

(OUT / "Limitations_Test_Set_Independence.txt").write_text(
    limitations_text + "\n", encoding="utf-8"
)

print("STEP 34.6 COMPLETED")
print("Historical independence:", history_status)
print("Independence verified:", independence_status)
print("Manuscript disclosure files created.")
print("No historical independence claim has been upgraded.")


In [ ]:

# ============================================================
# STEP 35.1: MODEL-SELECTION HISTORY DOCUMENTATION
# Evidence inventory + manuscript disclosure
# No model training or prediction changes
# ============================================================

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 35.1: MODEL-SELECTION HISTORY DOCUMENTATION")
print("=" * 70)

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
META = ROOT / "01_data/metadata"
OUT = ROOT / "07_manuscript_package"

OUT.mkdir(parents=True, exist_ok=True)

EVIDENCE_PATH = META / "model_selection_evidence_audit.json"
FINAL_AUDIT_PATH = META / "final_reproducibility_audit.json"

for path in [EVIDENCE_PATH, FINAL_AUDIT_PATH]:
    if not path.is_file():
        raise FileNotFoundError(str(path))

evidence = json.loads(
    EVIDENCE_PATH.read_text(encoding="utf-8")
)

audit = json.loads(
    FINAL_AUDIT_PATH.read_text(encoding="utf-8")
)

nested = evidence.get("nested_cv_evidence", {})
studies = evidence.get("optuna_studies", [])
historical = evidence.get("historical_files", [])

selection_status = audit.get(
    "verification_checks", {}
).get("Model-selection history verified")

# Do not convert unknown historical independence
# into a verified result using static records.

inventory = pd.DataFrame([
    {
        "Evidence": "Patient-disjoint nested CV",
        "Status": (
            "Available"
            if nested.get("available")
            else "Not available"
        ),
        "Interpretation":
            "Supports documented internal CV procedure"
    },
    {
        "Evidence": "Optuna study records",
        "Status": f"{len(studies)} study records",
        "Interpretation":
            "Documents available tuning analyses"
    },
    {
        "Evidence": "Historical artifact inventory",
        "Status": f"{len(historical)} files",
        "Interpretation":
            "Does not establish complete historical sequence"
    },
    {
        "Evidence": "Original pre-test protocol",
        "Status": "Not independently established",
        "Interpretation":
            "Requires dated research records"
    },
    {
        "Evidence": "First test inspection",
        "Status": "Not independently established",
        "Interpretation":
            "Cannot infer timing from saved code alone"
    },
    {
        "Evidence": "Post-test modification history",
        "Status": "Incomplete",
        "Interpretation":
            "Historical test-informed selection cannot be excluded"
    }
])

methods_text = (
    "The model-development partition contained 461 "
    "patients. Internal model assessment included "
    "patient-disjoint five-outer-fold and five-inner-fold "
    "nested cross-validation. A separate validation "
    "partition was also used during model development. "
    "The final frozen model was evaluated on an internal "
    "test partition of 99 patients."
)

limitations_text = (
    "The complete historical sequence of model selection, "
    "hyperparameter tuning, and initial test-set inspection "
    "could not be independently reconstructed. Test results "
    "were inspected during subsequent research revisions. "
    "Therefore, historical test-set independence cannot be "
    "confirmed, and reported internal test performance may "
    "be subject to selection-related optimism. Independent "
    "evaluation of a prespecified frozen pipeline in a new "
    "SCI cohort is recommended."
)

inventory_path = (
    OUT / "Model_Selection_Evidence_Inventory.csv"
)

text_path = (
    OUT / "Model_Selection_Methods_and_Limitations.txt"
)

report_path = (
    META / "model_selection_reporting_audit.json"
)

inventory.to_csv(inventory_path, index=False)

text_path.write_text(
    "METHODS\n\n" + methods_text
    + "\n\nLIMITATIONS\n\n" + limitations_text,
    encoding="utf-8"
)

report = {
    "updated_utc":
        datetime.now(timezone.utc).isoformat(),
    "nested_cv_available": bool(
        nested.get("available", False)
    ),
    "optuna_studies_found": len(studies),
    "historical_files_found": len(historical),
    "historical_independence_verified":
        selection_status,
    "reporting_disclosure_completed": True,
    "test_inspection_during_revisions": True,
    "model_retrained": False,
    "predictions_modified": False
}

report_path.write_text(
    json.dumps(report, indent=4),
    encoding="utf-8"
)

print("\nMODEL-SELECTION EVIDENCE")
display(inventory)

print("\nHISTORICAL INDEPENDENCE:", selection_status)
print("Disclosure prepared: Yes")

print("\nSAVED")
print(inventory_path)
print(text_path)
print(report_path)

print("\nSTEP 35.1 COMPLETED")


In [ ]:

# ============================================================
# STEP 35.2: TRIPOD+AI MANUSCRIPT PREPARATION
# Project-specific reporting inventory
# No model training
# ============================================================

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
import json

print("=" * 70)
print("STEP 35.2: TRIPOD+AI REPORTING PREPARATION")
print("=" * 70)

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
OUT = ROOT / "07_manuscript_package"
META = ROOT / "01_data/metadata"

OUT.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)

# Topic inventory, not official TRIPOD+AI numbering.

topics = [
    ("Title and abstract", "Manuscript review needed"),
    ("Clinical prediction objective", "Model purpose documented"),
    ("Intended population and setting", "Sygen cohort; describe eligibility"),
    ("Study design and data source", "Retrospective secondary analysis"),
    ("Participant eligibility", "Audit available"),
    ("Cohort flow and exclusions", "791 to 659 documented"),
    ("Outcome definition", "Nominal Week-26 TOTMOT26"),
    ("Predictor definition", "Seven raw inputs documented"),
    ("Predictor measurement timing", "Nominal visits; actual dates unverified"),
    ("Missing outcomes", "132 without Week-26 target"),
    ("Missing predictors", "Imputation pipeline; describe in Methods"),
    ("Sample size", "659 analytical patients"),
    ("Train-validation-test split", "461 / 99 / 99"),
    ("Preprocessing", "Pipeline and feature-order audit available"),
    ("Algorithm specification", "LightGBM parameters documented"),
    ("Model-selection history", "Historical independence unknown"),
    ("Internal validation", "Nested 5x5 GroupKFold documented"),
    ("Benchmark models", "AIS-only and baseline clinical"),
    ("Model performance", "MAE, RMSE and R2"),
    ("Statistical uncertainty", "Paired bootstrap intervals"),
    ("Prediction intervals", "Exploratory coverage analyses"),
    ("Calibration limitations", "Formal independence unverified"),
    ("Subgroup results", "AIS and missing Week-16 assessment"),
    ("Explainability", "SHAP and permutation importance"),
    ("Reproducibility", "Independent-session saved inference"),
    ("External validation", "Not performed"),
    ("Clinical limitations", "Discussion required"),
    ("Data availability and permissions", "Researcher verification required"),
    ("Code and artifact sharing", "Privacy review required"),
    ("Ethics and approvals", "Verify applicable permissions"),
    ("Funding and competing interests", "Author declarations required")
]

df = pd.DataFrame(
    topics,
    columns=["Reporting_Topic", "Evidence_or_Action"]
)

df["Manuscript_Section"] = ""
df["Page_or_Line"] = ""
df["Verified_by_Author"] = False
df["Notes"] = ""

CSV_PATH = OUT / "TRIPOD_AI_Preparation_Checklist.csv"
XLSX_PATH = OUT / "TRIPOD_AI_Preparation_Checklist.xlsx"

# Keep user-entered review columns if rerun.
if CSV_PATH.exists():
    previous = pd.read_csv(
        CSV_PATH,
        keep_default_na=False
    )

    if "Reporting_Topic" in previous:
        previous = previous.drop_duplicates(
            "Reporting_Topic"
        ).set_index("Reporting_Topic")

        for col in [
            "Manuscript_Section",
            "Page_or_Line",
            "Verified_by_Author",
            "Notes"
        ]:
            if col in previous.columns:
                df[col] = df["Reporting_Topic"].map(
                    previous[col]
                ).fillna(df[col])

df.to_csv(CSV_PATH, index=False)
df.to_excel(XLSX_PATH, index=False)

report = {
    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),
    "reporting_topics": len(df),
    "preparation_inventory_created": True,
    "official_tripod_ai_checklist_completed": False,
    "author_review_required": True
}

REPORT_PATH = (
    META / "tripod_ai_preparation_audit.json"
)

REPORT_PATH.write_text(
    json.dumps(report, indent=4),
    encoding="utf-8"
)

print("\nREPORTING INVENTORY")
display(df)

print("\nTopics:", len(df))
print("Official checklist completed: No")

print("\nSAVED")
print(CSV_PATH)
print(XLSX_PATH)
print(REPORT_PATH)

print("\nSTEP 35.2 COMPLETED")


In [ ]:

# ============================================================
# STEP 35.2A — TRIPOD+AI REPORTING READINESS TRACKER
# Placement: After STEP 35.2
# This is an internal tracker, not the official checklist.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
META = ROOT / "01_data/metadata"
OUT = ROOT / "07_manuscript_package/final_evidence"

META.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

items = [
    {
        "Domain": "Study design and data source",
        "Status": "PARTIAL",
        "Evidence_or_action": "Describe Sygen source, study design, and eligibility."
    },
    {
        "Domain": "Participants and cohort flow",
        "Status": "PARTIAL",
        "Evidence_or_action": "Verify exclusions and explain 791-to-659 cohort flow."
    },
    {
        "Domain": "Outcome definition",
        "Status": "ADDRESSED_REVIEW_REQUIRED",
        "Evidence_or_action": "Define nominal Week-26 total motor score and scoring range."
    },
    {
        "Domain": "Predictors and assessment timing",
        "Status": "PARTIAL",
        "Evidence_or_action": "Report nominal landmarks; individual examination chronology unverified."
    },
    {
        "Domain": "Missing data and exclusions",
        "Status": "PARTIAL",
        "Evidence_or_action": "Report outcome missingness, predictor missingness, and exclusions."
    },
    {
        "Domain": "Model development and hyperparameter selection",
        "Status": "PARTIAL",
        "Evidence_or_action": "Document search space, nested CV, seeds, and model-selection history."
    },
    {
        "Domain": "Internal validation",
        "Status": "ADDRESSED_REVIEW_REQUIRED",
        "Evidence_or_action": "Report patient-grouped nested CV and internal test evaluation separately."
    },
    {
        "Domain": "Historical test-set independence",
        "Status": "UNKNOWN",
        "Evidence_or_action": "Requires experiment logs or pre-test protocol; cannot be inferred from current notebook order."
    },
    {
        "Domain": "Performance measures",
        "Status": "ADDRESSED_REVIEW_REQUIRED",
        "Evidence_or_action": "Report MAE, RMSE, R2, paired differences, and uncertainty."
    },
    {
        "Domain": "Calibration and prediction intervals",
        "Status": "PARTIAL",
        "Evidence_or_action": "Do not claim formal 95% coverage; independent calibration not established."
    },
    {
        "Domain": "Explainability",
        "Status": "ADDRESSED_REVIEW_REQUIRED",
        "Evidence_or_action": "Describe SHAP/permutation procedures and avoid causal interpretation."
    },
    {
        "Domain": "External validation",
        "Status": "NOT_PERFORMED",
        "Evidence_or_action": "Explicitly state that external validation was not performed."
    },
    {
        "Domain": "Ethics and data permissions",
        "Status": "PENDING_CONFIRMATION",
        "Evidence_or_action": "Confirm the applicable ethics determination and dataset-use permissions."
    },
    {
        "Domain": "Code and artifact availability",
        "Status": "PARTIAL",
        "Evidence_or_action": "Verify reproducible paths, software versions, licenses, and data restrictions."
    },
    {
        "Domain": "Funding and conflicts of interest",
        "Status": "PENDING_AUTHOR_INPUT",
        "Evidence_or_action": "Complete using verified author declarations."
    }
]

tracker = pd.DataFrame(items)

csv_path = OUT / "TRIPOD_AI_Internal_Readiness_Tracker.csv"
xlsx_path = OUT / "TRIPOD_AI_Internal_Readiness_Tracker.xlsx"

tracker.to_csv(csv_path, index=False)
tracker.to_excel(xlsx_path, index=False)

report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "tracker_type": "Internal readiness tracker",
    "official_tripod_ai_checklist_completed": False,
    "items": len(tracker),
    "status_counts": tracker["Status"].value_counts().to_dict(),
    "requires_manual_author_review": True,
    "warning": (
        "This tracker is not the official TRIPOD+AI checklist and "
        "does not establish compliance. Authors must consult and complete "
        "the official checklist against the final manuscript."
    )
}

(META / "tripod_ai_readiness_audit.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("STEP 35.2A COMPLETED")
display(tracker)
print("\nOfficial checklist completed:", False)
print("Internal tracker:", csv_path)


In [ ]:

# ============================================================
# STEP 35.3 — FINAL DOCUMENTATION CONSISTENCY AUDIT
# FIXED VERSION
# ============================================================

import json
import pandas as pd
import numpy as np
import re

from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# STEP 1 — Paths
# ------------------------------------------------------------

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
META = ROOT / "01_data/metadata"
README = ROOT / "README.md"

CV_PATH = Path(
    "/kaggle/working/final_manuscript_results/"
    "nested_cv_all_landmarks/nested_cv_summary.csv"
)

REPRO_CANDIDATES = [
    Path(
        "/kaggle/input/datasets/shizansarkar/"
        "sci-independent-reproduction-evidence/"
        "clean_session_reproduction_result.json"
    ),
    META / "clean_session_reproduction_result.json"
]

REPRO_PATH = next(
    (p for p in REPRO_CANDIDATES if p.is_file()), None
)

if not ROOT.is_dir():
    raise FileNotFoundError(f"Project directory missing: {ROOT}")

if not README.is_file():
    raise FileNotFoundError(f"README missing: {README}")

if not CV_PATH.is_file():
    raise FileNotFoundError(f"Nested-CV summary missing: {CV_PATH}")

if REPRO_PATH is None:
    raise FileNotFoundError("Independent reproduction report missing.")

# ------------------------------------------------------------
# STEP 2 — Load and validate nested-CV summary
# ------------------------------------------------------------

cv = pd.read_csv(CV_PATH)

required_columns = {
    "Landmark", "MAE_Mean", "MAE_SD",
    "RMSE_Mean", "RMSE_SD", "R2_Mean", "R2_SD"
}

if not required_columns.issubset(cv.columns):
    raise ValueError(
        f"Unexpected nested-CV columns: {cv.columns.tolist()}"
    )

models = ["Model_A", "Model_B4", "Model_B8", "Model_B16"]

if cv["Landmark"].duplicated().any():
    raise ValueError("Duplicate landmarks in nested-CV summary.")

if set(cv["Landmark"]) != set(models):
    raise ValueError(
        f"Expected {models}; found {cv['Landmark'].tolist()}"
    )

numeric_columns = list(required_columns - {"Landmark"})
cv[numeric_columns] = cv[numeric_columns].apply(
    pd.to_numeric, errors="raise"
)

if not np.isfinite(cv[numeric_columns].to_numpy()).all():
    raise ValueError("Non-finite nested-CV values found.")

if (cv[["MAE_SD", "RMSE_SD", "R2_SD"]] < 0).any().any():
    raise ValueError("Negative SD detected.")

cv = cv.set_index("Landmark").loc[models].reset_index()

labels = {
    "Model_A": "Baseline",
    "Model_B4": "Week 4",
    "Model_B8": "Week 8",
    "Model_B16": "Week 16"
}

cv_results = {}
table_rows = []

for _, row in cv.iterrows():
    model = row["Landmark"]

    cv_results[model] = {
        "MAE_mean": float(row["MAE_Mean"]),
        "MAE_SD": float(row["MAE_SD"]),
        "RMSE_mean": float(row["RMSE_Mean"]),
        "RMSE_SD": float(row["RMSE_SD"]),
        "R2_mean": float(row["R2_Mean"]),
        "R2_SD": float(row["R2_SD"])
    }

    table_rows.append(
        f"| {labels[model]} "
        f"| {row['MAE_Mean']:.4f} ({row['MAE_SD']:.4f}) "
        f"| {row['RMSE_Mean']:.4f} ({row['RMSE_SD']:.4f}) "
        f"| {row['R2_Mean']:.4f} ({row['R2_SD']:.4f}) |"
    )

cv_table = "\n".join([
    "| Landmark | MAE, mean (SD) | RMSE, mean (SD) | R², mean (SD) |",
    "|---|---:|---:|---:|",
    *table_rows
])

print("Nested-CV source verified:", CV_PATH)
print(cv_table)

# ------------------------------------------------------------
# STEP 3 — Validate saved-model reproduction report
# ------------------------------------------------------------

repro = json.loads(
    REPRO_PATH.read_text(encoding="utf-8")
)

required_flags = [
    "artifact_hashes_verified",
    "raw_feature_order_verified",
    "processed_feature_order_verified",
    "predictions_match",
    "metrics_match",
    "reproduction_checks_passed"
]

if not all(repro.get(k) is True for k in required_flags):
    raise AssertionError("Reproduction verification failed.")

if int(repro.get("patient_count", -1)) != 99:
    raise ValueError("Expected 99 patients in reproduction report.")

test_metrics = repro.get("reproduced_metrics", {})

for metric in ["MAE", "RMSE", "R2"]:
    if metric not in test_metrics:
        raise KeyError(f"Missing reproduction metric: {metric}")

    test_metrics[metric] = float(test_metrics[metric])

    if not np.isfinite(test_metrics[metric]):
        raise ValueError(f"Invalid reproduction metric: {metric}")

max_diff = float(
    repro.get("maximum_prediction_difference", float("nan"))
)

# Correct formatting: no whitespace inside the format specifier.
max_diff_text = (
    f"{max_diff:.12g}" if np.isfinite(max_diff) else "Not reported"
)

# ------------------------------------------------------------
# STEP 4 — Construct README sections
# ------------------------------------------------------------

readme = README.read_text(encoding="utf-8")

cv_title = "## Latest Patient-Disjoint Nested Cross-Validation"

cv_body = f"""{cv_title}

Source: `{CV_PATH}`

{cv_table}

Values are means and standard deviations as reported in the saved
nested-CV summary. These are internal cross-validation estimates,
not external validation.
"""

test_title = "## Final Internal Test Performance"

test_body = f"""{test_title}

Saved Model_B16 inference was reproduced on the existing
99-patient internal test cohort.

| Metric | Reproduced result |
|---|---:|
| MAE | {test_metrics['MAE']:.6f} |
| RMSE | {test_metrics['RMSE']:.6f} |
| R² | {test_metrics['R2']:.6f} |

Maximum prediction difference: {max_diff_text}.

This verifies saved-model inference reproduction only. It does not
establish external validity or prove that the test set was never
inspected during historical model selection.
"""

# ------------------------------------------------------------
# STEP 5 — Replace existing sections or insert missing ones
# ------------------------------------------------------------

def replace_or_insert_section(text, title, body, before=None):
    lines = text.splitlines()
    target = title.lstrip("# ").strip().casefold()

    start = None
    heading_level = None

    for i, line in enumerate(lines):
        match = re.match(
            r"^\s{0,3}(#{1,6})\s+(.+?)\s*#*\s*$",
            line
        )
        if match and match.group(2).strip().casefold() == target:
            start = i
            heading_level = len(match.group(1))
            break

    body_lines = body.strip().splitlines()

    if start is not None:
        end = len(lines)

        for i in range(start + 1, len(lines)):
            match = re.match(r"^\s{0,3}(#{1,6})\s+", lines[i])

            if match and len(match.group(1)) <= heading_level:
                end = i
                break

        lines = lines[:start] + body_lines + lines[end:]

    else:
        insertion_point = len(lines)

        if before:
            for i, line in enumerate(lines):
                match = re.match(
                    r"^\s{0,3}#{1,6}\s+(.+?)\s*#*\s*$",
                    line
                )

                if (
                    match
                    and match.group(1).strip().casefold()
                    == before.casefold()
                ):
                    insertion_point = i
                    break

        lines[insertion_point:insertion_point] = (
            [""] + body_lines + [""]
        )

    return "\n".join(lines).rstrip() + "\n"


readme = replace_or_insert_section(
    readme, cv_title, cv_body
)

readme = replace_or_insert_section(
    readme,
    test_title,
    test_body,
    before="Test-Set Independence"
)

# ------------------------------------------------------------
# STEP 6 — Check README content before saving
# ------------------------------------------------------------

for label in labels.values():
    if label not in readme:
        raise AssertionError(f"README missing landmark: {label}")

for metric in ["MAE", "RMSE", "R2"]:
    expected = f"{test_metrics[metric]:.6f}"

    if expected not in readme:
        raise AssertionError(
            f"README missing reproduced metric {metric}: {expected}"
        )

# Save only after successful checks.
README.write_text(readme, encoding="utf-8")

# ------------------------------------------------------------
# STEP 7 — Save consistency audit
# ------------------------------------------------------------

report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "nested_cv_source": str(CV_PATH),
    "four_landmark_summary_verified": True,
    "nested_cv_metrics": cv_results,
    "independent_inference_reproduction_verified": True,
    "reproduced_test_metrics": {
        k: test_metrics[k] for k in ["MAE", "RMSE", "R2"]
    },
    "maximum_prediction_difference": (
        max_diff if np.isfinite(max_diff) else None
    ),
    "readme_updated": True,
    "model_retrained": False,
    "predictions_modified": False,
    "external_validation_established": False,
    "historical_test_set_independence_established": False
}

OUT = META / "q1_package_consistency_audit.json"
OUT.parent.mkdir(parents=True, exist_ok=True)

OUT.write_text(
    json.dumps(report, indent=2, allow_nan=False),
    encoding="utf-8"
)

# ------------------------------------------------------------
# STEP 8 — Final status
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 35.3 COMPLETED SUCCESSFULLY")
print("=" * 65)

print("\nNested-CV source:", CV_PATH)
print("\nNested-CV summary:")
print(cv_table)

print("\nInternal test metrics:")
for metric in ["MAE", "RMSE", "R2"]:
    print(f"{metric}: {test_metrics[metric]:.6f}")

print("Maximum prediction difference:", max_diff_text)
print("\nREADME updated:", README)
print("Audit saved:", OUT)
print("Model retrained: No")
print("Predictions modified: No")
print("External validation established: No")


In [ ]:

# ============================================================
# STEP 35.4: INCLUDED VS EXCLUDED COHORT COMPARISON
# Missing Week-26 Outcome | Selection Bias Assessment
# Descriptive analysis only | No model training
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("STEP 35.4: COHORT SELECTION BIAS ASSESSMENT")
print("=" * 70)

ROOT = Path(
    "/kaggle/working/SCI_Recovery_ML_Project"
)

SOURCE = (
    ROOT / "01_data/primary/raw/spss_v1_2.csv"
)

OUT = ROOT / "07_manuscript_package"
META = ROOT / "01_data/metadata"

OUT.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)

if not SOURCE.is_file():
    raise FileNotFoundError(str(SOURCE))

# Read only necessary columns from wide source dataset.
columns = [
    "PTID",
    "TOTMOT26",
    "AGE",
    "SEXCD",
    "AIS.1",
    "TOTMOT01"
]

df = pd.read_csv(
    SOURCE,
    usecols=columns,
    low_memory=False
)

assert len(df) == 791, (
    "Unexpected source cohort size."
)

assert df["PTID"].notna().all()
assert df["PTID"].is_unique

df["TOTMOT26"] = pd.to_numeric(
    df["TOTMOT26"],
    errors="coerce"
)

df["Cohort"] = np.where(
    df["TOTMOT26"].notna(),
    "Included",
    "Excluded"
)

n_included = int(
    (df["Cohort"] == "Included").sum()
)

n_excluded = int(
    (df["Cohort"] == "Excluded").sum()
)

assert n_included == 659
assert n_excluded == 132

# ------------------------------------------------------------
# 1. Continuous baseline characteristics
# ------------------------------------------------------------

continuous = ["AGE", "TOTMOT01"]
continuous_rows = []

for variable in continuous:

    values = pd.to_numeric(
        df[variable],
        errors="coerce"
    )

    included = values[
        df["Cohort"] == "Included"
    ].dropna()

    excluded = values[
        df["Cohort"] == "Excluded"
    ].dropna()

    mean_a = float(included.mean())
    mean_b = float(excluded.mean())

    sd_a = float(included.std(ddof=1))
    sd_b = float(excluded.std(ddof=1))

    pooled_sd = np.sqrt(
        (sd_a**2 + sd_b**2) / 2
    )

    smd = (
        (mean_a - mean_b) / pooled_sd
        if pooled_sd > 0
        else np.nan
    )

    continuous_rows.append({
        "Variable": variable,

        "Included_N": len(included),
        "Excluded_N": len(excluded),

        "Included_Mean": mean_a,
        "Excluded_Mean": mean_b,

        "Included_SD": sd_a,
        "Excluded_SD": sd_b,

        "Included_Median":
            float(included.median()),

        "Excluded_Median":
            float(excluded.median()),

        "Included_Missing":
            n_included - len(included),

        "Excluded_Missing":
            n_excluded - len(excluded),

        "Standardized_Mean_Difference":
            float(smd)
    })

continuous_table = pd.DataFrame(
    continuous_rows
)

# ------------------------------------------------------------
# 2. Categorical baseline characteristics
# ------------------------------------------------------------

categorical = ["SEXCD", "AIS.1"]
categorical_rows = []

for variable in categorical:

    values = (
        df[variable]
        .astype("string")
        .str.strip()
        .fillna("Missing")
    )

    values = values.replace("", "Missing")

    for category in sorted(
        values.unique().tolist()
    ):

        for cohort, total in [
            ("Included", n_included),
            ("Excluded", n_excluded)
        ]:

            mask = df["Cohort"].eq(cohort)

            count = int(
                (values[mask] == category).sum()
            )

            categorical_rows.append({
                "Variable": variable,
                "Category": str(category),
                "Cohort": cohort,
                "N": count,
                "Cohort_N": total,
                "Percent": 100 * count / total
            })

categorical_table = pd.DataFrame(
    categorical_rows
)

# ------------------------------------------------------------
# 3. Save results
# ------------------------------------------------------------

continuous_path = (
    OUT / "Table_Cohort_Selection_Continuous.csv"
)

categorical_path = (
    OUT / "Table_Cohort_Selection_Categorical.csv"
)

continuous_table.to_csv(
    continuous_path,
    index=False
)

categorical_table.to_csv(
    categorical_path,
    index=False
)

# ------------------------------------------------------------
# 4. Evidence-based interpretation
# ------------------------------------------------------------

report = {
    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "source_patients": len(df),
    "included_patients": n_included,
    "excluded_patients": n_excluded,

    "exclusion_reason":
        "Missing nominal Week-26 motor outcome",

    "baseline_comparison_completed": True,

    "continuous_variables": continuous,
    "categorical_variables": categorical,

    "selection_bias_ruled_out": False,

    "interpretation": (
        "Baseline characteristics were compared "
        "descriptively between included and excluded "
        "patients. Observed differences may indicate "
        "selection concerns but do not establish "
        "the missing-data mechanism or remove bias."
    ),

    "limitations": [
        "Only observed baseline variables were compared.",
        "The mechanism of missing Week-26 outcomes is unknown.",
        "Descriptive comparisons do not correct selection bias.",
        "No causal conclusion can be drawn."
    ]
}

REPORT_PATH = (
    META / "cohort_selection_bias_audit.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=4
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 5. Display
# ------------------------------------------------------------

print("\nCOHORT SUMMARY")
print("Source patients:", len(df))
print("Included:", n_included)
print("Excluded:", n_excluded)

print("\nCONTINUOUS CHARACTERISTICS")
display(continuous_table.round(4))

print("\nCATEGORICAL CHARACTERISTICS")
display(categorical_table.round(2))

print("\nINTERPRETATION")
print(report["interpretation"])

print("\nFILES SAVED")
print(continuous_path)
print(categorical_path)
print(REPORT_PATH)

print("\nNo model retraining performed.")
print("No patient predictions modified.")

print("\nSTEP 35.4 COMPLETED")


In [ ]:
# ============================================================
# STEP 36 FIX — ROBUST EVIDENCE FILE DISCOVERY
# Replace the existing source_csv() function and CV loading.
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")

def source_csv(relative):
    requested = ROOT / relative

    # Use the exact path first
    if requested.is_file():
        return pd.read_csv(requested)

    # Search project for the same filename
    matches = list(ROOT.rglob(Path(relative).name))
    matches = [p for p in matches if p.is_file()]

    if len(matches) == 1:
        print(f"Using discovered file: {matches[0]}")
        return pd.read_csv(matches[0])

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple files named {Path(relative).name} found:\n"
            + "\n".join(str(p) for p in matches)
            + "\nSet the correct canonical path explicitly."
        )

    raise FileNotFoundError(
        f"Evidence file not found: {Path(relative).name}\n"
        f"Expected path: {requested}\n"
        "Run the nested-CV export cell first. "
        "Do not substitute validation or test metrics for nested CV."
    )


# Load nested CV results using the robust resolver
cv = source_csv(
    "06_results/tables/dataset/Table15_Nested_CV_Mean_SD.csv"
)

print("Nested-CV table loaded:", cv.shape)
display(cv.head())

In [ ]:

# ============================================================
# STEP 36 FIX — RECOVER CANONICAL REGISTRY + VERIFY EXPORTS
# No model training | No fabricated or substituted metrics
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import json

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
PRIMARY = ROOT / "06_results/tables/primary"
AUDIT = ROOT / "06_results/audit"

PRIMARY.mkdir(parents=True, exist_ok=True)
AUDIT.mkdir(parents=True, exist_ok=True)

REGISTRY = PRIMARY / "Canonical_Evaluation_Registry.csv"
REGISTRY_NOTE = PRIMARY / "Canonical_Evaluation_Registry_README.txt"

# ------------------------------------------------------------
# 1. Search existing exports before rebuilding anything
# ------------------------------------------------------------

search_roots = [
    ROOT / "06_results",
    Path("/kaggle/working/final_manuscript_results"),
]

found = []

for base in search_roots:
    if base.exists():
        for p in base.rglob("Canonical_Evaluation_Registry.csv"):
            found.append(p)

if found and not REGISTRY.exists():
    source = found[0]
    REGISTRY.write_bytes(source.read_bytes())
    print("Recovered existing registry:", source)

# ------------------------------------------------------------
# 2. If unavailable, construct only from existing result objects
# ------------------------------------------------------------

if not REGISTRY.exists():

    required_objects = [
        "verified_results_df",
        "validation_result",
        "test_result",
    ]

    missing_objects = [
        name for name in required_objects
        if name not in globals()
    ]

    if missing_objects:
        raise FileNotFoundError(
            "Registry was not found and required in-memory results "
            f"are unavailable: {missing_objects}. "
            "Run STEP 14.9 and STEP 31.9, then rerun STEP 31.9B."
        )

    landmark_df = verified_results_df.copy()

    required_landmark_cols = {
        "Model", "Landmark", "Algorithm",
        "N_Validation", "MAE", "RMSE", "R2"
    }

    if not required_landmark_cols.issubset(landmark_df.columns):
        raise ValueError(
            "verified_results_df has an unexpected schema: "
            f"{landmark_df.columns.tolist()}"
        )

    rows = []

    for _, r in landmark_df.iterrows():
        rows.append({
            "Evaluation_Type": "Validation",
            "Analysis_Role": "Landmark comparison",
            "Model": r["Model"],
            "Landmark": r["Landmark"],
            "Algorithm": r["Algorithm"],
            "N_Evaluation": int(r["N_Validation"]),
            "N_Training_Partition": np.nan,
            "N_Outer_Folds": np.nan,
            "MAE": float(r["MAE"]),
            "RMSE": float(r["RMSE"]),
            "R2": float(r["R2"]),
            "MAE_SD": np.nan,
            "RMSE_SD": np.nan,
            "R2_SD": np.nan,
            "Source": "verified_results_df",
            "Interpretation_Note": "Validation landmark comparison"
        })

    def extract_metrics(obj, label):
        if isinstance(obj, pd.DataFrame):
            if obj.empty:
                raise ValueError(f"{label} is empty.")
            row = obj.iloc[0]
        elif isinstance(obj, dict):
            row = obj
        else:
            raise TypeError(
                f"{label} must be a DataFrame or dict; "
                f"received {type(obj).__name__}."
            )

        result = {}
        for metric in ["MAE", "RMSE", "R2"]:
            if metric not in row:
                raise KeyError(f"{label} missing {metric}.")
            result[metric] = float(row[metric])

        if not np.isfinite(list(result.values())).all():
            raise ValueError(f"{label} has invalid metrics.")

        return result

    # Optimized validation result
    val_metrics = extract_metrics(
        validation_result, "validation_result"
    )

    rows.append({
        "Evaluation_Type": "Validation",
        "Analysis_Role": "Optimized final model",
        "Model": "Optimized LightGBM",
        "Landmark": "Week 16",
        "Algorithm": "Optimized LightGBM",
        "N_Evaluation": (
            len(y_validation_B16_raw)
            if "y_validation_B16_raw" in globals()
            else len(validation_df)
            if "validation_df" in globals()
            else np.nan
        ),
        "N_Training_Partition": np.nan,
        "N_Outer_Folds": np.nan,
        **val_metrics,
        "MAE_SD": np.nan,
        "RMSE_SD": np.nan,
        "R2_SD": np.nan,
        "Source": "validation_result",
        "Interpretation_Note": "Optimized model validation"
    })

    # Nested-CV evidence: use existing primary file if available
    cv_candidates = [
        PRIMARY / "Table15_Nested_CV_Mean_SD.csv",
        PRIMARY / "Nested_CV_Mean_SD.csv",
        ROOT / "06_results/tables/dataset/Table15_Nested_CV_Mean_SD.csv",
        ROOT / "06_results/tables/primary/STEP15_8_OOF_Performance.csv",
    ]

    cv_path = next(
        (p for p in cv_candidates if p.is_file()),
        None
    )

    if cv_path is None:
        raise FileNotFoundError(
            "No nested-CV source table found. Checked:\n"
            + "\n".join(map(str, cv_candidates))
            + "\nExport the original nested-CV summary; "
              "do not substitute pooled OOF metrics for fold means."
        )

    cv = pd.read_csv(cv_path)

    if {"Metric", "Mean", "SD", "Landmark"}.issubset(cv.columns):
        b16 = cv[cv["Landmark"].astype(str).eq("Model_B16")]

        def cv_value(metric, field):
            m = b16[b16["Metric"].astype(str).str.upper().eq(metric)]
            if len(m) != 1:
                raise ValueError(
                    f"Expected one nested-CV {metric} row; found {len(m)}."
                )
            return float(m.iloc[0][field])

        rows.append({
            "Evaluation_Type": "Nested cross-validation",
            "Analysis_Role": "Internal resampling estimate",
            "Model": "Model_B16",
            "Landmark": "Week 16",
            "Algorithm": "LightGBM",
            "N_Evaluation": np.nan,
            "N_Training_Partition": 461,
            "N_Outer_Folds": 5,
            "MAE": cv_value("MAE", "Mean"),
            "RMSE": cv_value("RMSE", "Mean"),
            "R2": cv_value("R2", "Mean"),
            "MAE_SD": cv_value("MAE", "SD"),
            "RMSE_SD": cv_value("RMSE", "SD"),
            "R2_SD": cv_value("R2", "SD"),
            "Source": str(cv_path),
            "Interpretation_Note": (
                "Mean and SD across outer folds; not internal test."
            )
        })

    elif {
        "Landmark", "MAE_Mean", "MAE_SD",
        "RMSE_Mean", "RMSE_SD", "R2_Mean", "R2_SD"
    }.issubset(cv.columns):
        b16 = cv[cv["Landmark"].astype(str).eq("Model_B16")]

        if len(b16) != 1:
            raise ValueError("Expected one Model_B16 nested-CV row.")

        r = b16.iloc[0]

        rows.append({
            "Evaluation_Type": "Nested cross-validation",
            "Analysis_Role": "Internal resampling estimate",
            "Model": "Model_B16",
            "Landmark": "Week 16",
            "Algorithm": "LightGBM",
            "N_Evaluation": np.nan,
            "N_Training_Partition": 461,
            "N_Outer_Folds": 5,
            "MAE": float(r["MAE_Mean"]),
            "RMSE": float(r["RMSE_Mean"]),
            "R2": float(r["R2_Mean"]),
            "MAE_SD": float(r["MAE_SD"]),
            "RMSE_SD": float(r["RMSE_SD"]),
            "R2_SD": float(r["R2_SD"]),
            "Source": str(cv_path),
            "Interpretation_Note": (
                "Mean and SD across outer folds; not internal test."
            )
        })

    else:
        raise ValueError(
            f"Unrecognized nested-CV schema in {cv_path}: "
            f"{cv.columns.tolist()}"
        )

    # Internal test result
    test_metrics = extract_metrics(test_result, "test_result")

    rows.append({
        "Evaluation_Type": "Internal test",
        "Analysis_Role": "Optimized final model evaluation",
        "Model": "Optimized LightGBM",
        "Landmark": "Week 16",
        "Algorithm": "Optimized LightGBM",
        "N_Evaluation": (
            len(y_test_B16_raw)
            if "y_test_B16_raw" in globals()
            else len(test_df)
            if "test_df" in globals()
            else np.nan
        ),
        "N_Training_Partition": np.nan,
        "N_Outer_Folds": np.nan,
        **test_metrics,
        "MAE_SD": np.nan,
        "RMSE_SD": np.nan,
        "R2_SD": np.nan,
        "Source": "test_result",
        "Interpretation_Note": (
            "Internal test only; historical test independence UNKNOWN"
        )
    })

    registry = pd.DataFrame(rows)
    registry.to_csv(REGISTRY, index=False)

    REGISTRY_NOTE.write_text(
        "Canonical evaluation registry\n\n"
        "Validation, nested cross-validation and internal test results "
        "are separate evaluation procedures. Their metrics must not "
        "be substituted for one another.\n"
        "Historical test-set independence remains UNKNOWN. "
        "External validation has not been performed.\n",
        encoding="utf-8"
    )

# ------------------------------------------------------------
# 3. Final integrity checks
# ------------------------------------------------------------

registry = pd.read_csv(REGISTRY)

required = {
    "Evaluation_Type", "Analysis_Role", "Model", "Landmark",
    "MAE", "RMSE", "R2", "Source"
}

missing = required - set(registry.columns)

if missing:
    raise ValueError(f"Registry missing columns: {sorted(missing)}")

if registry.empty:
    raise ValueError("Registry is empty.")

for metric in ["MAE", "RMSE", "R2"]:
    values = pd.to_numeric(registry[metric], errors="coerce")
    if values.isna().any() or not np.isfinite(values).all():
        raise ValueError(f"Invalid registry metric: {metric}")

types = set(registry["Evaluation_Type"].astype(str))

for expected in [
    "Validation", "Nested cross-validation", "Internal test"
]:
    if expected not in types:
        raise ValueError(
            f"Registry is missing evaluation type: {expected}"
        )

if not REGISTRY_NOTE.exists():
    REGISTRY_NOTE.write_text(
        "Canonical evaluation registry\n"
        "Validation, nested CV and internal test results are distinct.\n"
        "Historical test-set independence remains UNKNOWN.\n"
        "External validation has not been performed.\n",
        encoding="utf-8"
    )

audit = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "registry_path": str(REGISTRY),
    "registry_readme_path": str(REGISTRY_NOTE),
    "registry_rows": len(registry),
    "evaluation_types": sorted(types),
    "historical_test_set_independence": "UNKNOWN",
    "external_validation_completed": False,
    "full_training_pipeline_reproduction_verified": False,
    "status": "CANONICAL_REGISTRY_VERIFIED"
}

audit_path = AUDIT / "STEP36_registry_recovery_audit.json"
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")

print("\nSTEP 36 REGISTRY CHECK PASSED")
print("CSV:", REGISTRY)
print("README:", REGISTRY_NOTE)
print("Rows:", len(registry))
print("Evaluation types:", sorted(types))
print("Audit:", audit_path)
print("Historical test independence: UNKNOWN")
print("External validation: NOT PERFORMED")


In [ ]:

# ============================================================
# STEP 36.1 — FINAL REGISTRY AND PACKAGE AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
PRIMARY = ROOT / "06_results/tables/primary"
AUDIT = ROOT / "06_results/audit"

REGISTRY = PRIMARY / "Canonical_Evaluation_Registry.csv"
REGISTRY_NOTE = PRIMARY / "Canonical_Evaluation_Registry_README.txt"

required_files = {
    "Canonical registry": REGISTRY,
    "Registry README": REGISTRY_NOTE,
    "Nested CV audit": AUDIT / "STEP15_8_nested_cv_integrity_audit.json",
    "Registry recovery audit": AUDIT / "STEP36_registry_recovery_audit.json",
}

checks = {}

for label, path in required_files.items():
    checks[label] = {
        "exists": path.is_file(),
        "path": str(path),
    }

if not REGISTRY.is_file():
    raise FileNotFoundError(f"Missing canonical registry: {REGISTRY}")

registry = pd.read_csv(REGISTRY)

required_columns = [
    "Evaluation_Type", "Analysis_Role", "Model",
    "Landmark", "MAE", "RMSE", "R2", "Source"
]

missing_columns = [
    col for col in required_columns
    if col not in registry.columns
]

if missing_columns:
    raise ValueError(f"Registry columns missing: {missing_columns}")

if registry.empty:
    raise ValueError("Registry is empty.")

for metric in ["MAE", "RMSE", "R2"]:
    values = pd.to_numeric(registry[metric], errors="coerce")
    if values.isna().any() or not values.map(
        lambda x: __import__("math").isfinite(x)
    ).all():
        raise ValueError(f"Invalid values in {metric}")

expected_types = {
    "Validation",
    "Nested cross-validation",
    "Internal test",
}

actual_types = set(registry["Evaluation_Type"].astype(str))

if not expected_types.issubset(actual_types):
    raise ValueError(
        f"Missing evaluation types: {expected_types - actual_types}"
    )

# Keep methodological uncertainties explicit.
audit_report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "registry_rows": int(len(registry)),
    "evaluation_types": sorted(actual_types),
    "required_files": checks,
    "historical_test_set_independence": "UNKNOWN",
    "assessment_chronology_verified": False,
    "formal_independent_calibration_verified": False,
    "external_validation_completed": False,
    "full_training_pipeline_reproduction_verified": False,
    "official_TRIPOD_AI_completion_verified": False,
    "interpretation": (
        "Registry and package file checks do not establish "
        "independent validation or full methodological verification."
    ),
}

out = AUDIT / "STEP36_1_final_registry_package_audit.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(audit_report, indent=2), encoding="utf-8")

print("=" * 65)
print("STEP 36.1 — FINAL REGISTRY AND PACKAGE AUDIT")
print("=" * 65)
print("Registry rows:", len(registry))
print("Evaluation types:", sorted(actual_types))

for label, result in checks.items():
    print(f"{label}: {'PASS' if result['exists'] else 'MISSING'}")

print("\nHistorical test independence: UNKNOWN")
print("External validation: NOT PERFORMED")
print("Full pipeline reproduction: NOT VERIFIED")
print("Audit saved:", out)


In [ ]:

# ============================================================
# STEP 36.2 — FINAL PROJECT ZIP EXPORT AND VERIFICATION
# ============================================================

from pathlib import Path
import zipfile
import json
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
ZIP_PATH = Path("/kaggle/working/SCI_Recovery_ML_Project_Final.zip")

required_files = [
    "06_results/tables/primary/Canonical_Evaluation_Registry.csv",
    "06_results/tables/primary/Canonical_Evaluation_Registry_README.txt",
    "06_results/audit/STEP36_registry_recovery_audit.json",
    "06_results/audit/STEP36_1_final_registry_package_audit.json",
    "06_results/audit/STEP15_8_nested_cv_integrity_audit.json",
]

if not ROOT.is_dir():
    raise FileNotFoundError(f"Project root missing: {ROOT}")

missing = [
    f for f in required_files
    if not (ROOT / f).is_file()
]

if missing:
    raise FileNotFoundError(
        "ZIP export stopped. Required files missing:\n"
        + "\n".join(missing)
    )

# Create a fresh ZIP from the current project contents.
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in ROOT.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(ROOT))

# Verify ZIP integrity and required members.
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    bad_member = archive.testzip()
    members = set(archive.namelist())

    missing_in_zip = [
        f for f in required_files if f not in members
    ]

if bad_member:
    raise RuntimeError(f"ZIP integrity failure: {bad_member}")

if missing_in_zip:
    raise RuntimeError(
        "Files missing from ZIP:\n" + "\n".join(missing_in_zip)
    )

report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "zip_path": str(ZIP_PATH),
    "zip_size_bytes": ZIP_PATH.stat().st_size,
    "files_in_archive": len(members),
    "required_files_verified": required_files,
    "zip_integrity_passed": True,
    "historical_test_set_independence": "UNKNOWN",
    "external_validation_completed": False,
    "full_training_pipeline_reproduction_verified": False,
    "status": "ZIP_CREATED_AND_VERIFIED"
}

audit_path = ROOT / "06_results/audit/STEP36_2_final_zip_audit.json"
audit_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("=" * 65)
print("STEP 36.2 — FINAL ZIP VERIFICATION")
print("=" * 65)
print("ZIP:", ZIP_PATH)
print("Size (MB):", round(ZIP_PATH.stat().st_size / (1024**2), 2))
print("Files archived:", len(members))
print("Required files verified:", len(required_files))
print("ZIP integrity: PASS")
print("Historical test independence: UNKNOWN")
print("External validation: NOT PERFORMED")
print("Full pipeline reproduction: NOT VERIFIED")
print("Audit:", audit_path)


In [ ]:

# ============================================================
# STEP 36.3 — CLINICAL INTERPRETATION OF WEEK-16 PREDICTIONS
# Descriptive analysis only; no clinical claim is inferred.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# Use the existing validation/test prediction table.
if "prediction_table" not in globals():
    raise NameError(
        "prediction_table not found. Run the original prediction-export cell."
    )

df = prediction_table.copy()

required = ["Observed_TOTMOT26", "Prediction_Model_B16"]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing prediction columns: {missing}")

df[required] = df[required].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=required).copy()

if df.empty:
    raise ValueError("No valid observed/predicted pairs.")

df["Error"] = df["Prediction_Model_B16"] - df["Observed_TOTMOT26"]
df["Absolute_Error"] = df["Error"].abs()

summary = pd.DataFrame([{
    "N": len(df),
    "MAE_points": df["Absolute_Error"].mean(),
    "Median_Absolute_Error": df["Absolute_Error"].median(),
    "Error_SD": df["Error"].std(ddof=1),
    "Mean_Error_Bias": df["Error"].mean(),
    "P90_Absolute_Error": df["Absolute_Error"].quantile(0.90),
    "Within_5_points_percent": (
        100 * (df["Absolute_Error"] <= 5).mean()
    ),
    "Within_10_points_percent": (
        100 * (df["Absolute_Error"] <= 10).mean()
    ),
}])

print("WEEK-16 PREDICTION ERROR SUMMARY")
display(summary.round(3))

# Save descriptive results.
out_dir = Path(
    "/kaggle/working/SCI_Recovery_ML_Project/"
    "06_results/tables/primary"
)
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "Week16_Clinical_Error_Interpretation.csv"
summary.to_csv(out_file, index=False)

print("Saved:", out_file)
print(
    "\nIMPORTANT: The 5- and 10-point bands are descriptive only. "
    "They are not validated clinical acceptability thresholds."
)
print(
    "Do not compare prediction MAE directly with MCID for patient change."
)


In [ ]:

# ============================================================
# STEP 37.1 — CONSOLIDATED CHRONOLOGY AUDIT
# Diagnostic only: date encoding is NOT source-verified.
# Replaces redundant STEP 37.1A–37.1H diagnostic cells.
# ============================================================

import json
import re
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
AUDIT_DIR = ROOT / "06_results" / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

if "raw_df" not in globals() or not isinstance(raw_df, pd.DataFrame):
    raise RuntimeError(
        "raw_df is unavailable. Load the original source dataset first."
    )

# Original timing-code variables
timing_cols = [
    "DTBAS", "DTWEEK4", "DTWEEK8",
    "DTWEEK16", "DTWEEK26"
]

# Follow-up visit-date candidates
followup_cols = [
    "DTFW00", "DTFW04", "DTFW08",
    "DTFW16", "DTFW26", "DTFW52"
]

nominal_intervals = {
    ("DTFW00", "DTFW04"): 28,
    ("DTFW04", "DTFW08"): 28,
    ("DTFW08", "DTFW16"): 56,
    ("DTFW16", "DTFW26"): 70,
    ("DTFW26", "DTFW52"): 182,
}

# ------------------------------------------------------------
# 2. DISCOVER DATE / ASSESSMENT COLUMNS
# ------------------------------------------------------------

keywords = [
    "DATE", "TIME", "VISIT", "DAY", "WEEK",
    "ASSESS", "EXAM", "FOLLOW", "BASELINE",
    "ENROLL", "INJURY", "DISCHARGE"
]

discovered_columns = [
    str(c) for c in raw_df.columns
    if any(k in str(c).upper() for k in keywords)
]

print("=" * 72)
print("STEP 37.1 — CONSOLIDATED CHRONOLOGY AUDIT")
print("=" * 72)

print("\nDataset rows:", len(raw_df))
print("PTID available:", "PTID" in raw_df.columns)
print("Discovered date/assessment columns:", discovered_columns)

# ------------------------------------------------------------
# 3. INSPECT ORIGINAL TIMING CODES
# ------------------------------------------------------------

available_timing = [
    c for c in timing_cols if c in raw_df.columns
]

missing_timing = [
    c for c in timing_cols if c not in raw_df.columns
]

print("\nOriginal timing variables available:", available_timing)
print("Original timing variables missing:", missing_timing)

timing_profiles = {}

for col in available_timing:
    s = pd.to_numeric(raw_df[col], errors="coerce")

    timing_profiles[col] = {
        "nonmissing": int(s.notna().sum()),
        "unique_values": int(s.nunique()),
        "minimum": (
            float(s.min()) if s.notna().any() else None
        ),
        "maximum": (
            float(s.max()) if s.notna().any() else None
        ),
    }

    print(
        f"{col}: nonmissing={s.notna().sum()}, "
        f"unique={s.nunique()}, "
        f"range=({s.min()}, {s.max()})"
    )

# Do not convert these codes to dates without a verified codebook.

# ------------------------------------------------------------
# 4. FOLLOW-UP DATE CANDIDATE CONVERSION
# ------------------------------------------------------------

available_followup = [
    c for c in followup_cols if c in raw_df.columns
]

missing_followup = [
    c for c in followup_cols if c not in raw_df.columns
]

print("\nFollow-up variables available:", available_followup)
print("Follow-up variables missing:", missing_followup)

date_results = []
anomaly_records = []
interval_records = []

if "PTID" not in raw_df.columns:
    print(
        "\nWARNING: PTID is absent. "
        "Patient-level anomaly identification is unavailable."
    )

if available_followup:
    work_cols = (
        (["PTID"] if "PTID" in raw_df.columns else [])
        + available_followup
    )

    df = raw_df[work_cols].copy()

    for col in available_followup:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Candidate SAS-date conversion only.
    candidate_dates = pd.DataFrame(index=df.index)

    for col in available_followup:
        candidate_dates[col] = pd.to_datetime(
            df[col],
            unit="D",
            origin="1960-01-01",
            errors="coerce"
        )

    for earlier, later in zip(
        available_followup[:-1],
        available_followup[1:]
    ):
        valid = (
            candidate_dates[earlier].notna()
            & candidate_dates[later].notna()
        )

        days = (
            candidate_dates.loc[valid, later]
            - candidate_dates.loc[valid, earlier]
        ).dt.days

        nominal = nominal_intervals.get((earlier, later))

        interval_results = {
            "interval": f"{earlier} -> {later}",
            "n_paired": int(valid.sum()),
            "same_date": int((days == 0).sum()),
            "later_date": int((days > 0).sum()),
            "earlier_date": int((days < 0).sum()),
            "median_days": (
                float(days.median()) if len(days) else None
            ),
            "min_days": (
                int(days.min()) if len(days) else None
            ),
            "max_days": (
                int(days.max()) if len(days) else None
            ),
            "nominal_interval_days": nominal,
        }

        if nominal is not None and len(days):
            interval_results["within_30_days_of_nominal"] = int(
                (abs(days - nominal) <= 30).sum()
            )
            interval_results["over_30_days_from_nominal"] = int(
                (abs(days - nominal) > 30).sum()
            )
        else:
            interval_results["within_30_days_of_nominal"] = None
            interval_results["over_30_days_from_nominal"] = None

        interval_records.append(interval_results)

        # Flag for manual review only; do not exclude automatically.
        for idx in days.index:
            observed = int(days.loc[idx])
            deviation = (
                abs(observed - nominal)
                if nominal is not None else None
            )

            non_increasing = observed <= 0
            large_deviation = (
                deviation > 30 if deviation is not None else False
            )

            if non_increasing or large_deviation:
                anomaly_records.append({
                    "PTID": (
                        df.loc[idx, "PTID"]
                        if "PTID" in df.columns else None
                    ),
                    "row_index": str(idx),
                    "interval": f"{earlier} -> {later}",
                    "earlier_variable": earlier,
                    "later_variable": later,
                    "earlier_candidate_date": str(
                        candidate_dates.loc[idx, earlier]
                    ),
                    "later_candidate_date": str(
                        candidate_dates.loc[idx, later]
                    ),
                    "observed_days": observed,
                    "nominal_days": nominal,
                    "absolute_deviation_days": deviation,
                    "non_increasing": non_increasing,
                    "over_30_days_from_nominal": large_deviation,
                    "requires_manual_review": True,
                })

    interval_summary = pd.DataFrame(interval_records)
    anomaly_df = pd.DataFrame(anomaly_records)

    print("\nFOLLOW-UP INTERVAL SUMMARY")
    display(interval_summary)

    print("\nCHRONOLOGY RECORDS FLAGGED FOR REVIEW")
    if anomaly_df.empty:
        print("No records flagged under the configured rules.")
    else:
        display(anomaly_df.head(100))

    interval_summary.to_csv(
        AUDIT_DIR / "STEP37_1_followup_interval_summary.csv",
        index=False
    )

    anomaly_df.to_csv(
        AUDIT_DIR / "STEP37_1_chronology_anomalies.csv",
        index=False
    )

else:
    interval_summary = pd.DataFrame()
    anomaly_df = pd.DataFrame()

    print(
        "\nNo DTFW follow-up variables found. "
        "Candidate follow-up interval analysis skipped."
    )

# ------------------------------------------------------------
# 5. SAVE CONSOLIDATED AUDIT
# ------------------------------------------------------------

report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_rows": int(len(raw_df)),
    "patient_id_available": "PTID" in raw_df.columns,
    "discovered_date_assessment_columns": discovered_columns,
    "timing_variables_available": available_timing,
    "timing_variables_missing": missing_timing,
    "timing_variable_profiles": timing_profiles,
    "followup_variables_available": available_followup,
    "followup_variables_missing": missing_followup,
    "date_encoding": "SAS date candidate; NOT source-verified",
    "nominal_intervals_days": {
        f"{a}->{b}": days
        for (a, b), days in nominal_intervals.items()
    },
    "interval_results": interval_records,
    "flagged_records": int(len(anomaly_df)),
    "assessment_chronology_verified": False,
    "status": "DIAGNOSTIC_ONLY_NOT_VERIFIED",
    "reason": (
        "The date encoding and variable semantics require confirmation "
        "from the original data dictionary or source documentation. "
        "Candidate dates and anomaly flags are not proof of actual "
        "assessment chronology."
    )
}

json_path = AUDIT_DIR / "STEP37_1_consolidated_chronology_audit.json"

json_path.write_text(
    json.dumps(report, indent=2, default=str),
    encoding="utf-8"
)

print("\nAudit files saved to:", AUDIT_DIR)
print("JSON:", json_path)
print("Assessment chronology remains NOT VERIFIED.")


In [ ]:

# ============================================================
# STEP 37.2 — CALIBRATION READINESS AND EVIDENCE AUDIT
# ============================================================

import numpy as np
import json
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
AUDIT_DIR = ROOT / "06_results/audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

alpha = 0.10

y_cal = globals().get("y_calibration")
p_cal = globals().get("pred_calibration")

result = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "nominal_coverage": 1 - alpha,
    "nominal_miscoverage": alpha,
    "independent_calibration_data_available": False,
    "patient_disjointness_verified": False,
    "calibration_protocol_prespecified": False,
    "formal_calibration_verified": False,
    "status": "NOT_READY"
}

if y_cal is not None and p_cal is not None:
    y = np.asarray(y_cal, dtype=float).ravel()
    p = np.asarray(p_cal, dtype=float).ravel()

    if len(y) != len(p) or len(y) == 0:
        raise ValueError("Calibration arrays must have equal nonzero length.")

    if not np.isfinite(y).all() or not np.isfinite(p).all():
        raise ValueError("Calibration arrays contain non-finite values.")

    scores = np.abs(y - p)
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))

    q = float(np.sort(scores)[k - 1]) if k <= n else None

    result.update({
        "n_calibration": int(n),
        "mean_absolute_residual": float(scores.mean()),
        "conformal_quantile_candidate": q,
        "status": "QUANTILE_CANDIDATE_ONLY",
        "reason": (
            "Independent patient separation and a prespecified "
            "calibration protocol must be verified before claiming "
            "formal prediction-interval coverage."
        )
    })
else:
    result["reason"] = (
        "Separate calibration targets and predictions are unavailable."
    )

out = AUDIT_DIR / "calibration_readiness_audit.json"
out.write_text(json.dumps(result, indent=2), encoding="utf-8")

print(json.dumps(result, indent=2))
print("Saved:", out)


In [ ]:

# ============================================================
# STEP 37.3 — CONSOLIDATED REPRODUCIBILITY & PIPELINE AUDIT
# Replaces previous STEP 37.3, 37.3A, 37.3B, 37.3C and 37.3D
# Static audit only: no model training or source execution.
# ============================================================

import sys
import json
import ast
import re
import hashlib
import platform
import subprocess
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
AUDIT = ROOT / "06_results" / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

SOURCE = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

# ------------------------------------------------------------
# 1. Environment snapshot
# ------------------------------------------------------------

packages = [
    "pandas", "numpy", "scikit-learn",
    "lightgbm", "shap", "joblib"
]

versions = {}

for package in packages:
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "show", package],
        capture_output=True, text=True
    )
    versions[package] = next(
        (
            line.split(":", 1)[1].strip()
            for line in proc.stdout.splitlines()
            if line.startswith("Version:")
        ),
        None
    )

freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True, text=True
)

(AUDIT / "requirements_snapshot.txt").write_text(
    freeze.stdout, encoding="utf-8"
)

# ------------------------------------------------------------
# 2. Discover and hash project source files
# ------------------------------------------------------------

excluded = {
    ".git", "__pycache__", ".ipynb_checkpoints",
    "venv", ".venv", "site-packages"
}
extensions = {".py", ".ipynb"}
source_hashes = {}
scan_errors = []

for folder in [ROOT, Path("/kaggle/working/.virtual_documents")]:
    if not folder.exists():
        continue

    try:
        for file in folder.rglob("*"):
            try:
                if not file.is_file():
                    continue
                if any(part in excluded for part in file.parts):
                    continue
                if file.suffix.lower() not in extensions:
                    continue

                source_hashes[str(file)] = hashlib.sha256(
                    file.read_bytes()
                ).hexdigest()

            except (OSError, PermissionError) as exc:
                scan_errors.append(f"{file}: {exc}")

    except (OSError, PermissionError) as exc:
        scan_errors.append(f"{folder}: {exc}")

# ------------------------------------------------------------
# 3. Read and identify the primary source
# ------------------------------------------------------------

if not SOURCE.is_file():
    raise FileNotFoundError(
        f"Source not found: {SOURCE}. Update SOURCE to the actual file."
    )

raw = SOURCE.read_bytes()

if not raw.strip():
    raise ValueError("Primary source file is empty.")

text = raw.decode("utf-8-sig", errors="strict")
lines = text.splitlines()

try:
    parsed = json.loads(text)
    if isinstance(parsed, dict) and isinstance(
        parsed.get("cells"), list
    ):
        source_format = "JUPYTER_NOTEBOOK"
        cells = parsed["cells"]
        code_sources = []

        for i, cell in enumerate(cells):
            if cell.get("cell_type") == "code":
                src = cell.get("source", "")
                if isinstance(src, list):
                    src = "".join(src)
                code_sources.append((i, src))
    else:
        raise ValueError("Not a valid notebook structure.")

except (json.JSONDecodeError, ValueError):
    source_format = "PYTHON_SCRIPT"
    code_sources = [(None, text)]
    cells = []

# ------------------------------------------------------------
# 4. Find STEP headings and map pipeline sections
# ------------------------------------------------------------

header_re = re.compile(
    r"^\s*#\s*(STEP\s+[\d.]+[A-Z]?(?:\.\d+)?\s*[:—-].*)$",
    re.IGNORECASE
)

headers = []

for line_no, line in enumerate(lines, 1):
    match = header_re.match(line)
    if match:
        headers.append({
            "line": line_no,
            "header": match.group(1).strip()
        })

categories = {
    "data_and_cohort": [
        "LOAD RAW", "COHORT", "EXCLUSION",
        "MISSING DATA", "DATA SPLIT"
    ],
    "model_development": [
        "MODEL TRAIN", "MODEL DEVELOPMENT",
        "HYPERPARAMETER", "NESTED CV",
        "NESTED CROSS", "LIGHTGBM"
    ],
    "validation_and_test": [
        "VALIDATION", "TEST SET", "TEST PERFORMANCE",
        "PATIENT OVERLAP", "SPLIT INTEGRITY"
    ],
    "landmark_analysis": [
        "LANDMARK", "REPRESENTATION COMPARISON",
        "WEEK4", "WEEK 4", "WEEK8", "WEEK 8",
        "WEEK16", "WEEK 16"
    ],
    "explainability_and_uncertainty": [
        "SHAP", "PERMUTATION IMPORTANCE",
        "CALIBRATION", "CONFORMAL",
        "PREDICTION UNCERTAINTY"
    ],
    "manuscript_exports_and_audits": [
        "MANUSCRIPT", "TABLE EXPORT",
        "FIGURE EXPORT", "FINAL ARTIFACT",
        "VERIFIED VALIDATION RESULTS"
    ]
}

mapping = {key: [] for key in categories}

for item in headers:
    heading = item["header"].upper()
    for category, terms in categories.items():
        if any(term in heading for term in terms):
            mapping[category].append(item)

# ------------------------------------------------------------
# 5. Static keyword and syntax inspection
# ------------------------------------------------------------

keywords = [
    "train_test_split", "GroupShuffleSplit", "GroupKFold",
    "GridSearchCV", "RandomizedSearchCV", "LGBMRegressor",
    "lightgbm", "shap", "conformal", "mean_absolute_error",
    "r2_score", "TOTMOT_change_4", "TOTMOT_change_8",
    "TOTMOT_change_16", "to_csv(", "savefig("
]

keyword_matches = []
syntax_issues = []
saved_errors = []

for cell_index, src in code_sources:
    found = [
        word for word in keywords
        if word.lower() in src.lower()
    ]
    if found:
        keyword_matches.append({
            "cell_index": cell_index,
            "keywords_found": found
        })

    try:
        ast.parse(src)
    except SyntaxError as exc:
        syntax_issues.append({
            "cell_index": cell_index,
            "line": exc.lineno,
            "error": str(exc)
        })

if source_format == "JUPYTER_NOTEBOOK":
    for i, cell in enumerate(cells):
        for output in cell.get("outputs", []):
            if output.get("output_type") == "error":
                saved_errors.append({
                    "cell_index": i,
                    "error_name": output.get("ename"),
                    "error_value": output.get("evalue")
                })

# Duplicate heading audit
seen = {}
for item in headers:
    seen.setdefault(item["header"].upper(), []).append(item["line"])

duplicates = {
    heading: positions
    for heading, positions in seen.items()
    if len(positions) > 1
}

# ------------------------------------------------------------
# 6. Save consolidated report
# ------------------------------------------------------------

report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(ROOT),
    "source_path": str(SOURCE),
    "source_format": source_format,
    "source_bytes": len(raw),
    "source_sha256": hashlib.sha256(raw).hexdigest(),
    "source_lines": len(lines),
    "step_headers_found": len(headers),
    "step_headers": headers,
    "pipeline_step_mapping": mapping,
    "duplicate_step_headings": duplicates,
    "keyword_match_count": len(keyword_matches),
    "keyword_matches": keyword_matches,
    "possible_syntax_issues": syntax_issues,
    "saved_execution_errors": saved_errors,
    "source_files_found": len(source_hashes),
    "source_hashes": source_hashes,
    "source_scan_errors": scan_errors,
    "python_version": sys.version,
    "platform": platform.platform(),
    "packages": versions,
    "requirements_snapshot": str(
        AUDIT / "requirements_snapshot.txt"
    ),
    "source_executed": False,
    "model_retrained": False,
    "test_predictions_recomputed": False,
    "full_training_pipeline_reproduction_verified": False,
    "assessment_chronology_verified": False,
    "formal_independent_calibration_verified": False,
    "external_validation_completed": False,
    "official_TRIPOD_AI_completion_verified": False,
    "historical_test_set_independence": "UNKNOWN",
    "status": "STATIC_AUDIT_COMPLETED_REPRODUCTION_NOT_VERIFIED",
    "interpretation": (
        "This audit inventories source, environment, STEP headings "
        "and code patterns. It does not establish successful execution, "
        "historical test independence, chronology, calibration, "
        "external validity, or end-to-end reproducibility."
    )
}

out = AUDIT / "STEP37_3_consolidated_reproducibility_pipeline_audit.json"
out.write_text(json.dumps(report, indent=2), encoding="utf-8")

# ------------------------------------------------------------
# 7. Concise console summary
# ------------------------------------------------------------

print("=" * 65)
print("STEP 37.3 — CONSOLIDATED AUDIT")
print("=" * 65)
print("Source format:", source_format)
print("Source lines:", len(lines))
print("STEP headers:", len(headers))
print("Source files hashed:", len(source_hashes))
print("Pipeline keyword groups:", len(keyword_matches))
print("Possible syntax issues:", len(syntax_issues))
print("Saved notebook errors:", len(saved_errors))

print("\nPIPELINE STEP COUNTS")
for category, items in mapping.items():
    print(f"{category}: {len(items)}")

print("\nKEY METHODOLOGICAL STATUS")
for key in [
    "full_training_pipeline_reproduction_verified",
    "assessment_chronology_verified",
    "formal_independent_calibration_verified",
    "external_validation_completed",
    "official_TRIPOD_AI_completion_verified",
    "historical_test_set_independence"
]:
    print(f"{key}: {report[key]}")

print("\nAudit saved:", out)
print("No model training or source execution performed.")


In [ ]:

# ============================================================
# STEP 37.4 — EXTERNAL VALIDATION EVIDENCE AUDIT
# ============================================================

import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
AUDIT_DIR = ROOT / "06_results/audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

external_df = globals().get("external_validation_df")

required_features = [
    "AGE", "SEXCD", "AIS.1", "TOTMOT01"
]

result = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "external_validation_completed": False,
    "external_cohort_independence_verified": False,
    "status": "NOT_PERFORMED"
}

if isinstance(external_df, pd.DataFrame):
    missing = [
        col for col in required_features
        if col not in external_df.columns
    ]

    result.update({
        "external_rows": int(len(external_df)),
        "missing_required_features": missing,
        "schema_check_passed": len(missing) == 0,
        "status": (
            "SCHEMA_READY_FOR_VALIDATION_REVIEW"
            if len(missing) == 0
            else "SCHEMA_INCOMPLETE"
        ),
        "reason": (
            "Schema compatibility does not establish cohort "
            "independence or successful external validation."
        )
    })
else:
    result["reason"] = (
        "No separate external validation DataFrame is available."
    )

out = AUDIT_DIR / "external_validation_evidence.json"
out.write_text(json.dumps(result, indent=2), encoding="utf-8")

print(json.dumps(result, indent=2))
print("Saved:", out)


In [ ]:
# ============================================================
# FINAL METADATA REFRESH
# ============================================================

import json
from pathlib import Path


final_status = {


"patient_split":

    "Verified using PTID",


"data_provenance":

    "Verified",


"cohort_derivation":

    "Verified",


"missingness_analysis":

    "Completed",


"learning_curve":

    "Completed",


"model_card":

    "Updated",


"external_validation":

    "Not performed"

}



path = Path(

"/kaggle/working/SCI_Recovery_ML_Project/"
"FINAL_STATUS.json"

)



path.write_text(

    json.dumps(
        final_status,
        indent=2
    ),

    encoding="utf-8"

)



print(
    "FINAL STATUS SAVED"
)

print(path) 

In [ ]:

# ============================================================
# FINAL STEP — FIXED PATIENT AUDIT + VERIFIED PROJECT ZIP
# Replace the entire existing final cell with this code.
# ============================================================

import json
import shutil
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("FINAL STEP — PATIENT AUDIT + PROJECT ZIP")
print("=" * 70)

ROOT = Path("/kaggle/working/SCI_Recovery_ML_Project")
AUDIT_DIR = ROOT / "06_results" / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_BASE = Path("/kaggle/working/SCI_Recovery_ML_Project_Final")
ZIP_PATH = Path(str(ZIP_BASE) + ".zip")

if not ROOT.is_dir():
    raise FileNotFoundError(f"Project directory missing: {ROOT}")

# ------------------------------------------------------------
# 1. Check that the three existing partitions are available
# ------------------------------------------------------------

required_vars = ["train_df", "validation_df", "test_df"]
missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    raise RuntimeError(
        f"Missing variables: {missing_vars}. "
        "Run the existing split-creation cells first."
    )

splits = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df
}

patient_sets = {}
split_details = {}
audit_errors = []

# ------------------------------------------------------------
# 2. Validate missing/blank IDs and duplicate patient IDs
# ------------------------------------------------------------

for name, frame in splits.items():

    if "PTID" not in frame.columns:
        raise KeyError(f"{name} is missing PTID.")

    normalized = frame["PTID"].astype("string").str.strip()

    # Correctly detect BOTH null and whitespace-only IDs.
    missing_mask = normalized.isna() | normalized.eq("")
    missing_or_blank_count = int(missing_mask.sum())

    valid_ids = normalized.loc[~missing_mask]
    duplicate_count = int(valid_ids.duplicated().sum())

    patient_sets[name] = set(valid_ids.tolist())

    split_details[name] = {
        "n_rows": int(len(frame)),
        "n_missing_or_blank_PTID": missing_or_blank_count,
        "n_unique_valid_PTID": int(valid_ids.nunique()),
        "n_duplicate_normalized_PTID": duplicate_count
    }

    # IMPORTANT: fail the audit on blank IDs as well as null IDs.
    if missing_or_blank_count > 0:
        audit_errors.append(
            f"{name}: {missing_or_blank_count} missing/blank PTID values"
        )

    if duplicate_count > 0:
        audit_errors.append(
            f"{name}: {duplicate_count} duplicate normalized PTIDs"
        )

# ------------------------------------------------------------
# 3. Pairwise patient overlap
# ------------------------------------------------------------

overlap = {
    "train_validation": len(
        patient_sets["train"] & patient_sets["validation"]
    ),
    "train_test": len(
        patient_sets["train"] & patient_sets["test"]
    ),
    "validation_test": len(
        patient_sets["validation"] & patient_sets["test"]
    )
}

no_overlap = all(v == 0 for v in overlap.values())

partition_passed = (
    no_overlap and len(audit_errors) == 0
)

audit = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "audit_name": "STEP 11.5.1 — Patient Split Integrity",
    "identifier": "PTID",
    "split_details": split_details,
    "overlap_counts": overlap,
    "no_patient_overlap": no_overlap,
    "partition_integrity_passed": partition_passed,
    "audit_errors": audit_errors,
    "historical_test_set_independence": "UNKNOWN",
    "external_validation_status": "NOT_ESTABLISHED",
    "interpretation": (
        "This audit checks current patient-level separation only. "
        "It does not establish historical test-set independence, "
        "external validity, or absence of all forms of data leakage."
    )
}

# Save the audit even if a check fails, so the ZIP records the status.
AUDIT_PATH = AUDIT_DIR / "split_integrity_audit.json"
AUDIT_PATH.write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8"
)

print("\nPATIENT SPLIT AUDIT")
print(json.dumps(audit, indent=2))

if partition_passed:
    print("\nCurrent patient-partition checks: PASSED")
else:
    print("\nWARNING: PATIENT-PARTITION AUDIT FAILED.")
    print("Review audit_errors before relying on these evaluations.")

# ------------------------------------------------------------
# 4. Create ZIP AFTER writing the audit
# ------------------------------------------------------------

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

archive_result = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=str(ROOT)
)

ZIP_PATH = Path(archive_result)

if not ZIP_PATH.is_file():
    raise RuntimeError("ZIP creation failed.")

# ------------------------------------------------------------
# 5. Verify archive integrity and audit inclusion
# ------------------------------------------------------------

with zipfile.ZipFile(ZIP_PATH, "r") as zf:

    corrupt_member = zf.testzip()

    if corrupt_member is not None:
        raise RuntimeError(
            f"Corrupt ZIP member: {corrupt_member}"
        )

    audit_member = "06_results/audit/split_integrity_audit.json"

    if audit_member not in zf.namelist():
        raise RuntimeError(
            f"Audit missing from ZIP: {audit_member}"
        )

    archived_audit_bytes = zf.read(audit_member)
    archived_audit = json.loads(
        archived_audit_bytes.decode("utf-8")
    )

    saved_hash = hashlib.sha256(AUDIT_PATH.read_bytes()).hexdigest()
    zipped_hash = hashlib.sha256(archived_audit_bytes).hexdigest()

    if saved_hash != zipped_hash:
        raise RuntimeError(
            "Saved audit and ZIP audit do not match."
        )

    archive_entries = len(zf.namelist())

print("\n" + "=" * 70)
print("FINAL PACKAGE VERIFICATION")
print("=" * 70)
print("ZIP:", ZIP_PATH)
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / 1024**2, 2))
print("Archive entries:", archive_entries)
print("ZIP integrity: PASSED")
print("Audit included in ZIP: YES")
print("Audit copy matches saved file: YES")
print("Patient partition audit:", "PASSED" if partition_passed else "FAILED")
print("Historical test-set independence: UNKNOWN")
print("External validation: NOT ESTABLISHED")
print("Full training reproduction: NOT VERIFIED")
print("\nZIP packaging completed.")
